# Cuaderno de análisis del Trabajo Fin de Máster

**¿Aporta la forma de la curva fuerza-profundidad, frente a los valores que publica el laboratorio, para reproducir la limpieza de un especialista en ensayos de nanoindentación?**

Este cuaderno contiene todo el análisis de la memoria: la comprobación del conjunto de datos, el efecto de limpiar, las reglas estadísticas, la construcción de las variables, los modelos, su interpretación, las comprobaciones de solidez y las ilustraciones. Está pensado para ejecutarse de principio a fin, en orden, en Google Colaboratory.

### Datos

- Conjunto: Henriksen et al. (2025), *A ground truth dataset for anomaly detection on load-displacement nanoindentation curves for NiFe electroplated samples*, publicado en el Materials Data Facility con licencia CC BY 4.0.
- La descarga se hace desde el Materials Data Facility mediante Globus, con una cuenta gratuita.
- Antes de ejecutar, subir el fichero comprimido `anomaly_detection_on_nanoindentation.zip` a `/content` (panel de archivos de Colab). La celda 1 lo descomprime en `/content/nanoindent`.
- Todas las rutas están bajo `/content`, y allí se guardan los resultados (ficheros `.csv` y `.npz`). La celda 18 guarda las figuras en `/content/figuras`. Para ejecutar fuera de Colab basta con sustituir `/content` por otra carpeta en todo el cuaderno.

### Cómo ejecutarlo

1. Ejecutar la celda 0 y comprobar las versiones. Si alguna no coincide con las registradas, la celda imprime la orden de instalación; después de instalar hay que reiniciar la sesión.
2. Ejecutar todas las celdas en orden y en la misma sesión (Entorno de ejecución > Ejecutar todas). Cada celda usa lo que han creado las anteriores: la celda 0 importa las bibliotecas y define las rutas y las funciones comunes; las celdas 2 y 3 leen las tablas del conjunto; la 5 construye la tabla de ensayos de las 139 películas analizadas, que usan las celdas 5 a 13; la 14 calcula las variables de la curva para los 9.600 ensayos, la 15 las añade a esa tabla, y las celdas 16, 17 y 18 usan los objetos de la 15 y de la 16.
3. La más lenta es la 16, que reentrena los modelos muchas veces, seguida de la 14, que lee las 9.600 curvas. La ejecución completa tarda unos minutos.
4. La celda 19 comprime la carpeta de figuras y la descarga (solo en Colab).

Los procesos aleatorios usan semillas fijas. Con las mismas versiones y el mismo entorno, la ejecución reproduce las cifras de la memoria. En otro entorno, algunas cifras por campaña pueden cambiar por diferencias numéricas del orden del redondeo (celda 16, bloque F).

### Versiones registradas

| Python | NumPy | pandas | SciPy | scikit-learn | Matplotlib | SHAP |
|---|---|---|---|---|---|---|
| 3.13.16 | 2.1.3 | 2.2.3 | 1.16.3 | 1.6.1 | 3.10.0 | 0.52.0 |

### Correspondencia entre fases, celdas y apartados de la memoria

| Fase | Objetivo | Celdas | Apartado |
|---|---|---|---|
| Entorno | | 0 | 4.1.3 |
| 1 | O1. Integridad del conjunto de datos | 1, 2, 3, 4 y 13 (A, D2, D3) | 4.3.1 |
| 2 | O2. Efecto de limpiar sobre el valor | 5 | 4.3.2 |
| 3 | O3. Lo que logran las reglas estadísticas | 6, 7, 8 y 13 (B) | 4.3.3 |
| 4 | O4. Lo que el valor final no muestra | 6, 8, 9, 10, 11 y 13 (C) | 4.3.4 |
| Complementario | Réplicas de proceso | 12 y 13 (D) | 5.4 y apéndice |
| 5 | O5. Variables de forma | 14 | 4.3.5 |
| 6 | O6. Comparación de los modelos A y B | 15, 16 (C) y 17 | 4.3.6 y 4.4.1 a 4.4.3 |
| 7 | O7. Interpretación del modelo B | 15 (4-bis) y 16 (A, I) | 4.4.4 |
| 8 | O8. Solidez del resultado | 15 (7) y 16 (B, D, E, F, G-H, J) | 4.4.5 |
| 9 | O9. Alcance y límites | 15 (6, 8) | 4.4.6 |
| Ilustraciones | | 18 y 19 | Toda la memoria |

Cada celda de código va precedida de una celda de texto que explica qué hace y con qué criterio. Los criterios que se indican como fijados antes de ejecutar se escribieron antes de ver el resultado correspondiente.

## Entorno de ejecución

### Celda 0. Entorno y elementos comunes

Importa todas las bibliotecas del cuaderno, fija las rutas y las listas de columnas que usan varias celdas, y define las funciones comunes:

- `z_mad` y `z_signed`: distancia de cada ensayo a la mediana de su película, en unidades de la desviación absoluta mediana multiplicada por 1,4826, en valor absoluto o con signo.
- `bins`: asigna cada valor a una franja según su orden, con el mismo número de valores en cada franja.
- `rho_vecinos`: correlación de rangos de una propiedad entre ensayos vecinos de la malla (a un paso de malla según su posición X-Y), una por película.
- `factor_vecinos`: factor que amplía el error típico del promedio de los 64 ensayos de una malla 8 × 8 cuando los vecinos inmediatos tienen correlación ρ, √(1 + 3,5 ρ), con ρ la mediana entre películas.

Al final imprime la versión de Python y de cada biblioteca junto a la registrada (memoria, apéndice II). Si alguna no coincide, imprime la orden para instalar la registrada; después de instalar hay que reiniciar la sesión. Si SHAP no está instalado, la celda lo instala.

In [1]:
import os, re, glob, json, time, shutil, zipfile, hashlib, warnings, itertools
import platform, textwrap, traceback
from math import comb
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, scipy, sklearn, matplotlib
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import Formatter, FixedLocator, NullLocator, MaxNLocator
from matplotlib.transforms import Bbox
try:
    import shap
except ImportError:
    os.system('pip -q install shap==0.52.0')
    import shap

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 200)

ZIP = '/content/anomaly_detection_on_nanoindentation.zip'
DEST = '/content/nanoindent'
RAIZ = os.path.join(DEST, 'nanoindentation_data_release')
CLAVE = ['sample_num', 'indent_num']
ANOM = ['fully_anomalous_sample', 'force_offset', 'displacement_offset',
        'unusual_unloading_curvature', 'unusual_loading_curvature',
        'tip_displacement_decreases_at_holding', 'too_deep']
POR_INDENT = ANOM[1:]
PROPS = ['hardness_H_GPa', 'reduced_modulus_Er_GPa']
SEED = 20260916


def sep(t):
    print('\n' + '=' * 94); print(t); print('=' * 94)


def z_mad(v):
    med = np.median(v); mad = np.median(np.abs(v - med))
    return np.abs(v - med) / (1.4826 * mad) if mad > 0 else np.zeros(len(v))


def z_signed(v):
    med = np.median(v); mad = np.median(np.abs(v - med))
    return (v - med) / (1.4826 * mad) if mad > 0 else np.zeros(len(v))


def bins(x, nq):
    if nq <= 1:
        return np.zeros(len(x), int)
    rk = np.argsort(np.argsort(x))
    return np.minimum((rk * nq) // max(len(x), 1), nq - 1)


def rho_vecinos(M_, prop):
    out = {}
    for s0, d0 in M_.groupby('sample_num'):
        if not {'X_mm', 'Y_mm'}.issubset(d0.columns) or len(d0) < 10:
            continue
        xy = d0[['X_mm', 'Y_mm']].astype(float).values
        v0 = d0[prop].astype(float)
        if not np.isfinite(xy).all() or not np.isfinite(v0.values).all():
            continue
        rk = v0.rank().values
        dd = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(-1))
        paso = np.min(dd[dd > 0])
        ii, jj = np.where(np.triu((dd > 0) & (dd < 1.2 * paso), 1))
        if len(ii) < 20:
            continue
        out[s0] = float(np.corrcoef(np.r_[rk[ii], rk[jj]], np.r_[rk[jj], rk[ii]])[0, 1])
    return pd.Series(out, dtype=float)


def factor_vecinos(rho):
    r = float(np.median(rho)) if len(rho) else 0.0
    return float(np.sqrt(1 + 3.5 * max(r, 0.0)))


REGISTRADAS = {'Python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3',
               'scikit-learn': '1.6.1', 'matplotlib': '3.10.0', 'shap': '0.52.0'}
ACTUAL = {'Python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
          'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__,
          'matplotlib': matplotlib.__version__, 'shap': shap.__version__}
print(platform.platform())
for k, v in ACTUAL.items():
    print(f'  {k:<13} {v:<10} registrada {REGISTRADAS[k]}' + ('' if v == REGISTRADAS[k] else '   <- distinta'))
dif = [f'{k}=={REGISTRADAS[k]}' for k in ACTUAL if k != 'Python' and ACTUAL[k] != REGISTRADAS[k]]
if dif:
    print('\nPara instalar las registradas (y reiniciar la sesion despues):')
    print('  !pip install ' + ' '.join(dif))

Linux-6.18.44-fc-v310-x86_64-with-glibc2.39
  Python        3.13.16    registrada 3.13.16
  numpy         2.1.3      registrada 2.1.3
  pandas        2.2.3      registrada 2.2.3
  scipy         1.16.3     registrada 1.16.3
  scikit-learn  1.6.1      registrada 1.6.1
  matplotlib    3.10.0     registrada 3.10.0
  shap          0.52.0     registrada 0.52.0


## Fase 1. Comprobación del conjunto de datos

Objetivo O1; memoria, apartado 4.3.1. Las celdas 1 a 4 no modelan nada: comprueban que las curvas están completas y sin registros dañados, que las marcas del especialista y los valores del laboratorio corresponden a los ensayos que describen, y qué variables de fabricación se confunden entre sí.

### Celda 1. Auditoría del conjunto de datos

Conjunto de Henriksen et al. (2025), publicado en el Materials Data Facility con licencia CC BY 4.0.

No modela nada. Descomprime el fichero en `/content/nanoindent`, lista su contenido, lee la documentación y las tablas, y comprueba si el dato es lo que dice ser: 150 películas con 64 ensayos cada una, reparto de las marcas por tipo y nivel, ficheros duplicados byte a byte y riesgo de fuga al partir los datos.

Las cuatro condiciones de aceptación se fijaron antes de abrir los ficheros (memoria, apartado 4.3.1):

1. Menos del 1 % de duplicados exactos.
2. Unas 150 películas y unas 9.600 curvas (en el código, al menos 140 películas y entre 9.000 y 10.200 filas en la tabla de marcas).
3. Al menos 50 ensayos de cada tipo de defecto en nivel 2 o 3 (reparto del bloque 3 y de la celda 2).
4. Cada película enlaza con sus datos de fabricación (celda 4).

Si falla la 1 o la 2, se descarta el conjunto. Si falla la 3, ese tipo de defecto se declara no evaluable. Si falla la 4, el trabajo sigue, pero sin describir el material ni separar las condiciones de fabricación. La celda imprime el resultado de las dos primeras.

In [2]:
sep('0 · CONTENIDO DEL ZIP')
os.makedirs(DEST, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    info = z.infolist()
    print(f'  entradas en el zip: {len(info)}   tamano comprimido: '
          f'{sum(i.compress_size for i in info)/1e6:.1f} MB   '
          f'descomprimido: {sum(i.file_size for i in info)/1e6:.1f} MB')
    ext = {}
    for i in info:
        if i.is_dir(): continue
        e = os.path.splitext(i.filename)[1].lower() or '(sin ext)'
        ext[e] = ext.get(e, 0) + 1
    print('  por extension:', dict(sorted(ext.items(), key=lambda kv: -kv[1])))
    z.extractall(DEST)

FIL = [f for f in glob.glob(os.path.join(DEST, '**', '*'), recursive=True) if os.path.isfile(f)]
FIL.sort(key=lambda f: -os.path.getsize(f))
print(f'\n--- ficheros extraidos: {len(FIL)} — los 25 mas grandes')
for f in FIL[:25]:
    print(f'  {os.path.getsize(f)/1e6:9.2f} MB  {os.path.relpath(f, DEST)}')
if len(FIL) > 25:
    print(f'  ... y {len(FIL)-25} mas')

sep('0.1 · README / DOCUMENTACION INCLUIDA')
txt = [f for f in FIL if os.path.splitext(f)[1].lower() in ('.md', '.txt', '.rst', '.json')
       and os.path.getsize(f) < 200_000]
if not txt:
    print('  (ninguno)')
for f in txt[:6]:
    print(f'\n  ===== {os.path.relpath(f, DEST)} =====')
    try:
        print(open(f, encoding='utf-8', errors='replace').read()[:3000])
    except Exception as e:
        print('  no legible:', e)

sep('1 · TABLAS (csv / xlsx / parquet)')
GRUPO_TAB = (('ficheros de curva', re.compile(r'_nanoindentation_\d+\.csv$')),
             ('ficheros de resumen por pelicula', re.compile(r'_summary\.csv$')))
RES_TAB = {g: dict(n=0, cols={}, filas={}, ej=None) for g, _ in GRUPO_TAB}
TABLAS = {}
for f in FIL:
    e = os.path.splitext(f)[1].lower()
    try:
        if   e == '.csv':      d = pd.read_csv(f)
        elif e in ('.xlsx', '.xls'): d = pd.read_excel(f)
        elif e == '.parquet':  d = pd.read_parquet(f)
        else: continue
    except Exception as ex:
        print(f'  {os.path.relpath(f, DEST)}: no legible ({type(ex).__name__}: {ex})'); continue
    g = next((g for g, rx in GRUPO_TAB if rx.search(os.path.basename(f))), None)
    if g:
        r = RES_TAB[g]; r['n'] += 1
        r['cols'][tuple(d.columns)] = r['cols'].get(tuple(d.columns), 0) + 1
        r['filas'][len(d)] = r['filas'].get(len(d), 0) + 1
        if r['ej'] is None:
            r['ej'] = (os.path.relpath(f, DEST), d.head(4))
        continue
    TABLAS[os.path.relpath(f, DEST)] = d
    print(f'\n  ===== {os.path.relpath(f, DEST)}  ({len(d)} filas x {len(d.columns)} cols)')
    print('   columnas:', list(d.columns))
    print(d.head(4).to_string()[:1800])
for g, r in RES_TAB.items():
    if not r['n']:
        continue
    print(f'\n  ===== {g}: {r["n"]} (se resumen; no se imprimen uno a uno)')
    for cols, k in r['cols'].items():
        print(f'   columnas {list(cols)}: {k} ficheros')
    print('   filas por fichero:', dict(sorted(r['filas'].items())))
    print(f'   ejemplo: {r["ej"][0]}')
    print(r['ej'][1].to_string()[:1800])

LAB = LABN = None
if TABLAS:
    LABN, LAB = max(TABLAS.items(), key=lambda kv: len(kv[1]))
    print(f'\n  >>> se toma como tabla de etiquetas: {LABN}  ({len(LAB)} filas)')

sep('2 · ESTRUCTURA 150 MUESTRAS x 64 INDENTACIONES')
col_s = col_i = None
cand_s = []
if LAB is not None:
    cand_s = [c for c in LAB.columns if re.search(r'sample|specimen|film|coupon', str(c), re.I)]
    cand_i = [c for c in LAB.columns if re.search(r'indent|test|curve|point|idx|index', str(c), re.I)]
    if not cand_s:
        cand_s = [c for c in LAB.columns if 100 <= LAB[c].nunique() <= 300]
    col_s = cand_s[0] if cand_s else None
    col_i = cand_i[0] if cand_i else None
    print(f'  columna de muestra   : {col_s}   (candidatas: {cand_s[:6]})')
    print(f'  columna de indent    : {col_i}   (candidatas: {cand_i[:6]})')
    print(f'  n_unicos por columna : '
          f'{ {c: int(LAB[c].nunique()) for c in LAB.columns[:20]} }')
    if col_s:
        g = LAB.groupby(col_s).size()
        print(f'\n  muestras distintas ......... {g.size:6d}   (esperado 150)')
        print(f'  filas totales .............. {len(LAB):6d}   (esperado 9600)')
        print(f'  indents/muestra: min={g.min()} p50={int(g.median())} max={g.max()}   (esperado 64)')
        raras = g[g != 64]
        print(f'  muestras que NO tienen 64 .. {len(raras)}')
        if len(raras): print('   ', dict(itertools.islice(raras.to_dict().items(), 12)))
        print(f'  filas duplicadas exactas en la tabla: {int(LAB.duplicated().sum())}')

sep('3 · REPARTO DE ANOMALIAS Y SEVERIDAD')
cols_a = []
if LAB is not None:
    for c in LAB.columns:
        cl = str(c).lower()
        if any(k.split('_')[0] in cl for k in ANOM) or re.search(
                r'anomal|label|level|severity|offset|curvature|deep|tip|note', cl):
            cols_a.append(c)
    print('  columnas relacionadas con anomalia:', cols_a)
    for c in cols_a:
        v = LAB[c]
        if v.dtype == object and v.astype(str).str.len().mean() > 40:
            nn = int((v.notna() & (v.astype(str).str.strip() != '')).sum())
            print(f'\n  {c}  [texto libre]  no vacios: {nn} ({100*nn/len(LAB):.1f} %)')
            for s in v.dropna().astype(str).head(3): print('      ·', s[:160])
            continue
        vc = v.value_counts(dropna=False)
        nz = int((v.notna() & ~v.astype(str).isin(['0', '0.0', 'nan', 'None', ''])).sum())
        print(f'\n  {c}   no-cero/no-vacio: {nz} ({100*nz/len(LAB):.2f} %)')
        print('     ', dict(itertools.islice(vc.to_dict().items(), 12)))

sep('4 · DESACUERDO ENTRE JUEGOS DE ETIQUETAS (ruido de etiqueta medido)')
if LAB is not None:
    def base(c):
        return re.sub(r'(primary|secondary|first|second|orig(inal)?|new|v?[12])', '',
                      str(c).lower()).strip(' _-')
    grupos = {}
    for c in cols_a: grupos.setdefault(base(c), []).append(c)
    hubo = False
    for b, cs in grupos.items():
        if len(cs) < 2: continue
        for a1, a2 in itertools.combinations(cs, 2):
            x, y = LAB[a1].astype(str), LAB[a2].astype(str)
            dif = int((x != y).sum())
            hubo = True
            print(f'  {b[:34]:34s} {a1} vs {a2}: {dif}/{len(LAB)} difieren '
                  f'({100*dif/len(LAB):.2f} %)')
    if not hubo:
        print('  No se detectaron pares primario/secundario por nombre en esta tabla.')

sep('5 · CURVAS: forma, huecos y DUPLICADOS EXACTOS')
def md5_filas(MC):
    return [hashlib.md5(np.ascontiguousarray(
        np.nan_to_num(np.asarray(r, dtype=float))).tobytes()).hexdigest() for r in MC]

CUR = CURN = None; ndup = None; h = None; n_cur = None
npys = [f for f in FIL if f.lower().endswith('.npy')]
for f in npys:
    try:
        a = np.load(f, allow_pickle=True)
    except Exception as ex:
        print(f'  {os.path.relpath(f,DEST)}: no cargable ({ex})'); continue
    print(f'  {os.path.relpath(f,DEST)}: shape={getattr(a,"shape",None)} dtype={a.dtype}')
    if CUR is None or (hasattr(a, 'shape') and a.size > getattr(CUR, 'size', 0)):
        CUR, CURN = a, os.path.relpath(f, DEST)

if CUR is None:
    print('\n  No hay .npy. Buscando curvas en ficheros sueltos (una curva por fichero)...')
    sueltos = [f for f in FIL if os.path.splitext(f)[1].lower() in ('.txt', '.csv', '.dat')
               and os.path.getsize(f) < 2_000_000]
    print(f'  candidatos: {len(sueltos)}')
    if len(sueltos) > 100:
        hs = {}
        for f in sueltos:
            hs.setdefault(hashlib.md5(open(f, 'rb').read()).hexdigest(), []).append(f)
        dups = {k: v for k, v in hs.items() if len(v) > 1}
        nd = sum(len(v) - 1 for v in dups.values())
        print(f'  *** ficheros duplicados byte a byte: {nd} de {len(sueltos)} '
              f'({100*nd/len(sueltos):.2f} %) en {len(dups)} grupos ***')
        ndup, n_cur = nd, len(sueltos)
else:
    A = np.asarray(CUR)
    print(f'\n  >>> se analiza {CURN}')
    if A.dtype == object:
        formas = [np.asarray(x).shape for x in A.ravel()[:200]]
        print('   array de objetos. formas de las 5 primeras:', formas[:5])
        print('   longitudes distintas en las 200 primeras:',
              sorted(set(f[0] for f in formas if f))[:12])
        MC = np.array([np.asarray(x, dtype=float).ravel()[:400] for x in A.ravel()])
    else:
        print(f'   shape={A.shape}  dtype={A.dtype}')
        fin = np.isfinite(A.astype(float))
        print(f'   NaN/inf: {100*(1-fin.mean()):.4f} %')
        MC = A.reshape(A.shape[0], -1).astype(float)
        if A.ndim == 3:
            for k in range(A.shape[-1]):
                v = A[..., k].astype(float); v = v[np.isfinite(v)]
                print(f'   canal {k}: min={v.min():.4g}  p50={np.median(v):.4g}  max={v.max():.4g}')
    print(f'\n   curvas: {MC.shape[0]}   valores por curva: {MC.shape[1]}')
    h = pd.Series(md5_filas(MC))
    ndup = int(h.duplicated().sum()); n_cur = len(h)
    print(f'\n   *** DUPLICADOS EXACTOS: {ndup} de {len(h)} ({100*ndup/len(h):.2f} %) ***')
    if ndup:
        gg = h.value_counts(); gg = gg[gg > 1]
        print(f'       grupos: {len(gg)}   tamano maximo: {int(gg.max())}')
    var = np.nanvar(MC, axis=1)
    print(f'   curvas de varianza cero (senal congelada): {int((var == 0).sum())}')
    print(f'   curvas con algun NaN: {int((~np.isfinite(MC)).any(axis=1).sum())}')
    if LAB is not None and len(LAB) == MC.shape[0]:
        print('   [OK] el numero de curvas coincide con el de filas de etiquetas')
    elif LAB is not None:
        print(f'   [AVISO] curvas={MC.shape[0]} pero filas de etiquetas={len(LAB)}')

sep('6 · PARTICION — donde esta la fuga')
if LAB is not None and col_s:
    n = LAB[col_s].nunique()
    print(f'  muestras: {n}  ->  70/15/15 por muestra deja ~{int(0.15*n)} en test')
    if cols_a:
        print('\n  casos por tipo de anomalia que caerian en un test del 15 % (estimacion orientativa):')
        for c in cols_a[:12]:
            v = LAB[c]
            if v.dtype == object and v.astype(str).str.len().mean() > 40: continue
            nz = int((v.notna() & ~v.astype(str).isin(['0', '0.0', 'nan', 'None', ''])).sum())
            print(f'    {str(c)[:44]:44s} {nz:5d} totales  ->  ~{int(0.15*nz):4d} en test')

sep('7 · CRITERIOS DE ACEPTACION')
ok1 = ok2 = None
try:    ok1 = (ndup / n_cur) < 0.01
except Exception: pass
try:    ok2 = (LAB[col_s].nunique() >= 140) and (9000 <= len(LAB) <= 10200)
except Exception: pass
print(f'  1. duplicados exactos < 1 % ................... {ok1}')
print(f'  2. ~150 muestras y ~9600 curvas ............... {ok2}')


0 · CONTENIDO DEL ZIP
  entradas en el zip: 10049   tamano comprimido: 80.8 MB   descomprimido: 191.6 MB
  por extension: {'.csv': 9757, '.txt': 2}



--- ficheros extraidos: 9759 — los 25 mas grandes
       1.22 MB  nanoindentation_data_release/updated_all_samples_labelled.csv
       0.26 MB  nanoindentation_data_release/anomalous_indent_labels_150.csv
       0.26 MB  nanoindentation_data_release/updated_anomalous_indent_labels_150.csv
       0.02 MB  nanoindentation_data_release/supplementary/electroplating_sample_metadata.csv
       0.02 MB  nanoindentation_data_release/NiFe3_nostir/cd_90/sample_6132/619_6132_nanoindentation_17.csv
       0.02 MB  nanoindentation_data_release/NiFe0_nostir/cd_175/sample_4158/609_4158_nanoindentation_33.csv
       0.02 MB  nanoindentation_data_release/NiFe3_nostir/cd_90/sample_6132/619_6132_nanoindentation_46.csv
       0.02 MB  nanoindentation_data_release/NiFe3_nostir/cd_90/sample_6132/619_6132_nanoindentation_37.csv
       0.02 MB  nanoindentation_data_release/NiFe3_nostir/cd_90/sample_6132/619_6132_nanoindentation_20.csv
       0.02 MB  nanoindentation_data_release/NiFe0_nostir/cd_175/sample_41


  ===== nanoindentation_data_release/supplementary/solution_metadata.csv  (7 filas x 22 cols)
   columnas: ['solution', 'date_made', 'primary_metal_salt', 'volume_mL', 'primary_metal_salt_conc_g/L', 'primary_metal_salt_conc_M', 'used_salt', 'Ni_salt_g', 'acid', 'acid_conc_g/L', 'acid_conc_M', 'acid_g', 'additive_1', 'additive1_conc_g/L', 'additive1_conc_M', 'additive1_add_g', 'additive2', 'additive2_stock_M', 'secondary_metal_salt', 'secondary_metal_salt_conc_g/L', 'secondary_metal_salt_conc_M', 'Fe_salt_g']
  solution  date_made primary_metal_salt  volume_mL  primary_metal_salt_conc_g/L  primary_metal_salt_conc_M     used_salt  Ni_salt_g   acid  acid_conc_g/L  acid_conc_M  acid_g additive_1  additive1_conc_g/L  additive1_conc_M  additive1_add_g additive2  additive2_stock_M secondary_metal_salt  secondary_metal_salt_conc_g/L  secondary_metal_salt_conc_M  Fe_salt_g
0   NiFe_0     220730       NiSO4*(H2O)7        200                        213.0                   0.758412  NiSO4*(H2O)6 

  *** ficheros duplicados byte a byte: 0 de 9759 (0.00 %) en 0 grupos ***

6 · PARTICION — donde esta la fuga
  muestras: 150  ->  70/15/15 por muestra deja ~22 en test

  casos por tipo de anomalia que caerian en un test del 15 % (estimacion orientativa):
    expert_note                                   3180 totales  ->  ~ 477 en test
    fully_anomalous_sample                         704 totales  ->  ~ 105 en test
    force_offset                                  1348 totales  ->  ~ 202 en test
    displacement_offset                            139 totales  ->  ~  20 en test
    unusual_unloading_curvature                    989 totales  ->  ~ 148 en test
    unusual_loading_curvature                      227 totales  ->  ~  34 en test
    tip_displacement_decreases_at_holding          223 totales  ->  ~  33 en test
    too_deep                                        84 totales  ->  ~  12 en test

7 · CRITERIOS DE ACEPTACION
  1. duplicados exactos < 1 % ................... True
  2

**Cómo leer la salida**

Los 64 ensayos de una película no son independientes. Partir al azar por ensayo pondría ensayos de una misma película en entrenamiento y en prueba, lo que es una fuga de información. La partición tiene que hacerse por película y, si las películas se agrupan por condición de fabricación, por esa condición. Las cifras del bloque 6 son solo una estimación orientativa: la validación final deja fuera una campaña entera (celda 15).

### Celda 2. Segundo juego de marcas, estructura de las curvas y primer contraste con la fabricación

- (a) Busca, entre las tablas de primer nivel, un segundo juego completo de marcas: como el fichero vigente se llama «updated_», se busca el original.
- (b) Cuántas películas, no ensayos, tienen cada tipo de defecto. Decide si una partición por película deja ejemplos suficientes de los tipos raros. Además, la columna `other`, que la celda 1 no cuenta.
- (c) Condiciones de fabricación: solución, densidad de corriente y campaña.
- (d) Proporción de ensayos con defecto frente a la densidad de corriente y frente a la solución.
- (e) Longitud y columnas de las curvas, en 400 elegidas al azar.
- (f) Enlace con los ficheros de resumen (dureza, módulo, deriva, posición X-Y).

Deja dos objetos que usan las celdas siguientes: `curvas`, la lista ordenada de los ficheros de curva, y `S`, los 150 ficheros de resumen en una sola tabla.

In [3]:
sep('a · TODOS LOS CSV DE PRIMER NIVEL (buscando un segundo juego de etiquetas)')
top = sorted(glob.glob(os.path.join(RAIZ, '*.csv')) +
             glob.glob(os.path.join(RAIZ, '*', '*.csv')))
top = [f for f in top if 'nanoindentation_' not in os.path.basename(f)]
for f in top:
    try:
        d = pd.read_csv(f, nrows=5)
        n = sum(1 for _ in open(f, encoding='utf-8', errors='replace')) - 1
        print(f'  {os.path.relpath(f, RAIZ):55s} {n:6d} filas  cols={list(d.columns)[:8]}')
    except Exception as e:
        print(f'  {os.path.relpath(f, RAIZ)}: {e}')

LABS = {}
for f in top:
    try:
        d = pd.read_csv(f)
    except Exception:
        continue
    if len(d) > 5000 and 'sample_num' in d.columns:
        LABS[os.path.basename(f)] = d
print('\n  tablas de etiquetas encontradas:', list(LABS.keys()))

if len(LABS) >= 2:
    ks = list(LABS.keys())
    A, B = LABS[ks[0]], LABS[ks[1]]
    print(f'\n  DESACUERDO {ks[0]} vs {ks[1]}:')
    if len(A) == len(B):
        for c in ANOM:
            if c in A.columns and c in B.columns:
                dif = int((A[c].astype(str) != B[c].astype(str)).sum())
                print(f'    {c:40s} {dif:5d}/{len(A)} difieren ({100*dif/len(A):5.2f} %)')
    else:
        print(f'    longitudes distintas ({len(A)} vs {len(B)}), hay que alinear por path')
else:
    print('  un solo juego completo de etiquetas (las dos revisiones se comparan en la celda 3)')

if not LABS:
    for f in glob.glob(os.path.join(RAIZ, '**', '*.csv'), recursive=True):
        if 'summary' in f or re.search(r'nanoindentation_\d+\.csv$', f): continue
        try:
            d = pd.read_csv(f)
        except Exception:
            continue
        if len(d) > 5000 and 'sample_num' in d.columns:
            LABS[os.path.relpath(f, RAIZ)] = d
    print('  (rescate) tablas encontradas:', list(LABS.keys()))
if not LABS:
    raise SystemExit('No se encontro ninguna tabla de etiquetas con >5000 filas y sample_num.')
pref = [k for k in LABS if os.path.basename(k).startswith('updated')]
LAB = LABS[pref[0]] if pref else list(LABS.values())[0]
print(f'\n  >>> se usa como tabla vigente: {pref[0] if pref else list(LABS)[0]}')

sep('b · CUANTAS MUESTRAS (no indentaciones) tiene cada anomalia')
print(f"{'anomalia':42s} {'indents>=2':>11s} {'muestras>=2':>12s} {'muestras>=1':>12s} "
      f"{'% muestras':>11s}")
ns = LAB.sample_num.nunique()
for c in ANOM:
    if c not in LAB.columns: continue
    v = pd.to_numeric(LAB[c], errors='coerce').fillna(0)
    n_ind = int((v >= 2).sum())
    m2 = LAB.loc[v >= 2, 'sample_num'].nunique()
    m1 = LAB.loc[v >= 1, 'sample_num'].nunique()
    print(f'  {c:40s} {n_ind:11d} {m2:12d} {m1:12d} {100*m2/ns:10.1f} %')
print(f'\n  total de muestras: {ns}')

if 'other' in LAB.columns:
    print('\n  columna `other`:')
    print('   ', dict(itertools.islice(LAB.other.value_counts(dropna=False).to_dict().items(), 14)))

sep('c · CONDICIONES DE FABRICACION (solucion, densidad de corriente y campaña)')
for c in ['solution', 'current_density', 'batch_num']:
    if c in LAB.columns:
        vc = LAB.groupby(c).sample_num.nunique()
        print(f'\n  {c}: {len(vc)} niveles · muestras por nivel:')
        print('   ', dict(vc.items()))

sep('d · EL CONTRASTE: ¿la anomalia depende de la densidad de corriente?')
LAB['_cd'] = pd.to_numeric(LAB.current_density, errors='coerce')
LAB['_mala'] = 0
for c in ANOM:
    if c in LAB.columns:
        LAB['_mala'] = np.maximum(LAB['_mala'],
                                  (pd.to_numeric(LAB[c], errors='coerce').fillna(0) >= 2).astype(int))
print(f"  indentaciones con al menos una anomalia de nivel >=2: "
      f"{int(LAB._mala.sum())} de {len(LAB)} ({100*LAB._mala.mean():.1f} %)")

bandas_cd = [0, 5, 15, 30, 50, 80, 125, 1e9]
LAB['_cdbin'] = pd.cut(LAB._cd, bandas_cd)
t = LAB.groupby('_cdbin', observed=True).agg(
    muestras=('sample_num', 'nunique'), indents=('_mala', 'size'),
    pct_anomalo=('_mala', lambda x: round(100 * x.mean(), 1)))
print('\n  --- por banda de densidad de corriente (mA/cm2) ---')
print(t.to_string())

print('\n  --- por bano ---')
t2 = LAB.groupby('solution', observed=True).agg(
    muestras=('sample_num', 'nunique'), indents=('_mala', 'size'),
    cd_min=('_cd', 'min'), cd_max=('_cd', 'max'),
    pct_anomalo=('_mala', lambda x: round(100 * x.mean(), 1)))
print(t2.to_string())

print('\n  --- por tipo de anomalia x banda de corriente (% de indentaciones) ---')
for c in ANOM:
    if c not in LAB.columns: continue
    v = (pd.to_numeric(LAB[c], errors='coerce').fillna(0) >= 2).astype(int)
    g = v.groupby(LAB._cdbin, observed=True).mean() * 100
    print(f'  {c:40s} ' + ' '.join(f'{x:6.1f}' for x in g.values))
print(f"  {'(bandas)':40s} " + ' '.join(f'{str(i):>6s}' for i in t.index.astype(str)))

sep('e · CURVAS: longitud, columnas y rangos')
curvas = sorted(glob.glob(os.path.join(RAIZ, '**', '*_nanoindentation_*.csv'), recursive=True))
curvas = [f for f in curvas if 'summary' not in os.path.basename(f)]
print(f'  ficheros de curva: {len(curvas)}   (esperado 9600)')
rs = np.random.default_rng(0)
mu = rs.choice(len(curvas), size=min(400, len(curvas)), replace=False)
largos, cols, mn, mx = [], set(), [], []
for i in mu:
    d = pd.read_csv(curvas[i])
    largos.append(len(d)); cols.add(tuple(d.columns))
    mn.append(d.depth_nm.min()); mx.append(d.depth_nm.max())
print(f'  longitudes (400 muestreadas): min={min(largos)} max={max(largos)} '
      f'distintas={sorted(set(largos))[:8]}')
print(f'  juegos de columnas distintos: {len(cols)} -> {list(cols)[0]}')
print(f'  profundidad: min de los minimos={min(mn):.1f} nm · max de los maximos={max(mx):.1f} nm')

sep('f · FICHEROS SUMMARY: variables derivadas por indentacion')
sums = sorted(glob.glob(os.path.join(RAIZ, '**', '*_summary.csv'), recursive=True))
print(f'  ficheros summary: {len(sums)}   (esperado 150)')
S = []
for f in sums:
    try:
        d = pd.read_csv(f)
        d['sample_num'] = int(re.search(r'sample_(\d+)', f).group(1)) if re.search(
            r'sample_(\d+)', f) else np.nan
        S.append(d)
    except Exception:
        pass
if S:
    S = pd.concat(S, ignore_index=True)
    print(f'  filas totales: {len(S)}   muestras: {S.sample_num.nunique()}')
    LS = LAB.merge(S, on=CLAVE, how='left', suffixes=('', '_s'))
    print(f'  cruce con etiquetas: {LS.hardness_H_GPa.notna().sum()} de {len(LS)} filas emparejadas')
    print('\n  medianas por estado (nivel >=2 en cualquier anomalia):')
    cols_i = ['hardness_H_GPa', 'reduced_modulus_Er_GPa', 'contact_depth_hc_nm',
              'max_displacement_hmax_nm', 'unloading_curve_m', 'unloading_curve_alpha',
              'drift_nm/s']
    cols_i = [c for c in cols_i if c in LS.columns]
    print(LS.groupby('_mala')[cols_i].median().round(3).to_string())
    print('\n  ¿estan las anomalias agrupadas en el espacio de la muestra?')
    if {'X_mm', 'Y_mm'}.issubset(LS.columns):
        g = LS.groupby('sample_num').agg(n_mala=('_mala', 'sum'))
        print(f'    muestras con 0 anomalias: {(g.n_mala == 0).sum()} de {len(g)}')
        print(f'    muestras con 64/64 anomalas: {(g.n_mala == 64).sum()}')
        print(f'    distribucion de n_mala por muestra (deciles): '
              f'{np.percentile(g.n_mala, [0,10,25,50,75,90,100]).round(1).tolist()}')

sep('VEREDICTO')
por_muestra = LAB.groupby('sample_num').size()
incompl = int((por_muestra != 64).sum())
print(f'  {ns} muestras x 64 = {len(LAB)} · {incompl} muestras incompletas -> '
      f'{"PASA" if (ns == 150) and (len(LAB) == 9600) and (incompl == 0) else "REVISAR"}')


a · TODOS LOS CSV DE PRIMER NIVEL (buscando un segundo juego de etiquetas)
  anomalous_indent_labels_150.csv                           3271 filas  cols=['solution', 'current_density', 'batch_num', 'sample_num', 'indent_num', 'expert_note', 'verify_zero_offset_type', 'fully_anomalous_sample']
  supplementary/constituent_metadata.csv                       5 filas  cols=['constituent', 'MS_g/mol']
  supplementary/electroplating_sample_metadata.csv           150 filas  cols=['batch_num', 'sample_num', 'acid', 'acid_stock_M', 'additive1', 'additive1_stock_M', 'additive2', 'additive2_stock_M']
  supplementary/formula_metadata.csv                          11 filas  cols=['Ni_formula_parameter', 'value', 'units', 'additional_notes']
  supplementary/solution_metadata.csv                          7 filas  cols=['solution', 'date_made', 'primary_metal_salt', 'volume_mL', 'primary_metal_salt_conc_g/L', 'primary_metal_salt_conc_M', 'used_salt', 'Ni_salt_g']
  updated_all_samples_labelled.csv      

  longitudes (400 muestreadas): min=400 max=400 distintas=[400]
  juegos de columnas distintos: 1 -> ('depth_nm', 'load_micro_N', 'time_s', 'depth_V', 'load_V')
  profundidad: min de los minimos=-95.5 nm · max de los maximos=1334.6 nm

f · FICHEROS SUMMARY: variables derivadas por indentacion
  ficheros summary: 150   (esperado 150)


  filas totales: 9600   muestras: 150
  cruce con etiquetas: 9600 de 9600 filas emparejadas

  medianas por estado (nivel >=2 en cualquier anomalia):
       hardness_H_GPa  reduced_modulus_Er_GPa  contact_depth_hc_nm  max_displacement_hmax_nm  unloading_curve_m  unloading_curve_alpha  drift_nm/s
_mala                                                                                                                                             
0               3.774                 124.517               66.655                   141.003              1.759                  3.570         0.0
1               3.401                 109.469               72.604                   173.552              2.309                  0.364         0.0

  ¿estan las anomalias agrupadas en el espacio de la muestra?
    muestras con 0 anomalias: 0 de 150
    muestras con 64/64 anomalas: 11
    distribucion de n_mala por muestra (deciles): [7.0, 10.0, 13.0, 18.0, 23.0, 32.1, 64.0]

VEREDICTO
  150 muestras x 64 =

**Cómo leer la salida**

Lo que decide si un tipo de defecto es evaluable por película es el número de películas que lo tienen, no el de ensayos (tabla b): con menos de unas 15, una prueba con el 15 % de las películas deja 2 o 3 con ese defecto. Las dos revisiones de las marcas solo contienen los ensayos marcados, por eso no aparecen aquí como juego completo; se comparan en la celda 3. La solución y la densidad de corriente vienen en la tabla de marcas, y la química completa del baño, en la carpeta `supplementary`; el enlace de cada película con sus datos de fabricación se comprueba en la celda 4.

### Celda 3. Cierre de la integridad de las curvas y de las marcas

No modela nada. Cierra, con un sí o un no por punto:

1. las dos revisiones de las marcas, alineadas por película y ensayo;
2. que la columna `path` apunte a ficheros que existen, sin que sobre ni falte ninguno;
3. valores no finitos en las 9.600 curvas;
4. tramos congelados (al menos 20 pasos seguidos sin cambio de profundidad) dentro de una curva;
5. la misma escala de tiempos en todas las curvas;
6. que todas alcancen la fuerza objetivo (más de 900 µN);
7. curvas casi idénticas (valores redondeados a la milésima), además de los duplicados byte a byte de la celda 1;
8. coherencia de las 11 películas marcadas como completamente anómalas, y cuántas marcas por ensayo llevan.

Lee las 9.600 curvas una vez y termina con un acta que resume cada punto. Deja `ALL` (la tabla de marcas vigente), `O` y `U` (las dos revisiones), `Oi` y `Ui` (las mismas, indexadas por película y ensayo) y `comun` (los ensayos presentes en las dos).

In [4]:
ACTA = []
def acta(punto, ok, detalle=''):
    ACTA.append((punto, ok, detalle))
    print(f'  [{"OK " if ok else "REV"}] {punto}' + (f'   -> {detalle}' if detalle else ''))

sep('1 · RUIDO DE ETIQUETA — primera pasada frente a la corregida')
ALL = pd.read_csv(os.path.join(RAIZ, 'updated_all_samples_labelled.csv'))
O = pd.read_csv(os.path.join(RAIZ, 'anomalous_indent_labels_150.csv'))
U = pd.read_csv(os.path.join(RAIZ, 'updated_anomalous_indent_labels_150.csv'))
print(f'  primera pasada : {len(O)} indentaciones anomalas')
print(f'  corregida      : {len(U)} indentaciones anomalas')
print(f'  columnas solo en la primera : {sorted(set(O.columns) - set(U.columns))}')
print(f'  columnas solo en la segunda : {sorted(set(U.columns) - set(O.columns))}')

kO = set(map(tuple, O[CLAVE].values))
kU = set(map(tuple, U[CLAVE].values))
print(f'\n  indentaciones que SOLO estan en la primera (se retiraron): {len(kO - kU)}')
print(f'  indentaciones que SOLO estan en la corregida (se anadieron): {len(kU - kO)}')
print(f'  presentes en las dos: {len(kO & kU)}')

comun = sorted(kO & kU)
Oi = O.set_index(CLAVE); Ui = U.set_index(CLAVE)
print('\n  entre las que estan en las dos, cuantas cambiaron de nivel:')
tot_dif = 0
for c in ANOM:
    if c in O.columns and c in U.columns:
        a = pd.to_numeric(Oi.loc[comun, c], errors='coerce').fillna(0).values
        b = pd.to_numeric(Ui.loc[comun, c], errors='coerce').fillna(0).values
        d = int((a != b).sum()); tot_dif += d
        subs = int(((a > 0) & (b == 0)).sum()); anad = int(((a == 0) & (b > 0)).sum())
        print(f'    {c:40s} {d:5d} cambios  (quitadas {subs}, puestas {anad})')
n_tot = len(kO | kU)
tasa = 100 * (len(kO ^ kU) + tot_dif) / max(n_tot, 1)
print(f'\n  TASA DE REVISION: {tasa:.1f} % de las indentaciones que alguna vez '
      f'fueron marcadas cambiaron entre las dos pasadas')
acta('Existen dos juegos de etiquetas comparables', True,
     f'{len(kO)} -> {len(kU)}; revision del {tasa:.1f} %')

sep('2 · LA COLUMNA `path` FRENTE AL ARBOL DE FICHEROS')
en_disco = set(os.path.relpath(f, RAIZ).replace('\\', '/') for f in curvas)
en_tabla = set(str(p).replace('\\', '/').lstrip('./') for p in ALL.path)
print(f'  curvas en disco : {len(en_disco)}')
print(f'  paths en tabla  : {len(en_tabla)}')
falt = en_tabla - en_disco; sobr = en_disco - en_tabla
print(f'  en la tabla pero NO en disco : {len(falt)}')
if falt: print('   ', list(falt)[:5])
print(f'  en disco pero NO en la tabla : {len(sobr)}')
if sobr: print('   ', list(sobr)[:5])
acta('Cada fila de la tabla tiene su fichero de curva', len(falt) == 0, f'{len(falt)} huerfanas')
acta('No hay curvas fuera del catalogo', len(sobr) == 0, f'{len(sobr)} sobrantes')

sep('3-7 · LECTURA DE LAS 9.600 CURVAS')
n = len(curvas)
nofin = 0; congel = 0; malt = 0; nopico = 0
t_ref = None; hashes = []; picos = []; largos = set()
for i, f in enumerate(curvas):
    if i % 2000 == 0: print(f'    {i}/{n}')
    d = pd.read_csv(f)
    largos.add(len(d))
    v = d[['depth_nm', 'load_micro_N']].to_numpy(float)
    if not np.isfinite(v).all(): nofin += 1
    dd = np.diff(v[:, 0])
    if len(dd):
        ceros = (dd == 0).astype(int)
        run = 0; mx = 0
        for z in ceros:
            run = run + 1 if z else 0
            mx = max(mx, run)
        if mx >= 20: congel += 1
    t = d.time_s.to_numpy(float)
    if t_ref is None: t_ref = t
    elif len(t) != len(t_ref) or not np.allclose(t, t_ref, atol=1e-9): malt += 1
    pk = float(np.nanmax(v[:, 1])); picos.append(pk)
    if pk < 900: nopico += 1
    hashes.append(hashlib.md5(np.round(v, 3).tobytes()).hexdigest())

print(f'\n  longitudes distintas: {sorted(largos)}')
acta('Todas las curvas tienen la misma longitud', len(largos) == 1, f'{sorted(largos)}')
acta('Sin valores no finitos', nofin == 0, f'{nofin} curvas con NaN/inf')
acta('Sin tramos congelados de 20+ muestras', congel == 0, f'{congel} curvas')
acta('Rejilla de tiempo identica en todas', malt == 0, f'{malt} curvas con otra rejilla')
picos = np.array(picos)
print(f'\n  carga maxima alcanzada: p1={np.percentile(picos,1):.1f} '
      f'mediana={np.median(picos):.1f} p99={np.percentile(picos,99):.1f} microN')
acta('Todas alcanzan la carga objetivo (>900 microN)', nopico == 0,
     f'{nopico} curvas por debajo')

s = pd.Series(hashes)
ndup = int(s.duplicated().sum())
print(f'\n  *** curvas CASI identicas (mismos valores redondeados a 0,001): '
      f'{ndup} de {n} ({100*ndup/n:.2f} %) ***')
if ndup:
    g = s.value_counts(); g = g[g > 1]
    print(f'      grupos: {len(g)}  tamano maximo: {int(g.max())}')
    ej = g.index[0]
    print('      ejemplo de grupo:', [os.path.relpath(curvas[j], RAIZ)
                                      for j in np.flatnonzero(s.values == ej)][:4])
acta('Sin curvas numericamente duplicadas', ndup == 0, f'{ndup} ({100*ndup/n:.2f} %)')

sep('8 · LAS 11 MUESTRAS COMPLETAMENTE ANOMALAS')
fa = pd.to_numeric(ALL.fully_anomalous_sample, errors='coerce').fillna(0)
ms = sorted(ALL.loc[fa >= 2, 'sample_num'].unique())
print(f'  muestras marcadas: {len(ms)} -> {ms}')
sub = ALL[ALL.sample_num.isin(ms)]
print(f'  indentaciones: {len(sub)}  (deberian ser {64*len(ms)})')
print('\n  ¿llevan ADEMAS etiquetas por indentacion?')
for c in POR_INDENT:
    v = pd.to_numeric(sub[c], errors='coerce').fillna(0)
    print(f'    {c:40s} {int((v>=2).sum()):4d} de {len(sub)}')
niv = sub[POR_INDENT].apply(pd.to_numeric, errors='coerce').fillna(0) >= 2
print(f'    {"total de marcas":40s} {int(niv.values.sum()):4d}, en {int(niv.any(axis=1).sum())} '
      f'de los {len(sub)} ensayos')
print('\n  condiciones de proceso de esas 11:')
print(sub.groupby('sample_num').agg(solution=('solution','first'),
      current_density=('current_density','first'), batch=('batch_num','first')).to_string())
acta('Las muestras completamente anomalas son coherentes (64/64)',
     len(sub) == 64*len(ms), f'{len(sub)} filas para {len(ms)} muestras')

sep('ACTA DE INTEGRIDAD')
ok = sum(1 for _, o, _ in ACTA if o)
for p, o, d in ACTA:
    print(f'  {"OK " if o else "REV"}  {p}' + (f'  ({d})' if d else ''))
print(f'\n  {ok} de {len(ACTA)} comprobaciones en verde.')


1 · RUIDO DE ETIQUETA — primera pasada frente a la corregida
  primera pasada : 3271 indentaciones anomalas
  corregida      : 3409 indentaciones anomalas
  columnas solo en la primera : ['verification_notes', 'verify_zero_offset_type']
  columnas solo en la segunda : []

  indentaciones que SOLO estan en la primera (se retiraron): 91
  indentaciones que SOLO estan en la corregida (se anadieron): 229
  presentes en las dos: 3180

  entre las que estan en las dos, cuantas cambiaron de nivel:
    fully_anomalous_sample                       3 cambios  (quitadas 0, puestas 3)
    force_offset                              1226 cambios  (quitadas 1, puestas 1225)
    displacement_offset                         23 cambios  (quitadas 6, puestas 3)
    unusual_unloading_curvature                116 cambios  (quitadas 0, puestas 70)
    unusual_loading_curvature                   33 cambios  (quitadas 2, puestas 9)
    tip_displacement_decreases_at_holding       88 cambios  (quitadas 1, puesta

    2000/9600


    4000/9600


    6000/9600


    8000/9600



  longitudes distintas: [400]
  [OK ] Todas las curvas tienen la misma longitud   -> [400]
  [OK ] Sin valores no finitos   -> 0 curvas con NaN/inf
  [OK ] Sin tramos congelados de 20+ muestras   -> 0 curvas
  [OK ] Rejilla de tiempo identica en todas   -> 0 curvas con otra rejilla

  carga maxima alcanzada: p1=998.2 mediana=998.4 p99=998.5 microN
  [OK ] Todas alcanzan la carga objetivo (>900 microN)   -> 0 curvas por debajo

  *** curvas CASI identicas (mismos valores redondeados a 0,001): 0 de 9600 (0.00 %) ***
  [OK ] Sin curvas numericamente duplicadas   -> 0 (0.00 %)

8 · LAS 11 MUESTRAS COMPLETAMENTE ANOMALAS
  muestras marcadas: 11 -> [np.int64(4141), np.int64(4147), np.int64(4154), np.int64(4158), np.int64(4267), np.int64(4273), np.int64(4279), np.int64(5946), np.int64(6132), np.int64(6146), np.int64(6392)]
  indentaciones: 704  (deberian ser 704)

  ¿llevan ADEMAS etiquetas por indentacion?
    force_offset                               21 de 704
    displacement_offset     

**Cómo leer la salida**

Un punto en REV no invalida el conjunto por sí solo: hay que leerlo y decidir si es un defecto o una propiedad del dato. Lo descalificante serían duplicados numéricos por encima del 1 % o filas de la tabla sin su curva.

### Celda 4. Notación de las marcas, densidad de corriente y metadatos de fabricación

1. Si los 1.225 desplazamientos de fuerza que aparecen en la segunda revisión son un cambio de notación (notas de texto «zero offset» que pasan a columna) o un cambio de criterio del especialista.
2. Si las 11 películas completamente anómalas se explican por la corriente alta o por la campaña.
3. Fecha y jornada de medida, deriva, posición X-Y y metadatos de fabricación de cada película. Los metadatos son el enlace de cada película con sus datos de fabricación, la cuarta condición de aceptación de la celda 1.

In [5]:
sep('1 · ¿MIGRACION DE TAXONOMIA O CAMBIO DE CRITERIO?')
a = pd.to_numeric(Oi.loc[comun, 'force_offset'], errors='coerce').fillna(0).values
b = pd.to_numeric(Ui.loc[comun, 'force_offset'], errors='coerce').fillna(0).values
puestas = np.array(comun, dtype=object)[(a == 0) & (b > 0)]
print(f'  force_offset que pasaron de 0 a >0: {len(puestas)}')

nota_o = Oi.loc[comun, 'expert_note'].astype(str).str.lower()
es_zero = nota_o.str.contains('zero offset|force offset', na=False).values
mask = (a == 0) & (b > 0)
print(f'\n  de esas, cuantas YA tenian una nota de texto de tipo "zero/force offset"')
print(f'  escrita en la PRIMERA pasada: {int((mask & es_zero).sum())} de {int(mask.sum())} '
      f'({100*(mask & es_zero).sum()/max(mask.sum(),1):.1f} %)')
print('\n  notas de texto mas frecuentes entre las que se anadieron:')
print('   ', dict(list(nota_o[mask].value_counts().head(8).items())))

print('\n  --- lo mismo para los demas tipos ---')
for c in ANOM:
    if c == 'force_offset' or c not in O.columns: continue
    x = pd.to_numeric(Oi.loc[comun, c], errors='coerce').fillna(0).values
    y = pd.to_numeric(Ui.loc[comun, c], errors='coerce').fillna(0).values
    m = (x == 0) & (y > 0)
    clave_txt = {'displacement_offset': 'zero offset|displacement offset',
                 'unusual_unloading_curvature': 'unload',
                 'unusual_loading_curvature': 'loading',
                 'tip_displacement_decreases_at_holding': 'drift|holding',
                 'too_deep': 'deep'}.get(c, c.split('_')[0])
    ya = nota_o.str.contains(clave_txt, na=False).values
    print(f'    {c:40s} puestas={int(m.sum()):5d}  con nota previa compatible='
          f'{int((m & ya).sum()):5d} ({100*(m & ya).sum()/max(m.sum(),1):5.1f} %)')

if 'verify_zero_offset_type' in O.columns:
    print('\n  columna `verify_zero_offset_type` de la primera pasada:')
    print('   ', dict(list(O.verify_zero_offset_type.value_counts(dropna=False).head(8).items())))
if 'verification_notes' in O.columns:
    vn = O.verification_notes.dropna().astype(str)
    print(f'\n  `verification_notes`: {len(vn)} no vacias. Ejemplos:')
    for s in vn.head(6): print('    ·', s[:150])

sep('2 · CONFUNDIDO DE CAMPAÑA — ¿corriente alta, o campañas tempranas?')
PEL = ALL.groupby('sample_num').agg(solution=('solution', 'first'),
                                    cd=('current_density', 'first'),
                                    batch=('batch_num', 'first'),
                                    fa=('fully_anomalous_sample', 'max')).reset_index()
PEL['fa'] = (pd.to_numeric(PEL.fa, errors='coerce').fillna(0) >= 2).astype(int)
PEL['cd_alta'] = (PEL.cd >= 90).astype(int)

print('  muestras por campaña y si son de corriente alta (>=90 mA/cm2):')
print(pd.crosstab(PEL.batch, PEL.cd_alta, margins=True).to_string())
print('\n  peliculas COMPLETAMENTE ANOMALAS por campaña x corriente alta:')
print(pd.crosstab([PEL.batch], [PEL.cd_alta], values=PEL.fa, aggfunc='sum').fillna(0).astype(int).to_string())
print('\n  tasa de fallo total (%) por campaña x corriente alta:')
t = PEL.groupby(['batch', 'cd_alta']).agg(n=('fa', 'size'), fallos=('fa', 'sum'))
t['pct'] = (100 * t.fallos / t.n).round(1)
print(t.to_string())

alto = PEL[PEL.cd_alta == 1]
print(f'\n  ENTRE LAS MUESTRAS DE CORRIENTE ALTA ({len(alto)} peliculas):')
print(alto.groupby('batch').agg(n=('fa', 'size'), fallos=('fa', 'sum')).assign(
    pct=lambda d: (100 * d.fallos / d.n).round(1)).to_string())

sep('3 · FECHA, DERIVA, POSICION Y ADITIVOS')
SF = S.assign(fecha=pd.to_datetime(S.date, format='%a %b %d %H:%M:%S %Y', errors='coerce'))
print(f'  fechas de adquisicion: {SF.fecha.min()}  ->  {SF.fecha.max()}')
print('\n  muestras por mes de medicion:')
print(SF.groupby(SF.fecha.dt.to_period("M")).sample_num.nunique().to_string())

MF = ALL.merge(SF, on=CLAVE, how='left')
MF['_mala'] = 0
for c in ANOM:
    MF['_mala'] = np.maximum(MF['_mala'],
                             (pd.to_numeric(MF[c], errors='coerce').fillna(0) >= 2).astype(int))
print('\n  tasa de anomalia por mes de medicion:')
g = MF.groupby(MF.fecha.dt.to_period('M')).agg(
    muestras=('sample_num', 'nunique'), indents=('_mala', 'size'),
    pct=('_mala', lambda x: round(100 * x.mean(), 1)))
print(g.to_string())

dia = MF.fecha.dt.strftime('%Y-%m-%d')
print(f'\n  jornadas de medida: {dia.nunique()} · peliculas medidas en mas de una jornada: '
      f'{int((dia.groupby(MF.sample_num).nunique() > 1).sum())}')
print('  peliculas por jornada y campaña:')
print(pd.crosstab(dia, MF.batch_num, values=MF.sample_num, aggfunc='nunique')
      .fillna(0).astype(int).rename_axis(index='jornada', columns='campaña').to_string())

print(f"\n  drift_nm/s: valores distintos = {SF['drift_nm/s'].nunique()} -> "
      f"{dict(list(SF['drift_nm/s'].value_counts().head(5).items()))}")

print('\n  posicion X-Y: ¿estan las anomalas agrupadas dentro de la pelicula?')
res = []
for s, d in MF.groupby('sample_num'):
    d = d.dropna(subset=['X_mm', 'Y_mm'])
    if d._mala.sum() < 3 or d._mala.sum() > len(d) - 3: continue
    A = d[d._mala == 1][['X_mm', 'Y_mm']].values
    B = d[d._mala == 0][['X_mm', 'Y_mm']].values
    da = np.mean([np.linalg.norm(A[i] - A[j]) for i in range(len(A)) for j in range(i+1, len(A))])
    db = np.mean([np.linalg.norm(B[i] - B[j]) for i in range(len(B)) for j in range(i+1, len(B))])
    res.append(da / db if db else np.nan)
res = np.array([r for r in res if r == r])
print(f'    razon (dispersion de las anomalas / dispersion de las sanas) en '
      f'{len(res)} peliculas:')
print(f'    mediana={np.median(res):.3f}  p10={np.percentile(res,10):.3f}  '
      f'p90={np.percentile(res,90):.3f}')

f_ad = os.path.join(RAIZ, 'supplementary', 'electroplating_sample_metadata.csv')
if os.path.exists(f_ad):
    AD = pd.read_csv(f_ad)
    print(f'\n  electroplating_sample_metadata.csv: {len(AD)} filas')
    print('   columnas:', list(AD.columns))
    for c in ['acid', 'additive1', 'additive2']:
        if c in AD.columns:
            print(f'   {c}: {dict(list(AD[c].value_counts(dropna=False).head(5).items()))}')
    ok = AD.sample_num.isin(ALL.sample_num.unique()).sum()
    print(f'   muestras que cruzan con la tabla de etiquetas: {ok} de {len(AD)}')


1 · ¿MIGRACION DE TAXONOMIA O CAMBIO DE CRITERIO?
  force_offset que pasaron de 0 a >0: 1225

  de esas, cuantas YA tenian una nota de texto de tipo "zero/force offset"
  escrita en la PRIMERA pasada: 1183 de 1225 (96.6 %)

  notas de texto mas frecuentes entre las que se anadieron:
    {'zero offset': 1032, 'zero offset too high': 94, 'zero offset, also to note, most of these are very deep and probably indicate a cracked sample': 18, 'curvy unload': 16, 'zero offset high': 7, 'zero offset and drifty at 1000 micronewtons': 5, 'displacement offset': 4, 'drift at 1000 micronewtons is wrong and some zero offset': 4}

  --- lo mismo para los demas tipos ---
    fully_anomalous_sample                   puestas=    3  con nota previa compatible=    0 (  0.0 %)
    displacement_offset                      puestas=    3  con nota previa compatible=    0 (  0.0 %)
    unusual_unloading_curvature              puestas=   70  con nota previa compatible=    1 (  1.4 %)
    unusual_loading_curvatur


  jornadas de medida: 11 · peliculas medidas en mas de una jornada: 0
  peliculas por jornada y campaña:
campaña     609  614  615  619  622  623  626
jornada                                      
2023-01-11   13    2    0    0    0    0    0
2023-01-12    5    1    6    4    0    0    0
2023-01-19    0    8    3    5    0    0    0
2023-01-24    0   28    0    4    0    0    0
2023-01-25    0   12    0    0    0    0    0
2023-04-12    0    0    0    0    0   14    0
2023-04-13    0    0    6    0    0    0    0
2023-05-01    0    0    0    0   16    0    0
2023-05-02    0    0    0    0    4    0    0
2023-05-30    0    0    0    0    0    0   15
2023-05-31    0    0    0    0    0    0    4

  drift_nm/s: valores distintos = 1 -> {0.0: 9600}

  posicion X-Y: ¿estan las anomalas agrupadas dentro de la pelicula?


    razon (dispersion de las anomalas / dispersion de las sanas) en 139 peliculas:
    mediana=1.301  p10=0.994  p90=1.472

  electroplating_sample_metadata.csv: 150 filas
   columnas: ['batch_num', 'sample_num', 'acid', 'acid_stock_M', 'additive1', 'additive1_stock_M', 'additive2', 'additive2_stock_M', 'current_A', 'current_density', 'date_plated', 'primary_metal_salt', 'primary_metal_salt_stock_M', 'secondary_metal_salt', 'secondary_metal_stock_M', 'solution', 'solution_date', 'solution_mix', 'temp_celsius', 'time_s', 'working_electrode', 'is_corrupt', 'corrupt_comments', 'Max  Ƞ', 'SS-V']
   acid: {'H3BO3': 110, 'H3BO4': 1, 'H3BO5': 1, 'H3BO6': 1, 'H3BO7': 1}
   additive1: {'Saccharin': 150}
   additive2: {'SLS': 150}
   muestras que cruzan con la tabla de etiquetas: 150 de 150


**Cómo leer la salida**

Si el fallo total aparece en todas las campañas con películas de corriente alta, la explicación es la corriente. Si aparece solo en algunas, aunque las demás también lleguen a corriente alta, no puede separarse de la campaña. La tasa de defectos por mes puede reflejar también el instrumento. Una razón de dispersión menor que 1 indicaría que los ensayos marcados están más juntos que los válidos, es decir, zonas malas dentro de la película. Si `drift_nm/s` tiene un solo valor, la columna no aporta.

## Fase 2. El efecto de limpiar sobre el valor publicado

Objetivo O2; memoria, apartado 4.3.2. Es una condición previa: si limpiar no cambiara el valor de las películas, el problema no tendría objeto.

### Celda 5. El efecto de limpiar sobre el valor publicado (condición previa, objetivo O2)

Mide cuánto cambian la dureza y el módulo reducido de cada película cuando se descartan los ensayos que el especialista marcó como anómalos. No entrena nada: compara la media de los 64 ensayos (el valor sin limpiar) con la media de los ensayos que el especialista conserva (el valor limpio). El bloque 1-bis da el resumen por campaña y el bloque 3 descarta un solo tipo de defecto cada vez.

Criterios fijados antes de ejecutar:

- D1. Un ensayo se descarta si tiene cualquiera de los seis tipos de defecto por ensayo en nivel 2 o superior. Los umbrales 1 y 3 se calculan como sensibilidad.
- D2. Las 11 películas completamente anómalas quedan fuera: apenas llevan marcas por ensayo (57 marcas en 30 de sus 704 ensayos, celda 3), de modo que su valor limpio mediría el etiquetado y no el dato.
- D3. Las películas con menos de 10 ensayos supervivientes se informan aparte.
- Condición previa: falla si la mediana del cambio relativo de la dureza, en valor absoluto, es menor del 1 % y la dispersión se estrecha en menos del 60 % de las películas. En otro caso se cumple.
- Bloque 3-bis, sentido de cada tipo. Para cada tipo se comparan los ensayos que solo llevan ese tipo con los ensayos sin ningún tipo de la misma película (diferencia relativa de medianas; películas con al menos 2 ensayos del tipo y 10 limpios). Un tipo es direccional si al menos 2/3 de sus películas tienen el mismo signo, la prueba de signos da p < 0,01 y el signo coincide en todas las campañas con al menos 3 películas, salvo una como mucho. Con menos de 10 películas no es evaluable.
- Bloque 3-bis, composición. En cada película, cada marca de un ensayo descartado suma +1 o −1 según el sentido de su tipo, estimado sin la campaña de la película (0 si el tipo no es direccional). La suma predice si el valor sube o baja y se compara con la regla trivial del sentido mayoritario en las otras campañas. Entran las películas cuyo cambio supera el error típico del promedio multiplicado por √(1 + 3,5 ρ), con ρ la mediana entre películas de la correlación de rangos entre ensayos vecinos de la malla.
- Lectura del bloque 3-bis. L1: los dos tipos más frecuentes son direccionales y de signo opuesto, y la composición acierta más que la regla trivial en conjunto y al menos igual en 5 de las 7 campañas. L2: al menos uno de los dos es direccional, pero no se cumple L1. L3: ninguno de los dos es direccional. El bloque 3-ter es descriptivo: da los valores reales de la cola alta de la dureza.

Deja la tabla `M` (139 películas, con la marca del especialista en `_y`), la lista `peli_fa` de películas excluidas, la lista de campañas `LOTES` (celdas 7 y 8), la tabla `NIV` de tipos en nivel 2 o superior por ensayo (celda 8) y el fichero `precondicion_por_pelicula.csv`, con una fila por película.

In [6]:
M = ALL.merge(S, on=CLAVE, how='left')
print(f'  filas cruzadas: {M[PROPS[0]].notna().sum()} de {len(M)}')
fa = pd.to_numeric(M.fully_anomalous_sample, errors='coerce').fillna(0)
peli_fa = sorted(M.loc[fa >= 2, 'sample_num'].unique())
M = M[~M.sample_num.isin(peli_fa)].copy()
M['_y'] = (M[POR_INDENT].apply(pd.to_numeric, errors='coerce').fillna(0) >= 2).any(axis=1).astype(int)
print(f'  D2: excluidas {len(peli_fa)} peliculas completamente anomalas -> quedan '
      f'{M.sample_num.nunique()} peliculas y {len(M)} indentaciones')


def tabla(umbral):
    mal = np.zeros(len(M), int)
    for c in POR_INDENT:
        mal = np.maximum(mal, (pd.to_numeric(M[c], errors='coerce').fillna(0) >= umbral).astype(int))
    T = M.assign(_mal=mal)
    out = []
    for s, d in T.groupby('sample_num'):
        limpio = d[d._mal == 0]
        r = dict(sample_num=s, n_total=len(d), n_limpias=len(limpio),
                 pct_descartado=round(100 * d._mal.mean(), 1),
                 solution=d.solution.iloc[0], cd=d.current_density.iloc[0],
                 batch=d.batch_num.iloc[0])
        for p in PROPS:
            v_all = d[p].astype(float); v_ok = limpio[p].astype(float)
            r[f'{p}_bruta'] = v_all.mean()
            r[f'{p}_limpia'] = v_ok.mean() if len(v_ok) >= 2 else np.nan
            r[f'{p}_sd_bruta'] = v_all.std()
            r[f'{p}_sd_limpia'] = v_ok.std() if len(v_ok) >= 2 else np.nan
        out.append(r)
    R = pd.DataFrame(out)
    for p in PROPS:
        R[f'{p}_dif_pct'] = 100 * (R[f'{p}_limpia'] - R[f'{p}_bruta']) / R[f'{p}_bruta']
        R[f'{p}_sd_pct'] = 100 * (R[f'{p}_sd_limpia'] - R[f'{p}_sd_bruta']) / R[f'{p}_sd_bruta']
    return R


sep('1 · CRITERIO PRINCIPAL (descarta nivel >= 2)')
R = tabla(2)
print(f'  peliculas analizadas: {len(R)}')
print(f'  indentaciones descartadas: mediana {R.pct_descartado.median():.1f} % '
      f'(min {R.pct_descartado.min():.1f} · max {R.pct_descartado.max():.1f})')
print(f'\n  supervivientes por pelicula: min={R.n_limpias.min()} '
      f'p10={np.percentile(R.n_limpias,10):.0f} mediana={R.n_limpias.median():.0f} '
      f'max={R.n_limpias.max()}')
pocas = R[R.n_limpias < 10]
print(f'  D3: peliculas con menos de 10 supervivientes: {len(pocas)}')
if len(pocas):
    print(pocas[['sample_num', 'n_limpias', 'pct_descartado', 'solution', 'cd']].to_string(index=False))

for p in PROPS:
    d = R[f'{p}_dif_pct'].dropna()
    sd = R[f'{p}_sd_pct'].dropna()
    print(f'\n  --- {p} ---')
    print(f'    desplazamiento de la MEDIA al limpiar (%):')
    print(f'      mediana={d.median():+.2f}   media={d.mean():+.2f}   '
          f'p10={np.percentile(d,10):+.2f}   p90={np.percentile(d,90):+.2f}   '
          f'max|.|={d.abs().max():.2f}')
    print(f'      peliculas donde SUBE: {int((d>0).sum())} de {len(d)} '
          f'({100*(d>0).mean():.0f} %)')
    print(f'      peliculas con cambio > 1 %: {int((d.abs()>1).sum())} '
          f'({100*(d.abs()>1).mean():.0f} %)')
    print(f'      peliculas con cambio > 5 %: {int((d.abs()>5).sum())} '
          f'({100*(d.abs()>5).mean():.0f} %)')
    print(f'    cambio de la DISPERSION al limpiar (%):')
    print(f'      mediana={sd.median():+.2f}   p10={np.percentile(sd,10):+.2f}   '
          f'p90={np.percentile(sd,90):+.2f}')
    print(f'      peliculas donde se ESTRECHA: {int((sd<0).sum())} de {len(sd)} '
          f'({100*(sd<0).mean():.0f} %)')

sep('1-bis · DESCARTES POR CAMPAÑA (descarta nivel >= 2)')
n_pelis = ALL.groupby('batch_num').sample_num.nunique()
print(f"   {'campaña':>7} {'peliculas':>10} {'analizadas':>11} {'ensayos':>8} "
      f"{'% descartado: mediana':>22} {'min':>6} {'max':>6}")
for b, g in list(R.groupby('batch')) + [('TOTAL', R)]:
    tot = n_pelis.sum() if b == 'TOTAL' else n_pelis[b]
    print(f'   {b:>7} {tot:>10} {len(g):>11} {int(g.n_total.sum()):>8} '
          f'{g.pct_descartado.median():>22.1f} {g.pct_descartado.min():>6.1f} {g.pct_descartado.max():>6.1f}')

sep('2 · SENSIBILIDAD AL UMBRAL DE DESCARTE')
print(f"{'umbral':>8} {'% descartado':>14} {'dureza: mediana':>17} {'dureza: p90':>13} "
      f"{'dispersion: mediana':>21} {'supervivientes min':>19}")
for u in (1, 2, 3):
    Ru = tabla(u)
    d = Ru['hardness_H_GPa_dif_pct'].dropna()
    sd = Ru['hardness_H_GPa_sd_pct'].dropna()
    print(f'{u:>8} {Ru.pct_descartado.median():>13.1f}% {d.median():>+16.2f}% '
          f'{np.percentile(d.abs(),90):>12.2f}% {sd.median():>+20.2f}% '
          f'{Ru.n_limpias.min():>19}')

sep('3 · ¿QUE TIPO DE ANOMALIA APORTA MAS A LA CORRECCION?')
print(f"{'tipo':>42} {'% descartado':>13} {'dureza mediana':>16} {'modulo mediano':>16}")
for c in POR_INDENT:
    mal = (pd.to_numeric(M[c], errors='coerce').fillna(0) >= 2).astype(int)
    T = M.assign(_mal=mal)
    dh, de, pc = [], [], []
    for s, d in T.groupby('sample_num'):
        ok = d[d._mal == 0]
        if len(ok) < 2: continue
        pc.append(100 * d._mal.mean())
        dh.append(100 * (ok.hardness_H_GPa.mean() - d.hardness_H_GPa.mean()) / d.hardness_H_GPa.mean())
        de.append(100 * (ok.reduced_modulus_Er_GPa.mean() - d.reduced_modulus_Er_GPa.mean())
                  / d.reduced_modulus_Er_GPa.mean())
    print(f'  {c:>40} {np.median(pc):>12.1f}% {np.median(dh):>+15.2f}% {np.median(de):>+15.2f}%')

sep('3-bis · ¿EN QUE SENTIDO EMPUJA CADA TIPO DE DEFECTO?')
NIV = M[POR_INDENT].apply(pd.to_numeric, errors='coerce').fillna(0) >= 2
N_TIPOS = NIV.sum(axis=1).values
LIMPIO = N_TIPOS == 0
GRUPOS = {s: M.index.get_indexer(ix) for s, ix in M.groupby('sample_num').groups.items()}
LOTE_DE = M.groupby('sample_num').batch_num.first().to_dict()
LOTES = sorted(M.batch_num.unique())


def dif_por_pelicula(p, tipo):
    excl = NIV[tipo].values & (N_TIPOS == 1)
    v_all = M[p].astype(float).values
    out = []
    for s, ii in GRUPOS.items():
        vt, vc = v_all[ii][excl[ii]], v_all[ii][LIMPIO[ii]]
        vt, vc = vt[np.isfinite(vt)], vc[np.isfinite(vc)]
        if len(vt) >= 2 and len(vc) >= 10 and np.median(vc) != 0:
            out.append((s, LOTE_DE[s], (np.median(vt) - np.median(vc)) / np.median(vc)))
    return pd.DataFrame(out, columns=['sample_num', 'batch', 'dif'])


def direccion(D):
    n = len(D)
    if n < 10:
        return dict(n=n, alto=None, bajo=None, p=None, lotes='', signo=0, estado='NO EVALUABLE')
    alto, bajo = int((D.dif > 0).sum()), int((D.dif < 0).sum())
    p_ = stats.binomtest(alto, alto + bajo, 0.5).pvalue if alto + bajo else 1.0
    sg = 1 if alto >= bajo else -1
    pl = D.groupby('batch').dif.agg(['median', 'size'])
    pl = pl[pl['size'] >= 3]
    discrepan = int((np.sign(pl['median']) != sg).sum())
    ok = (max(alto, bajo) / n >= 2 / 3) and (p_ < 0.01) and (discrepan <= 1)
    txt = ' '.join(f"{int(l)}{'+' if m > 0 else '-'}" for l, m in pl['median'].items())
    return dict(n=n, alto=alto, bajo=bajo, p=p_, lotes=txt, signo=sg if ok else 0,
                estado='DIRECCIONAL' if ok else 'sin sentido claro')


DIRS = {}
for p in PROPS:
    print(f'\n  --- {p} · ensayos de un solo tipo frente a ensayos limpios de su pelicula ---')
    print(f"    {'tipo':>40} {'pelis':>6} {'mas alto':>9} {'mas bajo':>9} {'p signos':>9}"
          f"   {'signo por campaña':<34} veredicto")
    DIRS[p] = {}
    for c in POR_INDENT:
        D = dif_por_pelicula(p, c)
        r = direccion(D)
        DIRS[p][c] = D
        f = lambda x, fmt: (fmt.format(x) if x is not None else '-')
        print(f"    {c:>40} {r['n']:>6} {f(r['alto'], '{:d}'):>9} {f(r['bajo'], '{:d}'):>9} "
              f"{f(r['p'], '{:.1e}'):>9}   {r['lotes']:<34} {r['estado']}")


def prediccion(p):
    sg_sin = {l: {c: direccion(DIRS[p][c][DIRS[p][c].batch != l])['signo'] for c in POR_INDENT}
              for l in LOTES}
    v_all = M[p].astype(float).values
    filas = []
    for s, ii in GRUPOS.items():
        v = v_all[ii]
        if not np.isfinite(v).all() or LIMPIO[ii].sum() < 2:
            continue
        A_, E_ = v.mean(), v[LIMPIO[ii]].mean()
        sem = np.std(v, ddof=1) / np.sqrt(len(v))
        lote = LOTE_DE[s]
        niv = NIV.values[ii]
        punt = sum(niv[:, j].sum() * sg_sin[lote][c] for j, c in enumerate(POR_INDENT))
        filas.append(dict(sample_num=s, batch=lote, obs=np.sign(E_ - A_),
                          pred=-np.sign(punt), fiable=abs(E_ - A_) >= sem * FAC5[p]))
    F = pd.DataFrame(filas)
    triv = {}
    for l in LOTES:
        o = F[(F.batch != l) & F.fiable].obs
        triv[l] = 1.0 if (o > 0).sum() > (o < 0).sum() else -1.0
    F['triv'] = F.batch.map(triv)
    return F


RHO5 = {p: rho_vecinos(M, p) for p in PROPS}
FAC5 = {p: factor_vecinos(RHO5[p]) for p in PROPS}
print('\n  ¿EXPLICA LA MEZCLA DE TIPOS EL SENTIDO DE LA CORRECCION DE CADA PELICULA?')
for p in PROPS:
    print(f'    {p}: correlacion entre vecinos, mediana {RHO5[p].median():.3f} · factor sobre el '
          f'error tipico {FAC5[p]:.3f}')
VEREDICTO_3BIS = {}
for p in PROPS:
    F = prediccion(p)
    G = F[F.fiable & (F.pred != 0)]
    print(f'\n  --- {p} ---   peliculas con correccion sobre el ruido: {int(F.fiable.sum())}'
          f' · con prediccion: {len(G)}')
    print(f"   {'campaña':>7} {'pelis':>6} {'composicion':>12} {'trivial':>9}")
    gana = 0
    for l in LOTES:
        g = G[G.batch == l]
        if not len(g):
            continue
        ac_c, ac_t = (g.pred == g.obs).mean(), (g.triv == g.obs).mean()
        gana += int(ac_c >= ac_t)
        print(f'    {int(l):>6} {len(g):>6} {100*ac_c:>11.0f}% {100*ac_t:>8.0f}%')
    ac_c, ac_t = (G.pred == G.obs).mean(), (G.triv == G.obs).mean()
    print(f"    {'TOTAL':>6} {len(G):>6} {100*ac_c:>11.0f}% {100*ac_t:>8.0f}%")
    print(f'    la composicion iguala o supera a la trivial en {gana} de '
          f'{G.batch.nunique()} campañas')
    VEREDICTO_3BIS[p] = (ac_c > ac_t, gana)

cuenta = NIV.sum().sort_values(ascending=False)
t1, t2 = cuenta.index[:2]
s1 = direccion(DIRS['hardness_H_GPa'][t1])['signo']
s2 = direccion(DIRS['hardness_H_GPa'][t2])['signo']
mejor, gana = VEREDICTO_3BIS['hardness_H_GPa']
print(f'\n  VEREDICTO (dureza). Tipos mas frecuentes: {t1} y {t2}.')
if s1 != 0 and s2 != 0 and s1 != s2 and mejor and gana >= 5:
    print('    L1 · los dos son direccionales y de signo opuesto, y la composicion explica el '
          'sentido de la correccion mejor que la regla trivial')
elif s1 != 0 or s2 != 0:
    print('    L2 · hay sesgo por tipo a nivel de ensayo, pero no explica por si solo el sentido '
          'de la correccion por pelicula')
else:
    print('    L3 · ninguno de los dos tipos mas frecuentes tiene sentido claro')

sep('3-ter · LA COLA ALTA DE LA DUREZA: VALORES REALES (descriptivo)')
H = M['hardness_H_GPa'].astype(float)
razon = H / M.groupby('sample_num')['hardness_H_GPa'].transform('median')
cand = M.groupby('sample_num')['hardness_H_GPa'].max().idxmax()
d0 = M[M.sample_num == cand]
print(f'  dureza individual mas alta de las 139 peliculas: {H.max():.2f} GPa')
print(f'  pelicula que la contiene: {cand} · campaña '
      f'{int(d0.batch_num.iloc[0])} · mediana de la pelicula {d0.hardness_H_GPa.median():.2f} GPa')
print(f"    {'indent':>7} {'dureza GPa':>11} {'x mediana':>10} {'modulo GPa':>11}   tipos (nivel >= 2)")
for i, r in d0.sort_values('hardness_H_GPa', ascending=False).head(6).iterrows():
    tipos = [c for c in POR_INDENT if NIV.loc[i, c]]
    print(f"    {int(r.indent_num):>7} {r.hardness_H_GPa:>11.2f} "
          f"{r.hardness_H_GPa / d0.hardness_H_GPa.median():>10.1f} "
          f"{r.reduced_modulus_Er_GPa:>11.1f}   {', '.join(tipos) if tipos else '(ninguno)'}")
print(f"\n    {'umbral':>14} {'ensayos':>8} {'marcados':>9} {'peliculas':>10}   tipos mas frecuentes")
for k in (2, 3, 5):
    sel = (razon > k).values
    tip = NIV[sel].sum().sort_values(ascending=False)
    tip = ', '.join(f'{c} {int(n)}' for c, n in tip.items() if n > 0)
    print(f"    {'> ' + str(k) + ' x mediana':>14} {int(sel.sum()):>8} "
          f"{int((sel & ~LIMPIO).sum()):>9} {M.sample_num[sel].nunique():>10}   {tip}")

sep('4 · VEREDICTO SOBRE LA PRECONDICION')
d = R['hardness_H_GPa_dif_pct'].dropna()
sd = R['hardness_H_GPa_sd_pct'].dropna()
med_abs = d.abs().median()
estrecha = (sd < 0).mean()
print(f'  desplazamiento relativo mediano de la dureza (valor absoluto): {med_abs:.2f} %')
print(f'  fraccion de peliculas donde la dispersion se estrecha: {100*estrecha:.0f} %')
ok = (med_abs >= 1.0) or (estrecha >= 0.6)
print(f'\n  ---> PRECONDICION {"SE CUMPLE" if ok else "FALLA"}')

R.to_csv('/content/precondicion_por_pelicula.csv', index=False)
print('\n  guardado: /content/precondicion_por_pelicula.csv (una fila por pelicula)')

  filas cruzadas: 9600 de 9600
  D2: excluidas 11 peliculas completamente anomalas -> quedan 139 peliculas y 8896 indentaciones

1 · CRITERIO PRINCIPAL (descarta nivel >= 2)
  peliculas analizadas: 139
  indentaciones descartadas: mediana 28.1 % (min 10.9 · max 59.4)

  supervivientes por pelicula: min=26 p10=38 mediana=46 max=57
  D3: peliculas con menos de 10 supervivientes: 0

  --- hardness_H_GPa ---
    desplazamiento de la MEDIA al limpiar (%):
      mediana=-0.98   media=-1.44   p10=-6.97   p90=+2.01   max|.|=34.21
      peliculas donde SUBE: 45 de 139 (32 %)
      peliculas con cambio > 1 %: 100 (72 %)
      peliculas con cambio > 5 %: 28 (20 %)
    cambio de la DISPERSION al limpiar (%):
      mediana=-19.79   p10=-45.69   p90=-2.25
      peliculas donde se ESTRECHA: 129 de 139 (93 %)

  --- reduced_modulus_Er_GPa ---
    desplazamiento de la MEDIA al limpiar (%):
      mediana=+1.93   media=+1.17   p10=-1.44   p90=+4.67   max|.|=74.24
      peliculas donde SUBE: 112 de 139 (8

       1          31.2%            -1.61%         8.54%               -20.63%                  24
       2          28.1%            -0.98%         8.50%               -19.79%                  26


       3           0.0%            +0.00%         1.04%                +0.00%                  59

3 · ¿QUE TIPO DE ANOMALIA APORTA MAS A LA CORRECCION?
                                      tipo  % descartado   dureza mediana   modulo mediano
                              force_offset         14.1%           -2.83%           +0.43%
                       displacement_offset          0.0%           +0.00%           +0.00%


               unusual_unloading_curvature         10.9%           +1.70%           +1.02%
                 unusual_loading_curvature          0.0%           +0.00%           +0.00%
     tip_displacement_decreases_at_holding          1.6%           -0.26%           +0.06%


                                  too_deep          0.0%           +0.00%           +0.00%

3-bis · ¿EN QUE SENTIDO EMPUJA CADA TIPO DE DEFECTO?

  --- hardness_H_GPa · ensayos de un solo tipo frente a ensayos limpios de su pelicula ---
                                        tipo  pelis  mas alto  mas bajo  p signos   signo por campaña                  veredicto
                                force_offset    115       105        10   4.0e-21   609+ 614+ 615+ 619+ 622+ 623+ 626+ DIRECCIONAL
                         displacement_offset     12         4         8   3.9e-01   615- 626+                          sin sentido claro
                 unusual_unloading_curvature    134         6       128   6.9e-31   609- 614- 615- 619- 622- 623- 626- DIRECCIONAL
                   unusual_loading_curvature     24         2        22   3.6e-05   615- 619- 622- 623-                DIRECCIONAL
       tip_displacement_decreases_at_holding     14        14         0   1.2e-04   609+ 614+ 619+      


  ¿EXPLICA LA MEZCLA DE TIPOS EL SENTIDO DE LA CORRECCION DE CADA PELICULA?
    hardness_H_GPa: correlacion entre vecinos, mediana 0.151 · factor sobre el error tipico 1.236
    reduced_modulus_Er_GPa: correlacion entre vecinos, mediana 0.296 · factor sobre el error tipico 1.426

  --- hardness_H_GPa ---   peliculas con correccion sobre el ruido: 46 · con prediccion: 46
   campaña  pelis  composicion   trivial
       609      5         100%      100%
       614     15         100%       87%
       615      6          67%       17%
       619      6         100%       17%
       622      4          75%       75%
       623      4         100%      100%
       626      6         100%      100%
     TOTAL     46          93%       72%
    la composicion iguala o supera a la trivial en 7 de 7 campañas

  --- reduced_modulus_Er_GPa ---   peliculas con correccion sobre el ruido: 63 · con prediccion: 36
   campaña  pelis  composicion   trivial
       609      9          89%       89%
       

**Cómo leer la salida**

- Bloque 2: si con el umbral 3 se obtiene casi el mismo efecto que con el 2, el problema de detección es más fácil, porque basta con encontrar las anomalías evidentes.
- Bloque 3-bis: «mas alto» cuenta las películas en que los ensayos de ese tipo tienen la propiedad más alta que los limpios. Si un tipo infla la propiedad, quitarlo la baja, y al revés. La línea del veredicto indica qué caso de la lectura (L1, L2 o L3) se cumple en la dureza.

## Fase 3. Lo que logran las reglas estadísticas

Objetivo O3; memoria, apartado 4.3.3. Las reglas que no aprenden de ejemplos fijan el mínimo que cualquier modelo debe superar. La celda 6 las mide en las 139 películas, la 7 en las películas evaluables y la 8 comprueba la sensibilidad al umbral de la regla de la desviación absoluta mediana.

### Celda 6. Las reglas estadísticas sobre las 139 películas (objetivos O3 y O4)

Antes de construir un detector hay que descartar que el problema se resuelva con estadística robusta y sin marcas. Para cada película se comparan cinco valores de la propiedad: A, media de los 64 ensayos; B, mediana de los 64; C, media recortada al 10 % por lado; D, media sin los ensayos con |z-MAD| > 3 dentro de la película; E, media de los ensayos que conserva el especialista, que es la referencia.

La métrica es la fracción recuperada de la corrección del especialista, 1 − |X − E| / |A − E|: vale 1 si la regla llega al valor del especialista, 0 si no mejora la media y es negativa si aleja el valor. Aquí se calcula en las 139 películas; la celda 7 la repite en las películas evaluables. Para el objetivo O4 se cuentan los ensayos descartados por el especialista que no son atípicos en dureza (|z-MAD| ≤ 3), en total y por tipo de defecto, con el número de películas que tienen al menos un ensayo de cada tipo en nivel 2 o superior.

Veredicto: si la mejor regla recupera 0,8 o más en dureza, gana la regla trivial; si no, y al menos la mitad de los descartados no son atípicos, el detector está justificado; en otro caso queda una zona intermedia. Añade a `M` la columna `_atip` (atípico en dureza) y guarda `competidor_trivial.csv`.

In [7]:
print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'{100*M._y.mean():.1f} % marcadas por el experto')

sep('1 · CUANTO RECUPERA CADA REGLA DE LA CORRECCION DEL EXPERTO')
filas = []
for s, d in M.groupby('sample_num'):
    r = dict(sample_num=s)
    for p in PROPS:
        v = d[p].astype(float).values
        ok = d.loc[d._y == 0, p].astype(float).values
        if len(ok) < 2 or not np.isfinite(v).all():
            continue
        A = np.mean(v)
        E = np.mean(ok)
        B = np.median(v)
        k = int(0.10 * len(v))
        C = np.mean(np.sort(v)[k:len(v) - k])
        D = np.mean(v[z_mad(v) <= 3])
        r[f'{p}_A'] = A; r[f'{p}_E'] = E
        den = abs(A - E)
        for nom, X in [('B', B), ('C', C), ('D', D)]:
            r[f'{p}_{nom}'] = X
            r[f'{p}_rec_{nom}'] = (1 - abs(X - E) / den) if den > 1e-9 else np.nan
    filas.append(r)
R = pd.DataFrame(filas)

for p in PROPS:
    print(f'\n  --- {p} ---')
    print(f"    {'regla':>34} {'mediana':>9} {'p25':>8} {'p75':>8} {'>0.5':>7} {'<0':>7}")
    for nom, txt in [('B', 'B · mediana de las 64'),
                     ('C', 'C · media recortada 10 % por lado'),
                     ('D', 'D · media tras quitar |z-MAD|>3')]:
        v = R[f'{p}_rec_{nom}'].dropna()
        print(f'    {txt:>34} {v.median():>9.2f} {np.percentile(v,25):>8.2f} '
              f'{np.percentile(v,75):>8.2f} {100*(v>0.5).mean():>6.0f}% {100*(v<0).mean():>6.0f}%')

sep('2 · ¿VE LA ESTADISTICA ROBUSTA LO QUE VIO EL EXPERTO?')
at = np.zeros(len(M), int)
for s, d in M.groupby('sample_num'):
    v = d['hardness_H_GPa'].astype(float).values
    at[M.sample_num.values == s] = (z_mad(v) > 3).astype(int)
M['_atip'] = at

tab = pd.crosstab(M._y, M._atip, rownames=['experto'], colnames=['atipica en dureza'])
print(tab.to_string())
n_mal = int(M._y.sum()); n_at = int(M._atip.sum())
solo_exp = int(((M._y == 1) & (M._atip == 0)).sum())
solo_at = int(((M._y == 0) & (M._atip == 1)).sum())
ambas = int(((M._y == 1) & (M._atip == 1)).sum())
print(f'\n  marcadas por el experto ................... {n_mal}')
print(f'  atipicas en dureza ....................... {n_at}')
print(f'  ambas cosas .............................. {ambas}')
print(f'\n  marcadas por el experto y NO atipicas: {solo_exp} '
      f'({100*solo_exp/max(n_mal,1):.1f} % de las marcadas)')
print(f'  atipicas que el experto NO marco: {solo_at} '
      f'({100*solo_at/max(n_at,1):.1f} % de las atipicas)')

print('\n  desglose por tipo de anomalia: ¿cuales son invisibles al valor de dureza?')
print(f"    {'tipo':>42} {'n':>6} {'% NO atipicas':>15} {'peliculas':>10}")
for c in POR_INDENT:
    v = (pd.to_numeric(M[c], errors='coerce').fillna(0) >= 2)
    if v.sum() == 0: continue
    inv = ((v) & (M._atip == 0)).sum()
    print(f'    {c:>42} {int(v.sum()):>6} {100*inv/v.sum():>14.1f} % {M.sample_num[v].nunique():>10}')
print(f"    {'cualquier tipo':>42} {n_mal:>6} {100*solo_exp/max(n_mal,1):>14.1f} % "
      f'{M.sample_num[M._y == 1].nunique():>10}')

sep('3 · EL TECHO: ¿cuanto se puede ganar sobre el mejor competidor trivial?')
for p in PROPS:
    mej = None
    for nom in ('B', 'C', 'D'):
        v = R[f'{p}_rec_{nom}'].dropna()
        if mej is None or v.median() > mej[1]: mej = (nom, v.median())
    print(f'  {p}: mejor regla trivial = {mej[0]} con {mej[1]:.2f} de la correccion')
    print(f'    margen que le queda a un modelo: {1-mej[1]:.2f} de la correccion')

sep('VEREDICTO')
b = R['hardness_H_GPa_rec_B'].dropna().median()
c = R['hardness_H_GPa_rec_C'].dropna().median()
d = R['hardness_H_GPa_rec_D'].dropna().median()
mejor = max(b, c, d)
frac_inv = solo_exp / max(n_mal, 1)
print(f'  mejor regla trivial sobre la dureza: recupera {mejor:.2f} de la correccion')
print(f'  fraccion de marcadas invisibles al valor: {100*frac_inv:.1f} %')
if mejor >= 0.8:
    print('\n  ---> LA REGLA TRIVIAL GANA: recupera 0.8 o mas de la correccion.')
elif frac_inv >= 0.5:
    print('\n  ---> EL DETECTOR ESTA JUSTIFICADO: mas de la mitad de las marcadas es invisible al valor final.')
else:
    print('\n  ---> ZONA INTERMEDIA: la regla trivial recupera menos de 0.8 y menos de la mitad de las '
          'marcadas es invisible al valor final.')

R.to_csv('/content/competidor_trivial.csv', index=False)
print('\n  guardado: /content/competidor_trivial.csv')

  139 peliculas · 8896 indentaciones · 27.7 % marcadas por el experto

1 · CUANTO RECUPERA CADA REGLA DE LA CORRECCION DEL EXPERTO



  --- hardness_H_GPa ---
                                 regla   mediana      p25      p75    >0.5      <0
                 B · mediana de las 64      0.09    -1.27     0.65     35%     47%
     C · media recortada 10 % por lado      0.25    -0.53     0.55     28%     38%
       D · media tras quitar |z-MAD|>3      0.00    -0.27     0.56     29%     32%

  --- reduced_modulus_Er_GPa ---
                                 regla   mediana      p25      p75    >0.5      <0
                 B · mediana de las 64     -0.30    -0.74     0.10     12%     68%
     C · media recortada 10 % por lado     -0.18    -0.38    -0.03      2%     78%
       D · media tras quitar |z-MAD|>3      0.00    -0.24     0.00      5%     37%

2 · ¿VE LA ESTADISTICA ROBUSTA LO QUE VIO EL EXPERTO?
atipica en dureza     0    1
experto                     
0                  6374   59
1                  2250  213

  marcadas por el experto ................... 2463
  atipicas en dureza ....................... 272
  am

**Cómo leer la salida**

- Bloque 1: una mediana cercana a 1 indica que la regla hace el trabajo del especialista y el detector sobra; cercana a 0, que no mejora; negativa, que aleja del valor correcto. La columna «<0» da la fracción de películas en que la regla empeora la medida.
- Bloque 2: los ensayos marcados por el especialista y no atípicos son los que ninguna regla sobre el valor final puede ver, y los que justifican mirar la forma de la curva. Los atípicos que el especialista no marcó son los que una regla robusta descartaría de más.

### Celda 7. Las reglas estadísticas en las películas evaluables (objetivo O3)

La celda 6 mide las reglas en las 139 películas, también en aquellas en que la corrección del especialista es menor que el ruido del propio promedio. En esas películas el cociente 1 − |X − E| / |A − E| divide por algo comparable al ruido y no significa nada (memoria, apartado 4.2.6). Los modelos se evalúan solo en las películas evaluables (celda 15), así que las reglas se miden en las mismas películas y se resumen igual: mediana por campaña y mediana de las siete. Se calcula para la dureza, la propiedad principal, y para el módulo reducido, la secundaria.

- Películas evaluables: |A − E| mayor o igual que el error típico del promedio multiplicado por √(1 + 3,5 ρ), con el mismo procedimiento que la celda 15.
- Reglas sin presupuesto fijado, las de la celda 6: mediana de los 64 ensayos, media recortada al 10 % por lado y media sin los ensayos con |z-MAD| > 3.
- Dos reglas de diagnóstico con el mismo número de descartes que el especialista en cada película: quitar los k valores más alejados de la mediana y quitar k al azar (mediana de 500 sorteos por película). Usan un dato que una regla real no conoce, cuántos ensayos descarta el especialista, y sirven para ver si con ese mismo gasto mirar solo el valor reproduce el cambio.
- Para cada regla: mediana por campaña en las evaluables, mediana de las siete, proporción de evaluables en que la regla aleja el valor y mediana en las 139 películas, que es la de la celda 6 y de la que sale el umbral 0,25 del criterio F1.

Lectura, fijada antes de ejecutar: es descriptiva y no decide nada. El resultado del objetivo O3 es la mejor regla sin presupuesto fijado en las películas evaluables (mediana de las siete campañas). Las dos reglas con el presupuesto del especialista son diagnóstico y no compiten con los modelos. Esta celda no modifica el umbral del criterio F1. Guarda `reglas_evaluables_<propiedad>.csv`.

In [8]:
SALIDA_7 = '/content/reglas_evaluables.csv'
N_AZAR_7 = 500

print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'{100*M._y.mean():.1f} % marcadas · {len(LOTES)} campañas')

REGLAS_7 = [('mediana', 'mediana de las 64', 'libre'),
             ('recortada', 'media recortada 10 % por lado', 'libre'),
             ('mad3', 'media sin |z-MAD| > 3', 'libre'),
             ('extremos_k', 'quitar los k mas alejados', 'igualado'),
             ('azar_k', 'quitar k al azar (mediana)', 'igualado')]

rng_7 = np.random.default_rng(SEED + 6)
TAB_7, CTRL_7 = {}, {}
for p_ in PROPS:
    rho_ = rho_vecinos(M, p_)
    fac_ = factor_vecinos(rho_)
    filas_ = {}
    for s_, d_ in M.groupby('sample_num'):
        v = d_[p_].astype(float).values; q = (d_._y == 1).values
        n = len(v); k = int(q.sum())
        if not np.isfinite(v).all() or k < 1 or n - k < 3:
            continue
        A_, E_ = v.mean(), v[~q].mean()
        den_ = abs(A_ - E_)
        sem_ = float(np.std(v, ddof=1) / np.sqrt(n))
        t = int(0.10 * n)
        X_ = {'mediana': np.median(v),
              'recortada': np.sort(v)[t:n - t].mean(),
              'mad3': v[z_mad(v) <= 3].mean()}
        orden_ = np.argsort(-np.abs(v - np.median(v)), kind='stable')
        keep_ = np.ones(n, bool); keep_[orden_[:k]] = False
        X_['extremos_k'] = v[keep_].mean()
        r_ = {nm_: (1 - abs(x_ - E_) / den_) if den_ > 1e-9 else np.nan for nm_, x_ in X_.items()}
        if den_ > 1e-9:
            perm_ = np.argsort(rng_7.random((N_AZAR_7, n)), axis=1)[:, :n - k]
            xa_ = v[perm_].mean(axis=1)
            r_['azar_k'] = float(np.median(1 - np.abs(xa_ - E_) / den_))
        else:
            r_['azar_k'] = np.nan
        filas_[s_] = dict(batch=d_.batch_num.iloc[0], den=den_, sem=sem_,
                          fiable=den_ >= sem_ * fac_, **{f'rec_{nm_}': r_[nm_] for nm_ in r_})
    tb_ = pd.DataFrame(filas_).T
    tb_['fiable'] = tb_['fiable'].astype(bool)
    for c_ in tb_.columns:
        if c_ not in ('fiable',):
            tb_[c_] = pd.to_numeric(tb_[c_], errors='coerce')
    TAB_7[p_] = tb_
    CTRL_7[p_] = dict(rho=float(rho_.median()), fac=fac_, n_ev=int(tb_['fiable'].sum()))

for p_ in PROPS:
    tb_ = TAB_7[p_]; c_ = CTRL_7[p_]
    sep(f'RECUPERACION DE CADA REGLA EN LAS PELICULAS EVALUABLES · {p_}'
        + ('  (principal)' if p_ == PROPS[0] else '  (secundaria)'))
    print(f'  correlacion entre vecinos {c_["rho"]:.3f} · factor {c_["fac"]:.3f} · '
          f'{c_["n_ev"]} de {len(tb_)} peliculas evaluables\n')
    print(f"   {'campaña':>7} {'evaluables':>11} " + ' '.join(f'{nm_:>11}' for nm_, _, _ in REGLAS_7))
    med_lote_ = {}
    for lt_ in LOTES:
        sel_ = tb_[(tb_['batch'] == lt_) & tb_['fiable']]
        med_lote_[lt_] = {nm_: sel_[f'rec_{nm_}'].median() for nm_, _, _ in REGLAS_7}
        print(f'    {int(lt_):>6} {len(sel_):>11} ' +
              ' '.join(f'{med_lote_[lt_][nm_]:>11.3f}' for nm_, _, _ in REGLAS_7))
    ML_ = pd.DataFrame(med_lote_).T
    glob_ = {nm_: float(np.nanmedian(ML_[nm_].astype(float))) for nm_, _, _ in REGLAS_7}
    print(f" {'GLOBAL (mediana de las campañas)':>33} " +
          ' '.join(f'{glob_[nm_]:>11.3f}' for nm_, _, _ in REGLAS_7))
    ev_ = tb_[tb_['fiable']]
    print(f'\n    proporcion de peliculas evaluables en que la regla aleja el valor (recuperacion < 0):')
    for nm_, txt_, pres_ in REGLAS_7:
        x_ = ev_[f'rec_{nm_}'].dropna()
        print(f'      {txt_:>34} ({pres_:>8}): {100*(x_ < 0).mean():5.1f} %   '
              f'[mediana por pelicula {x_.median():.3f}]')
    todas_ = {nm_: tb_[f'rec_{nm_}'].dropna().median() for nm_, _, _ in REGLAS_7}
    print(f'\n    sobre las {len(tb_)} peliculas, sin suelo (mediana por pelicula, como la celda 6):')
    print('      ' + ' · '.join(f'{nm_} {todas_[nm_]:.3f}' for nm_, _, _ in REGLAS_7))
    mejor_ = max([nm_ for nm_, _, pr_ in REGLAS_7 if pr_ == 'libre'], key=lambda z: glob_[z])
    print(f'\n    mejor regla de presupuesto libre en las evaluables: {mejor_} '
          f'({glob_[mejor_]:.3f}, mediana de las siete campañas)')
    tb_.assign(propiedad=p_).to_csv(SALIDA_7.replace('.csv', f'_{p_}.csv'))

print(f'\n  guardado: {SALIDA_7.replace(".csv", "_<propiedad>.csv")}')

del rng_7, TAB_7, CTRL_7

  139 peliculas · 8896 indentaciones · 27.7 % marcadas · 7 campañas



RECUPERACION DE CADA REGLA EN LAS PELICULAS EVALUABLES · hardness_H_GPa  (principal)
  correlacion entre vecinos 0.151 · factor 1.236 · 46 de 139 peliculas evaluables

   campaña  evaluables     mediana   recortada        mad3  extremos_k      azar_k
       609           5       0.853       0.478       0.510       0.764      -0.020
       614          15       0.598       0.367       0.440       0.588      -0.013
       615           6       0.266      -0.036       0.000       0.392      -0.011
       619           6       0.229      -0.007       0.193       0.069      -0.020
       622           4       0.602       0.243       0.135       0.449       0.015
       623           4       0.651       0.425       0.652       0.758      -0.016
       626           6       0.685       0.310       0.389       0.822      -0.013
  GLOBAL (mediana de las campañas)       0.602       0.310       0.389       0.588      -0.013

    proporcion de peliculas evaluables en que la regla aleja el valor (

### Celda 8. Sensibilidad al umbral de la regla de la desviación absoluta mediana (objetivos O3 y O4)

Las celdas 6 y 7 declaran atípico un ensayo con |z-MAD| > 3. Leys et al. (2013) recogen tres umbrales habituales, 3 (muy conservador), 2,5 (moderadamente conservador) y 2 (poco conservador), y recomiendan 2,5 por defecto. Esta celda recalcula con los tres las cifras que dependen del umbral:

- A. Ensayo a ensayo (dureza, 139 películas, como la celda 6): descartados por el especialista que no son atípicos, en número y en porcentaje, en total y por tipo de defecto; atípicos que el especialista no descarta.
- B. Valor de la película (dureza y módulo, como la celda 7): media sin los ensayos atípicos y su recuperación en las películas evaluables (mediana por campaña y de las siete), con intervalo del 95 % por remuestreo de películas dentro de cada campaña (2.000 veces); evaluables en que la regla aleja el valor; no evaluables en que lo aleja más que la incertidumbre corregida; distancia mediana al valor del especialista en las 139 películas (% de E); ensayos que aparta la regla por película; y diferencia con la mediana de los 64 ensayos. El remuestreo usa la misma semilla y el mismo orden que el bloque 8 de la celda 15, de modo que todos los umbrales y la mediana se comparan sobre los mismos remuestreos.

El umbral 3 es el fijado antes de ejecutar, y el 2,5, el que recomiendan Leys et al. (2013). El 2 se calcula solo como extremo de la sensibilidad, y con él la regla queda prácticamente donde la mediana de los 64 ensayos. El bloque de lectura indica si con 2,5 los descartados no atípicos siguen por encima del 50 % (objetivo O4) y si algún umbral supera a la mediana de los 64 ensayos en las evaluables (objetivo O3). Guarda `umbral_mad_sensibilidad_<propiedad>.csv`.

In [9]:
SALIDA_8 = '/content/umbral_mad_sensibilidad.csv'
NB_8 = 2000
UMBRALES_8 = (3.0, 2.5, 2.0)

print(f'  {M.sample_num.nunique()} peliculas · {len(M)} ensayos · '
      f'{int(M._y.sum())} descartados por el especialista · {len(LOTES)} campañas')

sep('A · DESCARTADOS QUE NO SON ATIPICOS EN DUREZA, SEGUN EL UMBRAL (139 peliculas)')
Z_8 = np.zeros(len(M))
for s_, ii_ in M.groupby('sample_num').indices.items():
    Z_8[ii_] = z_mad(M['hardness_H_GPa'].astype(float).values[ii_])
Y_8 = M._y.values == 1
RES_A = {}
print(f"    {'umbral':>7} {'descartados no atipicos':>26} {'atipicos':>9} "
      f"{'atipicos no descartados':>26}")
for u_ in UMBRALES_8:
    at_ = Z_8 > u_
    n_y, n_at = int(Y_8.sum()), int(at_.sum())
    no_at_ = int((Y_8 & ~at_).sum()); at_no_ = int((~Y_8 & at_).sum())
    RES_A[u_] = dict(no_atipicos=no_at_, descartados=n_y, pct=100 * no_at_ / n_y,
                     atipicos=n_at, atipicos_no_desc=at_no_,
                     pct_at=100 * at_no_ / max(n_at, 1))
    print(f'    {u_:>7.1f} {no_at_:>8} de {n_y} ({100*no_at_/n_y:5.1f} %) {n_at:>9} '
          f'{at_no_:>8} de {n_at} ({100*at_no_/max(n_at,1):5.1f} %)')
print('\n    por tipo de defecto, % de sus ensayos (nivel >= 2) que NO son atipicos:')
print(f"    {'tipo':>40} {'ensayos':>8} " + ' '.join(f'{"u = " + str(u_):>9}' for u_ in UMBRALES_8))
for c_ in POR_INDENT:
    m_ = NIV[c_].values
    if m_.sum() == 0:
        continue
    print(f'    {c_:>40} {int(m_.sum()):>8} ' +
          ' '.join(f'{100*(m_ & ~(Z_8 > u_)).sum()/m_.sum():>8.1f}%' for u_ in UMBRALES_8))
    RES_A[f'tipo_{c_}'] = 100 * (m_ & ~(Z_8 > 3.0)).sum() / m_.sum()

rng_8 = np.random.default_rng(SEED + 9)
RES_B = {}
for p_ in PROPS:
    fac_ = factor_vecinos(rho_vecinos(M, p_))
    filas_ = {}
    for s_, d_ in M.groupby('sample_num'):
        v = d_[p_].astype(float).values; q = (d_._y == 1).values
        n = len(v); k = int(q.sum())
        if not np.isfinite(v).all() or k < 1 or n - k < 3:
            continue
        A_, E_ = v.mean(), v[~q].mean()
        den_ = abs(A_ - E_)
        if den_ <= 0:
            continue
        sem_ = float(np.std(v, ddof=1) / np.sqrt(n))
        z_ = z_mad(v)
        fila_ = dict(batch=d_.batch_num.iloc[0], E=E_, den=den_, sem=sem_,
                     fiable=den_ >= sem_ * fac_,
                     rec_mediana=1 - abs(np.median(v) - E_) / den_)
        for u_ in UMBRALES_8:
            keep_ = z_ <= u_
            X_ = v[keep_].mean()
            fila_[f'rec_{u_}'] = 1 - abs(X_ - E_) / den_
            fila_[f'err_{u_}'] = abs(X_ - E_)
            fila_[f'quita_{u_}'] = int((~keep_).sum())
        filas_[s_] = fila_
    tb_ = pd.DataFrame(filas_).T
    tb_['fiable'] = tb_['fiable'].astype(bool)
    for c_ in tb_.columns:
        if c_ != 'fiable':
            tb_[c_] = pd.to_numeric(tb_[c_], errors='coerce')
    ev_ = tb_[tb_['fiable']]; ne_ = tb_[~tb_['fiable']]
    sep(f'B · LA REGLA CON CADA UMBRAL · {p_}' + ('  (principal)' if p_ == PROPS[0] else
                                                   '  (secundaria)'))
    print(f'  factor por vecinos {fac_:.3f} · {len(ev_)} evaluables de {len(tb_)} peliculas\n')
    SER_ = list(UMBRALES_8) + ['mediana']
    col_ = lambda s: f'rec_{s}'
    print(f"   {'campaña':>7} {'evaluables':>11} " + ' '.join(f'{"u = " + str(u_):>9}' for u_ in UMBRALES_8)
          + f" {'mediana 64':>11}")
    ML_ = {}
    for lt_ in LOTES:
        sel_ = ev_[ev_['batch'] == lt_]
        ML_[lt_] = {s: sel_[col_(s)].median() for s in SER_}
        print(f'    {int(lt_):>6} {len(sel_):>11} ' +
              ' '.join(f'{ML_[lt_][u_]:>9.3f}' for u_ in UMBRALES_8) + f" {ML_[lt_]['mediana']:>11.3f}")
    ML_ = pd.DataFrame(ML_).T.astype(float)
    res_p = {}
    boot_ = {s: np.full((NB_8, len(LOTES)), np.nan) for s in SER_}
    for j_, lt_ in enumerate(LOTES):
        sel_ = ev_[ev_['batch'] == lt_]
        if len(sel_) == 0:
            continue
        ix_ = rng_8.integers(0, len(sel_), size=(NB_8, len(sel_)))
        for s in SER_:
            boot_[s][:, j_] = np.nanmedian(sel_[col_(s)].values[ix_], axis=1)
    GB_ = {s: np.nanmedian(boot_[s], axis=1) for s in SER_}
    gm_ = float(np.nanmedian(ML_['mediana']))
    lom_, him_ = np.percentile(GB_['mediana'], [2.5, 97.5])
    res_p['mediana'] = dict(glob=gm_, lo=float(lom_), hi=float(him_))
    print(f"\n    {'umbral':>7} {'mediana campañas':>21} {'IC 95 %':>20} {'evaluables < 0':>15} "
          f"{'no evaluables estropeadas':>26} {'distancia en todas':>19} {'ensayos que aparta':>22}")
    for u_ in UMBRALES_8:
        g_ = float(np.nanmedian(ML_[u_]))
        lo_, hi_ = np.percentile(GB_[u_], [2.5, 97.5])
        dlo_, dhi_ = np.percentile(GB_[u_] - GB_['mediana'], [2.5, 97.5])
        neg_ = int((ev_[col_(u_)] < 0).sum())
        est_ = int((ne_[f'err_{u_}'] > ne_['sem'] * fac_).sum())
        dist_ = float((100 * tb_[f'err_{u_}'] / tb_['E']).median())
        q_ = tb_[f'quita_{u_}'].astype(float)
        res_p[u_] = dict(glob=g_, lo=float(lo_), hi=float(hi_), dif=g_ - gm_, dlo=float(dlo_),
                         dhi=float(dhi_), neg=neg_, n_ev=len(ev_), estropea=est_, n_ne=len(ne_),
                         dist=dist_, quita_med=float(q_.median()), quita_min=int(q_.min()),
                         quita_max=int(q_.max()))
        print(f'    {u_:>7.1f} {g_:>21.3f} {f"[{lo_:.3f}; {hi_:.3f}]":>20} '
              f'{f"{neg_} de {len(ev_)}":>15} {f"{est_} de {len(ne_)}":>26} {dist_:>18.1f}% '
              f'{f"mediana {q_.median():.0f} ({int(q_.min())} a {int(q_.max())})":>22}')
    print(f'    {"mediana de las 64":>17} {gm_:>11.3f} {f"[{lom_:.3f}; {him_:.3f}]":>20}')
    print('\n    diferencia con la mediana de las 64 (mismos remuestreos):')
    for u_ in UMBRALES_8:
        r_ = res_p[u_]
        print(f'      u = {u_:.1f}: {r_["dif"]:+.3f} [{r_["dlo"]:+.3f}; {r_["dhi"]:+.3f}]')
    RES_B[p_] = res_p
    tb_.assign(propiedad=p_).to_csv(SALIDA_8.replace('.csv', f'_{p_}.csv'))

sep('LECTURA')
p25_ = RES_A[2.5]['pct']
print(f'  O4 · descartados no atipicos con 2,5: {p25_:.1f} % (con 3: {RES_A[3.0]["pct"]:.1f} %).')
print('       ' + ('Siguen por encima del 50 %.' if p25_ > 50 else 'No pasan del 50 %.'))
RM_ = RES_B[PROPS[0]]['mediana']['glob']
print('  O3 · regla en las evaluables (dureza), mediana de las siete campañas: ' +
      ' · '.join(f'{u_:.1f} -> {RES_B[PROPS[0]][u_]["glob"]:.3f}' for u_ in UMBRALES_8) +
      f' (mediana de las 64: {RM_:.3f}).')
sup_ = [u_ for u_ in UMBRALES_8 if RES_B[PROPS[0]][u_]['glob'] > RM_]
if sup_:
    um_ = max(sup_, key=lambda z: RES_B[PROPS[0]][z]['glob'])
    print(f'       Con {um_:.1f} la regla supera a la mediana de las 64 en '
          f'{RES_B[PROPS[0]][um_]["glob"] - RM_:.3f}.')
else:
    print('       Ningun umbral supera a la mediana de las 64.')
print(f'\n  guardado: {SALIDA_8.replace(".csv", "_<propiedad>.csv")}')

del Z_8, Y_8, rng_8

  139 peliculas · 8896 ensayos · 2463 descartados por el especialista · 7 campañas

A · DESCARTADOS QUE NO SON ATIPICOS EN DUREZA, SEGUN EL UMBRAL (139 peliculas)
     umbral    descartados no atipicos  atipicos    atipicos no descartados
        3.0     2250 de 2463 ( 91.4 %)       272       59 de 272 ( 21.7 %)
        2.5     2141 de 2463 ( 86.9 %)       433      111 de 433 ( 25.6 %)
        2.0     1983 de 2463 ( 80.5 %)       749      269 de 749 ( 35.9 %)

    por tipo de defecto, % de sus ensayos (nivel >= 2) que NO son atipicos:
                                        tipo  ensayos   u = 3.0   u = 2.5   u = 2.0
                                force_offset     1232     87.3%     80.7%     73.0%
                         displacement_offset       99     70.7%     64.6%     57.6%
                 unusual_unloading_curvature      927     98.5%     97.6%     93.6%
                   unusual_loading_curvature      174     96.0%     93.7%     90.8%
       tip_displacement_decreases_at_ho


B · LA REGLA CON CADA UMBRAL · hardness_H_GPa  (principal)
  factor por vecinos 1.236 · 46 evaluables de 139 peliculas

   campaña  evaluables   u = 3.0   u = 2.5   u = 2.0  mediana 64
       609           5     0.510     0.802     0.802       0.853
       614          15     0.440     0.521     0.607       0.598
       615           6     0.000     0.002    -0.069       0.266
       619           6     0.193    -0.055    -0.014       0.229
       622           4     0.135     0.489     0.429       0.602
       623           4     0.652     0.670     0.791       0.651
       626           6     0.389     0.478     0.609       0.685



     umbral      mediana campañas              IC 95 %  evaluables < 0  no evaluables estropeadas  distancia en todas     ensayos que aparta
        3.0                 0.389       [0.176; 0.579]         6 de 46                   10 de 93                1.7%     mediana 2 (0 a 12)
        2.5                 0.489       [0.308; 0.705]         6 de 46                   19 de 93                1.5%     mediana 3 (0 a 14)
        2.0                 0.607       [0.364; 0.703]         8 de 46                   26 de 93                2.3%     mediana 5 (0 a 15)
    mediana de las 64       0.602       [0.355; 0.776]

    diferencia con la mediana de las 64 (mismos remuestreos):
      u = 3.0: -0.213 [-0.439; +0.014]
      u = 2.5: -0.114 [-0.262; +0.097]
      u = 2.0: +0.005 [-0.251; +0.113]



B · LA REGLA CON CADA UMBRAL · reduced_modulus_Er_GPa  (secundaria)
  factor por vecinos 1.426 · 63 evaluables de 139 peliculas

   campaña  evaluables   u = 3.0   u = 2.5   u = 2.0  mediana 64
       609           9     0.000     0.000    -0.084      -0.282
       614          34    -0.161    -0.288    -0.399      -0.292
       615           6     0.058     0.292     0.373       0.295
       619           6     0.000    -0.134    -0.091      -0.400
       622           1    -0.464    -0.764    -0.888      -0.611
       623           1     0.194     0.347     0.609       0.091
       626           6     0.000     0.224     0.432       0.204



     umbral      mediana campañas              IC 95 %  evaluables < 0  no evaluables estropeadas  distancia en todas     ensayos que aparta
        3.0                 0.000      [-0.053; 0.000]        30 de 63                    9 de 76                2.6%     mediana 1 (0 a 13)
        2.5                 0.000      [-0.189; 0.076]        37 de 63                   18 de 76                2.8%     mediana 2 (0 a 14)
        2.0                -0.084      [-0.222; 0.012]        43 de 63                   24 de 76                3.1%     mediana 4 (0 a 17)
    mediana de las 64      -0.282      [-0.345; 0.091]

    diferencia con la mediana de las 64 (mismos remuestreos):
      u = 3.0: +0.282 [-0.091; +0.337]
      u = 2.5: +0.282 [-0.099; +0.362]
      u = 2.0: +0.198 [-0.153; +0.265]

LECTURA
  O4 · descartados no atipicos con 2,5: 86.9 % (con 3: 91.4 %).
       Siguen por encima del 50 %.
  O3 · regla en las evaluables (dureza), mediana de las siete campañas: 3.0 -> 0.389 · 2.5 -

## Fase 4. Lo que el valor final no muestra

Objetivo O4; memoria, apartado 4.3.4. Se compara la limpieza del especialista con el azar que retira el mismo número de ensayos, primero sin más restricciones y después igualando la distancia a la mediana de los ensayos retirados.

### Celda 9. La limpieza del especialista frente al azar con el mismo número de descartes (objetivo O4)

El especialista descarta en torno al 28 % de los ensayos, y quitar más de una cuarta parte de los datos mueve la media aunque se quite al azar. Por eso, en cada película, el valor que deja su descarte se compara con el que deja quitar el mismo número de ensayos al azar, en 500 sorteos (prueba de permutación). Hay cuatro contrastes, cada uno más exigente que el anterior:

- N1, completo. El azar quita k ensayos cualesquiera de los 64.
- N2, sin los atípicos. Se retiran de los dos brazos los ensayos con |z-MAD| > 3 y se repite el contraste con las marcas que quedan.
- N3, misma magnitud de la desviación. Los ensayos se reparten en cuartiles de |z-MAD| y el azar quita en cada cuartil tantos como el especialista.
- N4, misma desviación con signo. Igual que N3, con cuartiles de la desviación con signo. Los dos brazos quitan ensayos igual de desviados y del mismo lado, y solo difieren en cuál eligen entre ensayos parecidos.

Bajo el azar, el 10 % de las películas cae fuera del intervalo central del 90 %. Hay señal si se cumplen tres condiciones: fracción fuera de al menos el 20 %, prueba binomial frente a 0,10 con p < 0,001 y mediana del z con signo de al menos 0,5 en valor absoluto. En otro caso, el resultado no se distingue del azar. Entre N1 y N2, un diagnóstico compara en dureza la desviación de los ensayos marcados y la de los no marcados.

La celda guarda `/content/nulo_correccion.csv`, con una fila por película, que lee la celda 10. Fija además `B = 500` sorteos, que usan también las celdas 10, 11 y 13.

In [10]:
B = 500
NQ = 4

print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'descarte del experto {100*M._y.mean():.1f} % · B={B} · semilla 20260910')


def resumen(e, nul):
    pct = 100 * ((nul < e).mean() + 0.5 * (nul == e).mean())
    fuera = int(e < np.percentile(nul, 5) or e > np.percentile(nul, 95))
    sd = nul.std()
    return [pct, fuera, (e - nul.mean()) / sd if sd > 0 else np.nan]


def nulo(v, quitar, pool, rng, B, strat=None):
    v = v[pool]; q = quitar[pool]
    n = len(v); k = int(q.sum())
    if k < 1 or n - k < 3:
        return None
    m_exp = v[~q].mean(); s_exp = v[~q].std(ddof=1)
    if strat is None:
        idx = np.argsort(rng.random((B, n)), axis=1)[:, :n - k]
    else:
        st = strat[pool]; cols = []
        for j in np.unique(st):
            pos = np.flatnonzero(st == j)
            nj = len(pos); mj = int(q[pos].sum())
            if nj - mj <= 0:
                continue
            cols.append(pos[np.argsort(rng.random((B, nj)), axis=1)[:, :nj - mj]])
        if not cols:
            return None
        idx = np.concatenate(cols, axis=1)
    sub = v[idx]
    return (resumen(m_exp, sub.mean(axis=1)) + resumen(s_exp, sub.std(axis=1, ddof=1))
            + [n, k])


rng = np.random.default_rng(20260910)
filas = []
diag = []
for s, d in M.groupby('sample_num'):
    quitar = (d._y == 1).values
    r = dict(sample_num=s, n=len(d), k=int(quitar.sum()),
             pct_desc=round(100 * quitar.mean(), 1))
    for p in PROPS:
        v = d[p].astype(float).values
        if not np.isfinite(v).all():
            continue
        zs = z_signed(v); z = np.abs(zs)
        pool = z <= 3
        todo = np.ones(len(v), bool)
        r[f'{p}_natip'] = int((~pool).sum())
        st_abs = np.full(len(v), -1); st_sgn = np.full(len(v), -1)
        st_abs[pool] = bins(z[pool], NQ)
        st_sgn[pool] = bins(zs[pool], NQ)
        for etiq, args in (('', dict(quitar=quitar, pool=todo, strat=None)),
                           ('I_', dict(quitar=quitar & pool, pool=pool, strat=None)),
                           ('E_', dict(quitar=quitar & pool, pool=pool, strat=st_abs)),
                           ('F_', dict(quitar=quitar & pool, pool=pool, strat=st_sgn))):
            o = nulo(v, rng=rng, B=B, **args)
            if o is None:
                continue
            (r[f'{p}_{etiq}m_pct'], r[f'{p}_{etiq}m_fuera'], r[f'{p}_{etiq}m_z'],
             r[f'{p}_{etiq}s_pct'], r[f'{p}_{etiq}s_fuera'], r[f'{p}_{etiq}s_z'],
             r[f'{p}_{etiq}n'], r[f'{p}_{etiq}k']) = o
        if p == 'hardness_H_GPa':
            diag.append(pd.DataFrame({'z': z[pool], 'zs': zs[pool],
                                      'marc': quitar[pool].astype(int)}))
    filas.append(r)
R = pd.DataFrame(filas)
D = pd.concat(diag, ignore_index=True) if diag else pd.DataFrame(columns=['z', 'marc'])


def _nev(suf):
    return R.get(f'hardness_H_GPa_{suf}m_pct', pd.Series(dtype=float)).notna().sum()


print(f'  peliculas evaluadas: N1={_nev("")} · N2={_nev("I_")} · N3={_nev("E_")} · N4={_nev("F_")}')


def informe(col, etiq):
    if f'{col}_pct' not in R.columns:
        print(f'\n  --- {etiq} ---  sin datos'); return None
    pc = R[f'{col}_pct'].dropna(); fu = R[f'{col}_fuera'].dropna(); z = R[f'{col}_z'].dropna()
    if len(pc) < 5:
        print(f'\n  --- {etiq} ---  solo {len(pc)} peliculas, no se evalua'); return None
    print(f'\n  --- {etiq} ---   ({len(pc)} peliculas)')
    dec = np.histogram(pc, bins=np.arange(0, 101, 10))[0]
    print('    reparto de los percentiles del experto dentro del nulo, por decil:')
    print('      ' + ' '.join(f'{x:4d}' for x in dec))
    print('      ' + ' '.join(f'{i*10:4d}' for i in range(10)) + '   (borde inferior del decil)')
    print(f'    esperado por decil si fuera ruido: {len(pc)/10:.1f}')
    ks = stats.kstest(pc / 100, 'uniform')
    nf = int(fu.sum()); frac = fu.mean(); bt = stats.binomtest(nf, len(fu), 0.10).pvalue
    print(f'    contraste contra uniforme (KS): D={ks.statistic:.3f}  p={ks.pvalue:.2e}')
    print(f'    FUERA del intervalo central del 90 %: {nf} de {len(fu)} ({100*frac:.1f} %)'
          f'   [esperado 10 %]   binomial p={bt:.2e}')
    print(f'    tamano del efecto (z respecto al nulo): mediana={z.median():+.2f}  '
          f'p10={np.percentile(z,10):+.2f}  p90={np.percentile(z,90):+.2f}')
    return dict(frac=100 * frac, z=z.median(), bino=bt,
                senal=(frac >= 0.20 and bt < 1e-3 and abs(z.median()) >= 0.5))


sep('N1 · NULO COMPLETO — ¿mueve el descarte del experto mas que quitar k al azar?')
n1m = {p: informe(f'{p}_m', f'{p} · MEDIA') for p in PROPS}
n1s = {p: informe(f'{p}_s', f'{p} · DESVIACION TIPICA') for p in PROPS}

sep('DIAGNOSTICO · ¿que aspecto tiene, EN VALOR, lo que el experto marca?')
if len(D):
    q0 = D[D.marc == 0]; q1 = D[D.marc == 1]
    print('  dentro del universo NO atipico (|z-MAD| <= 3) en dureza, agregando peliculas:')
    print(f"    {'':>12} {'n':>7} {'mediana |z|':>12} {'p75 |z|':>9} {'% |z|>2':>9} "
          f"{'% por encima':>13} {'mediana z':>11}")
    for nom, q in (('no marcadas', q0), ('marcadas', q1)):
        print(f'    {nom:>12} {len(q):>7} {q.z.median():>12.2f} {np.percentile(q.z,75):>9.2f} '
              f'{100*(q.z>2).mean():>8.1f} % {100*(q.zs>0).mean():>12.1f} % {q.zs.median():>11.2f}')
    print(f'    Mann-Whitney sobre |z| : p={stats.mannwhitneyu(q1.z, q0.z).pvalue:.2e}')
    print(f'    Mann-Whitney sobre  z  : p={stats.mannwhitneyu(q1.zs, q0.zs).pvalue:.2e}')

sep('N2 · SOLO LAS MARCAS INVISIBLES AL VALOR (fuera las atipicas, en los dos brazos)')
n2m = {p: informe(f'{p}_I_m', f'{p} · MEDIA') for p in PROPS}
n2s = {p: informe(f'{p}_I_s', f'{p} · DESVIACION TIPICA') for p in PROPS}

sep('N3 · ESTRATIFICADO POR |z-MAD| — el azar copia el perfil de MAGNITUD del experto')
n3m = {p: informe(f'{p}_E_m', f'{p} · MEDIA') for p in PROPS}
n3s = {p: informe(f'{p}_E_s', f'{p} · DESVIACION TIPICA') for p in PROPS}

sep('N4 · ESTRATIFICADO POR z CON SIGNO — el azar copia magnitud Y direccion')
n4m = {p: informe(f'{p}_F_m', f'{p} · MEDIA') for p in PROPS}
n4s = {p: informe(f'{p}_F_s', f'{p} · DESVIACION TIPICA') for p in PROPS}

sep('VEREDICTO')


def linea(t, r):
    if r is None:
        print(f'    {t:<30}: sin datos'); return False
    print(f'    {t:<30}: {r["frac"]:5.1f} % fuera · z mediano {r["z"]:+.2f} · '
          f'binomial p={r["bino"]:.1e}  ->  {"SENAL" if r["senal"] else "NO se distingue del azar"}')
    return r['senal']


marc = {}
for p in PROPS:
    print(f'\n  {p}')
    marc[p] = dict(
        n1m=linea('N1 media', n1m[p]),                n1s=linea('N1 dispersion', n1s[p]),
        n2m=linea('N2 media (invisibles)', n2m[p]),   n2s=linea('N2 dispersion (invisibles)', n2s[p]),
        n3m=linea('N3 media (|z| igualado)', n3m[p]), n3s=linea('N3 dispersion (|z| igualado)', n3s[p]),
        n4m=linea('N4 media (z con signo)', n4m[p]),  n4s=linea('N4 dispersion (z con signo)', n4s[p]))

h = marc['hardness_H_GPa']
sep('LECTURA AUTOMATICA SOBRE LA DUREZA (propiedad de referencia)')
if not (h['n1m'] or h['n1s']):
    print('  N1 no se distingue del azar ni en la media ni en la dispersion')
elif h['n4m'] or h['n4s']:
    q = ('media' if h['n4m'] else '') + (' y ' if h['n4m'] and h['n4s'] else '') + \
        ('dispersion' if h['n4s'] else '')
    print(f'  la senal sobrevive al nulo mas exigente (N4, en {q})')
elif h['n3m'] or h['n3s']:
    print('  la senal sobrevive a N3 (magnitud igualada) y no a N4 (direccion igualada)')
else:
    print('  hay senal en N1 y no sobrevive a N3 ni a N4')

R.to_csv('/content/nulo_correccion.csv', index=False)
print('\n  guardado: /content/nulo_correccion.csv (una fila por pelicula)')

  139 peliculas · 8896 indentaciones · descarte del experto 27.7 % · B=500 · semilla 20260910


  peliculas evaluadas: N1=139 · N2=139 · N3=139 · N4=139

N1 · NULO COMPLETO — ¿mueve el descarte del experto mas que quitar k al azar?

  --- hardness_H_GPa · MEDIA ---   (139 peliculas)
    reparto de los percentiles del experto dentro del nulo, por decil:
        53   11   14   11    5    2    5    6    4   28
         0   10   20   30   40   50   60   70   80   90   (borde inferior del decil)
    esperado por decil si fuera ruido: 13.9
    contraste contra uniforme (KS): D=0.312  p=1.60e-12
    FUERA del intervalo central del 90 %: 58 de 139 (41.7 %)   [esperado 10 %]   binomial p=1.64e-22
    tamano del efecto (z respecto al nulo): mediana=-0.77  p10=-2.95  p90=+2.51

  --- reduced_modulus_Er_GPa · MEDIA ---   (139 peliculas)
    reparto de los percentiles del experto dentro del nulo, por decil:
        15    3    2    3    4    3    9    8    7   85
         0   10   20   30   40   50   60   70   80   90   (borde inferior del decil)
    esperado por decil si fuera ruido: 13.9
   

**Cómo leer la salida**

Si el descarte del especialista se comportara como el azar, sus percentiles se repartirían por igual entre los deciles y en torno al 10 % de las películas caería fuera. En la desviación típica, la señal aparece como concentración en los deciles bajos, porque el especialista quitaría justo lo que más ensancha la distribución.

En el diagnóstico, una mediana de |z| mayor en las marcadas indica que N2 sigue influido por la magnitud, y eso lo cierra N3. Un porcentaje por encima de la mediana alejado de la mitad indicaría que el especialista quita preferentemente un lado de la distribución, y eso lo cierra N4.

N4 es el contraste que decide. Si la señal sobrevive, la elección del especialista usa información que no está en el valor publicado, sino en la forma de la curva. Si no sobrevive, su criterio equivale, para la propiedad publicada, a una regla de recorte sobre el valor.

### Celda 10. Barrido del número de franjas y lectura del contraste por magnitud (objetivo O4)

El bloque A estrecha las franjas de z con signo: 1, 2, 4, 8, 12, 16 y 24 franjas con el mismo número de ensayos. Con cuatro cuartiles, dentro de un cuartil central la magnitud sigue variando mucho, y el azar puede quitar ensayos claramente menos desviados que los del especialista. Al estrechar las franjas, el azar tiene menos ensayos entre los que elegir y la fracción fuera tiende a 0 por construcción. Por eso se da también la dispersión relativa del contraste, la del azar con franjas respecto a la del azar sin franjas. Se lee el mayor número de franjas que conserva al menos el 50 % de esa dispersión. Si ahí la fracción fuera sigue en el 20 % o más, la señal no se explica por lo extremo del valor; si cae hacia el 10 % antes de llegar a ese número de franjas, sí. Ese cociente no es una potencia estadística. La calibración del contraste se comprueba con un experto falso en la celda 13, bloque C (memoria, apartado 4.3.4, Tabla 5).

La celda 9 mide la materialidad con la mediana del z con signo, que se cancela si el efecto sube en unas películas y baja en otras. A2 da el veredicto del barrido con la mediana de |z|, y A3 relee del mismo modo la tabla de la celda 9 (`/content/nulo_correccion.csv`). El bloque B describe cómo son en valor los ensayos marcados frente a los no marcados, en las dos propiedades, y el bloque C reparte el signo del efecto sobre la media por solución, densidad de corriente y campaña.

In [11]:
CSV7 = '/content/nulo_correccion.csv'
NQS = [1, 2, 4, 8, 12, 16, 24]

print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'descarte {100*M._y.mean():.1f} % · B={B} · semilla 20260911')


def una(v, q, z, rng, nq):
    n = len(v); k = int(q.sum())
    if k < 1 or n - k < 3:
        return None
    st = bins(z, nq); cols = []
    for j in np.unique(st):
        pos = np.flatnonzero(st == j); nj = len(pos); mj = int(q[pos].sum())
        if nj - mj <= 0:
            continue
        cols.append(pos[np.argsort(rng.random((B, nj)), axis=1)[:, :nj - mj]])
    if not cols:
        return None
    sub = v[np.concatenate(cols, axis=1)]
    o = {}
    for nom, e, nul in (('m', v[~q].mean(), sub.mean(axis=1)),
                        ('s', v[~q].std(ddof=1), sub.std(axis=1, ddof=1))):
        sd = nul.std()
        o[f'{nom}_fuera'] = int(e < np.percentile(nul, 5) or e > np.percentile(nul, 95))
        o[f'{nom}_z'] = (e - nul.mean()) / sd if sd > 0 else np.nan
        o[f'{nom}_sd'] = sd
    return o


rng = np.random.default_rng(20260911)
acum = {}
for s, d in M.groupby('sample_num'):
    quitar = (d._y == 1).values
    for p in PROPS:
        v0 = d[p].astype(float).values
        if not np.isfinite(v0).all():
            continue
        zs = z_signed(v0); pool = np.abs(zs) <= 3
        v = v0[pool]; q = quitar[pool]; z = zs[pool]
        for nq in NQS:
            o = una(v, q, z, rng, nq)
            if o:
                acum.setdefault((p, nq), []).append(o)

sep('A · BARRIDO DEL NUMERO DE ESTRATOS (z con signo) — ¿es extremidad residual?')
filas = []
for p in PROPS:
    ref = {}
    for nq in NQS:
        L = acum.get((p, nq), [])
        if not L:
            continue
        T = pd.DataFrame(L)
        for w, etiq in (('m', 'MEDIA'), ('s', 'DISPERSION')):
            sd = T[f'{w}_sd'].median()
            if nq == NQS[0]:
                ref[w] = sd
            filas.append(dict(prop=p, est=etiq, nq=nq, n=len(T),
                              fuera=100 * T[f'{w}_fuera'].mean(),
                              z_abs=T[f'{w}_z'].abs().median(),
                              z_sgn=T[f'{w}_z'].median(),
                              pot=100 * sd / ref[w] if ref.get(w) else np.nan))
RES = pd.DataFrame(filas)
for p in PROPS:
    for etiq in ('MEDIA', 'DISPERSION'):
        t = RES[(RES.prop == p) & (RES.est == etiq)]
        if not len(t):
            continue
        print(f'\n  --- {p} · {etiq} ---')
        print(f"    {'estratos':>9} {'% fuera':>9} {'|z| mediano':>12} {'z mediano':>11} "
              f"{'disp. rel.':>10}   {'lectura':>8}")
        for _, r in t.iterrows():
            marca = 'ok' if r.pot >= 50 else 'disp. < 50 %'
            print(f"    {int(r.nq):>9} {r.fuera:>8.1f}% {r.z_abs:>12.2f} {r.z_sgn:>+11.2f} "
                  f"{r.pot:>9.0f}% {marca:>14}")

sep('A2 · VEREDICTO CON LA MEDIANA DE |z|')
print(f"    {'prop':>24} {'estad.':>11} {'estratos':>9} {'% fuera':>9} {'|z| med':>9}  veredicto")
for p in PROPS:
    for etiq in ('MEDIA', 'DISPERSION'):
        t = RES[(RES.prop == p) & (RES.est == etiq) & (RES.pot >= 50)]
        if not len(t):
            continue
        r = t.iloc[-1]
        ok = (r.fuera >= 20) and (r.z_abs >= 0.5)
        print(f"    {p:>24} {etiq:>11} {int(r.nq):>9} {r.fuera:>8.1f}% {r.z_abs:>9.2f}  "
              f"{'SENAL' if ok else 'no se sostiene'}")

sep('A3 · LA TABLA DE LA CELDA 9, RELEIDA CON |z| EN VEZ DE z CON SIGNO')
R7 = pd.read_csv(CSV7)
print(f"    {'prop':>24} {'nulo':>26} {'% fuera':>9} {'|z| med':>9} {'z med':>8}  veredicto")
for p in PROPS:
    for suf, nom in (('', 'N1 completo'), ('I_', 'N2 invisibles'),
                     ('E_', 'N3 |z| igualado'), ('F_', 'N4 z con signo')):
        for w, etiq in (('m', 'media'), ('s', 'dispersion')):
            cz, cf = f'{p}_{suf}{w}_z', f'{p}_{suf}{w}_fuera'
            if cz not in R7.columns:
                continue
            z = R7[cz].dropna(); fu = 100 * R7[cf].dropna().mean()
            ok = (fu >= 20) and (z.abs().median() >= 0.5)
            print(f"    {p:>24} {nom + ' · ' + etiq:>26} {fu:>8.1f}% "
                  f"{z.abs().median():>9.2f} {z.median():>+8.2f}  "
                  f"{'SENAL' if ok else 'no se sostiene'}")

sep('B · ¿QUE ASPECTO TIENE EN VALOR LO QUE MARCA EL EXPERTO? (las dos propiedades)')
print(f"    {'propiedad':>24} {'grupo':>12} {'n':>7} {'|z| med':>9} {'% |z|>2':>9} "
      f"{'% arriba':>9} {'z mediano':>10}")
for p in PROPS:
    zz, mm = [], []
    for s, d in M.groupby('sample_num'):
        v = d[p].astype(float).values
        if not np.isfinite(v).all():
            continue
        z = z_signed(v); pool = np.abs(z) <= 3
        zz.append(z[pool]); mm.append((d._y == 1).values[pool])
    z = np.concatenate(zz); m = np.concatenate(mm)
    for nom, sel in (('no marcadas', ~m), ('marcadas', m)):
        x = z[sel]
        print(f"    {p:>24} {nom:>12} {len(x):>7} {np.median(np.abs(x)):>9.2f} "
              f"{100*(np.abs(x)>2).mean():>8.1f}% {100*(x>0).mean():>8.1f}% {np.median(x):>10.2f}")
    print(f"    {'':>24} {'MannWhitney |z| p=':>12} {stats.mannwhitneyu(np.abs(z[m]), np.abs(z[~m])).pvalue:.1e}"
          f"   ·  sobre z con signo p={stats.mannwhitneyu(z[m], z[~m]).pvalue:.1e}")

sep('C · EL EFECTO SOBRE LA MEDIA NO TIENE DIRECCION COMUN: ¿de que depende el signo?')
meta = M.groupby('sample_num').agg(solution=('solution', 'first'),
                                   cd=('current_density', 'first'),
                                   batch=('batch_num', 'first')).reset_index()
R7 = R7.merge(meta, on='sample_num', how='left')
for p in PROPS:
    c = f'{p}_m_z'
    if c not in R7.columns:
        continue
    v = R7[c]
    print(f'\n  --- {p} · z de la MEDIA en el nulo completo (N1) ---')
    print(f'    peliculas con z < -1: {int((v < -1).sum())}   entre -1 y +1: '
          f'{int(v.between(-1,1).sum())}   con z > +1: {int((v > 1).sum())}')
    R7['_g'] = np.where(v < -1, 'baja', np.where(v > 1, 'sube', 'plano'))
    print('    reparto por solucion:')
    print(pd.crosstab(R7.solution, R7._g).to_string().replace('\n', '\n      '))
    print('    densidad de corriente mediana por grupo:')
    print(R7.groupby('_g').cd.median().to_string().replace('\n', '\n      '))
    print('    reparto por campaña:')
    print(pd.crosstab(R7.batch, R7._g).to_string().replace('\n', '\n      '))

  139 peliculas · 8896 indentaciones · descarte 27.7 % · B=500 · semilla 20260911



A · BARRIDO DEL NUMERO DE ESTRATOS (z con signo) — ¿es extremidad residual?

  --- hardness_H_GPa · MEDIA ---
     estratos   % fuera  |z| mediano   z mediano disp. rel.    lectura
            1     36.0%         1.22       -0.10       100%             ok
            2     25.2%         1.04       -0.30        60%             ok
            4     19.4%         0.96       -0.14        39%   disp. < 50 %
            8     15.1%         0.81       -0.08        23%   disp. < 50 %
           12     12.2%         0.80       +0.01        17%   disp. < 50 %
           16     10.8%         0.77       -0.02        13%   disp. < 50 %
           24      9.4%         0.76       +0.07         8%   disp. < 50 %

  --- hardness_H_GPa · DISPERSION ---
     estratos   % fuera  |z| mediano   z mediano disp. rel.    lectura
            1     57.6%         2.05       -2.05       100%             ok
            2     55.4%         1.99       -1.99        98%             ok
            4     41.7%         1

**Cómo leer la salida**

En la salida, las franjas aparecen como estratos. La columna «disp. rel.» es la dispersión del azar con estratos respecto a la del azar sin estratificar. Por debajo del 50 %, el contraste apenas tiene margen para detectar nada y un porcentaje fuera bajo deja de ser informativo; la lectura fijada es la última fila con «disp. rel.» de al menos el 50 %, que es la que recoge A2.

A3 usa los mismos nulos de la celda 9; cambia la condición de materialidad, que mira |z|, y no aplica la prueba binomial. Donde el veredicto difiere, la diferencia está en la etiqueta, no en la medida. En el bloque C, «baja» agrupa las películas con z < -1, «sube» las de z > +1 y «plano» las intermedias.

### Celda 11. El contraste dentro de cada campaña y las reglas con el mismo número de descartes (objetivos O3 y O4)

- A. Las películas no son independientes. Comparten campaña de electrodeposición, solución y probablemente el momento del etiquetado, y una prueba binomial sobre 139 películas exagera la evidencia si el efecto vive en dos o tres campañas. El bloque repite el contraste dentro de cada campaña, sin franjas y con 8 franjas (en la celda 10, el mayor número que conserva al menos el 50 % de la dispersión relativa en la desviación típica de la dureza), y comprueba si el signo del efecto es común a las campañas.
- B. Superar al azar no demuestra que el especialista supere a quitar los k ensayos más desviados, regla casi óptima por construcción para bajar la desviación típica. Cada regla recibe el mismo número de descartes k que el especialista: quitar los de mayor |z|, los de la cola alta, los de la cola baja o k al azar (200 sorteos). Se comparan en la desviación típica que consigue cada una (1) y en la distancia al valor del especialista (2), la única referencia que existe y la comparación en la que tiene que competir un modelo. La celda 13 (bloque B) recalcula la tabla (2) sobre los 64 ensayos.
- C. Cambio del criterio del especialista entre campañas: tasa de descarte y |z| mediano, en dureza, de lo marcado y de lo no marcado.

La celda guarda `/content/presupuesto_y_lotes.csv`, con una fila por película, y fija `BR = 200` sorteos, que usan también las celdas 12 y 13.

In [12]:
BR = 200
NQ_FINO = 8

print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'descarte {100*M._y.mean():.1f} % · semilla 20260912')

rng = np.random.default_rng(20260912)
filas = []
for s, d in M.groupby('sample_num'):
    quitar0 = (d._y == 1).values
    r = dict(sample_num=s, batch=d.batch_num.iloc[0], solution=d.solution.iloc[0],
             cd=d.current_density.iloc[0])
    for p in PROPS:
        v0 = d[p].astype(float).values
        if not np.isfinite(v0).all():
            continue
        zs = z_signed(v0); pool = np.abs(zs) <= 3
        v = v0[pool]; q = quitar0[pool]; z = zs[pool]
        n = len(v); k = int(q.sum())
        if k < 1 or n - k < 3:
            continue
        m_e = v[~q].mean(); s_e = v[~q].std(ddof=1)
        A = v.mean(); sA = v.std(ddof=1)
        r[f'{p}_n'] = n; r[f'{p}_k'] = k
        r[f'{p}_E'] = m_e; r[f'{p}_A'] = A
        r[f'{p}_sd_exp'] = s_e / sA if sA > 0 else np.nan

        for nq, suf in ((1, 'g'), (NQ_FINO, 'f')):
            st = bins(z, nq); cols = []
            for j in np.unique(st):
                pos = np.flatnonzero(st == j); nj = len(pos); mj = int(q[pos].sum())
                if nj - mj <= 0:
                    continue
                cols.append(pos[np.argsort(rng.random((B, nj)), axis=1)[:, :nj - mj]])
            if not cols:
                continue
            sim = v[np.concatenate(cols, axis=1)]
            for w, e, nul in (('m', m_e, sim.mean(axis=1)), ('s', s_e, sim.std(axis=1, ddof=1))):
                sd = nul.std()
                r[f'{p}_{suf}{w}_fuera'] = int(e < np.percentile(nul, 5) or e > np.percentile(nul, 95))
                r[f'{p}_{suf}{w}_z'] = (e - nul.mean()) / sd if sd > 0 else np.nan

        ordz = {'mas_abs': np.argsort(-np.abs(z)), 'cola_alta': np.argsort(-z),
                'cola_baja': np.argsort(z)}
        for nom, o in ordz.items():
            keep = np.ones(n, bool); keep[o[:k]] = False
            r[f'{p}_{nom}_sd'] = v[keep].std(ddof=1) / sA if sA > 0 else np.nan
            r[f'{p}_{nom}_m'] = v[keep].mean()
        idx = np.argsort(rng.random((BR, n)), axis=1)[:, :n - k]
        r[f'{p}_azar_sd'] = np.median(v[idx].std(axis=1, ddof=1)) / sA if sA > 0 else np.nan
        r[f'{p}_azar_m'] = np.median(v[idx].mean(axis=1))
    filas.append(r)
R = pd.DataFrame(filas)
print(f'  peliculas con resultado: {R[f"{PROPS[0]}_k"].notna().sum()}')

sep('A · ¿VIVE EL EFECTO EN LA MAYORIA DE LAS CAMPAÑAS, O EN DOS CAMPAÑAS?')
for p in PROPS:
    for suf, nqn in (('g', 1), ('f', NQ_FINO)):
        for w, etiq in (('m', 'MEDIA'), ('s', 'DISPERSION')):
            cf, cz = f'{p}_{suf}{w}_fuera', f'{p}_{suf}{w}_z'
            if cf not in R.columns:
                continue
            T = R.dropna(subset=[cf])
            g = T.groupby('batch').agg(n=(cf, 'size'), fuera=(cf, 'mean'), z=(cz, 'median'))
            g['fuera'] = (100 * g.fuera).round(1)
            lotes_ok = int((g.fuera >= 20).sum()); tot = len(g)
            sgn = np.sign(g.z.values); sgn = sgn[sgn != 0]
            pb = stats.binomtest(int((sgn > 0).sum()), len(sgn), 0.5).pvalue if len(sgn) else np.nan
            print(f'\n  {p} · {etiq} · estratos={nqn}   '
                  f'global {100*T[cf].mean():.1f} % fuera')
            print('    ' + ' '.join(f'{int(b):>6}' for b in g.index) + '   <- campaña')
            print('    ' + ' '.join(f'{x:>6.0f}' for x in g.n) + '   <- peliculas')
            print('    ' + ' '.join(f'{x:>6.0f}' for x in g.fuera) + '   <- % fuera')
            print('    ' + ' '.join(f'{x:>+6.1f}' for x in g.z) + '   <- z mediano')
            print(f'    campañas con >= 20 % fuera: {lotes_ok} de {tot}   ·   '
                  f'signo del z comun a las campañas: p={pb:.3f}')

sep('B · MISMO PRESUPUESTO k PARA TODOS: ¿le gana alguna regla de valor al experto?')
reglas = [('experto', 'exp'), ('quitar k mayor |z|', 'mas_abs'),
          ('quitar k de la cola alta', 'cola_alta'),
          ('quitar k de la cola baja', 'cola_baja'), ('quitar k al azar', 'azar')]
for p in PROPS:
    print(f'\n  --- {p} ---')
    print('  (1) DESVIACION TIPICA que consigue cada regla, en fraccion de la cruda '
          '(menor = mas estrecha)')
    print(f"      {'regla':>26} {'mediana':>9} {'p25':>8} {'p75':>8} {'% que baja mas que el experto':>31}")
    sd_exp = R[f'{p}_sd_exp']
    for nom, suf in reglas:
        c = f'{p}_sd_exp' if suf == 'exp' else f'{p}_{suf}_sd'
        if c not in R.columns:
            continue
        v = R[c].dropna()
        gana = '' if suf == 'exp' else f'{100*(R[c] < sd_exp).mean():>30.0f}%'
        print(f'      {nom:>26} {v.median():>9.3f} {np.percentile(v,25):>8.3f} '
              f'{np.percentile(v,75):>8.3f} {gana}')
    print('\n  (2) DISTANCIA AL VALOR DEL EXPERTO. recuperacion = 1 - |X-E| / |A-E|')
    print(f"      {'regla':>26} {'mediana':>9} {'p25':>8} {'p75':>8} {'% > 0.5':>9} {'% < 0':>8}")
    den = (R[f'{p}_A'] - R[f'{p}_E']).abs()
    for nom, suf in reglas[1:]:
        c = f'{p}_{suf}_m'
        if c not in R.columns:
            continue
        rec = (1 - (R[c] - R[f'{p}_E']).abs() / den).replace([np.inf, -np.inf], np.nan).dropna()
        print(f'      {nom:>26} {rec.median():>9.2f} {np.percentile(rec,25):>8.2f} '
              f'{np.percentile(rec,75):>8.2f} {100*(rec>0.5).mean():>8.0f}% {100*(rec<0).mean():>7.0f}%')

sep('C · ¿CAMBIO EL CRITERIO DEL EXPERTO ENTRE CAMPAÑAS?')
g = M.groupby('batch_num').agg(peliculas=('sample_num', 'nunique'), indents=('_y', 'size'),
                               pct=('_y', lambda x: round(100 * x.mean(), 1)))
g.index.name = 'campaña'
zz = []
for s, d in M.groupby('sample_num'):
    v = d['hardness_H_GPa'].astype(float).values
    if not np.isfinite(v).all():
        continue
    z = np.abs(z_signed(v)); pool = z <= 3
    zz.append(pd.DataFrame({'batch': d.batch_num.iloc[0], 'z': z[pool],
                            'marc': (d._y == 1).values[pool]}))
Z = pd.concat(zz, ignore_index=True)
g['z_marc'] = Z[Z.marc].groupby('batch').z.median().round(2)
g['z_nomarc'] = Z[~Z.marc].groupby('batch').z.median().round(2)
print(g.to_string())

R.to_csv('/content/presupuesto_y_lotes.csv', index=False)
print('\n  guardado: /content/presupuesto_y_lotes.csv')

  139 peliculas · 8896 indentaciones · descarte 27.7 % · semilla 20260912


  peliculas con resultado: 139

A · ¿VIVE EL EFECTO EN LA MAYORIA DE LAS CAMPAÑAS, O EN DOS CAMPAÑAS?

  hardness_H_GPa · MEDIA · estratos=1   global 35.3 % fuera
       609    614    615    619    622    623    626   <- campaña
        14     48     14     11     20     13     19   <- peliculas
        21     35     43     54     20     38     42   <- % fuera
      -1.0   -0.2   +0.6   +1.7   -0.5   +0.7   -1.2   <- z mediano
    campañas con >= 20 % fuera: 7 de 7   ·   signo del z comun a las campañas: p=1.000

  hardness_H_GPa · DISPERSION · estratos=1   global 57.6 % fuera
       609    614    615    619    622    623    626   <- campaña
        14     48     14     11     20     13     19   <- peliculas
        64     50     64     82     80     46     37   <- % fuera
      -2.1   -1.8   -3.1   -2.5   -2.3   -1.9   -1.1   <- z mediano
    campañas con >= 20 % fuera: 7 de 7   ·   signo del z comun a las campañas: p=0.016

  hardness_H_GPa · MEDIA · estratos=8   global 17.3 % fuera


**Cómo leer la salida**

En el bloque A, si el efecto solo supera el 20 % fuera en dos o tres campañas, es una propiedad de esas campañas y no del criterio del especialista. La prueba del signo común trabaja con las 7 campañas como observaciones, no con las 139 películas.

En el bloque B, la tabla (1) la gana quitar los k de mayor |z|, una regla hecha para minimizar la dispersión que no mira ninguna curva; por eso bajar la dispersión no puede ser la métrica. La tabla (2) mide la recuperación: 1 es llegar exactamente al valor del especialista, 0 no mejorar la media cruda y un valor negativo alejarse de él más que la media cruda. Si la mejor regla de valor se queda claramente por debajo de 1, ese hueco es el que tiene que llenar un modelo que mire la curva.

En el bloque C, si la tasa de descarte o el perfil de |z| de lo marcado cambiara mucho entre campañas, el criterio del especialista habría derivado con el tiempo, y la partición de entrenamiento y prueba tendría que hacerse por campaña.

## Análisis complementario. Réplicas de proceso

Fuera de las fases; memoria, apartado 5.4 y apéndice. Comprueba si la limpieza del especialista acerca los valores de películas fabricadas en las mismas condiciones.

### Celda 12. Réplicas de proceso: ¿acuerdo con el especialista o exactitud?

Análisis complementario (memoria, apartado 5.4 y apéndice). La dureza verdadera de las películas no se conoce. Pero si hay películas que deberían salir iguales, con la misma solución y la misma densidad de corriente, lo que difieren entre sí es error que no depende del juicio de nadie. La celda mide si la limpieza del especialista acerca esas películas y si lo hace mejor que las reglas que solo miran el valor, con el mismo número de descartes k (quitar los k de mayor |z|, de la cola alta, de la cola baja o al azar) o sin presupuesto fijado (mediana y media recortada al 10 %).

El estadístico es la discrepancia, la media de |xi - xj| entre todos los pares de películas del grupo, en fracción de la discrepancia sin limpiar. Descartar deja menos datos y más ruido en el promedio, así que la prueba es conservadora a favor de la hipótesis nula.

- C0. Con menos de 8 grupos de al menos 2 películas, el contraste no se evalúa.
- C1a, mejora absoluta. El especialista acerca las réplicas respecto a no limpiar en al menos el 60 % de los grupos, con Wilcoxon p < 0,05.
- C1b, mejora de la elección. El especialista las acerca más que quitar el mismo número al azar, que paga el mismo coste de ruido, en al menos el 60 % de los grupos, con Wilcoxon p < 0,05.
- C2. Si se cumple C1b y ninguna regla de valor lo iguala, hay exactitud y no solo acuerdo con una persona.
- C3. Si se cumple C1b pero alguna regla de valor lo iguala o lo supera, la regla basta.

La celda añade a `M` la columna `cd_r`, la densidad de corriente redondeada a dos decimales, y deja `IB` y `d_entre`, que usa la celda 13. Guarda `/content/replicas_hardness_H_GPa.csv` y `/content/replicas_reduced_modulus_Er_GPa.csv`.

In [13]:
MIN_GRUPOS = 8
REGLAS = ['experto', 'mas_abs', 'cola_alta', 'cola_baja', 'azar', 'mediana', 'recortada']
NOMBRE = {'experto': 'EXPERTO', 'mas_abs': 'quitar k mayor |z|',
          'cola_alta': 'quitar k cola alta', 'cola_baja': 'quitar k cola baja',
          'azar': 'quitar k al azar (nulo)', 'mediana': 'mediana (gasto libre)',
          'recortada': 'recortada 10 % (gasto libre)'}

M['cd_r'] = pd.to_numeric(M.current_density, errors='coerce').round(2)
print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'descarte {100*M._y.mean():.1f} % · semilla 20260913')


def discrepancia(xs):
    return float(np.mean([abs(a - b) for a, b in itertools.combinations(xs, 2)]))


sep('1 · ¿EXISTEN PELICULAS QUE DEBERIAN SALIR IGUALES?')
info = M.groupby('sample_num').agg(solution=('solution', 'first'), cd=('cd_r', 'first'),
                                   batch=('batch_num', 'first')).reset_index()
IB = info.set_index('sample_num')
g_dentro = info.groupby(['solution', 'cd', 'batch']).sample_num.apply(list)
g_entre = info.groupby(['solution', 'cd']).sample_num.apply(list)
d_dentro = {k: v for k, v in g_dentro.items() if len(v) >= 2}
d_entre = {k: v for k, v in g_entre.items() if len(v) >= 2}
print(f'  combinaciones solucion x corriente x campaña           : {len(g_dentro):>4} '
      f'· con 2 o mas peliculas: {len(d_dentro):>3} '
      f'({sum(len(v) for v in d_dentro.values())} peliculas)')
print(f'  combinaciones solucion x corriente (todas las campañas): {len(g_entre):>4} '
      f'· con 2 o mas peliculas: {len(d_entre):>3} '
      f'({sum(len(v) for v in d_entre.values())} peliculas)')
if d_entre:
    tam = pd.Series([len(v) for v in d_entre.values()]).value_counts().sort_index()
    print('  tamano de los grupos (solucion x corriente): '
          + ', '.join(f'{int(i)} peliculas x{int(n)}' for i, n in tam.items()))
    multi = sum(1 for v in d_entre.values() if IB.loc[v, 'batch'].nunique() > 1)
    print(f'  de esos grupos, {multi} tienen peliculas de CAMPAÑAS DISTINTAS (replicacion '
          f'independiente del bano) y {len(d_entre)-multi} son de la misma campaña')

USAR = d_entre if len(d_entre) >= len(d_dentro) else d_dentro
ETIQ = 'solucion x corriente' if USAR is d_entre else 'solucion x corriente x campaña'
print(f'\n  se usa la agrupacion por {ETIQ}: {len(USAR)} grupos')
HAY = len(USAR) >= MIN_GRUPOS
if not HAY:
    print(f'\n  ---> C0: hay menos de {MIN_GRUPOS} grupos; el contraste no se evalua.')


def valores_por_pelicula(rng):
    val = {p: {} for p in PROPS}
    azar = {p: {} for p in PROPS}
    for s, d in M.groupby('sample_num'):
        q = (d._y == 1).values
        for p in PROPS:
            v = d[p].astype(float).values
            n = len(v); k = int(q.sum())
            if not np.isfinite(v).all() or k < 1 or n - k < 3:
                continue
            z = z_signed(v)
            r = {'cruda': v.mean(), 'experto': v[~q].mean()}
            for nom, o in (('mas_abs', np.argsort(-np.abs(z))),
                           ('cola_alta', np.argsort(-z)), ('cola_baja', np.argsort(z))):
                keep = np.ones(n, bool); keep[o[:k]] = False
                r[nom] = v[keep].mean()
            r['mediana'] = np.median(v)
            t = int(0.10 * n); r['recortada'] = np.sort(v)[t:n - t].mean()
            val[p][s] = r
            azar[p][s] = v[np.argsort(rng.random((BR, n)), axis=1)[:, :n - k]].mean(axis=1)
    return val, azar


def contrastar(val, azar):
    for p in PROPS:
        filas = []
        for clave, muestras in USAR.items():
            ms = [s for s in muestras if s in val[p]]
            if len(ms) < 2:
                continue
            base = discrepancia([val[p][s]['cruda'] for s in ms])
            if not np.isfinite(base) or base <= 0:
                continue
            f = dict(grupo=str(clave), m=len(ms), cruda=base)
            for rg in REGLAS:
                if rg == 'azar':
                    col = np.vstack([azar[p][s] for s in ms])
                    f[rg] = float(np.median([discrepancia(col[:, b]) for b in range(col.shape[1])]))
                else:
                    f[rg] = discrepancia([val[p][s][rg] for s in ms])
            filas.append(f)
        T = pd.DataFrame(filas)
        if len(T) < MIN_GRUPOS:
            print(f'\n  {p}: solo {len(T)} grupos utilizables, no se evalua.')
            continue
        print(f'\n  --- {p} ---   {len(T)} grupos · {int(T.m.sum())} peliculas')
        print(f'    discrepancia cruda entre replicas: mediana {T.cruda.median():.3f} '
              f'(p25 {np.percentile(T.cruda, 25):.3f} · p75 {np.percentile(T.cruda, 75):.3f})')
        cab = f'{"regla":>30} {"razon mediana":>14} {"% grupos que acercan":>21} {"Wilcoxon p":>12}'
        print('\n    ' + cab)
        res = {}
        for rg in REGLAS:
            ratio = (T[rg] / T.cruda).dropna()
            acerca = 100 * (ratio < 1).mean()
            w = stats.wilcoxon(T[rg], T.cruda).pvalue if len(T) >= 6 else np.nan
            res[rg] = (ratio.median(), acerca, w)
            print(f'    {NOMBRE[rg]:>30} {ratio.median():>14.3f} {acerca:>20.0f}% {w:>12.2e}')
        print('\n    ¿le gana alguna regla de valor al experto, grupo a grupo?')
        for rg in REGLAS[1:]:
            mejor = 100 * (T[rg] < T['experto']).mean()
            w = stats.wilcoxon(T[rg], T['experto']).pvalue if len(T) >= 6 else np.nan
            print(f'      {NOMBRE[rg]:>30}: acerca mas que el experto en {mejor:>3.0f} % '
                  f'de los grupos (p={w:.2e})')
        rm, ra, rw = res['experto']
        gana_azar = 100 * (T['experto'] < T['azar']).mean()
        w_azar = stats.wilcoxon(T['experto'], T['azar']).pvalue if len(T) >= 6 else np.nan
        empatan = [NOMBRE[rg] for rg in ('mas_abs', 'cola_alta', 'cola_baja', 'mediana',
                                         'recortada')
                   if res[rg][0] <= rm or 100 * (T[rg] < T['experto']).mean() >= 50]
        c1a = ra >= 60 and rw < 0.05
        c1b = gana_azar >= 60 and w_azar < 0.05
        print(f'\n    contra el nulo de tirar la misma cantidad al azar: el experto acerca mas')
        print(f'    en {gana_azar:.0f} % de los grupos (Wilcoxon p={w_azar:.2e})')
        print(f'\n    VEREDICTO {p}:   C1a (mejora absoluta) = {"SI" if c1a else "NO"}   ·   '
              f'C1b (mejor que el azar) = {"SI" if c1b else "NO"}')
        if not c1b:
            print('      la eleccion del experto no acerca las replicas mas que tirar la misma '
                  'cantidad al azar')
        elif empatan:
            print(f'      C3: el experto le gana al azar, pero tambien lo hacen o lo igualan: '
                  f'{", ".join(empatan)}.')
        else:
            print('      C2: el experto acerca las replicas mas que el azar y ninguna regla de '
                  'valor lo iguala')
            if not c1a:
                print('      sin C1a: la eleccion es mejor que el azar, pero limpiar no acerca '
                      'las replicas en terminos absolutos')
        T.to_csv(f'/content/replicas_{p}.csv', index=False)
        print(f'      guardado: /content/replicas_{p}.csv')


if HAY:
    sep('2 · ¿ACERCA LA LIMPIEZA A LAS PELICULAS QUE DEBERIAN SER IGUALES?')
    contrastar(*valores_por_pelicula(np.random.default_rng(20260913)))

  139 peliculas · 8896 indentaciones · descarte 27.7 % · semilla 20260913

1 · ¿EXISTEN PELICULAS QUE DEBERIAN SALIR IGUALES?
  combinaciones solucion x corriente x campaña           :  119 · con 2 o mas peliculas:  20 (40 peliculas)
  combinaciones solucion x corriente (todas las campañas):  119 · con 2 o mas peliculas:  20 (40 peliculas)
  tamano de los grupos (solucion x corriente): 2 peliculas x20
  de esos grupos, 0 tienen peliculas de CAMPAÑAS DISTINTAS (replicacion independiente del bano) y 20 son de la misma campaña

  se usa la agrupacion por solucion x corriente: 20 grupos

2 · ¿ACERCA LA LIMPIEZA A LAS PELICULAS QUE DEBERIAN SER IGUALES?



  --- hardness_H_GPa ---   20 grupos · 40 peliculas
    discrepancia cruda entre replicas: mediana 0.154 (p25 0.073 · p75 0.298)

                             regla  razon mediana  % grupos que acercan   Wilcoxon p
                           EXPERTO          0.833                   65%     2.94e-01
                quitar k mayor |z|          1.220                   45%     9.85e-01
                quitar k cola alta          0.813                   70%     6.96e-02


                quitar k cola baja          1.675                   15%     5.86e-04
           quitar k al azar (nulo)          1.042                   30%     4.41e-02
             mediana (gasto libre)          1.005                   50%     9.56e-01
      recortada 10 % (gasto libre)          0.938                   60%     2.02e-01

    ¿le gana alguna regla de valor al experto, grupo a grupo?
                  quitar k mayor |z|: acerca mas que el experto en  45 % de los grupos (p=5.71e-01)
                  quitar k cola alta: acerca mas que el experto en  60 % de los grupos (p=1.89e-01)
                  quitar k cola baja: acerca mas que el experto en  25 % de los grupos (p=1.99e-03)
             quitar k al azar (nulo): acerca mas que el experto en  35 % de los grupos (p=2.02e-01)
               mediana (gasto libre): acerca mas que el experto en  45 % de los grupos (p=6.48e-01)
        recortada 10 % (gasto libre): acerca mas que el experto en  40 % de los grupos (p=5.71e-0

**Cómo leer la salida**

Una razón menor que 1 indica que las réplicas se acercan; igual a 1, que no cambia nada, y mayor que 1, que se separan. C1a mide si conviene limpiar con todo incluido; C1b, si la elección del especialista es mejor que el azar, que es la pregunta científica.

## Comprobaciones complementarias

Comprobaciones sobre cifras de las fases 1, 3 y 4 y sobre el propio conjunto: cambios de marca entre revisiones, reglas con el presupuesto del especialista, calibración del contraste con un experto falso, réplicas por campaña, profundidad máxima y ecuaciones del laboratorio.

### Celda 13. Comprobaciones complementarias

Comprobaciones sobre cifras de las celdas 3, 4, 6 y 10 a 12 y sobre el propio conjunto. La celda solo imprime y no guarda ficheros.

- A. Cambios de marca entre las dos revisiones. Se cuentan por curvas únicas sobre las 8.896 curvas del análisis, porque sumar cambios por categoría puede contar una curva varias veces. Las tablas por ensayo solo listan las curvas con alguna anotación, de modo que una curva ausente tiene todas sus categorías a 0 en esa revisión. Con ese supuesto, el objetivo binario reconstruido desde la segunda revisión debe coincidir con el del análisis. Se dan dos cifras de cambio: (1) con las columnas tal como están y (2) contando como descarte en la primera revisión el nivel 2 o más de «zero offset», que la segunda reparte entre desplazamiento de fuerza y de profundidad. La diferencia entre (1) y (2) es cambio de notación. La cifra (2) es la diferencia de criterio entre dos revisiones, no una tasa de error del especialista.
- B. Las reglas, con y sin el presupuesto del especialista, sobre los 64 ensayos. La tabla (2) de la celda 11 calcula la media sin limpiar y la del especialista sobre los ensayos con |z| ≤ 3, y resume el azar con la mediana de las medias antes de aplicar 1 − |X − E| / |A − E|, que no es lineal. Aquí se rehace sobre los 64 ensayos, con la métrica calculada en cada sorteo y resumida después.
- C. Calibración del contraste con un experto falso, que retira en cada franja tantos ensayos como el especialista, elegidos al azar, y por construcción cumple la hipótesis nula. Si queda en torno al 10 %, el contraste está bien calibrado (memoria, Tabla 5).
- D. Campaña de cada pareja de réplicas de la celda 12 y valores repetidos en las columnas de profundidad.
- D2. La profundidad máxima. Hay repetición sistemática de hmax si un valor aparece 3 o más veces en una película o si se repite entre ensayos consecutivos en más de 2 películas. Se comparan el valor por película del resumen y la profundidad máxima de cada curva. El rango del README, de 30 a 100 nm, describe los ensayos si al menos el 90 % de las medianas por película cae dentro. La penetración excesiva se explica como anomalía de valor si al menos 2/3 de los ensayos con `too_deep` superan el percentil 90 de la referencia; son pocos ensayos y la lectura es indicativa.
- D3. Ecuaciones y constantes con que el laboratorio calculó los valores del resumen (memoria, apartado 3.1 y Tabla 3). Una relación se da por la del laboratorio si se cumple a menos del 0,1 % en al menos el 99 % de los ensayos. La función de área se da por fija dentro de cada jornada si el ajuste la reproduce a menos del 0,1 % en todas las jornadas con 12 ensayos o más. Nada del análisis depende de este bloque.

In [14]:
REPS = 3
NQS_CAL = [1, 4, 8, 12]

print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'descarte {100*M._y.mean():.1f} % · semilla 20260914')

sep('A · CAMBIOS DE ETIQUETA ENTRE LAS DOS PASADAS (poblacion del analisis)')
IDX = pd.MultiIndex.from_frame(M[CLAVE])
enO, enU = IDX.isin(Oi.index), IDX.isin(Ui.index)
print(f'  curvas del analisis: {len(IDX)} · en las dos tablas: {int((enO & enU).sum())} · '
      f'solo en la 1a: {int((enO & ~enU).sum())} · solo en la 2a: {int((~enO & enU).sum())} · '
      f'en ninguna: {int((~enO & ~enU).sum())}')


def niveles(df, cols):
    cols = [c for c in cols if c in df.columns]
    return df.reindex(IDX)[cols].apply(pd.to_numeric, errors='coerce').fillna(0)


NO, NU = niveles(Oi, POR_INDENT), niveles(Ui, POR_INDENT)
bo = (NO >= 2).any(axis=1).values.astype(int)
bu = (NU >= 2).any(axis=1).values.astype(int)
n_disc = int((bu != M._y.values).sum())
print(f'  comprobacion del supuesto: objetivo de la 2a pasada reconstruido frente al del '
      f'analisis -> {n_disc} discrepancias '
      f'{"(supuesto confirmado)" if n_disc == 0 else "(supuesto no confirmado)"}')

print(f'\n    {"categoria":>42} {"altas":>7} {"bajas":>7} {"suben":>7} {"bajan":>7}')
cambio_alguna = np.zeros(len(IDX), bool)
tot_ev = 0
for c in POR_INDENT:
    if c not in NO.columns or c not in NU.columns:
        continue
    a, b = NO[c].values, NU[c].values
    alta = (a == 0) & (b > 0); baja = (a > 0) & (b == 0)
    sube = (a > 0) & (b > a); bajo = (a > 0) & (b < a) & (b > 0)
    cambio_alguna |= (a != b)
    tot_ev += int((a != b).sum())
    print(f'    {c:>42} {int(alta.sum()):>7} {int(baja.sum()):>7} '
          f'{int(sube.sum()):>7} {int(bajo.sum()):>7}')
print(f'\n  eventos sumando categorias: {tot_ev} · curvas unicas con algun cambio de nivel: '
      f'{int(cambio_alguna.sum())} ({100*cambio_alguna.mean():.1f} %)')


def cambio_objetivo(nom, x, y):
    ch = x != y
    print(f'    {nom:<46} {int(ch.sum()):>5} ({100*ch.mean():>4.1f} %)   de 0 a 1: '
          f'{int(((x == 0) & (y == 1)).sum()):>5}   de 1 a 0: {int(((x == 1) & (y == 0)).sum()):>4}'
          f'   descarte 1a {100*x.mean():>4.1f} % -> 2a {100*y.mean():>4.1f} %')
    return ch


print(f'\n  CAMBIO DEL OBJETIVO BINARIO, en curvas sobre las {len(IDX)}')
ch1 = cambio_objetivo('(1) columnas tal cual', bo, bu)
print(f'        de ellas, en curvas de las dos tablas: {int((ch1 & enO & enU).sum())} · '
      f'en curvas de una sola tabla: {int((ch1 & (enO ^ enU)).sum())}')
if 'verify_zero_offset_type' in Oi.columns:
    zo = (niveles(Oi, ['verify_zero_offset_type']).iloc[:, 0].values >= 2).astype(int)
    bo2 = np.maximum(bo, zo)
    ch2 = cambio_objetivo('(2) zero offset de la 1a como descarte', bo2, bu)
    print(f'        curvas con zero offset >= 2 en la 1a: {int(zo.sum())} · de ellas '
          f'descartadas en la 2a: {int((zo & bu).sum())}')
    print(f'        peliculas con algun cambio en (2): '
          f'{len(np.unique(M.sample_num.values[ch2]))} de {M.sample_num.nunique()}')
    nuevas = (bo2 == 0) & (bu == 1); quitadas = (bo2 == 1) & (bu == 0)
    t_n = (NU[nuevas] >= 2).sum()
    t_q = (NO[quitadas] >= 2).sum()
    solo_zo = int((quitadas & (bo == 0)).sum())
    print('        de 0 a 1, tipo que las marca en la 2a: ' +
          ' · '.join(f'{c} {int(n)}' for c, n in t_n.items() if n > 0))
    print('        de 1 a 0, tipo que las marcaba en la 1a: ' +
          ' · '.join(f'{c} {int(n)}' for c, n in t_q.items() if n > 0) +
          f' · solo zero offset {solo_zo}')
else:
    print('    la 1a pasada no trae verify_zero_offset_type: no se puede dar la cifra (2).')

sep('B · RECUPERACION SOBRE LAS 64 INDENTACIONES (misma poblacion que la celda 6)')
rng = np.random.default_rng(20260914)
filas = []
for s, d in M.groupby('sample_num'):
    q = (d._y == 1).values
    r = dict(sample_num=s)
    for p in PROPS:
        v = d[p].astype(float).values
        n = len(v); k = int(q.sum())
        if not np.isfinite(v).all() or k < 1 or n - k < 3:
            continue
        A = v.mean(); E = v[~q].mean(); den = abs(A - E)
        if den <= 1e-12:
            continue
        z = z_signed(v)
        rec = {}
        t = int(0.10 * n)
        rec['mediana'] = 1 - abs(np.median(v) - E) / den
        rec['recortada'] = 1 - abs(np.sort(v)[t:n - t].mean() - E) / den
        mad3 = v[np.abs(z) <= 3]
        rec['mad3'] = 1 - abs(mad3.mean() - E) / den if len(mad3) >= 2 else np.nan
        for nom, o in (('mas_abs', np.argsort(-np.abs(z))), ('cola_alta', np.argsort(-z)),
                       ('cola_baja', np.argsort(z))):
            keep = np.ones(n, bool); keep[o[:k]] = False
            rec[nom] = 1 - abs(v[keep].mean() - E) / den
        xs = v[np.argsort(rng.random((BR, n)), axis=1)[:, :n - k]].mean(axis=1)
        rec['azar'] = float(np.median(1 - np.abs(xs - E) / den))
        for kk, vv in rec.items():
            r[f'{p}_{kk}'] = vv
        r[f'{p}_pctk'] = 100 * k / n
        r[f'{p}_pct_mad3'] = 100 * (np.abs(z) > 3).mean()
    filas.append(r)
RB = pd.DataFrame(filas)

ORD = [('mediana', 'mediana', 'libre'), ('recortada', 'media recortada 10 %', 'libre'),
       ('mad3', 'quitar |z-MAD| > 3', 'libre'),
       ('mas_abs', 'quitar k mayor |z|', 'igualado'),
       ('cola_alta', 'quitar k cola alta', 'igualado'),
       ('cola_baja', 'quitar k cola baja', 'igualado'),
       ('azar', 'quitar k al azar (nulo)', 'igualado')]
for p in PROPS:
    print(f'\n  --- {p} ---   gasto del experto: {RB[f"{p}_pctk"].median():.1f} % · '
          f'gasto de la regla MAD: {RB[f"{p}_pct_mad3"].median():.1f} %')
    print(f'    {"regla":>26} {"gasto":>10} {"mediana":>9} {"p25":>8} {"p75":>8} '
          f'{"% > 0.5":>9} {"% < 0":>8}')
    for c, nom, gasto in ORD:
        col = f'{p}_{c}'
        if col not in RB.columns:
            continue
        v = RB[col].replace([np.inf, -np.inf], np.nan).dropna()
        print(f'    {nom:>26} {gasto:>10} {v.median():>9.2f} {np.percentile(v,25):>8.2f} '
              f'{np.percentile(v,75):>8.2f} {100*(v>0.5).mean():>8.0f}% {100*(v<0).mean():>7.0f}%')

sep('C · ¿ESTA CALIBRADO EL NULO ESTRATIFICADO? (experto falso bajo la hipotesis nula)')


def corre_nulo(v, q, z, nq, rng):
    n = len(v); k = int(q.sum())
    if k < 1 or n - k < 3:
        return None
    st = bins(z, nq); cols = []
    for j in np.unique(st):
        pos = np.flatnonzero(st == j); nj = len(pos); mj = int(q[pos].sum())
        if nj - mj <= 0:
            continue
        cols.append(pos[np.argsort(rng.random((B, nj)), axis=1)[:, :nj - mj]])
    if not cols:
        return None
    sub = v[np.concatenate(cols, axis=1)]
    out = []
    for e, nul in ((v[~q].mean(), sub.mean(axis=1)),
                   (v[~q].std(ddof=1), sub.std(axis=1, ddof=1))):
        out.append(int(e < np.percentile(nul, 5) or e > np.percentile(nul, 95)))
    return out


def experto_falso(q, z, nq, rng):
    st = bins(z, nq); f = np.zeros(len(q), bool)
    for j in np.unique(st):
        pos = np.flatnonzero(st == j); mj = int(q[pos].sum())
        if mj:
            f[rng.choice(pos, mj, replace=False)] = True
    return f


rng = np.random.default_rng(20260914 + 1)
acc = {}
for s, d in M.groupby('sample_num'):
    q0 = (d._y == 1).values
    for p in PROPS:
        v0 = d[p].astype(float).values
        if not np.isfinite(v0).all():
            continue
        zs = z_signed(v0); pool = np.abs(zs) <= 3
        v = v0[pool]; q = q0[pool]; z = zs[pool]
        for nq in NQS_CAL:
            o = corre_nulo(v, q, z, nq, rng)
            if o:
                acc.setdefault((p, nq, 'real'), []).append(o)
            for _ in range(REPS):
                qf = experto_falso(q, z, nq, rng)
                o = corre_nulo(v, qf, z, nq, rng)
                if o:
                    acc.setdefault((p, nq, 'falso'), []).append(o)

for p in PROPS:
    print(f'\n  --- {p} ---')
    print(f'    {"estratos":>9} {"% fuera MEDIA":>28} {"% fuera DISPERSION":>30}')
    print(f'    {"":>9} {"experto":>12} {"falso":>9} {"":>4} {"experto":>12} {"falso":>9}')
    for nq in NQS_CAL:
        R_ = np.array(acc.get((p, nq, 'real'), []))
        F_ = np.array(acc.get((p, nq, 'falso'), []))
        if not len(R_) or not len(F_):
            continue
        print(f'    {nq:>9} {100*R_[:,0].mean():>11.1f}% {100*F_[:,0].mean():>8.1f}% {"":>4} '
              f'{100*R_[:,1].mean():>11.1f}% {100*F_[:,1].mean():>8.1f}%')

sep('D · LAS 20 PAREJAS DE REPLICAS, Y LA DUDA DE LA PROFUNDIDAD')
lotes = pd.Series([IB.loc[v, 'batch'].iloc[0] for v in d_entre.values()]).value_counts().sort_index()
print(f'  parejas de replicas: {len(d_entre)} · reparto por campaña:')
print('    ' + ' · '.join(f'campaña {int(b)}: {int(n)}' for b, n in lotes.items()))
print(f'  -> {"TODAS de la misma campaña" if len(lotes)==1 else f"repartidas en {len(lotes)} campañas"}')

HMAX = 'max_displacement_hmax_nm'
cols_prof = [c for c in M.columns if 'depth' in c.lower() or c in ('num_points',)]
cols_prof += [c for c in (HMAX, 'final_displacement_hf_nm') if c in M.columns]
print(f'\n  columnas de profundidad / tamano: {cols_prof}')
for c in cols_prof:
    v = pd.to_numeric(M[c], errors='coerce').dropna()
    if not len(v):
        continue
    vc = v.value_counts()
    print(f'    {c:>24}: distintos={v.nunique():>6} · min={v.min():.4g} max={v.max():.4g} · '
          f'valor mas repetido={vc.index[0]:.6g} aparece {int(vc.iloc[0])} veces '
          f'({100*vc.iloc[0]/len(v):.1f} %)')

sep('D2 · LA PROFUNDIDAD MAXIMA hmax')
if HMAX in M.columns:
    hm = pd.to_numeric(M[HMAX], errors='coerce')
    sano = (M._y == 0).values

    rep_pel, max_rep, pel_cons, ejemplos = 0, 1, 0, []
    for s, d in M.assign(_h=hm).sort_values(CLAVE).groupby('sample_num'):
        vc = d._h.dropna().value_counts()
        if len(vc) and vc.iloc[0] > 1:
            rep_pel += 1
            max_rep = max(max_rep, int(vc.iloc[0]))
            ejemplos.append((s, vc.index[0], int(vc.iloc[0])))
        pel_cons += int((d._h.diff() == 0).any())
    print(f'  (a) peliculas con algun hmax repetido: {rep_pel} de {M.sample_num.nunique()} · '
          f'maximo de apariciones de un valor: {max_rep} · peliculas con repeticion entre '
          f'ensayos consecutivos: {pel_cons}')
    for s, val, k in ejemplos[:5]:
        print(f'        pelicula {s}: hmax = {val:.6g} nm aparece {k} veces')
    if max_rep >= 3 or pel_cons > 2:
        print('      LECTURA: repeticion sistematica. La columna del resumen no es una medida '
              'por ensayo.')
    else:
        print('      LECTURA: no hay repeticion sistematica en hmax.')

    num_res = [c for c in S.columns if c not in CLAVE
               and pd.api.types.is_numeric_dtype(S[c])]
    nu = M.groupby('sample_num')[num_res].nunique()
    const = [c for c in num_res if (nu[c] <= 1).all()]
    print(f'\n  (a2) columnas numericas del resumen constantes dentro de TODAS las peliculas: '
          f'{const}')

    t0 = time.time()
    hcur = np.full(len(M), np.nan)
    faltan = 0
    for i, ruta in enumerate(M['path'].astype(str).values):
        f = os.path.join(RAIZ, ruta)
        try:
            hcur[i] = np.nanmax(pd.read_csv(f, usecols=['depth_nm'])['depth_nm'].to_numpy(float))
        except Exception:
            faltan += 1
    print(f'\n  (a3) profundidad maxima leida de {int(np.isfinite(hcur).sum())} curvas '
          f'({faltan} sin leer) en {time.time() - t0:.0f} s')
    hc_res = pd.to_numeric(M['contact_depth_hc_nm'], errors='coerce').values
    okc = np.isfinite(hcur) & np.isfinite(hc_res)
    print(f'       control: hc del resumen por encima de la profundidad maxima de su curva en '
          f'{int((hc_res[okc] > hcur[okc]).sum())} de {int(okc.sum())} ensayos '
          f'(no descartados: {int((hc_res[okc & sano] > hcur[okc & sano]).sum())} de '
          f'{int((okc & sano).sum())})')
    filas = []
    for s, ii in M.groupby('sample_num').indices.items():
        val = hm.values[ii][0]; h = hcur[ii]
        if not np.isfinite(val) or not np.isfinite(h).any():
            continue
        j = int(np.nanargmin(np.abs(h - val)))
        filas.append(dict(sample_num=s, val=val, dmin=abs(h[j] - val),
                          indent=int(M.indent_num.values[ii][j]),
                          r_max=val / np.nanmax(h), r_med=val / np.nanmedian(h)))
    Q = pd.DataFrame(filas)
    Q['drel'] = Q.dmin / Q.val
    print(f'       distancia del hmax del resumen al ensayo mas proximo de su pelicula: mediana '
          f'{Q.dmin.median():.3g} nm · coincide a 0,01 nm en {int((Q.dmin <= 0.01).sum())} de '
          f'{len(Q)} peliculas · a menos del 1 % en {int((Q.drel <= 0.01).sum())}')
    vc = Q.loc[Q.drel <= 0.01, 'indent'].value_counts()
    if len(vc):
        print('       ensayo mas proximo, en las que coinciden a menos del 1 % '
              '(indent_num: peliculas): ' +
              ' · '.join(f'{int(k)}: {int(n)}' for k, n in vc.head(8).items()))
    print(f'       hmax del resumen / maximo de la pelicula: mediana {Q.r_max.median():.3f} '
          f'(p5 {Q.r_max.quantile(.05):.3f}, p95 {Q.r_max.quantile(.95):.3f}) · '
          f'/ mediana de la pelicula: mediana {Q.r_med.median():.3f}')

    med_pel = pd.Series(np.where(sano, hcur, np.nan), index=M.index).groupby(M.sample_num).median()
    vm = med_pel.dropna()
    dentro = vm.between(30, 100)
    p5, p50, p95 = np.percentile(vm, [5, 50, 95])
    print(f'\n  (b) mediana por pelicula de la profundidad maxima de la curva (no descartados): '
          f'min {vm.min():.1f} · p5 {p5:.1f} · mediana {p50:.1f} · p95 {p95:.1f} · max {vm.max():.1f} nm')
    print(f'      peliculas con la mediana entre 30 y 100 nm: {int(dentro.sum())} de '
          f'{len(vm)} ({100*dentro.mean():.1f} %)')
    for nom, sel in (('no descartados', sano), ('descartados', ~sano)):
        x = pd.Series(hcur[sel]).dropna()
        print(f'      ensayos {nom:<15} entre 30 y 100 nm: {100*x.between(30, 100).mean():5.1f} % · '
              f'por debajo: {100*(x < 30).mean():4.1f} % · por encima: {100*(x > 100).mean():4.1f} %')
    if dentro.mean() >= 0.90:
        print('      LECTURA: el rango del README describe los ensayos.')
    else:
        print(f'      LECTURA: el rango del README no describe los datos; rango observado '
              f'{p5:.0f} a {p95:.0f} nm (p5 a p95 de las medianas por pelicula).')

    if 'too_deep' in M.columns:
        td = (pd.to_numeric(M.too_deep, errors='coerce').fillna(0) >= 2).values
        otros = np.zeros(len(M), bool)
        for c in POR_INDENT:
            if c != 'too_deep':
                otros |= (pd.to_numeric(M[c], errors='coerce').fillna(0) >= 2).values
        ref_pel = M.sample_num.map(med_pel).values
        n_sano = pd.Series(sano, index=M.index).groupby(M.sample_num).transform('sum').values
        ok = np.isfinite(hcur) & np.isfinite(ref_pel) & (ref_pel > 0) & (n_sano >= 10)
        razon_h = np.where(ok, hcur / np.where(ok, ref_pel, 1), np.nan)
        ref = razon_h[sano & ok]
        p90 = np.percentile(ref, 90)
        print(f'\n  (c) profundidad maxima de la curva / mediana de los no descartados de su pelicula')
        print(f'      referencia, no descartados ({int((sano & ok).sum())} ensayos): '
              f'mediana {np.median(ref):.3f} · p90 {p90:.3f}')
        for nom, sel in (('too_deep (todos)', td & ok), ('too_deep sin otro tipo', td & ~otros & ok)):
            x = razon_h[sel]
            if not len(x):
                continue
            print(f'      {nom:<24} {len(x):>3} ensayos en {len(np.unique(M.sample_num.values[sel]))} '
                  f'peliculas · mediana {np.median(x):.3f} · por encima del p90 de la '
                  f'referencia: {int((x > p90).sum())} ({100*(x > p90).mean():.0f} %)')
        x = razon_h[td & ok]
        if len(x) and (x > p90).mean() >= 2 / 3:
            print('      LECTURA: los ensayos con penetracion excesiva son, en su mayoria, mas '
                  'profundos que sus vecinos sanos.')
        else:
            print('      LECTURA: los ensayos con penetracion excesiva no son, en su mayoria, mas '
                  'profundos que sus vecinos sanos.')
else:
    print(f'  no se encuentra la columna {HMAX}; se omite el bloque.')

sep('D3 · COMO CALCULO EL LABORATORIO LOS VALORES DEL RESUMEN')


def _d3_resumen(M, hcur):
    def col(c):
        return (pd.to_numeric(M[c], errors='coerce').to_numpy(float) if c in M.columns
                else np.full(len(M), np.nan))
    P, Sr, hc = col('max_load_Pmax_micro_N'), col('stiffness_S_micro_N/nm'), col('contact_depth_hc_nm')
    A, he, Er = col('proj_contact_area_A_nm^2'), col('effective_depth_heff_nm'), col('reduced_modulus_Er_GPa')
    H, al, hf, m = (col('hardness_H_GPa'), col('unloading_curve_alpha'),
                    col('final_displacement_hf_nm'), col('unloading_curve_m'))
    base = np.isfinite(P) & np.isfinite(Sr) & np.isfinite(A) & (P > 0) & (Sr > 0) & (A > 0)
    n = int(base.sum())
    print(f'  ensayos con fuerza, rigidez y area validas: {n} de {len(M)}')

    def cumple(r, tol=0.001):
        r = r[np.isfinite(r)]
        return (100 * np.mean(np.abs(r - 1) <= tol), float(np.max(np.abs(r - 1)))) if len(r) else (np.nan, np.nan)

    lect = {}
    r1 = np.where(base & np.isfinite(H) & (H > 0), P / A * 1e3 / H, np.nan)
    pc, mx = cumple(r1); lect['1'] = pc >= 99
    print(f'\n  (1) dureza = fuerza maxima / area: a menos del 0,1 % en {pc:.2f} % de los ensayos '
          f'(desviacion maxima {100*mx:.3g} %)')
    e = np.where(base & np.isfinite(he) & np.isfinite(hc), (he - hc) * Sr / P, np.nan)
    e_med = float(np.nanmedian(e))
    pc, mx = cumple(e / e_med); lect['2'] = pc >= 99
    print(f'  (2) e estimado = (prof. efectiva - prof. contacto) · rigidez / fuerza: mediana {e_med:.4f} · '
          f'p1 {np.nanpercentile(e, 1):.4f} · p99 {np.nanpercentile(e, 99):.4f} · '
          f'a menos del 0,1 % de la mediana en {pc:.2f} %')
    k = np.where(base & np.isfinite(Er) & (Er > 0), Er * np.sqrt(A) / (Sr * 1e3), np.nan)
    k_med = float(np.nanmedian(k))
    pc, mx = cumple(k / k_med); lect['3'] = pc >= 99
    print(f'  (3) k estimado = modulo · raiz(area) / rigidez: mediana {k_med:.5f} · p1 '
          f'{np.nanpercentile(k, 1):.5f} · p99 {np.nanpercentile(k, 99):.5f} · a menos del 0,1 % '
          f'de la mediana en {pc:.2f} % · raiz(pi)/2 = {np.sqrt(np.pi)/2:.5f} '
          f'(beta equivalente {np.sqrt(np.pi)/2/k_med:.5f})')
    ok4 = base & np.isfinite(al) & np.isfinite(hf) & np.isfinite(m) & np.isfinite(he) & (he > hf) & (al > 0)
    x = np.where(ok4, he - hf, np.nan)
    rS = np.where(ok4, m * al * x ** (m - 1) / Sr, np.nan)
    rP = np.where(ok4, al * x ** m / P, np.nan)
    pcS, _ = cumple(rS); pcS1, _ = cumple(rS, 0.01)
    pcP, _ = cumple(rP); pcP1, _ = cumple(rP, 0.01)
    lect['4'] = pcS >= 99 and pcP >= 99
    print(f'  (4) ajuste de la descarga evaluado en la profundidad efectiva ({int(ok4.sum())} ensayos): '
          f'pendiente = rigidez a menos del 0,1 % en {pcS:.2f} % (1 %: {pcS1:.2f} %) · '
          f'fuerza = fuerza maxima a menos del 0,1 % en {pcP:.2f} % (1 %: {pcP1:.2f} %)')
    ok5 = base & np.isfinite(hc) & (hc > 0)
    q = A[ok5] / hc[ok5] ** 2
    print(f'  (5) area / hc^2: min {q.min():.2f} · p5 {np.percentile(q, 5):.2f} · mediana '
          f'{np.median(q):.2f} · p95 {np.percentile(q, 95):.2f} · max {q.max():.2f} '
          f'(24,5 en una Berkovich ideal)')

    def ajusta(h, a):
        X = np.column_stack([h ** 2, h, h ** 0.5, h ** 0.25, h ** 0.125, h ** 0.0625])
        c, *_ = np.linalg.lstsq(X / a[:, None], np.ones(len(a)), rcond=None)
        return c, float(np.max(np.abs(X @ c / a - 1)))

    if 'date' in M.columns:
        txt = M['date'].astype(str)
        fecha = pd.to_datetime(txt, format='%a %b %d %H:%M:%S %Y', errors='coerce')
        if fecha.isna().all():
            fecha = pd.to_datetime(txt, errors='coerce')
        dia = np.where(fecha.notna(), fecha.dt.strftime('%Y-%m-%d'), txt.str[:10])
    else:
        dia = np.full(len(M), 'sin fecha')
    filas = []
    for d_ in pd.unique(dia[ok5]):
        s = ok5 & (dia == d_)
        if s.sum() >= 12:
            c, err = ajusta(hc[s], A[s])
            filas.append(dict(dia=d_, n=int(s.sum()), C0=c[0], err=err))
    J = pd.DataFrame(filas)
    cg, eg = ajusta(hc[ok5], A[ok5])
    n_pel_dias = int((pd.Series(dia, index=M.index).groupby(M.sample_num).nunique() > 1).sum())
    if len(J):
        lect['5'] = bool((J.err <= 0.001).all())
        print(f'      ajuste de la forma de Oliver y Pharr por jornada ({len(J)} jornadas con 12 ensayos o '
              f'mas, {int(J.n.sum())} ensayos): error relativo maximo mediana {J.err.median():.2g} · '
              f'peor jornada {J.err.max():.2g} · jornadas a menos del 0,1 %: {int((J.err <= 0.001).sum())} '
              f'de {len(J)}')
        print(f'      termino principal C0 por jornada: mediana {J.C0.median():.3f} · min {J.C0.min():.3f} · '
              f'max {J.C0.max():.3f}')
        print(f'      un unico ajuste para todo el conjunto: error relativo maximo {eg:.2g} · '
              f'peliculas medidas en mas de una jornada: {n_pel_dias}')
        for _, f in J.sort_values('dia').iterrows():
            print(f'        {f.dia}: {f.n:>5} ensayos · C0 {f.C0:7.3f} · error maximo {f.err:.1e}')
    else:
        lect['5'] = False
        print('      no hay jornadas con 12 ensayos o mas; se omite el ajuste por jornada')
    if hcur is not None and len(hcur) == len(M):
        dlt = np.where(np.isfinite(hcur) & np.isfinite(he), he - hcur, np.nan)
        dd = dlt[np.isfinite(dlt)]
        print(f'  (6) profundidad efectiva - profundidad maxima de la curva ({len(dd)} ensayos): mediana '
              f'{np.median(dd):.3g} nm · p5 {np.percentile(dd, 5):.3g} · p95 {np.percentile(dd, 95):.3g} · '
              f'a menos de 1 nm en {100*np.mean(np.abs(dd) < 1):.1f} %')
    else:
        print('  (6) no hay profundidad maxima de la curva (D2 a3); se omite')
    hcv = hc[ok5]
    pel25 = len(np.unique(M.sample_num.values[ok5][hcv < 25]))
    m_ok = m[np.isfinite(m)]
    print(f'  (7) fuerza maxima: {np.nanmin(P[base]):.1f} a {np.nanmax(P[base]):.1f} µN · exponente m: '
          f'{m_ok.min():.3f} a {m_ok.max():.3f} (mediana {np.median(m_ok):.3f}; en el valor minimo '
          f'{int(np.sum(m_ok <= m_ok.min() + 1e-6))}, en el maximo {int(np.sum(m_ok >= m_ok.max() - 1e-6))})')
    print(f'      profundidad de contacto: p1 {np.percentile(hcv, 1):.1f} · mediana {np.median(hcv):.1f} · '
          f'p99 {np.percentile(hcv, 99):.1f} · max {hcv.max():.1f} nm · por debajo de 25 nm: '
          f'{int((hcv < 25).sum())} ensayos en {pel25} peliculas')
    print('\n  LECTURA:')
    if lect['1'] and lect['2'] and lect['3']:
        print(f'   · El resumen sigue la cadena de Oliver y Pharr: dureza = fuerza / area; profundidad de '
              f'contacto = profundidad efectiva - {e_med:.3f} · fuerza / rigidez; modulo reducido = '
              f'{k_med:.4f} · rigidez / raiz(area).')
    else:
        fallan = ', '.join(f'({kk})' for kk in '123' if not lect[kk])
        print(f'   · No se cumplen en el 99 % de los ensayos: {fallan}.')
    print('   · Ajuste de la descarga evaluado en la profundidad efectiva: ' +
          ('se cumple.' if lect['4'] else 'no se cumple en el 99 % de los ensayos.'))
    if np.all(np.abs(q / 24.5 - 1) <= 0.001):
        print('   · Funcion de area ideal de una Berkovich, 24,5 · hc^2, sin terminos de calibracion.')
    elif lect['5']:
        print(f'   · Funcion de area calibrada, de la forma de Oliver y Pharr y termino principal '
              f'{J.C0.median():.2f}, fija dentro de cada jornada' +
              (', distinta entre jornadas (un unico ajuste no la reproduce).' if eg > 0.001 else
               '; un unico ajuste la reproduce en todo el conjunto.'))
    else:
        print('   · La funcion de area no queda reproducida a menos del 0,1 % en todas las jornadas.')


_d3_resumen(M, globals().get('hcur'))
del _d3_resumen

  139 peliculas · 8896 indentaciones · descarte 27.7 % · semilla 20260914

A · CAMBIOS DE ETIQUETA ENTRE LAS DOS PASADAS (poblacion del analisis)
  curvas del analisis: 8896 · en las dos tablas: 2514 · solo en la 1a: 91 · solo en la 2a: 191 · en ninguna: 6100
  comprobacion del supuesto: objetivo de la 2a pasada reconstruido frente al del analisis -> 0 discrepancias (supuesto confirmado)

                                     categoria   altas   bajas   suben   bajan
                                  force_offset    1261       7       0       0
                           displacement_offset       9      21       0      14
                   unusual_unloading_curvature      93       3      15      30
                     unusual_loading_curvature      74       6       0      22
         tip_displacement_decreases_at_holding     105       4       0      11
                                      too_deep      17       0       0       0

  eventos sumando categorias: 1692 · curvas unicas con


  --- hardness_H_GPa ---   gasto del experto: 28.1 % · gasto de la regla MAD: 3.1 %
                         regla      gasto   mediana      p25      p75   % > 0.5    % < 0
                       mediana      libre      0.09    -1.27     0.65       35%      47%
          media recortada 10 %      libre      0.25    -0.53     0.55       28%      38%
            quitar |z-MAD| > 3      libre      0.00    -0.27     0.56       29%      32%
            quitar k mayor |z|   igualado     -0.13    -2.14     0.67       35%      53%
            quitar k cola alta   igualado     -2.50    -7.50    -1.31        1%      91%
            quitar k cola baja   igualado     -3.13    -6.23    -1.64        3%      90%
       quitar k al azar (nulo)   igualado     -0.03    -0.25    -0.00        0%      76%

  --- reduced_modulus_Er_GPa ---   gasto del experto: 28.1 % · gasto de la regla MAD: 1.6 %
                         regla      gasto   mediana      p25      p75   % > 0.5    % < 0
                     


  --- hardness_H_GPa ---
     estratos                % fuera MEDIA             % fuera DISPERSION
                   experto     falso           experto     falso
            1        35.3%      8.2%             57.6%     12.0%
            4        20.9%      9.6%             42.4%     10.8%
            8        15.1%     10.3%             23.7%     11.8%
           12        11.5%     11.3%             16.5%     10.3%

  --- reduced_modulus_Er_GPa ---
     estratos                % fuera MEDIA             % fuera DISPERSION
                   experto     falso           experto     falso
            1        66.2%     10.8%             28.1%     10.1%
            4        24.5%     10.8%             23.7%     11.3%
            8        16.5%      8.9%             20.1%      8.6%
           12        15.1%     10.1%             13.7%     11.3%

D · LAS 20 PAREJAS DE REPLICAS, Y LA DUDA DE LA PROFUNDIDAD
  parejas de replicas: 20 · reparto por campaña:
    campaña 614: 20
  -> TODAS d


  (a3) profundidad maxima leida de 8896 curvas (0 sin leer) en 8 s
       control: hc del resumen por encima de la profundidad maxima de su curva en 0 de 8896 ensayos (no descartados: 0 de 6433)
       distancia del hmax del resumen al ensayo mas proximo de su pelicula: mediana 32.1 nm · coincide a 0,01 nm en 1 de 139 peliculas · a menos del 1 % en 29
       ensayo mas proximo, en las que coinciden a menos del 1 % (indent_num: peliculas): 44: 2 · 6: 2 · 22: 2 · 4: 2 · 19: 2 · 51: 1 · 33: 1 · 7: 1
       hmax del resumen / maximo de la pelicula: mediana 1.297 (p5 0.500, p95 4.337) · / mediana de la pelicula: mediana 1.792

  (b) mediana por pelicula de la profundidad maxima de la curva (no descartados): min 56.9 · p5 64.2 · mediana 78.4 · p95 98.1 · max 108.5 nm
      peliculas con la mediana entre 30 y 100 nm: 132 de 139 (95.0 %)
      ensayos no descartados  entre 30 y 100 nm:  92.0 % · por debajo:  0.0 % · por encima:  8.0 %
      ensayos descartados     entre 30 y 100 nm:  87.7 % ·

**Cómo leer la salida**

En el bloque A, una curva puede llevar más de un tipo, y por eso los desgloses por tipo suman más que el total. En el bloque B, la fila del azar es la referencia de las reglas con presupuesto igualado: cada regla igualada se juzga frente a ella, no frente a cero, y la columna «gasto» indica si la regla decide por sí misma cuántos ensayos quita (libre) o quita los mismos que el especialista (igualado). En el bloque C, si la columna «falso» se queda en torno al 10 % o por debajo en todos los niveles, el contraste está bien calibrado y la columna «experto» se lee directamente frente al 10 %; si sube al afinar los estratos, el contraste no vale a ese nivel. En el bloque D, un valor de profundidad que se repite en muchos ensayos indica que la columna no es una medida independiente por ensayo (saturada, cuantizada o con un valor por película) y no puede usarse como predictor sin explicarlo.

## Fase 5. Variables de la forma de la curva

Objetivo O5; memoria, apartado 4.3.5. La celda 14 lee las 9.600 curvas, calcula las variables de cada zona y las expresa en relación con su película. Es la más lenta junto con la 16.

### Celda 14. Extracción de las variables de la curva (objetivo O5)

Lee una sola vez las 9.600 curvas, incluidas las de las 11 películas completamente anómalas, que se filtran después, y construye una tabla con una fila por ensayo. No entrena ningún modelo ni usa las marcas para decidir nada. La posición X-Y y el número de orden no entran en la tabla como variables. La solución, la densidad de corriente y la campaña no entran como variables en ningún modelo: la campaña es el pliegue de validación y la solución va unida a ella.

- Fase 0. Abre unas pocas curvas, muestra su contenido y comprueba que existen las columnas calibradas de profundidad, fuerza y tiempo. Si faltan, se detiene sin extraer nada. Compara además esos canales con los de voltaje crudo, que podrían conservar el cero absoluto que la calibración borra.
- Fase 1. Calcula variables de la geometría de la carga, del sostenimiento (puntos con fuerza de al menos el 98 % del máximo) y de la descarga, organizadas según los tipos de defecto del especialista, y 40 puntos de la curva remuestreada (`crudo_*`), reservados para el análisis secundario. El modelo A usa solo los valores calculados por el laboratorio y el modelo B añade estas variables. Por eso, de la descarga se toma el desajuste del ajuste de potencia y no su exponente, que ya está en el resumen.
- Fase 1-bis. Decide si los parámetros del ajuste entre voltaje y canal calibrado podrían entrar en el modelo B, y guarda la decisión en `decision_voltaje.csv`. Solo podrían entrar si la mediana del R² del ajuste con dos canales supera 0,999 en profundidad y en carga (C1) y si el corte no correlaciona con el nivel de voltaje de la curva, con |r| < 0,5 (C2). Si más del 50 % de la varianza de una variable está entre campañas (C3), su versión cruda no entra en ningún modelo. La línea base previa al contacto no describe la forma de la curva. Por diseño del contraste queda fuera de A y de B y entra en el modelo C exploratorio, junto a la posición y el orden, con independencia de su AUC. Orden y posición se consideran no separables si la correlación mediana del orden con X o con Y dentro de la película es de al menos 0,9.
- Fase 2. Normaliza cada variable dentro de su película con la mediana y la desviación absoluta mediana, que son robustas frente a casi el 28 % de ensayos marcados.
- Fase 3. Revisa las variables constantes, vacías y con huecos, y calcula el AUC de cada variable de forma normalizada solo como diagnóstico. No se elimina ninguna variable por su AUC.

Guarda `variables_curva.csv` (variables crudas, variables normalizadas `z_*` y marca `_y` del especialista) y `decision_voltaje.csv`. Si la tabla ya existe con todas las filas, se lee de nuevo con precisión exacta y no se releen las curvas. `RECALCULAR = True` fuerza la extracción.

In [15]:
TABLA = '/content/variables_curva.csv'
DECISION = '/content/decision_voltaje.csv'
RECALCULAR = False
NRES = 20

sep('FASE 0 · ¿QUE HAY DENTRO DE UN FICHERO DE CURVA?')
if 'path' in ALL.columns:
    rutas = [os.path.join(RAIZ, str(p)) for p in ALL['path'].astype(str)]
else:
    rutas = sorted(glob.glob(os.path.join(RAIZ, '**', '*nanoindentation_*.csv'), recursive=True))
    rutas = [p for p in rutas if 'summary' not in os.path.basename(p)]
    print('  aviso: no hay columna `path`, se usa glob')
existen = [p for p in rutas[:50] if os.path.exists(p)]
print(f'  rutas declaradas: {len(rutas)} · de las 50 primeras existen {len(existen)}')
if not existen:
    print('  no se encuentra ningun fichero de curva; primeros ficheros del arbol:')
    for p in sorted(glob.glob(os.path.join(RAIZ, '**', '*.csv'), recursive=True))[:10]:
        print('    ', p.replace(RAIZ, '…'))
    raise SystemExit

ej = pd.read_csv(existen[0])
print(f'\n  ejemplo: …{existen[0][-60:]}')
print(f'  filas={len(ej)} · columnas={list(ej.columns)}')
print(f'  tipos: {dict(ej.dtypes.astype(str))}')
print('\n  primeras 3 filas:'); print(ej.head(3).to_string(index=False))
print('\n  ultimas 3 filas:');  print(ej.tail(3).to_string(index=False))

H, P, T = 'depth_nm', 'load_micro_N', 'time_s'
HV, PV = 'depth_V', 'load_V'
falta = [c for c in (H, P, T) if c not in ej.columns]
if falta:
    print(f'\n  faltan las columnas {falta}; columnas del fichero: {list(ej.columns)}')
    raise SystemExit
HAY_V = HV in ej.columns and PV in ej.columns
print(f'\n  ---> calibrados: profundidad="{H}"  carga="{P}"  tiempo="{T}"')
print(f'       voltaje crudo: {"si, " + HV + " y " + PV if HAY_V else "no disponible"}')

print('\n  PERFIL DE 3 CURVAS (20 posiciones equiespaciadas)')
for ruta in existen[:3]:
    c = pd.read_csv(ruta)
    h, p = c[H].to_numpy(float), c[P].to_numpy(float)
    idx = np.linspace(0, len(h) - 1, 20).astype(int)
    pm = np.nanmax(p); im = int(np.nanargmax(p))
    print(f'\n    …{ruta[-46:]}   n={len(h)}  Pmax={pm:.4g} en i={im} ({100*im/len(h):.0f} %)')
    print('      i    : ' + ' '.join(f'{i:>6}' for i in idx))
    print('      h(nm): ' + ' '.join(f'{h[i]:>6.1f}' for i in idx))
    print('      P(uN): ' + ' '.join(f'{p[i]:>6.1f}' for i in idx))
    mes = p >= 0.98 * pm
    j = np.flatnonzero(mes)
    print(f'      meseta >=98% Pmax: {len(j)} puntos ({100*len(j)/len(h):.1f} %)'
          + (f', de i={j[0]} a i={j[-1]}' if len(j) else ''))

if HAY_V:
    print('\n  ¿CONTIENE EL VOLTAJE ALGO QUE EL CANAL CALIBRADO NO TENGA?')
    print(f'\n    {"curva":>6} {"R2 prof":>9} {"pend prof":>11} {"corte prof":>12}'
          f' {"R2 carga":>10} {"pend carga":>11} {"corte carga":>12}')
    cortes_h, cortes_p = [], []
    for k, ruta in enumerate(existen[:8]):
        c = pd.read_csv(ruta)
        fila = [k]
        for a, b in ((H, HV), (P, PV)):
            x, y = c[b].to_numpy(float), c[a].to_numpy(float)
            ok = np.isfinite(x) & np.isfinite(y)
            m_, q_ = np.polyfit(x[ok], y[ok], 1)
            r2 = 1 - np.sum((y[ok] - (m_ * x[ok] + q_)) ** 2) / np.sum((y[ok] - y[ok].mean()) ** 2)
            fila += [r2, m_, q_]
            (cortes_h if a == H else cortes_p).append(q_)
        print(f'    {fila[0]:>6} {fila[1]:>9.6f} {fila[2]:>11.4g} {fila[3]:>12.5g}'
              f' {fila[4]:>10.6f} {fila[5]:>11.4g} {fila[6]:>12.5g}')
    ch, cp = np.array(cortes_h), np.array(cortes_p)
    print(f'\n    variacion del corte entre curvas -> profundidad: '
          f'{np.std(ch)/max(abs(np.mean(ch)),1e-12):.3%} del valor medio  ·  carga: '
          f'{np.std(cp)/max(abs(np.mean(cp)),1e-12):.3%}')

sep('FASE 1 · EXTRACCION DE VARIABLES DE FORMA')


def suaviza(y, w=5):
    if len(y) < w:
        return y.copy()
    k = np.ones(w) / w
    return np.convolve(np.pad(y, (w // 2, w // 2), mode='edge'), k, mode='valid')[:len(y)]


def ajuste_potencia(h, p, hf):
    x = h - hf
    ok = (x > 1e-9) & (p > 1e-12)
    if ok.sum() < 5:
        return np.nan, np.nan, np.nan, np.nan
    lx, ly = np.log(x[ok]), np.log(p[ok])
    m, b = np.polyfit(lx, ly, 1)
    pred = np.exp(b) * x[ok] ** m
    ss = np.sum((ly - ly.mean()) ** 2)
    r2 = 1 - np.sum((ly - (b + m * lx)) ** 2) / ss if ss > 0 else np.nan
    return m, b, r2, float(np.sqrt(np.mean((p[ok] - pred) ** 2)))


def variables(h, p, t, hv=None, pv=None):
    f = {}
    n = len(h)
    f['n_pts'] = n
    fin = np.isfinite(h) & np.isfinite(p)
    f['frac_no_finita'] = 1 - fin.mean()
    if fin.sum() < 30:
        return f
    h, p = h[fin], p[fin]
    t = t[fin] if t is not None else None
    pmax = float(np.nanmax(p)); imax = int(np.nanargmax(p))
    hmax = float(np.nanmax(h))
    if not np.isfinite(pmax) or pmax <= 0 or hmax <= 0:
        return f
    f['p_max'] = pmax; f['h_max'] = hmax
    f['t_total'] = float(t[-1] - t[0]) if t is not None else np.nan

    f['p_ini_frac'] = float(p[0] / pmax)
    f['p_ini_p5_frac'] = float(np.median(p[:max(3, n // 50)]) / pmax)
    f['h_ini_frac'] = float(h[0] / hmax)
    f['h_ini_p5_frac'] = float(np.median(h[:max(3, n // 50)]) / hmax)
    f['p_fin_frac'] = float(p[-1] / pmax)
    f['h_fin_frac'] = float(h[-1] / hmax)

    meseta = p >= 0.98 * pmax
    idx = np.flatnonzero(meseta)
    i0, i1 = (int(idx[0]), int(idx[-1])) if len(idx) else (imax, imax)
    car = slice(0, max(i0, 5)); sos = slice(i0, i1 + 1); des = slice(i1, n)
    f['frac_carga'] = (i0) / n; f['frac_sost'] = (i1 - i0 + 1) / n; f['frac_desc'] = (n - i1) / n

    d = np.diff(np.column_stack([h, p]), axis=0)
    f['frac_congelada'] = float(np.mean(np.all(np.abs(d) < 1e-12, axis=1)))
    f['ruido_rel'] = float(np.sqrt(np.mean((p - suaviza(p)) ** 2)) / pmax)

    hc, pc = h[car], p[car]
    if len(hc) > 10:
        m, b, r2, rr = ajuste_potencia(hc, pc, 0.0)
        f['carga_exp'] = m; f['carga_r2'] = r2; f['carga_resid_rel'] = rr / pmax if rr == rr else np.nan
        f['carga_no_monot'] = float(np.mean(np.diff(pc) < 0))
        ps = suaviza(pc)
        with np.errstate(all='ignore'):
            g2 = np.gradient(np.gradient(ps, hc), hc)
        f['carga_curv_med'] = float(np.nanmedian(g2)) if np.isfinite(g2).any() else np.nan
        f['carga_curv_iqr'] = float(np.nanpercentile(g2, 75) - np.nanpercentile(g2, 25)) \
            if np.isfinite(g2).any() else np.nan

    hs = h[sos]
    if len(hs) > 3:
        f['sost_fluencia_rel'] = float((hs[-1] - hs[0]) / hmax)
        f['sost_retroceso_rel'] = float((np.min(hs) - hs[0]) / hmax)
        f['sost_rango_rel'] = float((np.max(hs) - np.min(hs)) / hmax)
        f['sost_pendiente_neg'] = float(np.mean(np.diff(hs) < 0))
        f['sost_n'] = len(hs)

    hu, pu = h[des], p[des]
    if len(hu) > 10:
        alto = pu >= 0.5 * pmax
        hh, pp = hu[alto], pu[alto]
        if len(hh) > 8:
            mejor = (np.nan,) * 4; mejor_hf = np.nan; mejor_r2 = -np.inf
            for hf in np.linspace(0, 0.95 * float(np.min(hh)), 30):
                m, b, r2, rr = ajuste_potencia(hh, pp, hf)
                if r2 == r2 and r2 > mejor_r2:
                    mejor_r2 = r2; mejor = (m, b, r2, rr); mejor_hf = hf
            m, b, r2, rr = mejor
            f['desc_hf_rel'] = mejor_hf / hmax if mejor_hf == mejor_hf else np.nan
            f['desc_r2'] = r2
            f['desc_resid_rel'] = rr / pmax if rr == rr else np.nan
            x = hu - (mejor_hf if mejor_hf == mejor_hf else 0)
            ok = (x > 1e-9) & np.isfinite(x)
            if ok.sum() > 5 and m == m:
                pred = np.exp(b) * x[ok] ** m
                f['desc_resid_bajo_rel'] = float(np.sqrt(np.mean((pu[ok] - pred) ** 2)) / pmax)
                f['desc_sesgo_bajo_rel'] = float(np.mean(pu[ok] - pred) / pmax)
            if m == m and mejor_hf == mejor_hf:
                S_fit = np.exp(b) * m * max(hu[0] - mejor_hf, 1e-9) ** (m - 1)
                k = min(10, len(hu) - 1)
                S_loc = abs((pu[0] - pu[k]) / (hu[0] - hu[k])) if abs(hu[0] - hu[k]) > 1e-9 else np.nan
                f['rigidez_fit'] = float(S_fit)
                f['rigidez_razon'] = float(S_fit / S_loc) if S_loc and S_loc == S_loc else np.nan
        f['desc_no_monot'] = float(np.mean(np.diff(pu) > 0))
        ps = suaviza(pu)
        with np.errstate(all='ignore'):
            g2 = np.gradient(np.gradient(ps, hu), hu)
        f['desc_curv_med'] = float(np.nanmedian(g2)) if np.isfinite(g2).any() else np.nan

    if hv is not None and pv is not None:
        hv, pv = hv[fin], pv[fin]
        ok = np.isfinite(hv) & np.isfinite(pv)
        if ok.sum() > 30:
            HVv, PVv = hv[ok], pv[ok]
            for nom, y, x1, x2 in (('h', h[ok], HVv, PVv), ('p', p[ok], PVv, HVv)):
                ss = float(np.sum((y - y.mean()) ** 2))
                if ss <= 0:
                    continue
                m1, b1 = np.polyfit(x1, y, 1)
                f[f'v_{nom}_r2'] = float(1 - np.sum((y - (m1 * x1 + b1)) ** 2) / ss)
                f[f'v_{nom}_pend'] = float(m1); f[f'v_{nom}_corte'] = float(b1)
                X = np.column_stack([x1, x2, np.ones(len(y))])
                beta, *_ = np.linalg.lstsq(X, y, rcond=None)
                f[f'v_{nom}_r2_2c'] = float(1 - np.sum((y - X @ beta) ** 2) / ss)
                f[f'v_{nom}_corte_2c'] = float(beta[2])
                ry = float(y.max() - y.min())
                if ry > 0:
                    f[f'v_{nom}_peso1'] = float(abs(beta[0]) * (x1.max() - x1.min()) / ry)
                    f[f'v_{nom}_peso2'] = float(abs(beta[1]) * (x2.max() - x2.min()) / ry)
            f['v_h_nivel'] = float(np.median(HVv)); f['v_h_rango'] = float(HVv.max() - HVv.min())
            f['v_p_nivel'] = float(np.median(PVv)); f['v_p_rango'] = float(PVv.max() - PVv.min())
        pre = np.flatnonzero(p > 0.05 * pmax)
        j = int(pre[0]) if len(pre) else max(5, n // 20)
        j = max(5, min(j, n // 4))
        f['v_n_precontacto'] = j
        for nom, vol in (('h', hv), ('p', pv)):
            seg = vol[:j]; seg = seg[np.isfinite(seg)]
            if len(seg) > 4:
                f[f'v_{nom}_base'] = float(np.median(seg))
                f[f'v_{nom}_ruido'] = float(np.std(seg))
                f[f'v_{nom}_deriva'] = float(np.polyfit(np.arange(len(seg)), seg, 1)[0])

    try:
        gc = np.linspace(0, 1, NRES)
        pc_n = np.interp(gc, (h[car] - h[car][0]) / max(h[car][-1] - h[car][0], 1e-9),
                         p[car] / pmax) if len(hc) > 10 else np.full(NRES, np.nan)
        ordu = np.argsort(hu)
        pu_n = np.interp(gc, (hu[ordu] - hu[ordu][0]) / max(hu[ordu][-1] - hu[ordu][0], 1e-9),
                         pu[ordu] / pmax) if len(hu) > 10 else np.full(NRES, np.nan)
        for i in range(NRES):
            f[f'crudo_car_{i:02d}'] = float(pc_n[i]); f[f'crudo_des_{i:02d}'] = float(pu_n[i])
    except Exception:
        pass
    return f


CACHE = (not RECALCULAR) and os.path.exists(TABLA)
if CACHE:
    _T = pd.read_csv(TABLA, float_precision='round_trip')
    CACHE = len(_T) == len(ALL)
    if CACHE:
        print(f'  tabla leida de {TABLA} ({len(_T)} filas); no se releen las curvas')
        F = _T[[c for c in _T.columns if not c.startswith('z_') and c != '_y']].copy()

t0 = time.time(); filas = []; fallos = 0
if not CACHE:
    usecols = [c for c in (H, P, T) if c] + ([HV, PV] if HAY_V else [])
    for i, r in enumerate(ALL.itertuples(index=False)):
        ruta = os.path.join(RAIZ, str(getattr(r, 'path')))
        base = dict(sample_num=r.sample_num, indent_num=r.indent_num)
        try:
            c = pd.read_csv(ruta, usecols=usecols)
            base.update(variables(c[H].to_numpy(float), c[P].to_numpy(float),
                                  c[T].to_numpy(float) if T else None,
                                  c[HV].to_numpy(float) if HAY_V else None,
                                  c[PV].to_numpy(float) if HAY_V else None))
        except Exception as e:
            fallos += 1
            base['error'] = str(e)[:60]
        filas.append(base)
        if (i + 1) % 1000 == 0:
            print(f'    {i+1:>5} / {len(ALL)} curvas · {time.time()-t0:.0f} s · fallos={fallos}')
    F = pd.DataFrame(filas)
    print(f'  extraidas {len(F)} curvas en {time.time()-t0:.0f} s · fallos de lectura: {fallos}')

print('\n  reparto de puntos por tramo (mediana entre curvas):')
for c in ('frac_carga', 'frac_sost', 'frac_desc'):
    if c in F.columns:
        v = F[c].dropna()
        print(f'    {c:>12}: mediana {v.median():.3f} · p10 {np.percentile(v,10):.3f} '
              f'· p90 {np.percentile(v,90):.3f}')
if 'sost_n' in F.columns:
    sn = F['sost_n'].dropna()
    print(f'    puntos de sostenimiento: mediana {sn.median():.0f} · '
          f'curvas con menos de 5 puntos de meseta: {int((sn < 5).sum())} de {len(sn)}')
if 'v_h_r2' in F.columns:
    print('\n  ¿son los canales calibrados afines al voltaje, en las 9.600 curvas?')
    for c in ('v_h_r2', 'v_p_r2'):
        v = F[c].dropna()
        print(f'    {c:>8}: mediana {v.median():.6f} · minimo {v.min():.6f}')
    for c in ('v_h_corte', 'v_p_corte'):
        v = F[c].dropna()
        print(f'    {c:>10}: desviacion entre curvas = '
              f'{np.std(v)/max(abs(np.mean(v)),1e-12):.4%} del valor medio')

sep('FASE 1-bis · ¿ENTRAN LOS CANALES DE VOLTAJE EN LOS MODELOS?')
COLS_V = [c for c in F.columns if c.startswith('v_')]
USAR_V = False
if not COLS_V:
    print('  no hay canales de voltaje en los ficheros')
else:
    print('\n  1) ¿basta un canal, o hacen falta dos?')
    print(f"    {'canal':>14} {'R2 con 1':>12} {'R2 con 2':>12} {'mejora':>10} "
          f"{'peso 2o canal':>15}")
    for nom, et in (('h', 'profundidad'), ('p', 'carga')):
        a, b = f'v_{nom}_r2', f'v_{nom}_r2_2c'
        if a not in F.columns:
            continue
        va, vb = F[a].dropna(), F[b].dropna()
        w = F.get(f'v_{nom}_peso2', pd.Series(dtype=float)).dropna()
        print(f'    {et:>14} {va.median():>12.6f} {vb.median():>12.6f} '
              f'{(vb.median()-va.median()):>+10.6f} '
              f'{(w.median() if len(w) else np.nan):>14.2%}')
    print('\n  2) ¿es el corte un cero real o el residuo de extrapolar hasta V = 0?')
    print(f"    {'variable':>14} {'corr con el nivel de V':>24}")
    corr = {}
    for nom in ('h', 'p'):
        for suf in ('corte', 'corte_2c'):
            c, nv = f'v_{nom}_{suf}', f'v_{nom}_nivel'
            if c in F.columns and nv in F.columns:
                d_ = F[[c, nv]].dropna()
                corr[c] = d_[c].corr(d_[nv]) if len(d_) > 50 else np.nan
                print(f'    {c:>14} {corr[c]:>24.3f}')
    print('\n  3) ¿donde vive su variacion?')
    _cols = CLAVE + ['batch_num'] + [c for c in ('X_mm', 'Y_mm') if c in ALL.columns]
    BB = F.merge(ALL[_cols], on=CLAVE, how='left')
    print(f"    {'variable':>14} {'% var ENTRE peliculas':>23} {'% var ENTRE camp.':>19}")
    entre_lote = {}
    for c in ('v_h_corte', 'v_p_corte', 'v_h_corte_2c', 'v_p_corte_2c', 'v_h_base', 'v_p_base'):
        if c not in BB.columns:
            continue
        v = BB[c].astype(float); tot = float(v.var())
        if not np.isfinite(tot) or tot <= 0:
            continue
        f1 = 100 * float(BB.groupby('sample_num')[c].mean().var()) / tot
        f2 = 100 * float(BB.groupby('batch_num')[c].mean().var()) / tot
        entre_lote[c] = f2
        print(f'    {c:>14} {f1:>22.1f}% {f2:>18.1f}%')

    BXY = BB.merge(S[CLAVE + ['X_mm', 'Y_mm']], on=CLAVE, how='left')
    conf = {'x': [], 'y': []}
    for s_, d_ in BXY.groupby('sample_num'):
        for k, col in (('x', 'X_mm'), ('y', 'Y_mm')):
            if col in d_.columns and d_[col].std() > 0:
                conf[k].append(abs(d_['indent_num'].corr(d_[col])))
    cx = np.nanmedian(conf['x']) if conf['x'] else np.nan
    cy = np.nanmedian(conf['y']) if conf['y'] else np.nan
    no_sep = max([v for v in (cx, cy) if v == v], default=0) >= 0.9
    print('\n  4) la linea base de voltaje, ¿que sigue? (correlacion DENTRO de cada pelicula)')
    print(f'    posicion leida de los resumenes: {int(BXY.X_mm.notna().sum())}'
          f' ensayos · correlacion del orden con X: {cx:.2f} · con Y: {cy:.2f}')
    print(f"    {'variable':>12} {'con el ORDEN':>14} {'con X':>9} {'con Y':>9}   lectura")
    for c in ('v_h_base', 'v_p_base'):
        if c not in BXY.columns:
            continue
        rs = {k: [] for k in ('ord', 'x', 'y')}
        for s_, d_ in BXY.groupby('sample_num'):
            if len(d_) < 10 or d_[c].std() == 0:
                continue
            rs['ord'].append(d_[c].corr(d_['indent_num']))
            for k, col in (('x', 'X_mm'), ('y', 'Y_mm')):
                if col in d_.columns and d_[col].std() > 0:
                    rs[k].append(d_[c].corr(d_[col]))
        m = {k: (np.nanmedian(np.abs(v)) if v else np.nan) for k, v in rs.items()}
        pos = max([v for v in (m['x'], m['y']) if v == v], default=0)
        if no_sep:
            cual = 'no separable'
        else:
            cual = 'deriva de sesion' if m['ord'] > pos else 'topografia'
        print(f'    {c:>12} {m["ord"]:>14.2f} {m["x"]:>9.2f} {m["y"]:>9.2f}   {cual}')

    r2ok = all(F.get(f'v_{n}_r2_2c', pd.Series([0.0])).median() > 0.999 for n in ('h', 'p'))
    corte_ok = all(abs(v) < 0.5 for v in corr.values() if v == v)
    lote_ok = all(v < 50 for v in entre_lote.values())
    USAR_V = bool(r2ok and corte_ok)
    sep('VEREDICTO SOBRE EL VOLTAJE')
    print(f'  C1 relacion entendida (R2 > 0,999 con dos canales) : {"SI" if r2ok else "NO"}')
    print(f'  C2 el corte no es artefacto de extrapolacion       : {"SI" if corte_ok else "NO"}')
    print(f'  C3 su varianza no es identidad de campaña          : {"SI" if lote_ok else "NO"}')
    print('\n  parametros del ajuste del voltaje: '
          + ('dentro del modelo B, normalizados dentro de pelicula' if USAR_V else 'fuera de los tres modelos'))
    if not lote_ok:
        print('  linea base del voltaje: mas del 50 % de la varianza entre campañas en parte de sus variables;'
              ' en el modelo C solo entra normalizada dentro de pelicula')

pd.DataFrame([{'voltaje_ajuste_en_B': int(USAR_V), 'voltaje_base_en_B': 0,
               'voltaje_base_en_C': 1}]).to_csv(DECISION, index=False)
print(f'\n  guardado: {DECISION}')

sep('FASE 2 · NORMALIZACION DENTRO DE PELICULA (mediana y MAD)')
num = [c for c in F.columns if c not in ('sample_num', 'indent_num', 'error')
       and pd.api.types.is_numeric_dtype(F[c])]
G = F.groupby('sample_num')[num]
med = G.transform('median')
mad = G.transform(lambda x: np.median(np.abs(x - np.median(x))))
Z = (F[num] - med) / (1.4826 * mad.replace(0, np.nan))
Z = Z.replace([np.inf, -np.inf], np.nan).fillna(0.0)
Z.columns = [f'z_{c}' for c in num]
TAB = pd.concat([F[CLAVE], F[num], Z], axis=1)
print(f'  tabla final: {TAB.shape[0]} filas x {TAB.shape[1]} columnas '
      f'({len(num)} variables crudas + {len(num)} normalizadas)')

sep('FASE 3 · SALUD DE LAS VARIABLES (diagnostico, NO seleccion)')
cte = [c for c in num if F[c].nunique(dropna=True) <= 1]
vac = [c for c in num if F[c].notna().sum() == 0]
print(f'  variables constantes: {len(cte)}  {cte[:8]}')
print(f'  variables vacias    : {len(vac)}  {vac[:8]}')
frac_vacia = F[num].isna().mean().sort_values(ascending=False)
print('\n  variables con mas huecos:')
for c, v in frac_vacia.head(8).items():
    print(f'    {c:>26}: {100*v:5.1f} % vacia')

y = np.zeros(len(ALL), int)
for c in POR_INDENT:
    y = np.maximum(y, (pd.to_numeric(ALL[c], errors='coerce').fillna(0) >= 2).astype(int))
key = ALL[CLAVE].copy(); key['_y'] = y
TAB = TAB.merge(key, on=CLAVE, how='left')


def auc(x, yy):
    ok = np.isfinite(x)
    x, yy = x[ok], yy[ok]
    if yy.sum() < 5 or (1 - yy).sum() < 5:
        return np.nan
    r = pd.Series(x).rank().to_numpy()
    n1 = yy.sum(); n0 = len(yy) - n1
    return (r[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


print('\n  AUC univariado de las variables NORMALIZADAS (0,5 = no separa nada)')
res = []
for c in [f'z_{v}' for v in num if not v.startswith('crudo_')]:
    if c in TAB.columns:
        a = auc(TAB[c].to_numpy(float), TAB['_y'].to_numpy(int))
        if a == a:
            res.append((c, a, abs(a - .5)))
res.sort(key=lambda r: -r[2])
print(f"\n    {'variable':>30} {'AUC':>7}      {'variable':>30} {'AUC':>7}")
for i in range(0, min(20, len(res)), 2):
    a = res[i]; b = res[i + 1] if i + 1 < len(res) else ('', np.nan, 0)
    print(f'    {a[0]:>30} {a[1]:>7.3f}      {b[0]:>30} '
          f'{b[1]:>7.3f}' if b[0] else f'    {a[0]:>30} {a[1]:>7.3f}')
print(f'\n  variables de forma con |AUC-0,5| < 0,02 (no separan nada por si solas): '
      f'{sum(1 for r in res if r[2] < .02)} de {len(res)}')

TAB.to_csv(TABLA, index=False)
print(f'\n  guardado: {TABLA}  ({os.path.getsize(TABLA)/1e6:.1f} MB)')


FASE 0 · ¿QUE HAY DENTRO DE UN FICHERO DE CURVA?
  rutas declaradas: 9600 · de las 50 primeras existen 50

  ejemplo: …NiFe0_nostir/cd_1/sample_2658/609_2658_nanoindentation_0.csv
  filas=400 · columnas=['depth_nm', 'load_micro_N', 'time_s', 'depth_V', 'load_V']
  tipos: {'depth_nm': 'float64', 'load_micro_N': 'float64', 'time_s': 'float64', 'depth_V': 'float64', 'load_V': 'float64'}

  primeras 3 filas:
 depth_nm  load_micro_N   time_s  depth_V   load_V
 0.310378      1.098369 0.000000 0.022301 0.264639
-0.265487      1.672855 0.000998 0.022424 0.266212
 0.000000      2.589424 0.001997 0.022367 0.270605

  ultimas 3 filas:
 depth_nm  load_micro_N   time_s  depth_V   load_V
33.583210     70.398487 0.396365 0.015188 0.509508
33.147474     61.858052 0.397363 0.015282 0.489298
32.817549     53.538762 0.398362 0.015354 0.468861

  ---> calibrados: profundidad="depth_nm"  carga="load_micro_N"  tiempo="time_s"
       voltaje crudo: si, depth_V y load_V

  PERFIL DE 3 CURVAS (20 posiciones e

  tabla leida de /content/variables_curva.csv (9600 filas); no se releen las curvas

  reparto de puntos por tramo (mediana entre curvas):
      frac_carga: mediana 0.247 · p10 0.247 · p90 0.247
       frac_sost: mediana 0.512 · p10 0.512 · p90 0.512
       frac_desc: mediana 0.242 · p10 0.242 · p90 0.242
    puntos de sostenimiento: mediana 205 · curvas con menos de 5 puntos de meseta: 0 de 9599

  ¿son los canales calibrados afines al voltaje, en las 9.600 curvas?
      v_h_r2: mediana 0.999943 · minimo 0.997982
      v_p_r2: mediana 0.977260 · minimo 0.748713
     v_h_corte: desviacion entre curvas = 203.1261% del valor medio
     v_p_corte: desviacion entre curvas = 43.3912% del valor medio

FASE 1-bis · ¿ENTRAN LOS CANALES DE VOLTAJE EN LOS MODELOS?

  1) ¿basta un canal, o hacen falta dos?
             canal     R2 con 1     R2 con 2     mejora   peso 2o canal
       profundidad     0.999943     0.999996  +0.000053          1.19%
             carga     0.977260     0.979117  +0.0

        v_h_base           0.42      0.61      0.46   no separable
        v_p_base           0.39      0.60      0.43   no separable

VEREDICTO SOBRE EL VOLTAJE
  C1 relacion entendida (R2 > 0,999 con dos canales) : NO
  C2 el corte no es artefacto de extrapolacion       : NO
  C3 su varianza no es identidad de campaña          : SI

  parametros del ajuste del voltaje: fuera de los tres modelos

  guardado: /content/decision_voltaje.csv

FASE 2 · NORMALIZACION DENTRO DE PELICULA (mediana y MAD)


  tabla final: 9600 filas x 204 columnas (101 variables crudas + 101 normalizadas)

FASE 3 · SALUD DE LAS VARIABLES (diagnostico, NO seleccion)
  variables constantes: 4  ['n_pts', 'frac_no_finita', 't_total', 'frac_congelada']
  variables vacias    : 0  []

  variables con mas huecos:
                   rigidez_fit:   0.1 % vacia
                desc_resid_rel:   0.1 % vacia
           desc_sesgo_bajo_rel:   0.1 % vacia
           desc_resid_bajo_rel:   0.1 % vacia
                 rigidez_razon:   0.1 % vacia
                carga_curv_med:   0.0 % vacia
                carga_no_monot:   0.0 % vacia
                         h_max:   0.0 % vacia

  AUC univariado de las variables NORMALIZADAS (0,5 = no separa nada)

                          variable     AUC                            variable     AUC
                     z_desc_hf_rel   0.221                        z_h_fin_frac   0.234
                   z_p_ini_p5_frac   0.750                        z_p_ini_frac   0.735
            


  guardado: /content/variables_curva.csv  (34.0 MB)


**Cómo leer la salida**

En la comparación entre voltaje y canal calibrado, un corte casi igual en todas las curvas indicaría que el voltaje es redundante, y un corte que cambia de curva en curva, que conserva una referencia absoluta. Si los parámetros del ajuste quedan fuera de los modelos, la hipótesis de que el voltaje conserva el cero que la calibración borra no se sostiene con estos datos.

La malla se recorre por filas. El orden coincide con Y y, dentro de cada fila, la posición en X coincide con el orden dentro de la fila. Cuando la lectura es «no separable», con estos datos no se distingue una deriva del instrumento de una variación de la superficie.

En la fase 3, una variable con |AUC-0,5| < 0,02 no separa nada por sí sola.

## Fases 6 a 9. Modelos, interpretación, solidez y alcance

Objetivos O6 a O9; memoria, apartados 4.3.6 y 4.4. La celda 15 entrena y evalúa los modelos dejando una campaña fuera. Las celdas 16 y 17 necesitan sus objetos en la misma sesión: la 16 interpreta el modelo B y comprueba la solidez del resultado, y la 17 solo imprime las cifras de diagnóstico que usa el apartado 4.4.

### Celda 15. Los modelos

Objetivo O6 (memoria, apartados 4.3.6 y 4.4.1 a 4.4.3). Se amplía `M` con las variables de la curva de la celda 14 (`variables_curva.csv`) y se entrenan cuatro modelos que predicen, ensayo a ensayo, la marca del especialista. La respuesta a la pregunta del trabajo es la diferencia entre A y B. A recibe los valores calculados por el laboratorio (todas las columnas numéricas del fichero resumen salvo identificadores y metadatos, más `h_max` y `p_max`), normalizados dentro de cada película, sin los que quedan constantes; B, los de A más las variables de forma de la curva; C, los de B más variables del voltaje (línea base, ruido, deriva, nivel y rango), la posición X-Y y el orden (exploratorio); D, los de A más los 40 puntos de la curva remuestreada.

Criterios fijados antes de ejecutar:

- Partición dejando una campaña fuera (7 pliegues), nunca por película ni al azar. Hiperparámetros fijos (`HP`), sin ajuste. Umbral (D3): el que iguala, en las campañas de entrenamiento, la tasa de descarte predicha con la observada; se aplica sin cambios a la campaña evaluada.
- Métrica principal (D4): recuperación por película, 1 − |X − E| / |A − E|, con A la media sin limpiar, E la del especialista y X la del modelo. Solo cuentan las películas evaluables (D5): aquellas en que |A − E| alcanza el error típico del promedio multiplicado por √(1 + 3,5 ρ), con ρ la mediana entre películas de la correlación de rangos entre ensayos vecinos de la malla, calculada para cada propiedad. Si un método deja menos de 3 ensayos en una película, se conservan los 3 de menor probabilidad de descarte, o de menor puntuación en la regla del bloque 5 (D6). La dureza es la propiedad principal; el módulo reducido se evalúa con los mismos descartes, como resultado secundario (D7).
- Refutación (memoria, apartado 4.2.6): F1, B supera una recuperación mediana de 0,25 en al menos 5 de las 7 campañas; F2, B supera a A en al menos 5 de las 7; F3, el resultado no depende solo de la campaña 614.
- Complementos: regla de forma sin aprendizaje con dos magnitudes de la curva, la mayor de las dos y el umbral D3 (bloque 5, D8; R1 si B no la supera en 0,05 o más, R2 si queda 0,05 o más por debajo y B la supera en al menos 5 campañas, R3 en otro caso); error absoluto frente al especialista en las 139 películas (bloque 6, D9); intervalos del 95 % por remuestreo de películas dentro de cada campaña, 2.000 veces (bloque 7, D10); exhaustividad por tipo de defecto con umbral común y con el de cada pliegue, con una diferencia de referencia de 0,05 (bloques 4 y 4-bis, D11); mejor regla estadística en las películas evaluables, la mediana de los 64 ensayos y la media sin |z-MAD| > 3 (bloque 8, D12); veces que se aplica D6 (bloque 9).

Guarda `resultado_modelos.csv`, `resultado_modelos_reduced_modulus_Er_GPa.csv`, `resultado_regla_por_lote.csv`, `error_absoluto_por_pelicula.csv` y `mejor_regla_por_lote.csv`. Deja para las celdas siguientes, entre otros, `M` (ampliada), `X`, `JUEGOS`, `lotes`, `HP`, `recupera`, `B_`, `T`, `T9`, `TR`, `res`, `pred_all`, `UMB`, `EA`, `EA9`, `GL_IV`, `PT_IV`, `IV9` y `FAC_ESP`.

In [16]:
HP = dict(max_iter=200, learning_rate=0.05, max_depth=4, min_samples_leaf=50,
          l2_regularization=1.0, early_stopping=False, random_state=SEED)
NO_VAL = {'sample_num', 'indent_num', 'num_points', 'X_mm', 'Y_mm', 'drift_nm/s'}

V = pd.read_csv(TABLA)
M = M.merge(V.drop(columns='_y'), on=CLAVE, how='left', suffixes=('', '_v')).set_axis(M.index)
print(f'  {M.sample_num.nunique()} peliculas · {len(M)} indentaciones · '
      f'{100*M._y.mean():.1f} % marcadas · {M.batch_num.nunique()} campañas')


def zrob(df, cols):
    g = df.groupby('sample_num')[cols]
    med = g.transform('median')
    mad = g.transform(lambda x: np.median(np.abs(x - np.median(x))))
    z = (df[cols] - med) / (1.4826 * mad.replace(0, np.nan))
    return z.replace([np.inf, -np.inf], np.nan)


val_lab = [c for c in S.columns if c not in NO_VAL
           and pd.api.types.is_numeric_dtype(S[c]) and S[c].nunique(dropna=True) > 1]
print(f'\n  modelo A recibe {len(val_lab)} columnas del fichero resumen: {val_lab}')
Zval = zrob(M, val_lab); Zval.columns = [f'a_{c}' for c in val_lab]
for c in ('h_max', 'p_max'):
    if c in M.columns:
        Zval[f'a_{c}'] = zrob(M, [c])[c]
zz = [c for c in V.columns if c.startswith('z_')]
es_v = lambda c: c.startswith('z_v_')
es_crudo = lambda c: c.startswith('z_crudo_')
FORMA = [c for c in zz if not es_v(c) and not es_crudo(c)]
VOLT_BASE = [c for c in zz if es_v(c) and any(k in c for k in ('_base', '_ruido', '_deriva',
                                                               '_nivel', '_rango'))]
CRUDO = [c for c in zz if es_crudo(c)]
EXTRA_C = [c for c in ('X_mm', 'Y_mm', 'indent_num') if c in M.columns]
X = pd.concat([Zval, M[[c for c in zz if c in M.columns]], M[EXTRA_C]], axis=1)
X = X.loc[:, ~X.columns.duplicated()]
JUEGOS = {
    'A · solo valores derivados': list(Zval.columns),
    'B · valores + forma de la curva': list(Zval.columns) + [c for c in FORMA if c in X.columns],
    'C · B + voltaje base, posicion y orden': list(Zval.columns)
        + [c for c in FORMA if c in X.columns] + [c for c in VOLT_BASE if c in X.columns] + EXTRA_C,
    'D · valores + curva cruda remuestreada': list(Zval.columns) + [c for c in CRUDO if c in X.columns],
}
for k in JUEGOS:
    JUEGOS[k] = [c for c in JUEGOS[k] if c in X.columns and X[c].nunique(dropna=True) > 1]
print('\n  juegos de variables:')
for k, v in JUEGOS.items():
    print(f'    {k:>40}: {len(v)} variables')

PROP_PRIN, PROP_SEC = PROPS
PROPS_EV = [p for p in (PROP_PRIN, PROP_SEC) if p in M.columns]
RHO_ESP = {p_: rho_vecinos(M, p_) for p_ in PROPS_EV}
FAC_ESP = {p_: factor_vecinos(RHO_ESP[p_]) for p_ in PROPS_EV}
BASE = {}
for p_ in PROPS_EV:
    bp = {}
    for s_, d_ in M.groupby('sample_num'):
        v = d_[p_].astype(float).values; q = (d_._y == 1).values
        if not np.isfinite(v).all() or q.sum() < 1 or len(v) - q.sum() < 3:
            continue
        A_, E_ = v.mean(), v[~q].mean()
        t = int(0.10 * len(v))
        bp[s_] = dict(A=A_, E=E_, den=abs(A_ - E_),
                      sem=float(np.std(v, ddof=1) / np.sqrt(len(v))),
                      recorte=np.sort(v)[t:len(v) - t].mean(),
                      batch=d_.batch_num.iloc[0], n=len(v))
    BASE[p_] = bp
base_pel = BASE[PROP_PRIN]
B_ = pd.DataFrame(base_pel).T.apply(pd.to_numeric, errors='coerce')
B_['rec_base'] = 1 - (B_['recorte'] - B_['E']).abs() / B_['den']
B_['fiable'] = B_['den'] >= B_['sem'] * FAC_ESP[PROP_PRIN]
n_fiable = int(B_['fiable'].sum())
BTAB = {}
for p_ in PROPS_EV:
    b = pd.DataFrame(BASE[p_]).T.apply(pd.to_numeric, errors='coerce')
    b['rec_base'] = 1 - (b['recorte'] - b['E']).abs() / b['den']
    b['fiable'] = b['den'] >= b['sem'] * FAC_ESP[p_]
    BTAB[p_] = b
    print(f'  {p_:>24}: {len(b)} peliculas con metrica · '
          f'{int(b["fiable"].sum())} con correccion por encima del ruido (D5) · correlacion entre '
          f'vecinos {RHO_ESP[p_].median():.3f} · factor {FAC_ESP[p_]:.3f}')

D6_ACT = []


def informa_d6(regs_):
    if not regs_:
        print('  ninguna: ningun metodo deja menos de 3 ensayos en ninguna pelicula.')
        return
    d6_ = pd.DataFrame(regs_, columns=['contexto', 'pelicula', 'propiedad'])
    t6_ = d6_.groupby(['contexto', 'propiedad']).pelicula.agg(veces='size', peliculas='nunique')
    print(t6_.to_string())
    print(f'\n  en total: {len(d6_)} aplicaciones en {d6_.pelicula.nunique()} peliculas distintas')


def recupera(sub, pred, punt, prop_=None, ctx=''):
    prop_ = prop_ or PROP_PRIN
    base_pel = BASE[prop_]
    out = {}
    for s_, idx in sub.groupby('sample_num').groups.items():
        if s_ not in base_pel:
            continue
        d_ = sub.loc[idx]
        v = d_[prop_].astype(float).values
        p_ = pred[sub.index.get_indexer(idx)]
        quita = p_ == 1
        if (~quita).sum() < 3:
            pu_ = np.asarray(punt, dtype=float)[sub.index.get_indexer(idx)]
            orden = np.argsort(pu_, kind='stable')
            quita = np.ones(len(p_), bool); quita[orden[:3]] = False
            D6_ACT.append((ctx, s_, prop_))
        b = base_pel[s_]
        out[s_] = 1 - abs(v[~quita].mean() - b['E']) / b['den'] if b['den'] > 0 else np.nan
    return pd.Series(out)


sep('VALIDACION DEJANDO UNA CAMPAÑA FUERA')
lotes = sorted(M.batch_num.unique())
MULT = [0.5, 0.75, 1.0, 1.25, 1.5]
res = {k: {} for k in JUEGOS}
res_sec = {}
det = {k: {} for k in JUEGOS}
barr = {}
pred_all = {k: pd.Series(index=M.index, dtype=float) for k in JUEGOS}
UMB = {}
for lote in lotes:
    te = M.batch_num == lote
    for nom, cols in JUEGOS.items():
        Xtr, ytr = X.loc[~te, cols], M.loc[~te, '_y']
        Xte, yte = X.loc[te, cols], M.loc[te, '_y']
        mdl = HistGradientBoostingClassifier(**HP).fit(Xtr, ytr)
        ptr, pte = mdl.predict_proba(Xtr)[:, 1], mdl.predict_proba(Xte)[:, 1]
        umbral = np.quantile(ptr, 1 - ytr.mean())
        UMB.setdefault(nom, {})[lote] = float(umbral)
        pred = (pte >= umbral).astype(int)
        pred_all[nom].loc[te] = pte
        ctx_ = nom.split(' · ')[0]
        r = recupera(M.loc[te], pred, pte, ctx=f'{ctx_} · umbral declarado')
        res[nom][lote] = r
        for p_ in PROPS_EV[1:]:
            res_sec.setdefault((nom, p_), {})[lote] = recupera(M.loc[te], pred, pte, prop_=p_,
                                                               ctx=f'{ctx_} · umbral declarado')
        for mult in MULT:
            u2 = np.quantile(ptr, 1 - min(0.99, ytr.mean() * mult))
            barr.setdefault((nom, mult), {})[lote] = \
                recupera(M.loc[te], (pte >= u2).astype(int), pte, ctx=f'{ctx_} · umbral x{mult}')
        det[nom][lote] = dict(pr_auc=average_precision_score(yte, pte),
                              auc=roc_auc_score(yte, pte) if yte.nunique() > 1 else np.nan,
                              n=int(te.sum()), pos=float(yte.mean()))
    print(f'    campaña {lote}: {int(te.sum())} indentaciones · '
          f'{100*M.loc[te,"_y"].mean():.1f} % marcadas')

sep('1 · RECUPERACION DE LA CORRECCION, POR CAMPAÑA  (metrica principal)')
print(f"   {'campaña':>7} {'peliculas':>10} {'LINEA BASE':>11} " +
      ' '.join(f'{k.split(" · ")[0]:>8}' for k in JUEGOS))
tab = {}
for lote in lotes:
    pel = B_[(B_['batch'] == lote) & B_['fiable']].index
    lb = B_.loc[pel, 'rec_base'].median()
    fila = [lb]
    for nom in JUEGOS:
        r = res[nom][lote]
        fila.append(r.reindex(pel).median())
    tab[lote] = fila
    print(f'    {int(lote):>6} {len(pel):>10} {lb:>11.3f} ' +
          ' '.join(f'{v:>8.3f}' for v in fila[1:]))
T = pd.DataFrame(tab, index=['base'] + list(JUEGOS)).T
print(f"\n {'GLOBAL (mediana entre campañas)':>31} {T['base'].median():>11.3f} " +
      ' '.join(f'{T[k].median():>8.3f}' for k in JUEGOS))

sep('1-bis · ¿ES EL RESULTADO UNA ELECCION AFORTUNADA DE UMBRAL?')
print('  recuperacion mediana entre campañas, variando la tasa de descarte del modelo')
print(f"    {'modelo':>40} " + ' '.join(f'x{m:<5}' for m in MULT))
for nom in JUEGOS:
    fila = []
    for m_ in MULT:
        vals = []
        for lote in lotes:
            pel = B_[(B_['batch'] == lote) & B_['fiable']].index
            vals.append(barr[(nom, m_)][lote].reindex(pel).median())
        fila.append(np.median(vals))
    print(f'    {nom:>40} ' + ' '.join(f'{v:>6.3f}' for v in fila))

sep('1-ter · ¿PARA CUANTAS PELICULAS IMPORTA ESTO SIQUIERA?')
print(f'  peliculas totales                                        : {len(B_)}')
print(f'  con correccion MAYOR que el error tipico del promedio (D5) : {n_fiable}')
print(f'  con correccion MENOR que ese error (metrica sin sentido)  : {len(B_)-n_fiable}'
      f'  ({100*(len(B_)-n_fiable)/len(B_):.0f} %)')

sep('2 · LOS TRES CRITERIOS DE FALSACION')
gana_base = {k: int((T[k] > T['base']).sum()) for k in JUEGOS}
bA = 'A · solo valores derivados'; bB = 'B · valores + forma de la curva'
gana_A = int((T[bB] > T[bA]).sum())
F1_UMBRAL = 0.25
gana_025 = {k: int((T[k] > F1_UMBRAL).sum()) for k in JUEGOS}
print(f'  F1 · ¿supera cada modelo una recuperacion mediana de {F1_UMBRAL} en cada campaña?')
print(f"      {'campaña':>7} " + ' '.join(f'{k.split(" · ")[0]:>8}' for k in JUEGOS))
for lote in lotes:
    print(f'       {int(lote):>6} ' + ' '.join(f'{T.loc[lote, k]:>8.3f}' for k in JUEGOS))
for k in JUEGOS:
    print(f'       {k:>40}: supera {F1_UMBRAL} en {gana_025[k]} de {len(lotes)} campañas')
print(f'\n  F1 complementaria · ¿bate cada modelo a la linea base de su propia campaña?')
for k in JUEGOS:
    print(f'       {k:>40}: {gana_base[k]} de {len(lotes)} campañas')
print(f'\n  F2 · ¿bate B al modelo A? {gana_A} de {len(lotes)} campañas '
      f'(diferencia mediana {T[bB].median()-T[bA].median():+.3f})')
lote614 = 614 if 614 in T.index else T.index[int(np.argmax([(B_['batch'] == l).sum() for l in T.index]))]
resto = [l for l in T.index if l != lote614]
print(f'\n  F3 · campaña mayor ({int(lote614)}): B = {T.loc[lote614, bB]:.3f} · '
      f'resto de campañas: mediana {T.loc[resto, bB].median():.3f}')
ok1 = gana_025[bB] >= 5; ok1c = gana_base[bB] >= 5; ok2 = gana_A >= 5
ok3 = T.loc[resto, bB].median() > T.loc[resto, 'base'].median()
print(f'\n  F1 (B supera 0,25 en >=5 campañas)  : {"SE CUMPLE" if ok1 else "NO SE CUMPLE"}')
print(f'  F1 complementaria (B bate la linea base de su campaña en >=5): '
      f'{"se cumple" if ok1c else "no se cumple"}')
print(f'  F2 (B bate a A en >=5 campañas)     : {"SE CUMPLE" if ok2 else "NO SE CUMPLE"}')
print(f'  F3 (no vive en una sola campaña)   : {"SE CUMPLE" if ok3 else "NO SE CUMPLE"}')

sep('2-bis · LA OTRA PROPIEDAD DEL LABORATORIO (secundario, fuera de los criterios)')
for p_ in PROPS_EV[1:]:
    b = BTAB[p_]
    print(f'  {p_} · con las MISMAS indentaciones descartadas por cada modelo')
    print(f"   {'campaña':>7} {'peliculas':>10} {'LINEA BASE':>11} " +
          ' '.join(f'{k.split(" · ")[0]:>8}' for k in JUEGOS))
    filas_s = {}
    for lote in lotes:
        pel = b[(b['batch'] == lote) & b['fiable']].index
        lb = b.loc[pel, 'rec_base'].median()
        fila = [lb] + [res_sec[(nom, p_)][lote].reindex(pel).median() for nom in JUEGOS]
        filas_s[lote] = fila
        print(f'    {int(lote):>6} {len(pel):>10} {lb:>11.3f} ' +
              ' '.join(f'{v:>8.3f}' for v in fila[1:]))
    TS = pd.DataFrame(filas_s, index=['base'] + list(JUEGOS)).T
    print(f"    {'GLOBAL':>6} {'':>10} {TS['base'].median():>11.3f} " +
          ' '.join(f'{TS[k].median():>8.3f}' for k in JUEGOS))
    gb = int((TS[bB] > TS['base']).sum()); ga = int((TS[bB] > TS[bA]).sum())
    print(f'\n    B bate a la linea base en {gb} de {len(lotes)} campañas · '
          f'B bate a A en {ga} de {len(lotes)}')
    TS.to_csv(f'/content/resultado_modelos_{p_}.csv')

sep('3 · DETECCION PURA (secundario): ¿reproduce la etiqueta?')
print(f"    {'modelo':>40} {'PR-AUC medio':>14} {'AUC medio':>11}   [tasa de positivos "
      f"{100*M._y.mean():.1f} %]")
for nom in JUEGOS:
    pr = np.mean([det[nom][l]['pr_auc'] for l in lotes])
    au = np.nanmean([det[nom][l]['auc'] for l in lotes])
    print(f'    {nom:>40} {pr:>14.3f} {au:>11.3f}')

sep('4 · ¿DE QUE TIPO DE ANOMALIA VIENE LA GANANCIA DE B SOBRE A?')
print(f"    {'tipo':>42} {'n':>6} " + ' '.join(f'{k.split(" · ")[0]:>8}' for k in JUEGOS))
for c in POR_INDENT:
    m_ = (pd.to_numeric(M[c], errors='coerce').fillna(0) >= 2).values
    if m_.sum() < 20:
        continue
    fila = []
    for nom in JUEGOS:
        p_ = pred_all[nom].values
        u = np.nanquantile(p_, 1 - M._y.mean())
        fila.append(np.mean(p_[m_] >= u))
    print(f'    {c:>42} {int(m_.sum()):>6} ' + ' '.join(f'{v:>8.2f}' for v in fila))

sep('4-bis · LA MISMA TABLA CON EL UMBRAL DE CADA PLIEGUE (el que aplica el modelo, D3)')
print(f"    {'tipo':>42} {'n':>6} " + ' '.join(f'{k_.split(" · ")[0]:>8}' for k_ in JUEGOS))
DIF4 = {}
for cc_ in POR_INDENT:
    mm_ = (pd.to_numeric(M[cc_], errors='coerce').fillna(0) >= 2).values
    if mm_.sum() < 20:
        continue
    fila_ = []
    for nm_ in JUEGOS:
        pv_ = pred_all[nm_].values
        uu_ = M.batch_num.map(UMB[nm_]).values.astype(float)
        f4b_ = float(np.mean(pv_[mm_] >= uu_[mm_]))
        f4_ = float(np.mean(pv_[mm_] >= np.nanquantile(pv_, 1 - M._y.mean())))
        DIF4[(cc_, nm_)] = f4b_ - f4_
        fila_.append(f4b_)
    print(f'    {cc_:>42} {int(mm_.sum()):>6} ' + ' '.join(f'{v_:>8.2f}' for v_ in fila_))
dmax_ = max([abs(v_) for (cc_, nm_), v_ in DIF4.items() if nm_ in (bA, bB)], default=0.0)
print(f'\n  mayor diferencia frente al bloque 4, en A o en B: {dmax_:.3f}')
if dmax_ > 0.05:
    print('  LECTURA: el umbral de cada pliegue cambia la tabla por tipo.')
else:
    print('  LECTURA: el umbral de cada pliegue no cambia la tabla por tipo.')

pd.concat([T, pd.Series({k: gana_base[k] for k in JUEGOS}, name='lotes_gana_base')], axis=1) \
  .to_csv('/content/resultado_modelos.csv')
print('\n  guardado: /content/resultado_modelos.csv')

sep('5 · REGLA DE FORMA SIN APRENDIZAJE: DOS MAGNITUDES DE LA CURVA (D8)')
RG_VARS = ['z_p_ini_p5_frac', 'z_desc_resid_bajo_rel']
RES_RG, RES_RG_SEC, TR = {}, {}, None
falta_ = [v_ for v_ in RG_VARS if v_ not in M.columns]
if falta_:
    print(f'  faltan en la tabla de la celda 14: {falta_}; se omiten los bloques 5 a 7.')
else:
    SC_RG = M[RG_VARS].astype(float).fillna(0.0).max(axis=1).values
    for lt_ in lotes:
        te_ = (M.batch_num == lt_).values
        u_ = np.quantile(SC_RG[~te_], 1 - M.loc[~te_, '_y'].mean())
        pr_ = (SC_RG[te_] >= u_).astype(int)
        RES_RG[lt_] = recupera(M.loc[te_], pr_, SC_RG[te_], ctx='regla D8')
        for ps_ in PROPS_EV[1:]:
            RES_RG_SEC[(ps_, lt_)] = recupera(M.loc[te_], pr_, SC_RG[te_], prop_=ps_, ctx='regla D8')
    print(f'  variables: {RG_VARS} · puntuacion = la mayor de las dos')
    print(f"\n   {'campaña':>7} {'peliculas':>10} {'LINEA BASE':>11} {'A':>8} {'B':>8} {'REGLA':>8}")
    filas_ = {}
    for lt_ in lotes:
        pel_ = B_[(B_['batch'] == lt_) & B_['fiable']].index
        filas_[lt_] = dict(n=len(pel_), base=T.loc[lt_, 'base'], A=T.loc[lt_, bA],
                           B=T.loc[lt_, bB], regla=RES_RG[lt_].reindex(pel_).median())
        r_ = filas_[lt_]
        print(f"    {int(lt_):>6} {r_['n']:>10} {r_['base']:>11.3f} {r_['A']:>8.3f} "
              f"{r_['B']:>8.3f} {r_['regla']:>8.3f}")
    TR = pd.DataFrame(filas_).T
    g_ = TR[['base', 'A', 'B', 'regla']].median()
    print(f"    {'GLOBAL':>6} {'':>10} {g_['base']:>11.3f} {g_['A']:>8.3f} {g_['B']:>8.3f} "
          f"{g_['regla']:>8.3f}")
    n_BR = int((TR.B > TR.regla).sum()); n_RA = int((TR.regla > TR.A).sum())
    n_Rb = int((TR.regla > TR.base).sum())
    print(f'\n  B supera a la regla en {n_BR} de {len(lotes)} campañas · la regla supera a A en '
          f'{n_RA} · la regla supera a la linea base en {n_Rb}')
    for ps_ in PROPS_EV[1:]:
        bs_ = BTAB[ps_]
        med_rg = np.median([RES_RG_SEC[(ps_, lt_)].reindex(
            bs_[(bs_['batch'] == lt_) & bs_['fiable']].index).median() for lt_ in lotes])
        print(f'  {ps_} (secundario): linea base {TS["base"].median():.3f} · A '
              f'{TS[bA].median():.3f} · B {TS[bB].median():.3f} · regla {med_rg:.3f}')
    dif_ = g_['B'] - g_['regla']
    print(f'\n  diferencia B - regla en la mediana entre campañas: {dif_:+.3f}')
    if dif_ < 0.05:
        print('  LECTURA R1: dos magnitudes de la curva bastan para reproducir la correccion.')
    elif n_BR >= 5:
        print('  LECTURA R2: el aprendizaje aporta por encima de las dos magnitudes.')
    else:
        print('  LECTURA R3: resultado intermedio.')

if TR is not None:
    sep('6 · ERROR ABSOLUTO FRENTE AL EXPERTO, EN LAS 139 PELICULAS (D9)')
    EA = pd.DataFrame({'fiable': B_['fiable'].astype(bool), 'den': B_['den'],
                       'sem': B_['sem'], 'E': B_['E']})
    EA['ruido'] = EA['sem'] * FAC_ESP[PROP_PRIN]
    EA['cruda'] = EA['den']
    EA['base'] = (1 - B_['rec_base']) * EA['den']
    for et_, nm_ in (('A', bA), ('B', bB)):
        EA[et_] = (1 - pd.concat([res[nm_][lt_] for lt_ in lotes]).reindex(EA.index)) * EA['den']
    EA['regla'] = (1 - pd.concat([RES_RG[lt_] for lt_ in lotes]).reindex(EA.index)) * EA['den']
    MET_EA = ['cruda', 'base', 'A', 'B', 'regla']
    print('  |X - E| en GPa, mediana por grupo de peliculas (entre parentesis, en % de E)\n')
    print(f"    {'grupo':>26} {'n':>5} " + ' '.join(f'{m_:>15}' for m_ in MET_EA))
    for et_, sel_ in (('por encima del ruido', EA.fiable), ('por debajo del ruido', ~EA.fiable),
                      ('todas', EA.fiable | ~EA.fiable)):
        d_ea = EA[sel_]
        cel_ = [f'{d_ea[m_].median():.3f} ({100*(d_ea[m_]/d_ea.E).median():4.1f}%)' for m_ in MET_EA]
        print(f'    {et_:>26} {len(d_ea):>5} ' + ' '.join(f'{c_:>15}' for c_ in cel_))
    d_ea = EA[~EA.fiable]
    print(f'\n  peliculas por debajo del ruido ({len(d_ea)}): en cuantas cada metodo aleja el '
          f'valor del experto')
    print(f'  mas que el ruido del promedio (error tipico x {FAC_ESP[PROP_PRIN]:.3f}, el mismo de D5)')
    FR_EA = {}
    for m_ in MET_EA:
        n_al = int((d_ea[m_] > d_ea['ruido']).sum())
        FR_EA[m_] = n_al / len(d_ea) if len(d_ea) else np.nan
        print(f'    {m_:>8}: {n_al:>4} de {len(d_ea)} ({100*FR_EA[m_]:.0f} %)')
    if FR_EA['B'] <= FR_EA['base']:
        print('  LECTURA: donde la correccion no importa, B no aleja el valor en mas peliculas que')
        print('  la linea base.')
    else:
        print('  LECTURA: donde la correccion no importa, B aleja el valor en mas peliculas que la')
        print('  linea base.')

    sep('7 · INTERVALOS DE CONFIANZA DE LA MEDIANA ENTRE CAMPAÑAS (D10)')
    rng_iv = np.random.default_rng(SEED + 7)
    NB_IV = 2000
    SER_IV = {'base': B_['rec_base']}
    for nm_ in JUEGOS:
        SER_IV[nm_.split(' · ')[0]] = pd.concat([res[nm_][lt_] for lt_ in lotes])
    SER_IV['regla'] = pd.concat([RES_RG[lt_] for lt_ in lotes])
    MET_IV = list(SER_IV)
    BOOT = {m_: np.zeros((NB_IV, len(lotes))) for m_ in MET_IV}
    for j_, lt_ in enumerate(lotes):
        pel_ = B_[(B_['batch'] == lt_) & B_['fiable']].index
        if len(pel_) == 0:
            for m_ in MET_IV:
                BOOT[m_][:, j_] = np.nan
            continue
        V_ = np.column_stack([SER_IV[m_].reindex(pel_).values for m_ in MET_IV])
        ix_ = rng_iv.integers(0, len(pel_), size=(NB_IV, len(pel_)))
        for i_, m_ in enumerate(MET_IV):
            BOOT[m_][:, j_] = np.nanmedian(V_[:, i_][ix_], axis=1)
    GL_IV = {m_: np.nanmedian(BOOT[m_], axis=1) for m_ in MET_IV}
    PT_IV = {m_: float(np.median([SER_IV[m_].reindex(
        B_[(B_['batch'] == lt_) & B_['fiable']].index).median() for lt_ in lotes])) for m_ in MET_IV}
    print(f"    {'metodo':>16} {'mediana campañas':>20} {'IC 95 %':>20}")
    for m_ in MET_IV:
        lo_, hi_ = np.percentile(GL_IV[m_], [2.5, 97.5])
        print(f'    {m_:>16} {PT_IV[m_]:>20.3f} {f"[{lo_:.3f}, {hi_:.3f}]":>20}')
    for a_, b_ in (('B', 'A'), ('B', 'regla')):
        dd_ = GL_IV[a_] - GL_IV[b_]
        lo_, hi_ = np.percentile(dd_, [2.5, 97.5])
        print(f'    {a_ + " - " + b_:>16} {PT_IV[a_] - PT_IV[b_]:>+20.3f} '
              f'{f"[{lo_:+.3f}, {hi_:+.3f}]":>20}')
    n_ba_ = int((T[bB] > T[bA]).sum()); nl_ = len(lotes)
    p_min = 2 * 0.5 ** nl_
    k_ba = max(n_ba_, nl_ - n_ba_)
    p_obs = min(1.0, 2 * sum(comb(nl_, i_) for i_ in range(k_ba, nl_ + 1)) / 2 ** nl_)
    print(f'\n  B supera a A en {n_ba_} de {nl_} campañas: prueba de signos bilateral p = {p_obs:.4f}.')
    print(f'  Con {nl_} campañas no puede bajar de p = {p_min:.4f}, que solo se alcanza con {nl_} de {nl_}.')

    TR.to_csv('/content/resultado_regla_por_lote.csv')
    EA.to_csv('/content/error_absoluto_por_pelicula.csv')
    print('\n  guardado: /content/resultado_regla_por_lote.csv · /content/error_absoluto_por_pelicula.csv')

sep('8 · LA MEJOR REGLA ESTADISTICA EN LAS PELICULAS EVALUABLES (D12)')
REG_EV = {}
for p_ in PROPS_EV:
    fl_ = {}
    for s_, d_ in M.groupby('sample_num'):
        if s_ not in BASE[p_] or BASE[p_][s_]['den'] <= 0:
            continue
        v = d_[p_].astype(float).values; b = BASE[p_][s_]
        xm_ = float(np.median(v)); xd_ = float(v[z_mad(v) <= 3].mean())
        fl_[s_] = dict(batch=b['batch'], E=b['E'], den=b['den'], sem=b['sem'],
                       rec_mediana=1 - abs(xm_ - b['E']) / b['den'],
                       rec_mad3=1 - abs(xd_ - b['E']) / b['den'])
    df_ = pd.DataFrame(fl_).T.apply(pd.to_numeric, errors='coerce')
    df_['fiable'] = (df_['den'] >= df_['sem'] * FAC_ESP[p_]).astype(bool)
    REG_EV[p_] = df_

R9_ = REG_EV[PROP_PRIN]
T9 = {}
for lt_ in lotes:
    pel_ = R9_[(R9_['batch'] == lt_) & R9_['fiable']].index
    T9[lt_] = dict(n=len(pel_), base=float(T.loc[lt_, 'base']),
                   mediana=float(R9_.loc[pel_, 'rec_mediana'].median()),
                   mad3=float(R9_.loc[pel_, 'rec_mad3'].median()),
                   A=float(T.loc[lt_, bA]), B=float(T.loc[lt_, bB]),
                   regla=float(TR.loc[lt_, 'regla']) if TR is not None else np.nan)
T9 = pd.DataFrame(T9).T
print(f"   {'campaña':>7} {'pelis':>6} {'recortada':>10} {'mediana':>8} {'|z|<=3':>8} {'A':>8} "
      f"{'B':>8} {'regla D8':>9}")
for lt_, r_ in T9.iterrows():
    print(f"    {int(lt_):>6} {int(r_.n):>6} {r_.base:>10.3f} {r_.mediana:>8.3f} {r_.mad3:>8.3f} "
          f"{r_.A:>8.3f} {r_.B:>8.3f} {r_.regla:>9.3f}")
G9 = T9[['base', 'mediana', 'mad3', 'A', 'B', 'regla']].median()
print(f"    {'GLOBAL':>6} {'':>6} {G9['base']:>10.3f} {G9['mediana']:>8.3f} {G9['mad3']:>8.3f} "
      f"{G9['A']:>8.3f} {G9['B']:>8.3f} {G9['regla']:>9.3f}")
for et_, c_ in (('B', 'B'), ('A', 'A'), ('regla D8', 'regla')):
    print(f'  {et_} supera a la mediana en {int((T9[c_] > T9["mediana"]).sum())} de {len(lotes)} campañas '
          f'(diferencia en la mediana de las campañas {G9[c_] - G9["mediana"]:+.3f})')

R9m_ = REG_EV[PROP_SEC]
med_m_ = {lt_: float(R9m_.loc[(R9m_['batch'] == lt_) & R9m_['fiable'], 'rec_mediana'].median())
          for lt_ in lotes}
B_m_ = {lt_: float(res_sec[(bB, PROP_SEC)][lt_].reindex(
    R9m_[(R9m_['batch'] == lt_) & R9m_['fiable']].index).median()) for lt_ in lotes}
print(f'\n  {PROP_SEC} (secundario): mediana de las campañas, regla mediana '
      f'{np.nanmedian(list(med_m_.values())):.3f} · B {np.nanmedian(list(B_m_.values())):.3f} · '
      f'B supera a la mediana en {sum(B_m_[l_] > med_m_[l_] for l_ in lotes)} de {len(lotes)} campañas')

rng9_ = np.random.default_rng(SEED + 9)
NB9_ = 2000
SER9_ = {'mediana': R9_['rec_mediana'], 'mad3': R9_['rec_mad3'],
         'A': pd.concat([res[bA][lt_] for lt_ in lotes]),
         'B': pd.concat([res[bB][lt_] for lt_ in lotes])}
BOOT9_ = {m_: np.zeros((NB9_, len(lotes))) for m_ in SER9_}
for j_, lt_ in enumerate(lotes):
    pel_ = B_[(B_['batch'] == lt_) & B_['fiable']].index
    if len(pel_) == 0:
        for m_ in SER9_:
            BOOT9_[m_][:, j_] = np.nan
        continue
    V9_ = np.column_stack([SER9_[m_].reindex(pel_).values for m_ in SER9_])
    ix_ = rng9_.integers(0, len(pel_), size=(NB9_, len(pel_)))
    for i_, m_ in enumerate(SER9_):
        BOOT9_[m_][:, j_] = np.nanmedian(V9_[:, i_][ix_], axis=1)
GL9_ = {m_: np.nanmedian(BOOT9_[m_], axis=1) for m_ in SER9_}
IV9 = {}
print(f"\n    {'metodo':>16} {'mediana campañas':>20} {'IC 95 %':>20}")
for m_ in ('mediana', 'mad3'):
    lo_, hi_ = np.percentile(GL9_[m_], [2.5, 97.5]); IV9[m_] = (float(G9[m_]), lo_, hi_)
    print(f'    {m_:>16} {G9[m_]:>20.3f} {f"[{lo_:.3f}, {hi_:.3f}]":>20}')
for a_, b_ in (('B', 'mediana'), ('A', 'mediana')):
    dd_ = GL9_[a_] - GL9_[b_]; lo_, hi_ = np.percentile(dd_, [2.5, 97.5])
    IV9[f'{a_}-{b_}'] = (float(G9[a_] - G9[b_]), lo_, hi_)
    print(f'    {a_ + " - " + b_:>16} {G9[a_] - G9[b_]:>+20.3f} {f"[{lo_:+.3f}, {hi_:+.3f}]":>20}')

EA9 = pd.DataFrame({'fiable': R9_['fiable'], 'den': R9_['den'], 'E': R9_['E'],
                    'ruido': R9_['sem'] * FAC_ESP[PROP_PRIN]})
EA9['mediana'] = (1 - R9_['rec_mediana']) * EA9['den']
EA9['mad3'] = (1 - R9_['rec_mad3']) * EA9['den']
print('\n  |X - E| en GPa, mediana por grupo de peliculas (entre parentesis, en % de E)')
for et_, sel_ in (('por encima del ruido', EA9.fiable), ('por debajo del ruido', ~EA9.fiable),
                  ('todas', EA9.fiable | ~EA9.fiable)):
    d9_ = EA9[sel_]
    print(f'    {et_:>22} {len(d9_):>4} · ' + ' · '.join(
        f'{m_} {d9_[m_].median():.3f} ({100*(d9_[m_]/d9_.E).median():.1f} %)' for m_ in ('mediana', 'mad3')))
d9_ = EA9[~EA9.fiable]
for m_ in ('mediana', 'mad3'):
    print(f'  por debajo del ruido: {m_} aleja el valor mas que el ruido en '
          f'{int((d9_[m_] > d9_["ruido"]).sum())} de {len(d9_)} peliculas')
T9.to_csv('/content/mejor_regla_por_lote.csv')
print('\n  guardado: /content/mejor_regla_por_lote.csv')

sep('9 · ¿CUANTAS VECES SE APLICA LA SALVAGUARDA DE TRES ENSAYOS (D6) EN ESTA CELDA?')
informa_d6(D6_ACT)

  139 peliculas · 8896 indentaciones · 27.7 % marcadas · 7 campañas

  modelo A recibe 11 columnas del fichero resumen: ['contact_depth_hc_nm', 'max_load_Pmax_micro_N', 'stiffness_S_micro_N/nm', 'proj_contact_area_A_nm^2', 'max_displacement_hmax_nm', 'effective_depth_heff_nm', 'reduced_modulus_Er_GPa', 'hardness_H_GPa', 'unloading_curve_alpha', 'final_displacement_hf_nm', 'unloading_curve_m']



  juegos de variables:
                  A · solo valores derivados: 12 variables
             B · valores + forma de la curva: 38 variables
      C · B + voltaje base, posicion y orden: 51 variables
      D · valores + curva cruda remuestreada: 52 variables


            hardness_H_GPa: 139 peliculas con metrica · 46 con correccion por encima del ruido (D5) · correlacion entre vecinos 0.151 · factor 1.236
    reduced_modulus_Er_GPa: 139 peliculas con metrica · 63 con correccion por encima del ruido (D5) · correlacion entre vecinos 0.296 · factor 1.426

VALIDACION DEJANDO UNA CAMPAÑA FUERA


    campaña 609: 896 indentaciones · 27.9 % marcadas


    campaña 614: 3072 indentaciones · 26.2 % marcadas


    campaña 615: 896 indentaciones · 33.8 % marcadas


    campaña 619: 704 indentaciones · 25.4 % marcadas


    campaña 622: 1280 indentaciones · 25.9 % marcadas


    campaña 623: 832 indentaciones · 26.3 % marcadas


    campaña 626: 1216 indentaciones · 30.8 % marcadas

1 · RECUPERACION DE LA CORRECCION, POR CAMPAÑA  (metrica principal)
   campaña  peliculas  LINEA BASE        A        B        C        D
       609          5       0.478    0.723    0.932    0.946    0.839
       614         15       0.367    0.622    0.925    0.943    0.858
       615          6      -0.036    0.263    0.695    0.729    0.378
       619          6      -0.007    0.545    0.537    0.743    0.675
       622          4       0.243    0.553    0.611    0.680    0.611
       623          4       0.425    0.874    0.957    0.857    0.789
       626          6       0.310    0.806    0.921    0.916    0.919

 GLOBAL (mediana entre campañas)       0.310    0.622    0.921    0.857    0.789

1-bis · ¿ES EL RESULTADO UNA ELECCION AFORTUNADA DE UMBRAL?
  recuperacion mediana entre campañas, variando la tasa de descarte del modelo
                                      modelo x0.5   x0.75  x1.0   x1.25  x1.5  
               

  variables: ['z_p_ini_p5_frac', 'z_desc_resid_bajo_rel'] · puntuacion = la mayor de las dos

   campaña  peliculas  LINEA BASE        A        B    REGLA
       609          5       0.478    0.723    0.932    0.962
       614         15       0.367    0.622    0.925    0.769
       615          6      -0.036    0.263    0.695    0.170
       619          6      -0.007    0.545    0.537    0.089
       622          4       0.243    0.553    0.611    0.609
       623          4       0.425    0.874    0.957    0.918
       626          6       0.310    0.806    0.921    0.937
    GLOBAL                  0.310    0.622    0.921    0.769

  B supera a la regla en 5 de 7 campañas · la regla supera a A en 5 · la regla supera a la linea base en 7
  reduced_modulus_Er_GPa (secundario): linea base -0.032 · A 0.637 · B 0.910 · regla 0.666

  diferencia B - regla en la mediana entre campañas: +0.152
  LECTURA R2: el aprendizaje aporta por encima de las dos magnitudes.

6 · ERROR ABSOLUTO FRENTE 

**Cómo leer la salida**

- Bloque 1-bis: si la ventaja de B sobre A se mantiene en toda la fila, no depende del umbral; si solo aparece con la tasa de descarte de D3, el resultado es frágil.
- Bloque 1-ter: en las películas no evaluables, la limpieza del especialista mueve el valor de la película menos que el ruido del propio promedio. Ahí la métrica no tiene sentido, y por eso las tablas de recuperación se calculan solo en las evaluables, que son el alcance real del trabajo.
- Bloque 2-bis: el módulo reducido no entra en los criterios F1 a F3, fijados sobre la dureza. Informa de lo que hace cada modelo con la otra propiedad del laboratorio, con los mismos descartes.
- Bloques 4 y 4-bis: exhaustividad por tipo de defecto (fracción de los ensayos marcados de ese tipo que el modelo descarta) en la campaña que queda fuera del entrenamiento en cada pliegue. El bloque 4 usa un umbral común, el cuantil de todas las probabilidades fuera de pliegue a la tasa global de descarte; el 4-bis, el umbral de cada pliegue (D3). Si toda la ventaja de B sobre A estuviera en un solo tipo, la conclusión sería más estrecha que «la forma de la curva aporta información».
- Bloque 6: en el recuento de películas no evaluables, la media sin limpiar da 0 por definición.
- Bloque 7: el intervalo recoge la variación entre películas dentro de cada campaña; las campañas no se remuestrean.
- Bloque 8: lectura descriptiva, que no cambia F1 a F3. Los intervalos de A, B y la media recortada son los del bloque 7.

### Celda 16. Interpretación del modelo B y comprobaciones de solidez

Objetivos O7 y O8 (memoria, apartados 4.4.4 y 4.4.5). Usa los objetos de la celda 15 (`M`, `X`, `JUEGOS`, `lotes`, `HP`, `recupera`, `B_` y otros), con la misma partición, el mismo umbral D3 y la misma salvaguarda D6, y al final imprime cuántas veces se aplica esta. No se elimina ninguna variable por su importancia: es interpretación, no selección. Ninguna comprobación sustituye al resultado principal de la celda 15.

- A1 y A2 · Importancia por permutación medida en la campaña retenida de cada pliegue (caída de la precisión media, PR-AUC), también agrupada por familia de variable; valores SHAP sobre 600 ensayos de la campaña retenida más grande, con un explicador genérico si `TreeExplainer` falla.
- B · Sensibilidad al nivel de gravedad (≥ 1, ≥ 2 y ≥ 3), con el criterio D5; el principal sigue siendo ≥ 2.
- C · Modelo reducido: las variables de A más las dos de la regla del bloque 5. Q1 si su mediana queda a menos de 0,05 de la de B; Q2 si queda 0,05 o más por debajo. Aprender con esas dos magnitudes aporta por sí mismo si el reducido supera a la regla en 0,05 o más.
- D · Ensayos de marca estable entre las dos revisiones (mismo objetivo binario en ambas, con el nivel ≥ 2 de `verify_zero_offset_type` de la primera). Se reentrenan A y B solo con ellos; el resultado no depende de la segunda revisión si la mediana de B cambia menos de 0,05 y B supera a A en al menos 5 de las 7 campañas.
- E · Base del factor por vecinos de D5: correlación entre vecinos por película, factor y películas evaluables sin él. Solo informa.
- F · Estabilidad numérica: 10 repeticiones con un ruido relativo de 5e-16 sobre las entradas de A y B (la tabla de la celda 14, leída sin pérdida), normalizadas de nuevo. Estable si B supera a A en al menos 5 de las 7 campañas en al menos 9 de las 10 repeticiones.
- G-H · Umbral de evaluabilidad: sin factor por vecinos y con dos veces el error típico multiplicado por ese factor, con las mismas predicciones. Se mantiene si B supera a A en la mediana y en al menos 5/7 de las campañas con películas evaluables.
- I · Reentrenar sin cada familia de variables de forma. Una zona aporta información que las demás no sustituyen si su retirada baja la PR-AUC media en 0,02 o más.
- J · Campañas 614 y 626 (misma solución): cada una se evalúa entrenando sin la otra y frente a un grupo de comparación del mismo tamaño, que conserva la campaña hermana y quita al azar, de las otras cinco campañas, tantas películas como tiene la hermana (20 repeticiones). Si B cae por debajo del mínimo del grupo, es un límite del alcance en esa campaña.

Guarda `importancia.csv`, `shap_valores.npz`, `sensibilidad_nivel.csv`, `estabilidad_numerica.csv`, `robustez_umbral_evaluable.csv`, `reentrenar_sin_zona.csv` y `lotes_misma_solucion.csv`; la celda 18 lee `importancia.csv`, `shap_valores.npz`, `reentrenar_sin_zona.csv` y `estabilidad_numerica.csv`. Deja `ex` y `MED_RED`, que usa la celda 17, e `imp`, que la celda 18 usa si no encuentra `importancia.csv`.

In [17]:
D6_N0 = len(D6_ACT)
SALIDA_IMP = '/content/importancia.csv'
SALIDA_SHAP = '/content/shap_valores.npz'
SALIDA_SENS = '/content/sensibilidad_nivel.csv'

sep('A1 · IMPORTANCIA POR PERMUTACION, MEDIDA EN LA CAMPAÑA RETENIDA')
cols = JUEGOS[bB]
imp = pd.DataFrame(0.0, index=cols, columns=lotes)
for lote in lotes:
    te = M.batch_num == lote
    mdl = HistGradientBoostingClassifier(**HP).fit(X.loc[~te, cols], M.loc[~te, '_y'])
    r = permutation_importance(mdl, X.loc[te, cols], M.loc[te, '_y'], n_repeats=5,
                               random_state=SEED, scoring='average_precision', n_jobs=-1)
    imp[lote] = r.importances_mean
imp['media'] = imp[lotes].mean(axis=1)
imp['min_lote'] = imp[lotes].min(axis=1)
imp = imp.sort_values('media', ascending=False)
print(f"    {'variable':>28} {'caida media de PR-AUC':>23} {'peor campaña':>13}")
for c, r in imp.head(18).iterrows():
    print(f'    {c:>28} {r["media"]:>23.4f} {r["min_lote"]:>13.4f}')

FAM = {
    'valor del laboratorio': lambda c: c.startswith('a_'),
    'arranque de la curva (offset)': lambda c: any(k in c for k in ('p_ini', 'h_ini')),
    'final de la curva': lambda c: any(k in c for k in ('p_fin', 'h_fin')),
    'tramo de carga': lambda c: '_carga' in c or c.startswith('z_carga'),
    'sostenimiento': lambda c: 'sost' in c,
    'tramo de descarga': lambda c: 'desc' in c or 'rigidez' in c,
    'calidad / ruido': lambda c: any(k in c for k in ('ruido', 'congel', 'no_finita', 'n_pts')),
    'reparto de tramos': lambda c: 'frac_' in c and 'ini' not in c and 'fin' not in c,
}


def familia(c):
    for k, f in FAM.items():
        if f(c):
            return k
    return 'otras'


imp['familia'] = [familia(c) for c in imp.index]
g = imp.groupby('familia')['media'].agg(['sum', 'max', 'size']).sort_values('sum', ascending=False)
print(f"\n  agrupado por familia de variable:")
print(f"    {'familia':>32} {'caida total':>12} {'mejor variable':>15} {'n':>4}")
for k, r in g.iterrows():
    print(f'    {k:>32} {r["sum"]:>12.4f} {r["max"]:>15.4f} {int(r["size"]):>4}')
imp.to_csv(SALIDA_IMP)
print(f'\n  guardado: {SALIDA_IMP}')

sep('A2 · VALORES SHAP (direccion, no solo peso)')
try:
    te = M.batch_num == lotes[np.argmax([(M.batch_num == l).sum() for l in lotes])]
    mdl = HistGradientBoostingClassifier(**HP).fit(X.loc[~te, cols], M.loc[~te, '_y'])
    idx = X.loc[te].sample(min(600, int(te.sum())), random_state=SEED).index
    Xs = X.loc[idx, cols].fillna(X.loc[~te, cols].median())
    try:
        ex = shap.TreeExplainer(mdl)
        sv = ex.shap_values(Xs)
    except Exception:
        bg = shap.sample(X.loc[~te, cols].fillna(X.loc[~te, cols].median()), 80,
                         random_state=SEED)
        ex = shap.Explainer(lambda z: mdl.predict_proba(z)[:, 1], bg)
        sv = ex(Xs, max_evals=2 * len(cols) + 1).values
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[..., -1]
    np.savez(SALIDA_SHAP, valores=sv, datos=Xs.to_numpy(float),
             columnas=np.array(cols, dtype=object),
             y=M.loc[idx, '_y'].to_numpy(int))
    med = pd.Series(np.abs(sv).mean(axis=0), index=cols).sort_values(ascending=False)
    print(f'  calculado sobre {len(Xs)} indentaciones de la campaña retenida mas grande.')
    print(f"\n    {'variable':>28} {'|SHAP| medio':>14} {'empuja hacia':>14}")
    for c in med.head(12).index:
        j = cols.index(c)
        r = np.corrcoef(Xs[c].to_numpy(float), sv[:, j])[0, 1]
        d = 'anomala si sube' if r > 0 else 'anomala si baja'
        print(f'    {c:>28} {med[c]:>14.4f} {d:>14}')
    print(f'\n  guardado: {SALIDA_SHAP}')
except Exception as e:
    print(f'  no se pudo calcular SHAP: {str(e)[:120]}')

sep('B · ¿AGUANTA EL RESULTADO SI CAMBIA EL NIVEL DE SEVERIDAD?')
filas = []
POR_CAMP_NIV = {}
for niv in (1, 2, 3):
    yy = np.zeros(len(M), int)
    for c in POR_INDENT:
        yy = np.maximum(yy, np.asarray(
            (pd.to_numeric(M[c], errors='coerce').fillna(0) >= niv).astype(int)))
    yy = np.asarray(yy).astype(int)
    bp = {}
    for s_, d_ in M.groupby('sample_num'):
        v = d_['hardness_H_GPa'].astype(float).values
        q = yy[M.index.get_indexer(d_.index)] == 1
        if not np.isfinite(v).all() or q.sum() < 1 or len(v) - q.sum() < 3:
            continue
        A_, E_ = v.mean(), v[~q].mean()
        t = int(0.10 * len(v))
        bp[s_] = dict(E=E_, den=abs(A_ - E_), sem=float(np.std(v, ddof=1) / np.sqrt(len(v))),
                      recorte=np.sort(v)[t:len(v) - t].mean(), batch=d_.batch_num.iloc[0])
    BB_ = pd.DataFrame(bp).T.apply(pd.to_numeric, errors='coerce')
    BB_['fiable'] = BB_['den'] >= BB_['sem'] * FAC_ESP.get('hardness_H_GPa', 1.0)
    BB_['rec_base'] = 1 - (BB_['recorte'] - BB_['E']).abs() / BB_['den']
    out = {}
    for nom in (bA, bB):
        por_lote = {}
        for lote in lotes:
            te = M.batch_num == lote
            mdl = HistGradientBoostingClassifier(**HP).fit(X.loc[~te, JUEGOS[nom]], yy[~te.values])
            ptr = mdl.predict_proba(X.loc[~te, JUEGOS[nom]])[:, 1]
            pte = mdl.predict_proba(X.loc[te, JUEGOS[nom]])[:, 1]
            u = np.quantile(ptr, 1 - yy[~te.values].mean())
            pred = (pte >= u).astype(int)
            rr = {}
            sub = M.loc[te]
            for s_, ii in sub.groupby('sample_num').groups.items():
                if s_ not in bp:
                    continue
                d_ = sub.loc[ii]; v = d_['hardness_H_GPa'].astype(float).values
                p_ = pred[sub.index.get_indexer(ii)]
                quita = p_ == 1
                if (~quita).sum() < 3:
                    o = np.argsort(pte[sub.index.get_indexer(ii)], kind='stable')
                    quita = np.ones(len(p_), bool); quita[o[:3]] = False
                    D6_ACT.append((f'B · nivel >= {niv} · {nom.split(" · ")[0]}', s_, 'hardness_H_GPa'))
                rr[s_] = 1 - abs(v[~quita].mean() - bp[s_]['E']) / bp[s_]['den'] \
                    if bp[s_]['den'] > 0 else np.nan
            pel = BB_[(BB_['batch'] == lote) & BB_['fiable']].index
            por_lote[lote] = pd.Series(rr).reindex(pel).median()
        out[nom] = por_lote
    base_l = {l: BB_[(BB_['batch'] == l) & BB_['fiable']]['rec_base'].median() for l in lotes}
    gA = sum(out[bA][l] > base_l[l] for l in lotes)
    gB = sum(out[bB][l] > base_l[l] for l in lotes)
    gBA = sum(out[bB][l] > out[bA][l] for l in lotes)
    n_lev_ = int(sum(((BB_['batch'] == l) & BB_['fiable']).any() for l in lotes))
    n_pev_ = int(BB_['fiable'].sum())
    POR_CAMP_NIV[niv] = dict(base=base_l, A=out[bA], B=out[bB],
                             n={l: int(((BB_['batch'] == l) & BB_['fiable']).sum()) for l in lotes})
    filas.append(dict(nivel=niv, pct_marcado=100 * yy.mean(), peliculas=n_pev_,
                      lotes_eval=n_lev_,
                      base=np.nanmedian(np.asarray(list(base_l.values()), float)),
                      A=np.nanmedian(np.asarray(list(out[bA].values()), float)),
                      B=np.nanmedian(np.asarray(list(out[bB].values()), float)),
                      B_bate_base=gB, B_bate_A=gBA, A_bate_base=gA))
    print(f'    nivel >= {niv}: {100*yy.mean():.1f} % marcadas · {n_pev_} '
          f'{"pelicula" if n_pev_ == 1 else "peliculas"} con metrica en {n_lev_} '
          f'{"campaña" if n_lev_ == 1 else "campañas"}')
SENS = pd.DataFrame(filas)
print(f"\n    {'nivel':>6} {'% marcado':>10} {'peliculas':>10} {'base':>7} {'A':>7} {'B':>7} "
      f"{'B>base':>8} {'B>A':>6}")
for _, r in SENS.iterrows():
    if int(r.lotes_eval) < 5:
        print(f'    {int(r.nivel):>6} {r.pct_marcado:>9.1f}% {int(r.peliculas):>10}   no evaluable: '
              f'{int(r.peliculas)} {"pelicula" if int(r.peliculas) == 1 else "peliculas"} con '
              f'metrica en {int(r.lotes_eval)} de {len(lotes)} campañas')
        continue
    print(f'    {int(r.nivel):>6} {r.pct_marcado:>9.1f}% {int(r.peliculas):>10} '
          f'{r.base:>7.3f} {r.A:>7.3f} {r.B:>7.3f} {int(r.B_bate_base)}/{len(lotes):<6} '
          f'{int(r.B_bate_A)}/{len(lotes)}')
print('\n    valor de cada campaña (peliculas evaluables), en los niveles con metrica en al'
      '\n    menos 5 campañas ("-" = campaña sin peliculas evaluables):')
for _, r in SENS.iterrows():
    if int(r.lotes_eval) < 5:
        continue
    pc_ = POR_CAMP_NIV[int(r.nivel)]
    print(f'\n    nivel >= {int(r.nivel)}')
    print(f"   {'campaña':>7} {'pelis':>6} {'base':>7} {'A':>7} {'B':>7}")
    for l_ in lotes:
        vals_ = ' '.join(f'{x:>7.3f}' if x == x else f"{'-':>7}"
                         for x in (pc_['base'][l_], pc_['A'][l_], pc_['B'][l_]))
        print(f'    {int(l_):>6} {pc_["n"][l_]:>6} {vals_}')
SENS.to_csv(SALIDA_SENS, index=False)
print(f'\n  guardado: {SALIDA_SENS}')


def lolo_rec(Xm, cols_, filas_tr=None, ctx_=''):
    rec_, med_ = {}, {}
    for lt_ in lotes:
        te_ = (M.batch_num == lt_).values
        tr_ = ~te_ if filas_tr is None else (~te_ & filas_tr)
        mdl_ = HistGradientBoostingClassifier(**HP).fit(Xm.loc[tr_, cols_], M.loc[tr_, '_y'])
        ptr_ = mdl_.predict_proba(Xm.loc[tr_, cols_])[:, 1]
        pte_ = mdl_.predict_proba(Xm.loc[te_, cols_])[:, 1]
        u_ = np.quantile(ptr_, 1 - M.loc[tr_, '_y'].mean())
        r_ = recupera(M.loc[te_], (pte_ >= u_).astype(int), pte_, ctx=ctx_)
        rec_[lt_] = r_
        med_[lt_] = r_.reindex(B_[(B_['batch'] == lt_) & B_['fiable']].index).median()
    return rec_, pd.Series(med_)


GL = lambda s_: float(np.nanmedian(np.asarray(s_, dtype=float)))
TA_, TB_ = T[bA].astype(float), T[bB].astype(float)

sep('C · MODELO REDUCIDO: VALORES DEL LABORATORIO + LAS DOS MAGNITUDES DE LA REGLA')
if TR is None:
    print('  sin la regla del bloque 5 de la celda 15; se omite el bloque.')
else:
    cols_red = list(JUEGOS[bA]) + [v_ for v_ in RG_VARS if v_ in X.columns]
    _, MED_RED = lolo_rec(X, cols_red, ctx_='C · reducido')
    print(f'  variables: las {len(JUEGOS[bA])} de A + {[v_ for v_ in RG_VARS if v_ in X.columns]}\n')
    print(f"   {'campaña':>7} {'A':>8} {'REDUCIDO':>9} {'B':>8} {'REGLA':>8}")
    for lt_ in lotes:
        print(f'    {int(lt_):>6} {TA_[lt_]:>8.3f} {MED_RED[lt_]:>9.3f} {TB_[lt_]:>8.3f} '
              f'{float(TR.loc[lt_, "regla"]):>8.3f}')
    g_red, g_B, g_A, g_rg = GL(MED_RED), GL(TB_), GL(TA_), GL(TR['regla'].astype(float))
    print(f"    {'GLOBAL':>6} {g_A:>8.3f} {g_red:>9.3f} {g_B:>8.3f} {g_rg:>8.3f}")
    print(f'\n  el reducido supera a A en {int((MED_RED > TA_).sum())} de {len(lotes)} campañas · '
          f'B supera al reducido en {int((TB_ > MED_RED).sum())} · el reducido supera a la '
          f'regla en {int((MED_RED > TR["regla"].astype(float)).sum())}')
    if g_B - g_red < 0.05:
        print(f'  LECTURA Q1 (B - reducido = {g_B - g_red:+.3f}): la ventaja de B sobre A se explica')
        print('  con las dos magnitudes mas el aprendizaje; las demas variables de forma aportan poco.')
    else:
        print(f'  LECTURA Q2 (B - reducido = {g_B - g_red:+.3f}): las demas variables de forma aportan.')
    if g_red - g_rg >= 0.05:
        print(f'  reducido - regla = {g_red - g_rg:+.3f}: aprender sobre las dos magnitudes, junto a los')
        print('  valores del laboratorio, aporta por encima de la regla fija.')
    else:
        print(f'  reducido - regla = {g_red - g_rg:+.3f}: aprender sobre las dos magnitudes no aporta')
        print('  por encima de la regla fija.')

sep('D · CIRCULARIDAD: ¿SE SOSTIENE EL RESULTADO SIN LA SEGUNDA PASADA?')
IX_ = pd.MultiIndex.from_frame(M[CLAVE])
cl_ = [c_ for c_ in POR_INDENT + ['verify_zero_offset_type'] if c_ in Oi.columns]
NO_ = Oi.reindex(IX_)[cl_].apply(pd.to_numeric, errors='coerce').fillna(0)
bo_ = (NO_ >= 2).any(axis=1).values.astype(int)
ESTABLE = bo_ == M['_y'].values
print(f'  ensayos estables entre pasadas: {int(ESTABLE.sum())} de {len(M)} · cambiados: '
      f'{int((~ESTABLE).sum())}')
_, MED_CA = lolo_rec(X, JUEGOS[bA], ESTABLE, ctx_='D · A, marca estable')
_, MED_CB = lolo_rec(X, JUEGOS[bB], ESTABLE, ctx_='D · B, marca estable')
print(f"\n   {'campaña':>7} {'A (15)':>8} {'A estables':>11} {'B (15)':>8} {'B estables':>11}")
for lt_ in lotes:
    print(f'    {int(lt_):>6} {TA_[lt_]:>8.3f} {MED_CA[lt_]:>11.3f} {TB_[lt_]:>8.3f} '
          f'{MED_CB[lt_]:>11.3f}')
print(f"    {'GLOBAL':>6} {GL(TA_):>8.3f} {GL(MED_CA):>11.3f} {GL(TB_):>8.3f} {GL(MED_CB):>11.3f}")
n_cba = int((MED_CB > MED_CA).sum()); d_cb = GL(MED_CB) - GL(TB_)
print(f'\n  entrenando solo con estables: B supera a A en {n_cba} de {len(lotes)} campañas · '
      f'cambio de la mediana de B: {d_cb:+.3f}')
if abs(d_cb) < 0.05 and n_cba >= 5:
    print('  LECTURA: el resultado no depende de la segunda pasada: se mantiene sin ella.')
else:
    print('  LECTURA: el resultado depende en parte de la segunda pasada.')

sep('E · DEPENDENCIA ESPACIAL ENTRE ENSAYOS VECINOS (informe; la correccion se aplica en la 15)')
for ps_ in PROPS_EV:
    r_ = RHO_ESP[ps_]; bt_ = BTAB[ps_]
    print(f'  {ps_}: {len(r_)} peliculas · correlacion de rangos entre vecinos: mediana '
          f'{r_.median():.3f} · p25 {r_.quantile(.25):.3f} · p75 {r_.quantile(.75):.3f} · '
          f'positiva en {int((r_ > 0).sum())}')
    print(f'      factor sobre el error tipico {FAC_ESP[ps_]:.3f} · peliculas evaluables '
          f'{int(bt_["fiable"].sum())} (sin la correccion serian {int((bt_["den"] >= bt_["sem"]).sum())})')

sep('F · ESTABILIDAD NUMERICA DE A Y B (perturbacion del orden del redondeo)')
V_ex = pd.read_csv(TABLA, float_precision='round_trip')
z_forma = [c_ for c_ in JUEGOS[bB] if c_.startswith('z_')]
crudas_ = [c_[2:] for c_ in z_forma]
falta_c = [c_ for c_ in crudas_ if c_ not in V_ex.columns]
if falta_c:
    print(f'  faltan columnas crudas en la tabla de la celda 14: {falta_c[:5]}; se omite el bloque.')
else:
    RAW_B = M[CLAVE].merge(V_ex[CLAVE + crudas_], on=CLAVE, how='left')
    RAW_B.index = M.index
    raw_A = [c_ for c_ in val_lab] + [c_ for c_ in ('h_max', 'p_max') if c_ in M.columns]
    RAW_A = M[raw_A].astype(float)
    GRP_ = M['sample_num']

    def z12(df_):
        g0_ = df_.groupby(GRP_)
        md_ = g0_.transform('median')
        ma_ = g0_.transform(lambda x: np.median(np.abs(x - np.median(x))))
        z0_ = (df_ - md_) / (1.4826 * ma_.replace(0, np.nan))
        return z0_.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    def construye(eps_, rng_):
        ra_ = RAW_A * (1 + eps_ * rng_.standard_normal(RAW_A.shape)) if eps_ else RAW_A
        rb_ = RAW_B[crudas_].astype(float)
        rb_ = rb_ * (1 + eps_ * rng_.standard_normal(rb_.shape)) if eps_ else rb_
        za_ = zrob(pd.concat([M[['sample_num']], ra_], axis=1), val_lab)
        za_.columns = [f'a_{c_}' for c_ in val_lab]
        for c_ in ('h_max', 'p_max'):
            if c_ in ra_.columns:
                za_[f'a_{c_}'] = zrob(pd.concat([M[['sample_num']], ra_], axis=1), [c_])[c_]
        zb_ = z12(rb_); zb_.columns = z_forma
        return pd.concat([za_, zb_], axis=1)

    rng_f = np.random.default_rng(SEED + 15)
    EPS_ = 5e-16
    X0_ = construye(0.0, rng_f)
    X1_ = construye(EPS_, rng_f)
    dz_ = (X1_[z_forma] - X0_[z_forma]).abs()
    deg_ = dz_.max().sort_values(ascending=False)
    deg_ = deg_[deg_ > 1e-6]
    print(f'  diagnostico: variables de B cuya version normalizada cambia mas de 1e-6 con la')
    print(f'  perturbacion: {len(deg_)} de {len(z_forma)}')
    for c_, v_ in deg_.head(10).items():
        npel_ = int((dz_[c_] > 1e-6).groupby(GRP_).any().sum())
        print(f'    {c_:>30}: cambio maximo {v_:.3g} · peliculas afectadas {npel_}')
    filas_f = []
    for rep_ in range(0, 11):
        Xr_ = X0_ if rep_ == 0 else construye(EPS_, rng_f)
        _, mA_ = lolo_rec(Xr_, [c_ for c_ in JUEGOS[bA] if c_ in Xr_.columns], ctx_='F · A, estabilidad')
        _, mB_ = lolo_rec(Xr_, [c_ for c_ in JUEGOS[bB] if c_ in Xr_.columns], ctx_='F · B, estabilidad')
        filas_f.append(dict(rep=rep_, A=GL(mA_), B=GL(mB_), BmA=GL(mB_) - GL(mA_),
                            gana=int((mB_ > mA_).sum()), **{f'B_{int(l_)}': mB_[l_] for l_ in lotes}))
        print(f'    repeticion {rep_:>2}{" (sin ruido, tabla leida sin perdida)" if rep_ == 0 else "":<38} '
              f'A {GL(mA_):.3f} · B {GL(mB_):.3f} · B supera a A en {int((mB_ > mA_).sum())} de {len(lotes)}')
    ESTAB = pd.DataFrame(filas_f)
    ER_ = ESTAB[ESTAB.rep > 0]
    print(f'\n  en las 10 repeticiones con ruido: B global de {ER_.B.min():.3f} a {ER_.B.max():.3f} · '
          f'B - A de {ER_.BmA.min():+.3f} a {ER_.BmA.max():+.3f}')
    print('  rango de B por campaña: ' + ' · '.join(
        f'{int(l_)} {ER_[f"B_{int(l_)}"].min():.2f}-{ER_[f"B_{int(l_)}"].max():.2f}' for l_ in lotes))
    n_ok = int((ER_.gana >= 5).sum())
    print(f'  repeticiones en que B supera a A en al menos 5 de {len(lotes)} campañas: {n_ok} de 10')
    if n_ok >= 9:
        print('  LECTURA: la ventaja de B sobre A es estable frente al redondeo.')
    else:
        print('  LECTURA: la ventaja de B sobre A no es estable frente al redondeo.')
    ESTAB.to_csv('/content/estabilidad_numerica.csv', index=False)
    print('\n  guardado: /content/estabilidad_numerica.csv')

sep('G-H · ¿DEPENDE EL RESULTADO DEL UMBRAL DE EVALUABILIDAD?')
FAC_GH_ = float(FAC_ESP[PROP_PRIN])
VAR_GH_ = [('sin correccion por vecinos', 1.0), ('principal (D5, control)', FAC_GH_),
           ('dos errores tipicos corregidos', 2.0 * FAC_GH_)]
recA_gh_ = pd.concat([res[bA][l_] for l_ in lotes])
recB_gh_ = pd.concat([res[bB][l_] for l_ in lotes])
filas_gh_ = []
for et_, f_ in VAR_GH_:
    ev_ = (B_['den'] >= B_['sem'] * f_).astype(bool)
    print(f'\n  suelo: {et_} (error tipico x {f_:.3f}) · {int(ev_.sum())} peliculas evaluables')
    print(f"   {'campaña':>7} {'peliculas':>10} {'base':>8} {'A':>8} {'B':>8}")
    pl_ = {}
    for lt_ in lotes:
        pel_ = B_[(B_['batch'] == lt_) & ev_].index
        if len(pel_) == 0:
            print(f'    {int(lt_):>6} {0:>10}   sin peliculas evaluables (no cuenta)')
            continue
        pl_[lt_] = dict(n=len(pel_), base=B_.loc[pel_, 'rec_base'].median(),
                        A=recA_gh_.reindex(pel_).median(), B=recB_gh_.reindex(pel_).median())
        r_ = pl_[lt_]
        print(f"    {int(lt_):>6} {r_['n']:>10} {r_['base']:>8.3f} {r_['A']:>8.3f} {r_['B']:>8.3f}")
    TP_ = pd.DataFrame(pl_).T
    n_ev_ = len(TP_)
    if n_ev_ == 0:
        print('    ninguna campaña conserva peliculas evaluables con este suelo: no evaluable')
        filas_gh_.append(dict(suelo=et_, factor=f_, peliculas=int(ev_.sum()), lotes_eval=0,
                              base=np.nan, A=np.nan, B=np.nan, B_supera_A=0, se_mantiene=False))
        continue
    gA_, gB_, gbase_ = (float(np.nanmedian(TP_[c_].astype(float))) for c_ in ('A', 'B', 'base'))
    gana_ = int((TP_['B'] > TP_['A']).sum())
    se_mantiene_ = (gB_ > gA_) and (n_ev_ > 0) and (gana_ / n_ev_ >= 5 / 7 - 1e-9)
    print(f"    {'GLOBAL':>6} {'':>10} {gbase_:>8.3f} {gA_:>8.3f} {gB_:>8.3f}")
    print(f'    B supera a A en {gana_} de {n_ev_} campañas con peliculas evaluables · '
          f'{"SE MANTIENE" if se_mantiene_ else "NO SE MANTIENE"}')
    filas_gh_.append(dict(suelo=et_, factor=f_, peliculas=int(ev_.sum()), lotes_eval=n_ev_,
                          base=gbase_, A=gA_, B=gB_, B_supera_A=gana_, se_mantiene=se_mantiene_))
UMBR_TAB = pd.DataFrame(filas_gh_)
UMBR_TAB.to_csv('/content/robustez_umbral_evaluable.csv', index=False)
print('\n  guardado: /content/robustez_umbral_evaluable.csv')
del FAC_GH_, VAR_GH_, recA_gh_, recB_gh_, filas_gh_, TP_, pl_

sep('I · REENTRENAR SIN CADA ZONA DE LA CURVA')


def lolo_det_(cols_):
    pr_, med_ = [], {}
    for lt_ in lotes:
        te_ = (M.batch_num == lt_).values
        mdl_ = HistGradientBoostingClassifier(**HP).fit(X.loc[~te_, cols_], M.loc[~te_, '_y'])
        ptr_ = mdl_.predict_proba(X.loc[~te_, cols_])[:, 1]
        pte_ = mdl_.predict_proba(X.loc[te_, cols_])[:, 1]
        pr_.append(average_precision_score(M.loc[te_, '_y'], pte_))
        u_ = np.quantile(ptr_, 1 - M.loc[~te_, '_y'].mean())
        r_ = recupera(M.loc[te_], (pte_ >= u_).astype(int), pte_, ctx='I · sin zona')
        med_[lt_] = r_.reindex(B_[(B_['batch'] == lt_) & B_['fiable']].index).median()
    return float(np.mean(pr_)), float(np.nanmedian(np.asarray(list(med_.values()), float)))


colsB_ = list(JUEGOS[bB])
forma_ = [c_ for c_ in colsB_ if not c_.startswith('a_')]
fams_ = sorted({familia(c_) for c_ in forma_})
peso_ = imp.groupby('familia')['media'].sum() if 'familia' in imp.columns else pd.Series(dtype=float)
tot_peso_ = float(peso_.sum()) if len(peso_) else np.nan
pr_full_, rec_full_ = lolo_det_(colsB_)
pr_13_ = float(np.mean([det[bB][l_]['pr_auc'] for l_ in lotes]))
print(f'  B completo, reentrenado aqui: PR-AUC medio {pr_full_:.4f} (celda 15: {pr_13_:.4f}) · '
      f'recuperacion {rec_full_:.3f} (celda 15: {float(T[bB].median()):.3f})')
print(f'\n    {"zona retirada":>32} {"n var":>6} {"peso A1":>8} {"PR-AUC":>8} {"perdida":>8} '
      f'{"recup.":>7}   lectura')
filas_i_ = []
for f_ in fams_:
    quita_ = [c_ for c_ in forma_ if familia(c_) == f_]
    pr_i_, rec_i_ = lolo_det_([c_ for c_ in colsB_ if c_ not in quita_])
    perd_ = pr_full_ - pr_i_
    w_ = float(peso_.get(f_, np.nan)) / tot_peso_ if tot_peso_ and np.isfinite(tot_peso_) else np.nan
    lect_ = 'aporta lo que las demas no sustituyen' if perd_ >= 0.02 else 'lo recogen otras zonas'
    filas_i_.append(dict(zona=f_, n_var=len(quita_), peso_A1=w_, pr_auc=pr_i_, perdida=perd_,
                         recuperacion=rec_i_, aporta_unico=perd_ >= 0.02))
    print(f'    {f_:>32} {len(quita_):>6} {100*w_:>7.1f}% {pr_i_:>8.4f} {perd_:>+8.4f} '
          f'{rec_i_:>7.3f}   {lect_}')
ZONAS_TAB = pd.DataFrame(filas_i_)
for f_ in ('otras', 'calidad / ruido'):
    m_ = [c_ for c_ in JUEGOS[bB] if not c_.startswith('a_') and familia(c_) == f_]
    if m_:
        print(f'    variables de la familia "{f_}": {m_}')
if not ZONAS_TAB['aporta_unico'].any():
    print('\n  LECTURA: ninguna zona baja el PR-AUC en 0,02 o mas al retirarla; la informacion esta'
          ' repartida entre zonas.')
else:
    z_ = ', '.join(ZONAS_TAB.loc[ZONAS_TAB['aporta_unico'], 'zona'])
    print(f'\n  LECTURA: aportan informacion que las demas zonas no sustituyen: {z_}.')
ZONAS_TAB.to_csv('/content/reentrenar_sin_zona.csv', index=False)
print('\n  guardado: /content/reentrenar_sin_zona.csv')
del colsB_, forma_, fams_, peso_, tot_peso_, pr_full_, rec_full_, pr_13_, filas_i_

sep('J · CAMPAÑAS QUE COMPARTEN SOLUCION: 614 Y 626, CADA UNA SIN LA OTRA')
PAR_J_ = (614, 626)
if not all(l_ in lotes for l_ in PAR_J_):
    print(f'  no estan las dos campañas {PAR_J_}; se omite el bloque.')
    HERM_TAB = None
else:
    npel_j_ = M.groupby('batch_num').sample_num.nunique()
    rng_j_ = np.random.default_rng(SEED + 26)
    NREP_J_ = 20

    def eval_lote_(lt_, tr_):
        te_ = (M.batch_num == lt_).values
        pel_ = B_[(B_['batch'] == lt_) & B_['fiable']].index
        o_ = {}
        for et_, nm_ in (('A', bA), ('B', bB)):
            c_ = JUEGOS[nm_]
            mdl_ = HistGradientBoostingClassifier(**HP).fit(X.loc[tr_, c_], M.loc[tr_, '_y'])
            ptr_ = mdl_.predict_proba(X.loc[tr_, c_])[:, 1]
            pte_ = mdl_.predict_proba(X.loc[te_, c_])[:, 1]
            u_ = np.quantile(ptr_, 1 - M.loc[tr_, '_y'].mean())
            o_[et_] = float(recupera(M.loc[te_], (pte_ >= u_).astype(int), pte_,
                                     ctx=f'J · {et_}').reindex(pel_).median())
            o_[f'pr_{et_}'] = float(average_precision_score(M.loc[te_, '_y'], pte_))
        o_['pel_entrena'] = int(M.loc[tr_, 'sample_num'].nunique())
        return o_

    filas_j_ = []
    for lt_, herm_ in ((626, 614), (614, 626)):
        te_ = (M.batch_num == lt_).values
        prueba_ = eval_lote_(lt_, ~te_ & (M.batch_num != herm_).values)
        otros_ = M.loc[~te_ & (M.batch_num != herm_).values, 'sample_num'].unique()
        ctl_ = []
        for rep_ in range(NREP_J_):
            fuera_ = rng_j_.choice(otros_, size=int(npel_j_[herm_]), replace=False)
            ctl_.append(eval_lote_(lt_, ~te_ & ~M.sample_num.isin(fuera_).values))
        CB_ = np.array([c_['B'] for c_ in ctl_]); CA_ = np.array([c_['A'] for c_ in ctl_])
        if prueba_['B'] < CB_.min():
            lect_ = 'POR DEBAJO del control: limite del alcance en esta campaña'
        elif prueba_['B'] > CB_.max():
            lect_ = 'por encima del control: compartir solucion no explica el resultado'
        else:
            lect_ = 'dentro del control: compartir solucion no explica el resultado'
        n_pel_ev_ = int(((B_['batch'] == lt_) & B_['fiable']).sum())
        print(f'\n  campaña evaluada {lt_} ({n_pel_ev_} peliculas evaluables) · campaña hermana {herm_}')
        print(f'    validacion principal (celda 15, con la campaña hermana en entrenamiento): A {float(T.loc[lt_, bA]):.3f} · '
              f'B {float(T.loc[lt_, bB]):.3f}')
        print(f'    prueba, sin la campaña hermana ({prueba_["pel_entrena"]} peliculas de entrenamiento): '
              f'A {prueba_["A"]:.3f} · B {prueba_["B"]:.3f} · PR-AUC B {prueba_["pr_B"]:.3f}')
        pe_ctl_ = sorted({c_['pel_entrena'] for c_ in ctl_})
        print(f'    control, con la campaña hermana y {pe_ctl_} peliculas de entrenamiento ({NREP_J_} repeticiones):')
        print(f'      B de {CB_.min():.3f} a {CB_.max():.3f} (mediana {np.median(CB_):.3f}) · '
              f'A de {CA_.min():.3f} a {CA_.max():.3f}')
        print(f'    LECTURA: {lect_}')
        filas_j_.append(dict(lote=lt_, hermano=herm_, evaluables=n_pel_ev_,
                             A_principal=float(T.loc[lt_, bA]), B_principal=float(T.loc[lt_, bB]),
                             A_prueba=prueba_['A'], B_prueba=prueba_['B'], prB_prueba=prueba_['pr_B'],
                             pel_entrena_prueba=prueba_['pel_entrena'],
                             B_control_min=CB_.min(), B_control_med=float(np.median(CB_)),
                             B_control_max=CB_.max(), lectura=lect_))
    HERM_TAB = pd.DataFrame(filas_j_)
    HERM_TAB.to_csv('/content/lotes_misma_solucion.csv', index=False)
    print('\n  guardado: /content/lotes_misma_solucion.csv')
    del npel_j_, rng_j_, filas_j_, CB_, CA_, ctl_, prueba_, otros_

sep('D6 · ¿CUANTAS VECES SE APLICA LA SALVAGUARDA DE TRES ENSAYOS EN ESTA CELDA?')
informa_d6(D6_ACT[D6_N0:])


A1 · IMPORTANCIA POR PERMUTACION, MEDIDA EN LA CAMPAÑA RETENIDA


                        variable   caida media de PR-AUC  peor campaña
                 z_p_ini_p5_frac                  0.1166        0.0415
           z_desc_sesgo_bajo_rel                  0.0815        0.0541
                    z_p_ini_frac                  0.0359        0.0210
                   z_desc_hf_rel                  0.0214        0.0141
                     z_carga_exp                  0.0065        0.0019
             z_sost_fluencia_rel                  0.0060        0.0014
                      z_carga_r2                  0.0039       -0.0006
                         a_h_max                  0.0036        0.0002
                z_carga_curv_iqr                  0.0035        0.0011
                   z_rigidez_fit                  0.0034        0.0009
                 z_h_ini_p5_frac                  0.0032       -0.0000
                    z_h_ini_frac                  0.0032        0.0011
                    z_h_fin_frac                  0.0030        0.0006
      

  calculado sobre 600 indentaciones de la campaña retenida mas grande.

                        variable   |SHAP| medio   empuja hacia
                 z_p_ini_p5_frac         0.9122 anomala si sube
           z_desc_sesgo_bajo_rel         0.7340 anomala si sube
                    z_p_ini_frac         0.6541 anomala si sube
                   z_desc_hf_rel         0.5762 anomala si baja
                      z_carga_r2         0.3247 anomala si baja
                   z_rigidez_fit         0.2357 anomala si sube
                    z_h_fin_frac         0.2238 anomala si baja
             z_sost_fluencia_rel         0.2176 anomala si sube
                z_carga_curv_iqr         0.2100 anomala si sube
               z_carga_resid_rel         0.1729 anomala si sube
                     z_carga_exp         0.1608 anomala si sube
            z_sost_pendiente_neg         0.1495 anomala si baja

  guardado: /content/shap_valores.npz

B · ¿AGUANTA EL RESULTADO SI CAMBIA EL NIVEL DE SEVERIDAD

    nivel >= 1: 30.4 % marcadas · 47 peliculas con metrica en 7 campañas


    nivel >= 2: 27.7 % marcadas · 46 peliculas con metrica en 7 campañas


    nivel >= 3: 1.1 % marcadas · 1 pelicula con metrica en 1 campaña

     nivel  % marcado  peliculas    base       A       B   B>base    B>A
         1      30.4%         47   0.329   0.712   0.889 7/7      6/7
         2      27.7%         46   0.310   0.622   0.921 7/7      6/7
         3       1.1%          1   no evaluable: 1 pelicula con metrica en 1 de 7 campañas

    valor de cada campaña (peliculas evaluables), en los niveles con metrica en al
    menos 5 campañas ("-" = campaña sin peliculas evaluables):

    nivel >= 1
   campaña  pelis    base       A       B
       609      6   0.458   0.769   0.968
       614     15   0.415   0.712   0.862
       615      7  -0.017   0.381   0.823
       619      5   0.078   0.526   0.889
       622      4   0.252   0.644   0.487
       623      4   0.425   0.829   0.952
       626      6   0.329   0.802   0.923

    nivel >= 2
   campaña  pelis    base       A       B
       609      5   0.478   0.723   0.932
       614     15   0.367  

... (20 outputs left out)

                      tramo de carga      6     4.9%   0.9573  +0.0079   0.917   lo recogen otras zonas


                   tramo de descarga      8    34.9%   0.9611  +0.0041   0.858   lo recogen otras zonas
    variables de la familia "otras": ['z_p_max', 'z_h_max']
    variables de la familia "calidad / ruido": ['z_ruido_rel']

  LECTURA: aportan informacion que las demas zonas no sustituyen: arranque de la curva (offset).

  guardado: /content/reentrenar_sin_zona.csv

J · CAMPAÑAS QUE COMPARTEN SOLUCION: 614 Y 626, CADA UNA SIN LA OTRA



  campaña evaluada 626 (6 peliculas evaluables) · campaña hermana 614
    validacion principal (celda 15, con la campaña hermana en entrenamiento): A 0.806 · B 0.921
    prueba, sin la campaña hermana (72 peliculas de entrenamiento): A 0.724 · B 0.928 · PR-AUC B 0.962
    control, con la campaña hermana y [72] peliculas de entrenamiento (20 repeticiones):
      B de 0.890 a 0.975 (mediana 0.932) · A de 0.614 a 0.802
    LECTURA: dentro del control: compartir solucion no explica el resultado



  campaña evaluada 614 (15 peliculas evaluables) · campaña hermana 626
    validacion principal (celda 15, con la campaña hermana en entrenamiento): A 0.622 · B 0.925
    prueba, sin la campaña hermana (72 peliculas de entrenamiento): A 0.597 · B 0.862 · PR-AUC B 0.989
    control, con la campaña hermana y [72] peliculas de entrenamiento (20 repeticiones):
      B de 0.852 a 0.949 (mediana 0.877) · A de 0.504 a 0.722
    LECTURA: dentro del control: compartir solucion no explica el resultado

  guardado: /content/lotes_misma_solucion.csv

D6 · ¿CUANTAS VECES SE APLICA LA SALVAGUARDA DE TRES ENSAYOS EN ESTA CELDA?
  ninguna: ningun metodo deja menos de 3 ensayos en ninguna pelicula.


**Cómo leer la salida**

- A1: en cada pliegue se baraja una variable cada vez en la campaña retenida y se mide cuánto empeora la detección; lo que baja mucho es aquello de lo que el modelo depende para generalizar, no solo para memorizar. La columna «peor campaña» importa tanto como la media. Una variable con media alta y alguna campaña en cero funciona en unas campañas y no en otras. Si SHAP no pudiera calcularse, A1 ya dice en qué se fija el modelo, y con una medida fuera de muestra.
- B: el nivel ≥ 1 descarta también lo dudoso; el nivel ≥ 3, solo lo evidente. Si en los niveles evaluables B supera a la línea base y a A en la mayoría de las campañas, el resultado no depende de dónde se pone el corte de gravedad. Si solo se sostuviera en el nivel 2, el hallazgo valdría para las anomalías de gravedad media o alta, no para el criterio completo del especialista.
- F: la ventaja de B sobre A puede ser estable aunque las cifras sueltas por campaña varíen más.
- I: «peso A1» es la proporción de la caída total de PR-AUC de A1 (todas las familias, incluidos los valores del laboratorio) que corresponde a esa zona. Una zona puede pesar en A1 aunque su información la recojan otras.
- J: los criterios F1 a F3 se evalúan solo con la validación principal de la celda 15; esta comprobación se informa aparte.

### Celda 17. Cifras de diagnóstico para el apartado 4.4

Solo imprime: no entrena ningún modelo, no cambia ningún objeto que use la celda 18 y no decide nada. Reorganiza cifras que ya calculan las celdas 15 y 16, y añade unas pocas comprobaciones que ninguna otra celda imprime. La recuperación de B por campaña forma dos grupos y, con siete valores así, la diferencia entre las medianas globales de dos métodos puede no coincidir con la diferencia típica campaña a campaña. Por eso cada comparación se da de varias formas: valores por campaña, recuento, diferencia de medianas globales y mediana de las diferencias por campaña.

1. Dureza: recuperación por campaña (películas evaluables) de todos los métodos y mediana de las siete.
2. Comparaciones por pares: campañas a favor, diferencia de medianas globales, mediana de las diferencias, la menor y la mayor.
3. F3: B en la campaña 614 y, en las otras seis, mediana de B, de A, de la media recortada y de la mediana de los 64 ensayos.
4. Detección por campaña: PR-AUC y AUC de A, B, C y D, y proporción de ensayos descartados por el especialista.
5. Barrido del umbral por campaña: A y B en cada campaña y multiplicador.
6. Módulo reducido por campaña: media recortada, mediana, A, B, C y D.
7. Variables de forma: en cuántas películas cada una queda a 0 en toda la película, por desviación absoluta mediana nula o por algún ensayo sin valor, y variables de cada modelo.
8. SHAP: explicador usado en la celda 16 (A2) y su valor esperado.

In [18]:
kC_17 = [k_ for k_ in JUEGOS if k_.startswith('C')][0]
kD_17 = [k_ for k_ in JUEGOS if k_.startswith('D')][0]
L_17 = list(lotes)
fmt3 = lambda v: f'{v:>8.3f}' if pd.notna(v) else f'{"-":>8}'

sep('1 · DUREZA: RECUPERACION POR CAMPAÑA (peliculas evaluables) Y MEDIANA DE LAS SIETE')
H_17 = pd.DataFrame({
    'recortada': T['base'].astype(float),
    'mediana': T9['mediana'].astype(float),
    'mad3': T9['mad3'].astype(float),
    'regla': TR['regla'].astype(float),
    'A': T[bA].astype(float),
    'reducido': pd.Series(MED_RED, dtype=float),
    'D': T[kD_17].astype(float),
    'C': T[kC_17].astype(float),
    'B': T[bB].astype(float)}).reindex(L_17)
N_17 = T9['n'].astype(int).reindex(L_17)
print(f"    {'campaña':>8} {'pelis':>6} " + ' '.join(f'{c_:>9}' for c_ in H_17.columns))
for lt_ in L_17:
    print(f'    {int(lt_):>8} {int(N_17[lt_]):>6} ' +
          ' '.join(f'{H_17.loc[lt_, c_]:>9.3f}' for c_ in H_17.columns))
G_17 = H_17.median()
print(f"    {'MEDIANA':>8} {int(N_17.sum()):>6} " + ' '.join(f'{G_17[c_]:>9.3f}' for c_ in H_17.columns))
print('\n    B ordenado de menor a mayor: ' + ' · '.join(f'{v:.3f}' for v in sorted(H_17['B'])))

sep('2 · COMPARACIONES POR PARES (dureza, peliculas evaluables)')
PARES_17 = [('B', 'A'), ('B', 'mediana'), ('A', 'mediana'), ('B', 'regla'), ('B', 'reducido'),
             ('reducido', 'regla'), ('reducido', 'A'), ('regla', 'A'), ('regla', 'mediana'),
             ('C', 'B'), ('D', 'B'), ('D', 'A'), ('B', 'recortada')]
print(f"    {'comparacion':>20} {'a favor':>8} {'dif. medianas':>14} {'mediana dif.':>13} "
      f"{'menor':>8} {'mayor':>8}   diferencias por campaña")
CMP_17 = {}
for x_, y_ in PARES_17:
    d_ = H_17[x_] - H_17[y_]
    r_ = dict(n=int((d_ > 0).sum()), dg=float(G_17[x_] - G_17[y_]), md=float(d_.median()),
              mn=float(d_.min()), mx=float(d_.max()))
    CMP_17[(x_, y_)] = r_
    print(f"    {x_ + ' - ' + y_:>20} {str(r_['n']) + ' de 7':>8} {r_['dg']:>+14.3f} "
          f"{r_['md']:>+13.3f} {r_['mn']:>+8.3f} {r_['mx']:>+8.3f}   " +
          ' '.join(f'{v:+.3f}' for v in d_))

sep('3 · F3: LA CAMPAÑA 614 FRENTE A LAS OTRAS SEIS')
rest_17 = [l_ for l_ in L_17 if int(l_) != 614]
F3_17 = dict(B614=float(H_17.loc[614, 'B']), Bresto=float(H_17.loc[rest_17, 'B'].median()),
              recresto=float(H_17.loc[rest_17, 'recortada'].median()),
              medresto=float(H_17.loc[rest_17, 'mediana'].median()),
              Aresto=float(H_17.loc[rest_17, 'A'].median()))
print(f"  B en la 614: {F3_17['B614']:.3f}")
print(f"  en las otras seis, mediana de B {F3_17['Bresto']:.3f} · de la media recortada "
      f"{F3_17['recresto']:.3f} · de la mediana de las 64 {F3_17['medresto']:.3f} · de A "
      f"{F3_17['Aresto']:.3f}")
print('  B en las otras seis, ordenado: ' + ' · '.join(f'{v:.3f}' for v in sorted(H_17.loc[rest_17, 'B'])))

sep('4 · DETECCION ENSAYO A ENSAYO, POR CAMPAÑA (todos los ensayos de la campaña evaluada)')
MODS_17 = [('A', bA), ('B', bB), ('C', kC_17), ('D', kD_17)]
DET_17 = {}
for med_ in ('pr_auc', 'auc'):
    print(f"\n  {'PR-AUC' if med_ == 'pr_auc' else 'AUC'}")
    print(f"    {'campaña':>8} {'ensayos':>8} {'% desc.':>8} " + ' '.join(f'{m_:>8}' for m_, _ in MODS_17))
    for lt_ in L_17:
        d0_ = det[bA][lt_]
        print(f"    {int(lt_):>8} {d0_['n']:>8} {100*d0_['pos']:>7.1f}% " +
              ' '.join(fmt3(det[k_][lt_][med_]) for _, k_ in MODS_17))
    medias_ = {m_: float(np.nanmean([det[k_][l_][med_] for l_ in L_17])) for m_, k_ in MODS_17}
    DET_17[med_] = medias_
    print(f"    {'MEDIA':>8} {'':>8} {'':>8} " + ' '.join(f'{medias_[m_]:>8.3f}' for m_, _ in MODS_17))
    nBA_ = sum(det[bB][l_][med_] > det[bA][l_][med_] for l_ in L_17)
    print(f'    B supera a A en {nBA_} de 7 campañas')

sep('5 · BARRIDO DEL UMBRAL, POR CAMPAÑA (dureza, peliculas evaluables)')
SW_17 = {}
for m_ in MULT:
    fa_, fb_ = [], []
    for lt_ in L_17:
        pel_ = B_[(B_['batch'] == lt_) & B_['fiable']].index
        fa_.append(barr[(bA, m_)][lt_].reindex(pel_).median())
        fb_.append(barr[(bB, m_)][lt_].reindex(pel_).median())
    fa_, fb_ = np.array(fa_, float), np.array(fb_, float)
    SW_17[m_] = dict(A=float(np.median(fa_)), B=float(np.median(fb_)), n=int((fb_ > fa_).sum()),
                      md=float(np.median(fb_ - fa_)))
    print(f'  x{m_:<4}  A: ' + ' '.join(f'{v:.3f}' for v in fa_) + f'   -> mediana {np.median(fa_):.3f}')
    print('         B: ' + ' '.join(f'{v:.3f}' for v in fb_) + f'   -> mediana {np.median(fb_):.3f}')
    print(f'         B supera a A en {(fb_ > fa_).sum()} de 7 · mediana de las diferencias '
          f'{np.median(fb_ - fa_):+.3f} · B ordenado: ' + ' '.join(f'{v:.3f}' for v in sorted(fb_)))
print('  (campañas en el orden ' + ', '.join(str(int(l_)) for l_ in L_17) + ')')

sep('6 · MODULO REDUCIDO: RECUPERACION POR CAMPAÑA (sus peliculas evaluables)')
bm_ = BTAB[PROP_SEC]; rm_ = REG_EV[PROP_SEC]
MOD_17 = {}
for lt_ in L_17:
    pel_ = bm_[(bm_['batch'] == lt_) & bm_['fiable']].index
    MOD_17[lt_] = dict(n=len(pel_), recortada=bm_.loc[pel_, 'rec_base'].median(),
                        mediana=rm_.reindex(pel_)['rec_mediana'].median(),
                        A=res_sec[(bA, PROP_SEC)][lt_].reindex(pel_).median(),
                        D=res_sec[(kD_17, PROP_SEC)][lt_].reindex(pel_).median(),
                        C=res_sec[(kC_17, PROP_SEC)][lt_].reindex(pel_).median(),
                        B=res_sec[(bB, PROP_SEC)][lt_].reindex(pel_).median())
MOD_17 = pd.DataFrame(MOD_17).T.astype(float)
cols_m_ = ['recortada', 'mediana', 'A', 'D', 'C', 'B']
print(f"    {'campaña':>8} {'pelis':>6} " + ' '.join(f'{c_:>9}' for c_ in cols_m_))
for lt_ in L_17:
    print(f"    {int(lt_):>8} {int(MOD_17.loc[lt_, 'n']):>6} " +
          ' '.join(f'{MOD_17.loc[lt_, c_]:>9.3f}' for c_ in cols_m_))
GM_17 = MOD_17[cols_m_].median()
print(f"    {'MEDIANA':>8} {int(MOD_17['n'].sum()):>6} " + ' '.join(f'{GM_17[c_]:>9.3f}' for c_ in cols_m_))
for x_, y_ in (('B', 'A'), ('B', 'mediana'), ('A', 'mediana')):
    d_ = MOD_17[x_] - MOD_17[y_]
    print(f'    {x_} - {y_}: a favor {int((d_ > 0).sum())} de 7 · dif. medianas '
          f'{GM_17[x_] - GM_17[y_]:+.3f} · mediana dif. {d_.median():+.3f} · '
          f'de {d_.min():+.3f} a {d_.max():+.3f}')

sep('7 · VARIABLES DE FORMA: PELICULAS EN QUE LA VARIABLE QUEDA A 0 EN TODA LA PELICULA')
cand_17 = [c_ for c_ in V.columns if c_ not in ('sample_num', 'indent_num', '_y', 'error')
            and not c_.startswith(('z_', 'v_', 'crudo_')) and c_ not in ('p_max', 'h_max')
            and pd.api.types.is_numeric_dtype(V[c_])]
enB_17 = set(JUEGOS[bB])
VF_17 = {}
for c_ in cand_17:
    cero_ = hueco_ = 0; z0_ok_ = True
    for s_, d_ in M.groupby('sample_num'):
        x_ = d_[c_].to_numpy(float)
        med_ = np.median(x_); mad_ = np.median(np.abs(x_ - med_))
        if np.isnan(mad_):
            hueco_ += 1; afect_ = True
        elif mad_ == 0:
            cero_ += 1; afect_ = True
        else:
            afect_ = False
        if afect_ and f'z_{c_}' in d_.columns:
            z0_ok_ &= bool(np.all(d_[f'z_{c_}'].to_numpy(float) == 0))
    VF_17[c_] = dict(cero=cero_, hueco=hueco_, enB=f'z_{c_}' in enB_17,
                      constante=int(V[c_].nunique(dropna=True) <= 1), z0=z0_ok_)
VF_17 = pd.DataFrame(VF_17).T
npel_17 = M.sample_num.nunique()
print(f"    {'variable':>24} {'en B':>5} {'constante*':>10} {'pel. con 0 (DAM nula)':>22} "
      f"{'pel. con 0 (hueco)':>19} {'z = 0 comprobado':>17}")
for c_, r_ in VF_17.sort_values(['enB', 'cero'], ascending=[True, False]).iterrows():
    print(f"    {c_:>24} {'si' if r_['enB'] else 'no':>5} {'si' if r_['constante'] else 'no':>10} "
          f"{int(r_['cero']):>12} de {npel_17:<6} {int(r_['hueco']):>9} de {npel_17:<6} "
          f"{'si' if r_['z0'] else 'NO':>17}")
print('  * constante en las 9.600 curvas extraidas por la celda 14')
print(f'\n  variables de forma calculadas: {len(cand_17)} · en B: {int(VF_17["enB"].sum())} · '
      f'constantes en bruto: {int(VF_17["constante"].sum())} · '
      f'a 0 en todas las peliculas sin ser constantes: '
      f'{int(((VF_17["cero"] + VF_17["hueco"] == npel_17) & (VF_17["constante"] == 0)).sum())}')
print('  variables por modelo: ' + ' · '.join(f'{k_.split(" · ")[0]} {len(v_)}' for k_, v_ in JUEGOS.items()))
rep_ = [c_ for c_ in JUEGOS[bB] if c_ in ('z_h_max', 'z_p_max')]
print(f'  repetidas en B (valores de A con otro nombre): {rep_}')

sep('8 · SHAP: EXPLICADOR USADO EN LA CELDA 16 (A2)')
if 'ex' in globals():
    print(f'  explicador: {type(ex).__module__}.{type(ex).__name__}')
    ev_ = getattr(ex, 'expected_value', None)
    print(f'  valor esperado: {ev_}')
else:
    print('  no hay explicador en la sesion (la celda 16 no calculo SHAP)')


1 · DUREZA: RECUPERACION POR CAMPAÑA (peliculas evaluables) Y MEDIANA DE LAS SIETE
     campaña  pelis recortada   mediana      mad3     regla         A  reducido         D         C         B
         609      5     0.478     0.853     0.510     0.962     0.723     0.838     0.839     0.946     0.932
         614     15     0.367     0.598     0.440     0.769     0.622     0.904     0.858     0.943     0.925
         615      6    -0.036     0.266     0.000     0.170     0.263     0.631     0.378     0.729     0.695
         619      6    -0.007     0.229     0.193     0.089     0.545     0.830     0.675     0.743     0.537
         622      4     0.243     0.602     0.135     0.609     0.553     0.686     0.611     0.680     0.611
         623      4     0.425     0.651     0.652     0.918     0.874     0.823     0.789     0.857     0.957
         626      6     0.310     0.685     0.389     0.937     0.806     0.887     0.919     0.916     0.921
     MEDIANA     46     0.310     0.

                    variable  en B constante*  pel. con 0 (DAM nula)  pel. con 0 (hueco)  z = 0 comprobado
                       n_pts    no         si          139 de 139            0 de 139                   si
              frac_no_finita    no         si          139 de 139            0 de 139                   si
                     t_total    no         si          139 de 139            0 de 139                   si
                  frac_carga    no         no          139 de 139            0 de 139                   si
                   frac_sost    no         no          139 de 139            0 de 139                   si
                   frac_desc    no         no          139 de 139            0 de 139                   si
              frac_congelada    no         si          139 de 139            0 de 139                   si
          sost_retroceso_rel    no         no          139 de 139            0 de 139                   si
                      sost_n    no   

**Cómo leer la salida**

- Bloque 7: la celda 14 divide por la desviación absoluta mediana de cada película. Si vale 0, o si algún ensayo de la película no tiene valor (la mediana con huecos no se calcula), la variable normalizada queda a 0 en todos los ensayos de esa película.
- Bloque 8: con `TreeExplainer`, los valores SHAP están en la escala de la salida bruta del modelo (log-odds); el explicador genérico de respaldo trabaja sobre la probabilidad.
- Todo es descriptivo: no cambia ningún criterio ni ninguna lectura de las celdas 15 y 16 (F1 a F3, R1 a R3, Q1 y Q2).

## Ilustraciones de la memoria

La celda 18 dibuja las ilustraciones de la memoria con los objetos de las celdas 14, 15 y 16, que deben haberse ejecutado antes en la misma sesión. No calcula ningún resultado nuevo. Guarda cada figura en `/content/figuras` con el número que lleva en la memoria.

### Celda 18. Las ilustraciones de la memoria

Dibuja las 17 ilustraciones de la memoria, numeradas como en ella, y guarda cada una en `/content/figuras` con ese número (`ilustracion_01_anatomia_ensayo.png` a `ilustracion_17_vecinos_de_la_rejilla.png`). No calcula ningún resultado nuevo: dibuja lo que calculan las celdas 15 y 16 y, cuando hace falta para dibujar, recalcula algo pequeño con las mismas reglas. Imprime una línea por ilustración guardada y, al final, cuántas se han dibujado; si una falla, indica el error y sigue con la siguiente.

Necesita las celdas 14, 15 y 16 ejecutadas antes en la misma sesión; de la 14 usa la función que ajusta la ley de potencias. Lee los ficheros que guarda la celda 16 en `/content` (`importancia.csv`, `shap_valores.npz`, `reentrenar_sin_zona.csv` y `estabilidad_numerica.csv`) y, de la carpeta de datos, las curvas de los ensayos que dibuja.

Criterios:

- **Estilo.** Colores principales: naranja, modelo B; azul, modelo A; gris, reglas y referencias; tinta oscura y gris claro, ensayos descartados por el especialista y válidos. Naranja y azul se distinguen también con daltonismo; el rojo no se usa. En la ilustración 8, violeta, verde azulado y amarillo distinguen los tipos de defecto. Rótulos con mayúscula inicial, coma decimal y los nombres de la memoria («Modelo A», «Modelo B», «Mediana de los 64 ensayos», «campaña»), sin título general dentro de la figura. Todas tienen el ancho de texto de la página (16 cm), para que la letra tenga el mismo tamaño en todas.
- **Ensayo de ejemplo (ilustraciones 1 y 10).** En la primera película evaluable, el ensayo sin ninguna marca, de ningún nivel, cuya profundidad máxima está más cerca de la mediana de los ensayos sin marca de esa película.
- **Ejemplos de defecto (ilustración 4).** Para cada tipo, entre los ensayos con ese tipo en nivel 2 o más y ningún otro, el de nivel más alto; a igual nivel, el de la película con más ensayos válidos. En el desplazamiento de fuerza, a igual nivel se prefiere el de mayor fuerza al comienzo respecto a su película.
- **Valor con los descartes del modelo (ilustración 5).** La misma regla de la celda 15: si quedan menos de 3 ensayos, se conservan los 3 de menor probabilidad.
- **Sentido de cada tipo de defecto (ilustración 7).** Criterio de la celda 5: al menos dos tercios de las películas con el mismo signo, prueba de signos con p < 0,01 y el signo de la mediana por campaña (campañas con al menos 3 películas) igual al mayoritario en todas salvo una como mucho.
- **Película de las reglas (ilustración 8).** La del ensayo de ejemplo de las ilustraciones 1 y 10, que es la primera película evaluable. Las tres reglas se calculan como en la celda 7: la mediana de los 64 ensayos, la media de los ensayos que la regla de la desviación absoluta mediana con umbral 3 considera normales y la media recortada al 10 % por cada lado.
- **Ensayos de la ilustración 11.** En la película del ejemplo de desplazamiento de fuerza de la ilustración 4, ese mismo ensayo y, para la curvatura anómala de la descarga, el ensayo con ese tipo en el nivel más alto y ningún otro; a igual nivel, el de número menor. El ajuste de la descarga se calcula como en la celda 14.
- **Número de variables por zona (ilustración 10).** Se cuentan las variables de forma del modelo B según la zona que les asigna la celda 16.
- **Esquemas sin datos (ilustraciones 2, 3 y 17).** El contacto, las fases del trabajo y los vecinos de la rejilla. El panel b de la ilustración 12 es un ejemplo con valores inventados, que muestra cómo se fija el umbral de descarte.


In [19]:
DIR = '/content/figuras'
F_IMP = '/content/importancia.csv'
F_SHAP = '/content/shap_valores.npz'
F_ZONA = '/content/reentrenar_sin_zona.csv'
F_ESTAB = '/content/estabilidad_numerica.csv'
os.makedirs(DIR, exist_ok=True)

FICH = {1: 'ilustracion_01_anatomia_ensayo', 2: 'ilustracion_02_esquema_contacto',
        3: 'ilustracion_03_fases_del_trabajo', 4: 'ilustracion_04_tipos_de_defecto',
        5: 'ilustracion_05_como_se_mide_el_acierto', 6: 'ilustracion_06_cambio_al_limpiar',
        7: 'ilustracion_07_sentido_por_tipo', 8: 'ilustracion_08_reglas_en_una_pelicula',
        9: 'ilustracion_09_descartados_no_extremos', 10: 'ilustracion_10_lo_que_recibe_cada_modelo',
        11: 'ilustracion_11_arranque_y_descarga', 12: 'ilustracion_12_validacion',
        13: 'ilustracion_13_resultado_principal', 14: 'ilustracion_14_distancia_al_especialista',
        15: 'ilustracion_15_de_donde_viene_la_ventaja', 16: 'ilustracion_16_como_decide',
        17: 'ilustracion_17_vecinos_de_la_rejilla'}

TINTA, SEC, MUDO, EJE, REJ = '#0b0b0b', '#52514e', '#898781', '#c3c2b7', '#e1e0d9'
AZUL, NARANJA = '#2a78d6', '#eb6834'
DESC, VALIDO, REF = '#2b2a27', '#cfcdc6', '#8f8d86'
VALIDO_P = '#b4b2ab'
BANDA = '#ebe9e3'
ANCHO = 6.3

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.03,
    'font.size': 8.5, 'axes.labelsize': 8.5, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.edgecolor': EJE, 'axes.linewidth': 0.8, 'axes.labelcolor': SEC, 'text.color': TINTA,
    'xtick.color': SEC, 'ytick.color': SEC, 'xtick.major.size': 3, 'ytick.major.size': 3,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False,
    'legend.frameon': False, 'legend.fontsize': 8, 'lines.solid_capstyle': 'round',
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.titlesize': 8.5,
})

def nf(x, d=2, signo=False):
    if x is None or not np.isfinite(x):
        return '–'
    if round(float(x), d) == 0:
        x = 0.0
    s = f'{x:+,.{d}f}' if signo else f'{x:,.{d}f}'
    return s.replace(',', '§').replace('.', ',').replace('§', '.').replace('-', '−')

class FmtES(Formatter):
    def __init__(self, d=None, pre='', suf=''):
        self.d, self.pre, self.suf = d, pre, suf

    def __call__(self, x, pos=None):
        d = self.d
        if d is None:
            d = 0
            for v in getattr(self, 'locs', []):
                for k in range(4):
                    if abs(v * 10 ** k - round(v * 10 ** k)) < 1e-6:
                        d = max(d, k)
                        break
                else:
                    d = 3
        return self.pre + nf(x, d) + self.suf

def eje_es(ax, eje='both', d=None):
    if eje in ('x', 'both'):
        ax.xaxis.set_major_formatter(FmtES(d))
    if eje in ('y', 'both'):
        ax.yaxis.set_major_formatter(FmtES(d))

def envuelve(t, n):
    return '\n'.join(textwrap.wrap(t, n))

def caja(fig):
    r = fig.canvas.get_renderer()
    bbs = [fig.get_tightbbox(r)]
    for a in fig.axes:
        for lab in (a.xaxis.label, a.yaxis.label, a.title, getattr(a, '_left_title', a.title),
                    getattr(a, '_right_title', a.title)):
            if lab.get_visible() and lab.get_text():
                bbs.append(lab.get_window_extent(r).transformed(fig.dpi_scale_trans.inverted()))
    return Bbox.union(bbs)

def fija_ancho(fig):
    for _ in range(10):
        fig.canvas.draw()
        w = caja(fig).width
        if abs(w - ANCHO) < 0.01:
            break
        W, H = fig.get_size_inches()
        fig.set_size_inches(max(W + 1.15 * (ANCHO - w), 1.0), H)

HECHAS = {}

def guarda(fig, n):
    fija_ancho(fig)
    fig.canvas.draw()
    f = os.path.join(DIR, f'{FICH[n]}.png')
    fig.savefig(f, bbox_inches=caja(fig).padded(0.03))
    plt.close(fig)
    HECHAS[n] = os.path.basename(f)
    print(f'  Ilustracion {n:>2} · {os.path.basename(f)}')

def fallo(n):
    print(f'  [Ilustracion {n}] no se pudo dibujar:')
    print('    ' + traceback.format_exc().strip().split('\n')[-1][:160])
    plt.close('all')

class Rotulos:
    G = 2.0

    def __init__(self, ax, xs, ys, peso=1.0):
        self.ax = ax
        self.r = ax.figure.canvas.get_renderer()
        self.A = ax.get_window_extent(self.r)
        self.nx, self.ny = int(self.A.width / self.G) + 1, int(self.A.height / self.G) + 1
        self.W = np.zeros((self.ny, self.nx))
        self.cajas = []
        self.anade(xs, ys, peso)

    def anade(self, xs, ys, peso=1.0):
        xs, ys = np.asarray(xs, float), np.asarray(ys, float)
        ok = np.isfinite(xs) & np.isfinite(ys)
        if ok.any():
            P = self.ax.transData.transform(np.column_stack([xs[ok], ys[ok]]))
            d = np.hypot(*np.diff(P, axis=0).T)
            k = np.maximum(1, np.ceil(d / self.G)).astype(int)
            k[d > 60] = 1
            Q = [P[:1]] + [P[i] + np.outer(np.arange(1, k[i] + 1) / k[i], P[i + 1] - P[i])
                           for i in range(len(k))]
            P = np.vstack(Q)
            ix = ((P[:, 0] - self.A.x0) / self.G).astype(int)
            iy = ((P[:, 1] - self.A.y0) / self.G).astype(int)
            v = (ix >= 0) & (ix < self.nx) & (iy >= 0) & (iy < self.ny)
            np.add.at(self.W, (iy[v], ix[v]), peso)
        self.S = np.pad(self.W.cumsum(0).cumsum(1), ((1, 0), (1, 0)))
        D = self.W.copy()
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                D = np.maximum(D, np.roll(np.roll(self.W, dy, 0), dx, 1))
        self.D = np.minimum(D, 1.0) * (D > 0)

    def _suma(self, x0, x1, y0, y1):
        i0 = int(np.clip((x0 - self.A.x0) / self.G, 0, self.nx))
        i1 = int(np.clip((x1 - self.A.x0) / self.G, 0, self.nx))
        j0 = int(np.clip((y0 - self.A.y0) / self.G, 0, self.ny))
        j1 = int(np.clip((y1 - self.A.y0) / self.G, 0, self.ny))
        return self.S[j1, i1] - self.S[j0, i1] - self.S[j1, i0] + self.S[j0, i0]

    def _cruces(self, cx, cy, ax_, ay_, caja_):
        L = np.hypot(ax_ - cx, ay_ - cy)
        if L < 1:
            return 0.0
        t = np.arange(0, L, 2.5) / L
        px, py = cx + t * (ax_ - cx), cy + t * (ay_ - cy)
        x0, x1, y0, y1 = caja_
        cerca = max(10.0, 0.12 * L)
        fuera = ~((px >= x0) & (px <= x1) & (py >= y0) & (py <= y1)) & \
            (np.hypot(ax_ - px, ay_ - py) > cerca)
        ix = ((px[fuera] - self.A.x0) / self.G).astype(int).clip(0, self.nx - 1)
        iy = ((py[fuera] - self.A.y0) / self.G).astype(int).clip(0, self.ny - 1)
        return float(self.D[iy, ix].sum())

    def _busca(self, w, h, ancla=None, n=28):
        A = self.A
        mejor, pm = None, np.inf
        for fx in np.linspace(0, 1, n):
            for fy in np.linspace(0, 1, n):
                cx = A.x0 + w / 2 + fx * (A.width - w)
                cy = A.y0 + h / 2 + fy * (A.height - h)
                cj = (cx - w / 2, cx + w / 2, cy - h / 2, cy + h / 2)
                pen = 3.0 * self._suma(*cj)
                for (a0, a1, b0, b1) in self.cajas:
                    ox = max(0, min(cj[1], a1) - max(cj[0], a0))
                    oy = max(0, min(cj[3], b1) - max(cj[2], b0))
                    pen += 5 * ox * oy
                if ancla is not None:
                    dd_ = np.hypot(cx - ancla[0], cy - ancla[1]) / A.width
                    pen += 50 * dd_ + 120 * max(0.0, dd_ - 0.35)
                    if cj[0] <= ancla[0] <= cj[1] and cj[2] <= ancla[1] <= cj[3]:
                        pen += 400
                    pen += 12 * self._cruces(cx, cy, ancla[0], ancla[1], cj)
                    for (a0, a1, b0, b1) in self.cajas:
                        for t in np.linspace(0.15, 0.95, 6):
                            px, py = cx + t * (ancla[0] - cx), cy + t * (ancla[1] - cy)
                            if a0 <= px <= a1 and b0 <= py <= b1:
                                pen += 40
                if pen < pm:
                    pm, mejor = pen, (cx, cy) + cj
        return mejor

    def pon(self, txt, xy, fs=7.5, color=SEC, fondo=False):
        ax = self.ax
        t = ax.text(0, 0, txt, fontsize=fs, ha='center', va='center', transform=ax.transAxes)
        bb = t.get_window_extent(self.r); t.remove()
        m = 4
        a = ax.transData.transform(xy)
        cx, cy, x0, x1, y0, y1 = self._busca(bb.width + 2 * m, bb.height + 2 * m, a)
        self.cajas.append((x0, x1, y0, y1))
        fx, fy = ax.transAxes.inverted().transform((cx, cy))
        ap = dict(arrowstyle='-', color=SEC, lw=0.6, shrinkA=1, shrinkB=2)
        bbox = dict(boxstyle='square,pad=0.12', fc='white', ec='none', alpha=0.85) if fondo else None
        ax.annotate(txt, xy=xy, xytext=(fx, fy), textcoords='axes fraction', fontsize=fs, color=color,
                    ha='center', va='center', arrowprops=ap, zorder=6, bbox=bbox)

    def hueco(self, wf, hf):
        A = self.A
        cx, cy, x0, x1, y0, y1 = self._busca(wf * A.width, hf * A.height)
        self.cajas.append((x0, x1, y0, y1))
        return (x0 - A.x0) / A.width, (y0 - A.y0) / A.height

def rotulo(ax, txt, xy, xyt, ha='center', fs=7.5):
    ax.annotate(txt, xy=xy, xytext=xyt, textcoords='data', ha=ha, va='center', fontsize=fs,
                color=SEC, arrowprops=dict(arrowstyle='-', color=SEC, lw=0.6, shrinkA=2, shrinkB=2),
                zorder=6)

def marcas_hasta(ax, vmax, eje='y'):
    a = ax.yaxis if eje == 'y' else ax.xaxis
    lim = ax.get_ylim() if eje == 'y' else ax.get_xlim()
    tk = [v for v in MaxNLocator(6).tick_values(*lim) if lim[0] <= v <= vmax * 1.05]
    a.set_major_locator(FixedLocator(tk))

def tarjeta(ax, x0, y0, w, h):
    ax.add_patch(Rectangle((x0, y0), w, h, transform=ax.transAxes, fc='white', ec=EJE, lw=0.5,
                           zorder=4, clip_on=False))

def sin_bordes(ax, lados=('top', 'right')):
    for sp in lados:
        ax.spines[sp].set_visible(False)

POR6 = ['force_offset', 'displacement_offset', 'unusual_loading_curvature',
        'tip_displacement_decreases_at_holding', 'unusual_unloading_curvature', 'too_deep']
TIPO = {'force_offset': 'Desplazamiento de fuerza',
        'displacement_offset': 'Desplazamiento de profundidad',
        'unusual_unloading_curvature': 'Curvatura anómala de la descarga',
        'unusual_loading_curvature': 'Curvatura anómala de la carga',
        'tip_displacement_decreases_at_holding': 'Retroceso de la punta',
        'too_deep': 'Penetración excesiva'}
ZONA = {'arranque de la curva (offset)': 'Arranque', 'tramo de carga': 'Carga',
        'sostenimiento': 'Sostenimiento', 'tramo de descarga': 'Descarga',
        'final de la curva': 'Final', 'calidad / ruido': 'Calidad (ruido)',
        'valor del laboratorio': 'Valores del laboratorio', 'reparto de tramos': 'Otras',
        'otras': 'Otras'}
ORDEN_ZONA = ['Valores del laboratorio', 'Arranque', 'Carga', 'Sostenimiento', 'Descarga',
              'Final', 'Calidad (ruido)', 'Otras']
VAR = {
    'p_ini_frac': 'fuerza en el primer punto', 'p_ini_p5_frac': 'fuerza al comienzo de la curva',
    'h_ini_frac': 'profundidad en el primer punto',
    'h_ini_p5_frac': 'profundidad al comienzo de la curva',
    'p_fin_frac': 'fuerza en el último punto', 'h_fin_frac': 'profundidad en el último punto',
    'carga_exp': 'exponente de la carga', 'carga_r2': 'calidad del ajuste de la carga',
    'carga_resid_rel': 'desajuste de la carga',
    'carga_no_monot': 'pasos en que la fuerza baja en la carga',
    'carga_curv_med': 'curvatura de la carga',
    'carga_curv_iqr': 'dispersión de la curvatura de la carga',
    'sost_fluencia_rel': 'avance de la profundidad al sostener',
    'sost_retroceso_rel': 'mayor retroceso al sostener',
    'sost_rango_rel': 'recorrido de la profundidad al sostener',
    'sost_pendiente_neg': 'pasos en que la profundidad baja al sostener',
    'desc_hf_rel': 'profundidad que queda al retirar la fuerza',
    'desc_r2': 'calidad del ajuste de la descarga', 'desc_resid_rel': 'desajuste de la mitad superior de la descarga',
    'desc_resid_bajo_rel': 'desajuste de toda la descarga',
    'desc_sesgo_bajo_rel': 'desajuste con signo de la descarga',
    'rigidez_fit': 'rigidez del ajuste de la descarga', 'rigidez_razon': 'relación entre las dos rigideces',
    'desc_no_monot': 'pasos en que la fuerza sube en la descarga', 'desc_curv_med': 'curvatura de la descarga',
    'ruido_rel': 'ruido de la fuerza', 'h_max': 'profundidad máxima', 'p_max': 'fuerza máxima'}

def nombre_var(c):
    base = c[2:] if c.startswith(('z_', 'a_')) else c
    return VAR.get(base, base.replace('_', ' '))

def lista_es(xs):
    xs = list(xs)
    return xs[0] if len(xs) == 1 else ', '.join(xs[:-1]) + ' y ' + xs[-1]

def mayus(s):
    return s[:1].upper() + s[1:]

H_, P_, T_ = 'depth_nm', 'load_micro_N', 'time_s'
NIV6 = M[POR6].apply(pd.to_numeric, errors='coerce').fillna(0)
VALIDOS = (M['_y'] == 0).values
FAC_H = float(FAC_ESP.get('hardness_H_GPa', 1.0))
FIAB = B_['fiable'].astype(bool)
N_FIAB = int(FIAB.sum())

def curva(s, i):
    r = ALL[(ALL.sample_num == s) & (ALL.indent_num == i)]
    c = pd.read_csv(os.path.join(RAIZ, str(r.path.iloc[0])), usecols=[H_, P_, T_])
    return c[H_].to_numpy(float), c[P_].to_numpy(float), c[T_].to_numpy(float)

def tramos(p):
    pm = np.nanmax(p); j = np.flatnonzero(p >= 0.98 * pm)
    return int(j[0]), int(j[-1])

def en_fuerza(p, frac, rama):
    i0, i1 = tramos(p)
    idx = np.arange(0, i0 + 1) if rama == 'carga' else np.arange(i1, len(p))
    return int(idx[np.argmin(np.abs(p[idx] - frac * np.nanmax(p)))])

def ensayo_tipico():
    s = int(B_[FIAB].index.min())
    lim = ((NIV6 == 0).all(axis=1) & (M['_y'] == 0)).values
    d = M[(M.sample_num == s).values & lim]
    hm = d['h_max'] if 'h_max' in d.columns else d['contact_depth_hc_nm']
    i = int(d.loc[(hm - hm.median()).abs().idxmin(), 'indent_num'])
    return s, i, M[(M.sample_num == s) & (M.indent_num == i)].iloc[0]

print(f'  figuras en {DIR}\n')
S0, I0, R0 = ensayo_tipico()

try:
    h, p, t = curva(S0, I0)
    i0, i1 = tramos(p)
    pm, hm = float(p[i0:i1 + 1].mean()), float(h[i1])
    rig = float(R0['stiffness_S_micro_N/nm'])
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(ANCHO, 2.7), sharey=True,
                                 gridspec_kw=dict(width_ratios=[1.45, 1], wspace=0.12))
    ax.plot(h, p, color=TINTA, lw=1.3, zorder=3)
    tg_p = np.array([0.35 * pm, pm]); tg_h = hm + (tg_p - pm) / rig
    ax.plot(tg_h, tg_p, color=SEC, lw=0.9, ls=(0, (3, 2)), zorder=2)
    ax.plot([hm], [pm], 'o', ms=4.5, color=TINTA, zorder=4)
    H0 = min(0.0, float(np.nanmin(h))); SP = hm - H0
    ax.set_xlim(H0, H0 + 1.3 * SP); ax.set_ylim(0, 1.25 * pm)
    ax.set_xlabel('Profundidad (nm)'); ax.set_ylabel('Fuerza (µN)'); eje_es(ax)
    ax.set_title('(a) Fuerza frente a profundidad', loc='left', color=TINTA, fontsize=8, pad=6)
    bx.axvspan(t[i0], t[i1], color=BANDA, lw=0, zorder=0)
    bx.plot(t, p, color=TINTA, lw=1.3, zorder=3)
    for a_, b_, txt in ((t[0], t[i0], 'Carga'), (t[i0], t[i1], 'Sostenimiento'),
                        (t[i1], t[-1], 'Descarga')):
        bx.text((a_ + b_) / 2, 1.04 * pm, f'{txt}\n{nf(b_ - a_, 2)} s', ha='center', va='bottom',
                fontsize=7.5, color=SEC)
    bx.set_xlabel('Tiempo (s)'); eje_es(bx); bx.tick_params(labelleft=False)
    bx.set_title('(b) El mismo ensayo frente al tiempo', loc='left', color=TINTA, fontsize=8, pad=6)
    marcas_hasta(ax, pm)
    fija_ancho(fig)
    jm = (i0 + i1) // 2
    rotulo(ax, 'Profundidad\nmáxima', (hm, pm), (hm + 0.07 * SP, 0.84 * pm), ha='left')
    rotulo(ax, 'Sostenimiento:\nfuerza máxima constante', (h[jm], pm), (h[jm] - 0.05 * SP, 1.15 * pm))
    xt = hm - 0.4 * pm / rig
    rotulo(ax, 'Pendiente\ninicial de\nla descarga', (xt, 0.6 * pm), (xt + 0.1 * SP, 0.48 * pm), ha='left')
    k = en_fuerza(p, 0.55, 'carga')
    rotulo(ax, 'Carga', (h[k], p[k]), (h[k] - 0.12 * SP, p[k] + 0.12 * pm), ha='right')
    k = en_fuerza(p, 0.2, 'descarga')
    rotulo(ax, 'Descarga', (h[k], p[k]), (h[k] + 0.1 * SP, p[k] - 0.04 * pm), ha='left')
    guarda(fig, 1)
except Exception:
    fallo(1)

try:
    HMAX_, HC_, HF_ = 1.0, 0.75, 0.5
    HS_ = HMAX_ - HC_
    A_ = 2.1
    PEND_ = HC_ / A_
    YTOP_ = 0.42
    XTOP_ = (HMAX_ + YTOP_) / PEND_
    XS_, XC_, L_ = 5.4, 5.95, 1.0
    fig, ax = plt.subplots(figsize=(ANCHO, 2.6))
    ax.set_xlim(-7.3, 7.3); ax.set_ylim(-1.32, 0.46)
    ax.axis('off')
    # Película bajo la fuerza máxima (gris muy claro) y punta
    x_ = np.linspace(A_, XS_, 200)
    y_ = -HS_ * np.exp(-(x_ - A_) / L_)
    xm_ = np.r_[-XC_ - 0.05, -XS_, -x_[::-1], 0, x_, XS_, XC_ + 0.05]
    ym_ = np.r_[0, 0, y_[::-1], -HMAX_, y_, 0, 0]
    ax.fill_between(xm_, ym_, -1.25, color='#f4f3ef', lw=0, zorder=0)
    ax.fill([0, -XTOP_, XTOP_], [-HMAX_, YTOP_, YTOP_], color=BANDA, lw=0, zorder=1)
    ax.plot([-XTOP_, 0, XTOP_], [YTOP_, -HMAX_, YTOP_], color=SEC, lw=0.8, zorder=2)
    ax.text(0, 0.24, 'Punta', ha='center', va='center', fontsize=7.5, color=SEC, zorder=6)
    ax.text(XC_ - 0.4, -1.12, 'Película', ha='right', va='center', fontsize=7.5, color=SEC, zorder=6)
    # Hueco sin contacto: entre la superficie hundida y la cara de la punta o el nivel inicial
    for sg in (-1, 1):
        techo = np.minimum(0.0, -HMAX_ + PEND_ * x_)
        ax.fill_between(sg * x_, y_, techo, where=techo > y_, facecolor='none', edgecolor=MUDO, hatch='////',
                        lw=0, zorder=2)
    ax.plot([-XC_ - 0.05, XC_ + 0.05], [0, 0], color=MUDO, lw=0.8, ls=(0, (1, 2)), zorder=3)
    ax.plot(x_, y_, color=TINTA, lw=1.3, zorder=4)
    ax.plot(-x_, y_, color=TINTA, lw=1.3, zorder=4)
    # Zona en contacto: las caras de la punta por debajo del borde del contacto, con trazo grueso
    ax.plot([-A_, 0, A_], [-HS_, -HMAX_, -HS_], color=TINTA, lw=2.6, zorder=4, solid_joinstyle='miter')
    ax.plot([-A_, A_], [-HS_, -HS_], 'o', ms=3.6, color=TINTA, zorder=5)
    ax.plot([-A_, 0, A_], [0, -HF_, 0], color=TINTA, lw=1.1, ls=(0, (4, 2.5)), zorder=4)

    def cota(xc, y0, y1, txt, lado):
        ax.annotate('', xy=(xc, y1), xytext=(xc, y0),
                    arrowprops=dict(arrowstyle='<|-|>', color=SEC, lw=0.7, mutation_scale=6,
                                    shrinkA=0, shrinkB=0), zorder=5)
        dx = 0.14 if lado == 'der' else -0.14
        ax.text(xc + dx, (y0 + y1) / 2, txt, ha='left' if lado == 'der' else 'right', va='center',
                fontsize=8.5, color=TINTA, zorder=6)

    for x0_, x1_, yy_ in ((-XC_ - 0.05, 0, -HMAX_), (0, XC_ + 0.05, -HMAX_), (A_, XC_ + 0.05, -HS_)):
        ax.plot([x0_, x1_], [yy_, yy_], color=MUDO, lw=0.6, ls=(0, (1, 2)), zorder=0)
    cota(-XC_, 0, -HMAX_, r'$h_{max}$', 'izq')
    cota(XC_, 0, -HS_, r'$\varepsilon P_{max}/S$', 'der')
    cota(XC_, -HS_, -HMAX_, r'$h_c$', 'der')
    cota(0, 0, -HF_, r'$h_f$', 'der')
    # Rótulos de las partes del contacto, en el lado izquierdo
    xg_ = -(A_ + 0.45); yg_ = 0.5 * (-HS_ * np.exp(-0.45 / L_) + min(0.0, -HMAX_ + PEND_ * (A_ + 0.45)))
    rotulo(ax, 'Hueco sin contacto', (xg_, yg_), (-5.15, 0.27), ha='center', fs=7.5)
    rotulo(ax, 'Borde del contacto', (-A_, -HS_), (-4.35, -0.62), ha='center', fs=7.5)
    xz_ = -0.45 * A_; rotulo(ax, 'Zona en contacto', (xz_, -HMAX_ + PEND_ * 0.45 * A_), (-2.75, -1.12), ha='center', fs=7.5)
    ley = [Line2D([], [], color=MUDO, lw=0.8, ls=(0, (1, 2))), Line2D([], [], color=TINTA, lw=1.3),
           Line2D([], [], color=TINTA, lw=1.1, ls=(0, (4, 2.5)))]
    ax.legend(ley, ['Superficie inicial', 'Superficie bajo la fuerza máxima', 'Superficie tras la descarga'],
              loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=3, fontsize=7.5, handlelength=2.6,
              columnspacing=2.2)
    guarda(fig, 2)
except Exception:
    fallo(2)

try:
    FASES = [('O1', '¿Pueden usarse los datos?', '4.3.1'),
             ('O2', '¿Cambia el valor publicado al limpiar?', '4.3.2'),
             ('O3', '¿Qué parte de la corrección logran las reglas estadísticas?', '4.3.3'),
             ('O4', '¿Se limita el especialista a quitar valores extremos?', '4.3.4'),
             ('O5', '¿Cómo describir la forma de la curva con variables?', '4.3.5'),
             ('O6', '¿Mejora la forma de la curva el resultado en campañas no vistas?', '4.3.6 y\n4.4.1 a 4.4.3'),
             ('O7', '¿En qué se apoya el modelo que la usa?', '4.4.4'),
             ('O8', '¿Depende el resultado de las decisiones del análisis?', '4.4.5'),
             ('O9', '¿Hasta dónde llega el resultado?', '4.4.6')]
    fig, ax = plt.subplots(figsize=(ANCHO, 3.35))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    H3, G3, TOP = 0.128, 0.034, 0.835
    COLS = [(0.0, 0.465, FASES[:4], '¿Tiene sentido el problema?', 'Objetivos O1 a O4, apartados 4.3.1 a 4.3.4'),
            (0.535, 1.0, FASES[4:], '¿Aporta la forma de la curva?', 'Objetivos O5 a O9, apartados 4.3.5 a 4.4.6')]
    cajas3 = {}
    for x0, x1, fases, tit, sub in COLS:
        ax.text(x0, 0.995, tit, fontsize=8.5, color=TINTA, va='top', ha='left')
        ax.text(x0, 0.925, sub, fontsize=7.5, color=SEC, va='top', ha='left')
        for k, (o, q, ap) in enumerate(fases):
            yt = TOP - k * (H3 + G3)
            ax.add_patch(Rectangle((x0, yt - H3), x1 - x0, H3, fc=BANDA, ec='none', zorder=1))
            ax.text(x0 + 0.012, yt - H3 / 2, o, fontsize=7.5, color=SEC, va='center', ha='left', zorder=2)
            ax.text(x0 + 0.058, yt - H3 / 2, envuelve(q, 34), fontsize=7.5, color=TINTA, va='center', ha='left',
                    linespacing=1.15, zorder=2)
            ax.text(x1 - 0.012, yt - H3 / 2, ap, fontsize=7, color=SEC, va='center', ha='right',
                    linespacing=1.15, zorder=2)
            cajas3[o] = (x0, x1, yt - H3, yt)
            if k:
                ax.annotate('', xy=((x0 + x1) / 2, yt), xytext=((x0 + x1) / 2, yt + G3),
                            arrowprops=dict(arrowstyle='-|>', color=MUDO, lw=0.8, mutation_scale=7), zorder=2)
    a0, a1, b0, b1 = cajas3['O4']; c0, c1, d0, d1 = cajas3['O5']
    ax.annotate('', xy=(c0, (d0 + d1) / 2), xytext=(a1, (b0 + b1) / 2),
                arrowprops=dict(arrowstyle='-|>', color=MUDO, lw=0.8, mutation_scale=7,
                                connectionstyle='arc3,rad=0.0'), zorder=2)
    guarda(fig, 3)
except Exception:
    fallo(3)

try:
    Z_INI = 'z_p_ini_p5_frac'

    def elige(c):
        lv = NIV6[c]; otros = (NIV6.drop(columns=c) >= 2).any(axis=1)
        cand = M[(lv >= 2) & ~otros]
        if not len(cand):
            cand = M[lv >= 2]
        nval = M[M['_y'] == 0].groupby('sample_num').size()
        cand = cand.assign(_lv=lv[cand.index], _nv=cand.sample_num.map(nval).fillna(0))
        orden, asc = ['_lv', '_nv', 'sample_num', 'indent_num'], [False, False, True, True]
        if c == 'force_offset' and Z_INI in cand.columns:
            cand = cand.assign(_z=pd.to_numeric(cand[Z_INI], errors='coerce').fillna(-np.inf))
            orden, asc = ['_lv', '_z'] + orden[1:], [False, False] + asc[1:]
        r = cand.sort_values(orden, ascending=asc).iloc[0]
        return int(r.sample_num), int(r.indent_num)

    def mayor_desvio(h, p, val, rama):
        def rama_hp(hh, pp):
            a0, a1 = tramos(pp)
            idx = np.arange(0, a0 + 1) if rama == 'carga' else np.arange(a1, len(pp))
            o = np.argsort(pp[idx])
            return pp[idx][o], hh[idx][o], idx[o]
        pd_, hd_, id_ = rama_hp(h, p)
        ref = [np.interp(pd_, *rama_hp(hh, pp)[:2], left=np.nan, right=np.nan) for hh, pp in val]
        ref = np.nanmedian(np.vstack(ref), axis=0)
        dv = np.abs(hd_ - ref); dv[~np.isfinite(dv)] = -1
        dv[(pd_ < 0.08 * np.nanmax(p)) | (pd_ > 0.95 * np.nanmax(p))] = -1
        return int(id_[int(np.argmax(dv))])

    NOTA = {'force_offset': ('La fuerza inicial\nno es nula', 'zoom'),
            'displacement_offset': ('Tramo inicial\ncasi sin fuerza', 'ini'),
            'unusual_loading_curvature': ('La carga\ncambia de forma', 'carga'),
            'tip_displacement_decreases_at_holding': ('La punta retrocede\nal sostener la fuerza', 'sost'),
            'unusual_unloading_curvature': ('Descarga con\notra curvatura', 'descarga'),
            'too_deep': ('Llega más\nhondo', 'max')}
    EJ = {}
    for c in POR6:
        s, i = elige(c)
        val = [curva(s, int(ii)) for ii in M[(M.sample_num == s) & (M['_y'] == 0)].indent_num]
        EJ[c] = (s, i, curva(s, i), val)
    pms = [np.nanmax(EJ[c][2][1]) for c in POR6]
    comparte = max(pms) / min(pms) < 1.3
    fig, axs = plt.subplots(2, 3, figsize=(ANCHO, 4.7), sharey=comparte,
                            gridspec_kw=dict(hspace=0.62, wspace=0.10 if comparte else 0.32))
    for ax, c in zip(axs.ravel(), POR6):
        s, i, (h, p, t), val = EJ[c]
        for hh, pp, _ in val:
            ax.plot(hh, pp, color=VALIDO, lw=0.5, zorder=1)
        ax.plot(h, p, color=DESC, lw=1.3, zorder=3)
        hx = max([np.nanmax(h)] + [np.nanmax(v[0]) for v in val])
        hn = min([0, np.nanmin(h)] + [np.nanmin(v[0]) for v in val])
        ax.set_xlim(hn, hn + 1.18 * (hx - hn))
        ax.set_ylim(min(0, np.nanmin(p)), 1.12 * (max(pms) if comparte else np.nanmax(p)))
        ax.set_title(envuelve(TIPO[c], 22), loc='left', color=TINTA, fontsize=8, pad=4)
        eje_es(ax)
    for ax in axs[:, 0]:
        ax.set_ylabel('Fuerza (µN)')
    for ax in axs[1, :]:
        ax.set_xlabel('Profundidad (nm)')
    fig.legend(handles=[Line2D([], [], color=VALIDO, lw=1.6, label='Ensayos válidos de la misma película'),
                        Line2D([], [], color=DESC, lw=1.6, label='Ensayo con el defecto')],
               loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.0))
    fija_ancho(fig)
    for ax, c in zip(axs.ravel(), POR6):
        s, i, (h, p, t), val = EJ[c]
        txt, donde = NOTA[c]
        hv_, pv_ = np.concatenate([v[0] for v in val]), np.concatenate([v[1] for v in val])
        if donde == 'zoom':
            R = Rotulos(ax, h, p)
            R.anade(hv_, pv_, peso=0.04)
            caja_ = R._busca(0.60 * R.A.width, 0.58 * R.A.height)
            if R._suma(*caja_[2:]) > 1.0:
                x0_, x1_ = ax.get_xlim()
                ax.set_xlim(x0_, x0_ + (x1_ - x0_) / 1.18 / 0.38)
        R = Rotulos(ax, h, p)
        R.anade(hv_, pv_, peso=0.04)
        i0, i1 = tramos(p)
        if donde == 'sost':
            x0, y0 = R.hueco(0.60, 0.46)
            tarjeta(ax, x0, y0, 0.60, 0.46)
            ins = ax.inset_axes([x0 + 0.22, y0 + 0.05, 0.36, 0.25], zorder=5)
            for hh, pp, tt in val:
                a0, a1 = tramos(pp)
                ins.plot(tt[a0:a1 + 1] - tt[a0], hh[a0:a1 + 1] - hh[a0], color=VALIDO, lw=0.5)
            ins.plot(t[i0:i1 + 1] - t[i0], h[i0:i1 + 1] - h[i0], color=DESC, lw=1.1)
            ins.set_title(txt, fontsize=6.5, color=SEC, pad=2)
            ins.tick_params(labelsize=6, length=2); ins.set_xticks([])
            ins.set_ylabel('Profundidad\n(nm)', fontsize=6, labelpad=1)
            sin_bordes(ins)
            eje_es(ins, 'y')
            continue
        if donde == 'zoom':
            nz = 2 * max(3, len(p) // 50)
            x0, y0 = R.hueco(0.60, 0.58)
            tarjeta(ax, x0, y0, 0.60, 0.58)
            ins = ax.inset_axes([x0 + 0.22, y0 + 0.17, 0.36, 0.27], zorder=5)
            for hh, pp, _ in val:
                ins.plot(hh[:nz], pp[:nz], color=VALIDO, lw=0.5)
            ins.plot(h[:nz], p[:nz], color=DESC, lw=1.1)
            ins.axhline(0, color=EJE, lw=0.6, zorder=0)
            ins.set_title(txt, fontsize=6.5, color=SEC, pad=2)
            ins.tick_params(labelsize=6, length=2)
            ins.set_ylabel('Fuerza\n(µN)', fontsize=6, labelpad=1)
            ins.set_xlabel('Profundidad (nm)', fontsize=6, labelpad=1)
            sin_bordes(ins)
            eje_es(ins)
            continue
        if donde == 'ini':
            j5 = int(np.argmax(p > 0.05 * np.nanmax(p)))
            j = j5 // 2 if j5 > 4 else 0
            xy = (h[j], p[j])
        elif donde == 'max':
            k = int(np.nanargmax(h)); xy = (h[k], p[k])
        else:
            k = mayor_desvio(h, p, [(v[0], v[1]) for v in val], donde); xy = (h[k], p[k])
        R.pon(txt, xy, fs=7, fondo=True)
    guarda(fig, 4)
except Exception:
    fallo(4)

try:
    def valor_modelo(s, nom):
        d = M[M.sample_num == s]; lt = LOTE_DE[s]
        pp = pred_all[nom].loc[d.index].values
        pr = (pp >= UMB[nom][lt]).astype(int)
        quita = pr == 1
        if (~quita).sum() < 3:
            o = np.argsort(pp, kind='stable')
            quita = np.ones(len(pr), bool); quita[o[:3]] = False
        return float(d['hardness_H_GPa'].astype(float).values[~quita].mean())

    fiab = B_[FIAB].astype(float)
    rB = pd.concat([res[bB][l] for l in lotes]).reindex(fiab.index)
    objetivo = float(np.nanmedian(rB))
    Xs_ = {s: valor_modelo(s, bB) for s in fiab.index}
    entre = [s for s in fiab.index
             if np.sign(Xs_[s] - fiab.loc[s, 'A']) == np.sign(fiab.loc[s, 'E'] - fiab.loc[s, 'A'])
             and abs(Xs_[s] - fiab.loc[s, 'A']) <= abs(fiab.loc[s, 'E'] - fiab.loc[s, 'A'])]
    cand = (rB.reindex(entre if entre else fiab.index) - objetivo).abs().sort_values().index[:10]
    s1 = max(cand, key=lambda s: float(fiab.loc[s, 'den'] / (fiab.loc[s, 'sem'] * FAC_H)))
    nofi = B_[~FIAB].astype(float)
    s2 = (nofi['den'] / (nofi['sem'] * FAC_H)).sort_values().index[len(nofi) // 2]
    b1, b2 = B_.loc[s1].astype(float), B_.loc[s2].astype(float)
    X1 = Xs_[s1]
    rec1 = 1 - abs(X1 - b1['E']) / abs(b1['A'] - b1['E'])
    fig, axs = plt.subplots(2, 1, figsize=(ANCHO, 3.5),
                            gridspec_kw=dict(hspace=0.75, height_ratios=[1.35, 1]))
    ax = axs[0]; A1, E1, w1 = b1['A'], b1['E'], b1['sem'] * FAC_H
    ax.add_patch(Rectangle((A1 - w1, -0.32), 2 * w1, 0.64, color=BANDA, lw=0, zorder=0))
    ax.axhline(0, color=EJE, lw=0.8, zorder=1)
    ax.plot([A1], [0], 'o', ms=7, color=REF, zorder=3)
    ax.plot([E1], [0], 'D', ms=6, color=DESC, zorder=4)
    ax.text(A1, 0.42, r'Media sin limpiar ($V_0$)', ha='center', va='bottom', fontsize=7.5, color=SEC)
    ax.text(E1, 0.42, r'Valor del especialista ($V_e$)', ha='center', va='bottom', fontsize=7.5, color=SEC)
    y1, y2 = -1.0, -1.75
    ax.plot([A1, E1], [y1, y1], color=REF, lw=5, solid_capstyle='butt', zorder=2)
    ax.plot([A1, X1], [y2, y2], color=NARANJA, lw=5, solid_capstyle='butt', zorder=2)
    ax.plot([X1], [y2], '|', ms=11, mew=1.4, color=NARANJA, zorder=3)
    ax.text(X1 + (-1 if X1 < A1 else 1) * 0.012 * abs(E1 - A1), y2, r'$V_m$',
            ha='right' if X1 < A1 else 'left', va='center', fontsize=7.5, color=TINTA)
    for xv in (A1, E1):
        ax.plot([xv, xv], [y2 - 0.25, 0], color=EJE, lw=0.6, ls=':', zorder=0)
    ax.text((A1 + E1) / 2, y1 + 0.2, f'Corrección del especialista: {nf(abs(E1 - A1), 2)} GPa',
            ha='center', va='bottom', fontsize=7.5, color=SEC)
    ax.text((A1 + X1) / 2, y2 + 0.2, f'Lo que corrige el modelo B con sus descartes: el {nf(100 * rec1, 0)} %',
            ha='center', va='bottom', fontsize=7.5, color=TINTA)
    lo_, hi_ = min(A1 - w1, E1, X1), max(A1 + w1, E1, X1); pad = 0.18 * (hi_ - lo_)
    ax.set_xlim(lo_ - pad, hi_ + pad); ax.set_ylim(-2.15, 1.0)
    ax.set_title('(a) Película en la que limpiar importa', loc='left', color=TINTA, fontsize=8, pad=2)
    bx = axs[1]; A2, E2_, w2 = b2['A'], b2['E'], b2['sem'] * FAC_H
    bx.add_patch(Rectangle((A2 - w2, -0.32), 2 * w2, 0.64, color=BANDA, lw=0, zorder=0))
    bx.axhline(0, color=EJE, lw=0.8, zorder=1)
    bx.plot([A2], [0], 'o', ms=7, color=REF, zorder=3)
    bx.plot([E2_], [0], 'D', ms=6, color=DESC, zorder=4)
    bx.text(A2, 0.42, r'$V_0$', ha='center', va='bottom', fontsize=7.5, color=SEC)
    bx.text(E2_, 0.42, r'$V_e$', ha='center', va='bottom', fontsize=7.5, color=SEC)
    bx.text(A2, -0.55, f'La corrección ({nf(abs(E2_ - A2), 2)} GPa) es menor que la incertidumbre del '
            f'promedio (± {nf(w2, 2)} GPa):\nen esta película el acierto no se puede medir',
            ha='center', va='top', fontsize=7.5, color=SEC)
    bx.set_xlim(A2 - 1.35 * w2, A2 + 1.35 * w2); bx.set_ylim(-1.5, 1.0)
    bx.set_title('(b) Película en la que limpiar no importa', loc='left', color=TINTA, fontsize=8, pad=2)
    for a_ in axs:
        a_.set_yticks([]); a_.spines['left'].set_visible(False)
        a_.set_xlabel('Dureza de la película (GPa)', labelpad=2)
        a_.xaxis.set_major_locator(MaxNLocator(6)); eje_es(a_, 'x')
    axs[0].legend(handles=[Patch(color=BANDA, label='Incertidumbre del promedio de 64 ensayos')],
                  loc='lower right', bbox_to_anchor=(1.0, 0.98), fontsize=7.5, borderaxespad=0)
    guarda(fig, 5)
except Exception:
    fallo(5)

try:
    D = B_[['A', 'E', 'sem']].astype(float).copy()
    D['fiable'] = FIAB.values
    D['rel'] = 100 * (D['E'] - D['A']) / D['A']
    D['banda'] = 100 * D['sem'] * FAC_H / D['A']
    D = D.sort_values('rel').reset_index(drop=True)
    fi = D['fiable'].values
    q1, q99 = np.percentile(D['rel'], [1, 99])
    ext = max(abs(q1), abs(q99), D['banda'].max())
    lo = min(D['rel'].min(), -D['banda'].max()); hi = max(D['rel'].max(), D['banda'].max())
    recorta = (D['rel'].min() < -2 * ext) or (D['rel'].max() > 2 * ext)
    if recorta:
        lo, hi = max(lo, -1.3 * ext), min(hi, 1.3 * ext)
    lo, hi = lo - 0.06 * (hi - lo), hi + 0.06 * (hi - lo)
    x = np.arange(len(D))
    fig, ax = plt.subplots(figsize=(ANCHO, 2.6))
    ax.bar(x, 2 * D['banda'], bottom=-D['banda'], width=1.0, color=BANDA, lw=0, zorder=1)
    ax.bar(x, D['rel'].clip(lo, hi), width=0.62, color=np.where(fi, DESC, VALIDO_P), lw=0, zorder=2)
    ax.axhline(0, color=SEC, lw=0.7, zorder=3)
    ax.set_ylim(lo, hi); ax.set_xlim(-1, len(D))
    ax.set_xticks([]); ax.spines['bottom'].set_visible(False)
    nb, ns = int((D['rel'] < 0).sum()), int((D['rel'] > 0).sum())
    ax.set_xlabel(f'Las {len(D)} películas, de la mayor bajada a la mayor subida: '
                  f'baja en {nb} y sube en {ns}')
    ax.set_ylabel('Cambio de la dureza\nal limpiar (%)')
    eje_es(ax, 'y')
    if recorta:
        fuera = D['rel'][(D['rel'] < lo) | (D['rel'] > hi)]
        ax.text(0.01, 0.98, 'Fuera de escala: ' + lista_es([nf(v, 0) + ' %' for v in fuera]),
                transform=ax.transAxes, fontsize=7, color=SEC, va='top')
    ax.legend(handles=[Patch(color=DESC, label=f'El cambio supera la incertidumbre ({int(fi.sum())})'),
                       Patch(color=VALIDO_P, label=f'No la supera ({int((~fi).sum())})'),
                       Patch(color=BANDA, label='Incertidumbre del promedio')],
              loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=3, fontsize=7.5,
              handlelength=1.2, columnspacing=1.2)
    guarda(fig, 6)
except Exception:
    fallo(6)

try:
    Hh = M['hardness_H_GPa'].astype(float).values
    ntip = (NIV6 >= 2).sum(axis=1).values
    filas = []
    for c in POR6:
        solo = (NIV6[c] >= 2).values & (ntip == 1)
        for s, ii in M.groupby('sample_num').indices.items():
            vt, vc = Hh[ii][solo[ii]], Hh[ii][VALIDOS[ii]]
            vt, vc = vt[np.isfinite(vt)], vc[np.isfinite(vc)]
            if len(vt) >= 2 and len(vc) >= 10 and np.median(vc) > 0:
                filas.append((c, s, int(LOTE_DE[s]), np.median(vt) / np.median(vc)))
    MEC = pd.DataFrame(filas, columns=['tipo', 'pel', 'batch', 'razon'])

    def veredicto5(D_):
        n = len(D_); dif = D_.razon - 1
        alto, bajo = int((dif > 0).sum()), int((dif < 0).sum())
        p_ = stats.binomtest(alto, alto + bajo, 0.5).pvalue if alto + bajo else 1.0
        sg = 1 if alto >= bajo else -1
        pl = D_.assign(dif=dif).groupby('batch').dif.agg(['median', 'size'])
        pl = pl[pl['size'] >= 3]
        discrepan = int((np.sign(pl['median']) != sg).sum())
        ok = (max(alto, bajo) / n >= 2 / 3) and (p_ < 0.01) and (discrepan <= 1)
        return dict(n=n, alto=alto, bajo=bajo, p=p_, direccional=ok, signo=sg)

    cnt = MEC.groupby('tipo').size()
    MEC = MEC[MEC.tipo.isin(cnt[cnt >= 10].index)]
    VER6 = {c: veredicto5(MEC[MEC.tipo == c]) for c in MEC.tipo.unique()}
    orden = MEC.groupby('tipo').razon.median().sort_values().index.tolist()
    e = max(np.log(1.25), np.percentile(np.abs(np.log(MEC.razon.values)), 98) * 1.08)
    LIM = (np.exp(-e), np.exp(e))
    CAND = ([0.25, 0.33, 0.5, 1, 2, 3, 4] if e > np.log(2.2) else
            [0.5, 0.67, 1, 1.5, 2] if e > np.log(1.8) else
            [0.67, 0.8, 1, 1.25, 1.5] if e > np.log(1.3) else [0.8, 0.9, 1, 1.1, 1.25])
    tk = [v for v in CAND if LIM[0] <= v <= LIM[1]]
    fig, ax = plt.subplots(figsize=(ANCHO, 0.55 + 0.5 * len(orden)))
    rng = np.random.default_rng(20261002)
    for k, c in enumerate(orden):
        r_all = MEC[MEC.tipo == c].razon.values
        r = np.clip(r_all, LIM[0] * 1.01, LIM[1] / 1.01)
        ax.scatter(r, k + rng.uniform(-0.16, 0.16, len(r)), s=8, color=VALIDO_P, alpha=0.8, lw=0, zorder=2)
        med = float(np.median(r_all))
        ax.plot([med, med], [k - 0.28, k + 0.28], color=DESC, lw=2.2, zorder=3)
        v = VER6[c]
        if v['direccional']:
            txt = (f'Más alta en {v["alto"]} de {v["n"]} películas' if v['signo'] > 0
                   else f'Más baja en {v["bajo"]} de {v["n"]} películas')
        else:
            txt = ('Sin sentido claro: ' + (f'más alta en {v["alto"]} de {v["n"]} películas'
                                            if v['alto'] >= v['bajo'] else
                                            f'más baja en {v["bajo"]} de {v["n"]} películas'))
        ax.text(-0.015, k + 0.14, TIPO[c], transform=ax.get_yaxis_transform(), ha='right', va='center',
                fontsize=8, color=TINTA)
        ax.text(-0.015, k - 0.20, txt, transform=ax.get_yaxis_transform(), ha='right', va='center',
                fontsize=7, color=SEC)
    ax.axvline(1, color=EJE, lw=0.9, zorder=1)
    ax.set_xscale('log'); ax.set_xlim(*LIM)
    ax.xaxis.set_major_locator(FixedLocator(tk)); ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xticklabels(['×' + nf(v, 2 if v in (0.67, 0.33, 1.25) else (0 if float(v).is_integer() else 1))
                        for v in tk])
    ax.set_yticks(range(len(orden))); ax.set_yticklabels([]); ax.tick_params(axis='y', length=0)
    ax.spines['left'].set_visible(False)
    ax.set_ylim(-0.6, len(orden) - 0.4)
    ax.set_xlabel('Dureza de los ensayos con el defecto,\ndividida por la de los ensayos válidos de su película')
    ax.text(1.0, 1.0, 'Más baja ←   → Más alta', transform=ax.get_xaxis_transform(), ha='center',
            va='bottom', fontsize=7.5, color=SEC)
    guarda(fig, 7)
except Exception:
    fallo(7)

try:
    # Película: la del ensayo de ejemplo de las ilustraciones 1 y 10 (la primera película evaluable).
    TIPO_COL = {'force_offset': '#6250d6', 'unusual_unloading_curvature': '#1baf7a',
                'tip_displacement_decreases_at_holding': '#eda100', 'displacement_offset': '#e87ba4',
                'unusual_loading_curvature': '#008300', 'too_deep': MUDO, 'varios': DESC}
    s8 = S0
    d8 = M[M.sample_num == s8].sort_values('indent_num')
    v8 = d8['hardness_H_GPa'].astype(float).values
    y8 = d8['_y'].values == 1
    n8 = NIV6.loc[d8.index]
    tipo8 = []
    for k in range(len(v8)):
        ts = [c for c in POR6 if n8.iloc[k][c] >= 2]
        tipo8.append('valido' if not y8[k] else (ts[0] if len(ts) == 1 else 'varios'))
    tipo8 = np.array(tipo8)
    V0_, Ve_ = float(v8.mean()), float(v8[~y8].mean())
    med8 = float(np.median(v8)); esc8 = 1.4826 * float(np.median(np.abs(v8 - med8)))
    z8 = (v8 - med8) / esc8
    dam8 = float(v8[np.abs(z8) <= 3].mean())
    t8 = int(0.10 * len(v8)); rec8 = float(np.sort(v8)[t8:len(v8) - t8].mean())
    assert abs(V0_ - float(B_.loc[s8, 'A'])) < 1e-9 and abs(Ve_ - float(B_.loc[s8, 'E'])) < 1e-9
    Rf = lambda x: 1 - abs(x - Ve_) / abs(V0_ - Ve_)
    kd8 = int((np.abs(z8[y8]) <= 3).sum())
    print(f'    ilustración 8: película {s8}, V0 {V0_:.3f}, Ve {Ve_:.3f}; mediana {med8:.3f} (R {Rf(med8):.3f}), '
          f'desv. abs. mediana {dam8:.3f} (R {Rf(dam8):.3f}), media recortada {rec8:.3f} (R {Rf(rec8):.3f}); '
          f'{kd8} de {int(y8.sum())} descartes dentro de la franja; borde superior {med8 + 3 * esc8:.3f}')
    fig, (ax, bx) = plt.subplots(2, 1, figsize=(ANCHO, 3.9),
                                 gridspec_kw=dict(hspace=1.55, height_ratios=[1.25, 0.8]))
    W8 = 0.1
    x0_ = np.floor((v8.min() - 0.15) / W8) * W8
    x1_ = np.ceil((v8.max() + 0.15) / W8) * W8
    ax.axvspan(x0_, med8 + 3 * esc8, color=BANDA, lw=0, zorder=0)
    orden_t = ['valido'] + POR6 + ['varios']
    bins8 = np.floor((v8 - x0_) / W8).astype(int)
    alto8 = 1
    for b in np.unique(bins8):
        sel = np.flatnonzero(bins8 == b)
        sel = sel[np.argsort([orden_t.index(tipo8[i]) for i in sel], kind='stable')]
        for j, i in enumerate(sel):
            ax.scatter(x0_ + (b + 0.5) * W8, j + 1, s=26, lw=0, zorder=3,
                       color=VALIDO_P if tipo8[i] == 'valido' else TIPO_COL[tipo8[i]])
        alto8 = max(alto8, len(sel))
    ax.axvline(med8, color=SEC, lw=0.8, ls=(0, (3, 2)), zorder=1)
    ax.text(med8, alto8 + 1.25, ' Mediana', ha='left', va='center', fontsize=7, color=SEC)
    ax.set_xlim(x0_, x1_); ax.set_ylim(0.3, alto8 + 1.9)
    ax.set_yticks([]); ax.spines['left'].set_visible(False)
    ax.set_xlabel('Dureza del ensayo (GPa)', labelpad=2); eje_es(ax, 'x')
    ax.set_title(f'(a) Los 64 ensayos de la película {s8}', loc='left', color=TINTA, fontsize=8, pad=4)
    ley8 = [Line2D([], [], marker='o', ls='', ms=4.5, color=VALIDO_P,
                   label=f'Válido ({int((tipo8 == "valido").sum())})')]
    for c in POR6 + ['varios']:
        k = int((tipo8 == c).sum())
        if k:
            ley8.append(Line2D([], [], marker='o', ls='', ms=4.5, color=TIPO_COL[c],
                               label=f'{TIPO.get(c, "Más de un tipo")} ({k})'))
    ley8.append(Patch(color=BANDA, label='Zona que la regla con umbral 3 considera normal'))
    ax.legend(handles=ley8, loc='upper left', bbox_to_anchor=(0.0, -0.40), ncol=2, fontsize=7.5,
              handletextpad=0.3, columnspacing=2.0, borderaxespad=0)
    PT8 = [(Ve_, r'Valor del especialista ($V_e$)', 'D', DESC, 6, 0.95),
           (med8, f'Mediana de los 64 ensayos: {nf(Rf(med8), 2)}', 'o', SEC, 5.5, -0.95),
           (dam8, f'Desviación absoluta mediana: {nf(Rf(dam8), 2)}', 'o', SEC, 5.5, 1.9),
           (rec8, f'Media recortada: {nf(Rf(rec8), 2)}', 'o', SEC, 5.5, -0.95),
           (V0_, r'Media sin limpiar ($V_0$)', 'o', REF, 7, 0.95)]
    bx.axhline(0, color=EJE, lw=0.8, zorder=1)
    for xv, txt, mk, col, ms, yl in PT8:
        bx.plot([xv, xv], [0, yl * 0.78], color=EJE, lw=0.6, zorder=0)
        bx.plot([xv], [0], mk, ms=ms, color=col, zorder=3)
        bx.text(xv, yl, txt, ha='center', va='center', fontsize=7.5, color=TINTA if mk == 'D' else SEC)
    vals8 = [p[0] for p in PT8]
    lo8 = np.floor((min(vals8) - 0.06) * 20) / 20; hi8 = np.ceil((max(vals8) + 0.06) * 20) / 20
    bx.set_xlim(lo8, hi8); bx.set_ylim(-1.35, 2.3)
    bx.set_yticks([]); bx.spines['left'].set_visible(False)
    bx.xaxis.set_major_locator(FixedLocator(np.round(np.arange(lo8, hi8 + 1e-9, 0.1), 2))); eje_es(bx, 'x', 1)
    bx.set_xlabel('Dureza de la película (GPa)', labelpad=2)
    bx.set_title('(b) El valor de la película con cada regla, en una escala ampliada', loc='left', color=TINTA,
                 fontsize=8, pad=4)
    guarda(fig, 8)
except Exception:
    fallo(8)

try:
    def zrob1(v):
        m = np.median(v); mad = np.median(np.abs(v - m))
        return (v - m) / (1.4826 * mad) if mad > 0 else np.zeros(len(v))
    Z = np.full(len(M), np.nan)
    for s, ii in M.groupby('sample_num').indices.items():
        v = M['hardness_H_GPa'].astype(float).values[ii]
        if np.isfinite(v).all():
            Z[ii] = zrob1(v)
    yb = M['_y'].values == 1
    ok = np.isfinite(Z)
    dentro_d = 100 * np.mean(np.abs(Z[ok & yb]) <= 3)
    dentro_v = 100 * np.mean(np.abs(Z[ok & ~yb]) <= 3)
    L = 8
    cortes = np.arange(-L, L + 0.51, 0.5)
    fig, axs = plt.subplots(2, 1, figsize=(ANCHO, 3.3), sharex=True, gridspec_kw=dict(hspace=0.28))
    for ax, sel, col, et, pct in ((axs[0], ok & yb, DESC, 'Ensayos descartados por el especialista', dentro_d),
                                  (axs[1], ok & ~yb, VALIDO_P, 'Ensayos válidos', dentro_v)):
        z = np.clip(Z[sel], -L + 0.01, L - 0.01)
        w = np.full(len(z), 100 / len(z))
        ax.axvspan(-3, 3, color=BANDA, lw=0, zorder=0)
        ax.hist(z, bins=cortes, weights=w, color=col, lw=0, rwidth=0.86, zorder=2)
        ax.text(0.01, 0.95, f'{et} ({nf(sel.sum(), 0)})', transform=ax.transAxes, fontsize=8,
                color=TINTA, va='top')
        ax.text(0.99, 0.95, f'{nf(pct, 1)} % dentro de la franja', transform=ax.transAxes, fontsize=8,
                color=TINTA, va='top', ha='right')
        ax.set_ylabel('% del grupo'); eje_es(ax, 'y')
        ax.set_ylim(0, ax.get_ylim()[1] * 1.18)
    axs[0].text(0, 1.02, 'Zona que la regla con umbral 3 considera normal',
                transform=axs[0].get_xaxis_transform(), ha='center', va='bottom', fontsize=7.5, color=SEC)
    axs[1].set_xticks([-8, -6, -3, 0, 3, 6, 8])
    axs[1].set_xticklabels(['≤ −8', '−6', '−3', '0', '3', '6', '≥ 8'])
    axs[1].set_xlabel('Distancia a la dureza típica de su película (en unidades de dispersión)')
    guarda(fig, 9)
except Exception:
    fallo(9)

try:
    # Cuántas variables de forma recibe el modelo B de cada zona, según la familia que les asigna la celda 16
    IMP_ = pd.read_csv(F_IMP, index_col=0)
    FAM_ = {'arranque de la curva (offset)': 'Arranque', 'tramo de carga': 'Carga', 'sostenimiento': 'Sostenimiento',
            'tramo de descarga': 'Descarga', 'final de la curva': 'Final', 'calidad / ruido': 'Ruido'}
    forma_B = [c for c in JUEGOS[bB] if c not in JUEGOS[bA]]
    NZ = pd.Series([FAM_.get(IMP_.loc[c, 'familia']) for c in forma_B if c in IMP_.index]).value_counts()
    print(f'    ilustración 10: variables de forma de B por zona {NZ.to_dict()} · total {int(NZ.sum())}')
    h, p, t = curva(S0, I0); i0, i1 = tramos(p)
    pm, hm = float(p[i0:i1 + 1].mean()), float(h[i1]); n = len(p)
    rig = float(R0['stiffness_S_micro_N/nm']); hc = float(R0['contact_depth_hc_nm'])
    hfr = float(R0['final_displacement_hf_nm'])
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(ANCHO, 2.9), sharey=True, gridspec_kw=dict(wspace=0.10))
    ax.plot(h, p, color=VALIDO, lw=1.4, zorder=1)
    tg_p = np.array([0.30 * pm, pm]); tg_h = hm + (tg_p - pm) / rig
    ax.plot(tg_h, tg_p, color=AZUL, lw=1.1, zorder=2)
    ax.plot([hm], [pm], 'o', ms=4.5, color=AZUL, zorder=3)
    for xv in (hc, hfr):
        if np.isfinite(xv):
            ax.plot([xv, xv], [0, 0.06 * pm], color=AZUL, lw=1.6, zorder=3, solid_capstyle='butt')
    ax.set_title('(a) Modelo A: los valores que el laboratorio\ncalcula a partir de la curva',
                 loc='left', color=TINTA, fontsize=8, pad=5)
    bx.plot(h, p, color=SEC, lw=0.8, zorder=1)
    k0 = max(3, n // 40); kf = n - max(3, n // 40)
    ZON = [('arranque', slice(0, k0 + 1)), ('carga', slice(k0 + 2, i0 - 1)),
           ('sostenimiento', slice(i0 + 1, i1)), ('descarga', slice(i1 + 2, kf - 2)),
           ('final', slice(kf, n))]
    for et, sl in ZON:
        bx.plot(h[sl], p[sl], color=NARANJA, lw=3.0, zorder=2, solid_capstyle='butt')
    bx.set_title('(b) Modelo B: además, cómo es la curva\nen cada una de sus zonas',
                 loc='left', color=TINTA, fontsize=8, pad=5)
    H0 = min(0.0, float(np.nanmin(h))); SP = hm - H0
    for a_ in (ax, bx):
        a_.set_xlabel('Profundidad (nm)'); a_.set_ylim(0, 1.22 * pm)
        a_.set_xlim(H0, H0 + 1.3 * SP); eje_es(a_)
    ax.set_ylabel('Fuerza (µN)'); bx.tick_params(labelleft=False)
    marcas_hasta(ax, pm)
    fija_ancho(fig)
    R = Rotulos(ax, h, p); R.anade(tg_h, tg_p)
    R.pon('Fuerza y profundidad\nmáximas', (hm, pm), fs=7)
    R.pon('Rigidez\n(pendiente)', (hm - 0.4 * pm / rig, 0.6 * pm), fs=7)
    if np.isfinite(hc):
        R.pon('Profundidad de\ncontacto ($h_c$)', (hc, 0.06 * pm), fs=7)
    if np.isfinite(hfr):
        R.pon('Profundidad que queda al\nretirar la fuerza ($h_f$)', (hfr, 0.06 * pm), fs=7)
    R2 = Rotulos(bx, h, p)
    jm = (i0 + i1) // 2; jf = n - 1 - (n - kf) // 2
    cz = lambda z_: int(NZ.get(z_, 0))
    for txt_, k_ in ((f'Arranque ({cz("Arranque")})', k0 // 2),
                     (f'Carga ({cz("Carga")})', en_fuerza(p, 0.62, 'carga')),
                     (f'Sostenimiento ({cz("Sostenimiento")})', jm),
                     (f'Descarga ({cz("Descarga")}),\nsobre todo\nsu parte baja', en_fuerza(p, 0.3, 'descarga')),
                     (f'Final ({cz("Final")})', jf)):
        R2.pon(txt_, (h[k_], p[k_]), fs=7)
    bx.text(0.02, 0.985, f'Ruido de toda la curva ({cz("Ruido")})', transform=bx.transAxes, ha='left', va='top',
            fontsize=7, color=SEC)
    guarda(fig, 10)
except Exception:
    fallo(10)

try:
    # Película: la del ejemplo de desplazamiento de fuerza de la ilustración 4. En ella, para cada uno de los dos
    # defectos, el ensayo con ese defecto en el nivel más alto y ningún otro (a igual nivel, el de número menor).
    s11, i_fo = EJ['force_offset'][0], EJ['force_offset'][1]
    d11 = M[M.sample_num == s11]
    niv11 = NIV6.loc[d11.index]
    solo_u = (niv11['unusual_unloading_curvature'] >= 2) & ((niv11.drop(columns='unusual_unloading_curvature') >= 2).sum(axis=1) == 0)
    cand_u = d11[solo_u.values].assign(_lv=niv11.loc[solo_u.values, 'unusual_unloading_curvature'].values)
    i_uc = int(cand_u.sort_values(['_lv', 'indent_num'], ascending=[False, True]).iloc[0].indent_num)
    lv_fo = int(niv11.loc[d11.indent_num.values == i_fo, 'force_offset'].iloc[0])
    lv_uc = int(niv11.loc[d11.indent_num.values == i_uc, 'unusual_unloading_curvature'].iloc[0])
    zf = lambda i, c: float(d11.loc[d11.indent_num == i, c].iloc[0])
    val11 = [curva(s11, int(ii)) for ii in d11[d11['_y'] == 0].indent_num]
    print(f'    ilustración 11: película {s11}; ensayo {i_fo} (desplazamiento de fuerza, nivel {lv_fo}) y ensayo {i_uc} '
          f'(curvatura anómala de la descarga, nivel {lv_uc}); {len(val11)} válidos; '
          f'fuerza al comienzo en unidades de dispersión {zf(i_fo, "z_p_ini_p5_frac"):.2f}; desajuste con signo '
          f'{zf(i_uc, "z_desc_sesgo_bajo_rel"):.2f} y sin signo {zf(i_uc, "z_desc_resid_bajo_rel"):.2f}')
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(ANCHO, 2.9), gridspec_kw=dict(wspace=0.32))
    # (a) arranque frente al tiempo
    T_MS = 30
    h, p, t = curva(s11, i_fo)
    n5 = max(3, len(p) // 50)
    for hh, pp, tt in val11:
        k = (tt - tt[0]) * 1000 <= T_MS
        ax.plot((tt[k] - tt[0]) * 1000, pp[k], color=VALIDO, lw=0.5, zorder=1)
    k = (t - t[0]) * 1000 <= T_MS
    ax.plot((t[k] - t[0]) * 1000, p[k], color=DESC, lw=1.4, zorder=3)
    ax.axvspan(0, (t[n5 - 1] - t[0]) * 1000, color=BANDA, lw=0, zorder=0)
    ax.text((t[n5 - 1] - t[0]) * 500, 1.0, f'Primeros\n{n5} puntos', transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=7, color=SEC)
    ax.set_xlim(0, T_MS); ax.set_ylim(bottom=min(0, ax.get_ylim()[0]))
    ax.set_xlabel('Tiempo desde el comienzo (ms)'); ax.set_ylabel('Fuerza (µN)'); eje_es(ax)
    ax.set_title(f'(a) Arranque del ensayo {i_fo},\ncon desplazamiento de fuerza', loc='left', color=TINTA,
                 fontsize=8, pad=5)
    # (b) descarga, alineada en la profundidad máxima y en proporción a la fuerza máxima
    for hh, pp, tt in val11:
        a0_, a1_ = tramos(pp)
        bx.plot(hh[a1_:] - np.nanmax(hh), pp[a1_:] / np.nanmax(pp), color=VALIDO, lw=0.5, zorder=1)
    h, p, t = curva(s11, i_uc)
    a0_, a1_ = tramos(p); pmx = float(np.nanmax(p)); hmx = float(np.nanmax(h))
    hu, pu = h[a1_:], p[a1_:]
    bx.plot(hu - hmx, pu / pmx, color=DESC, lw=1.4, zorder=3)
    # Ajuste de la ecuación (1) a los puntos de la descarga con al menos la mitad de la fuerza máxima,
    # con la misma búsqueda de h_f que la celda 14, y su prolongación a toda la descarga
    alto = pu >= 0.5 * pmx
    hh_, pp_ = hu[alto], pu[alto]
    mejor_r2, mejor = -np.inf, None
    for hf_ in np.linspace(0, 0.95 * float(np.min(hh_)), 30):
        m_, b_, r2_, rr_ = ajuste_potencia(hh_, pp_, hf_)
        if r2_ == r2_ and r2_ > mejor_r2:
            mejor_r2, mejor = r2_, (m_, b_, hf_)
    m_, b_, hf_ = mejor
    xx = np.linspace(float(np.min(hu)), float(np.max(hu)), 200)
    ok_ = xx - hf_ > 1e-9
    bx.plot(xx[ok_] - hmx, np.exp(b_) * (xx[ok_] - hf_) ** m_ / pmx, color=SEC, lw=1.0, ls=(0, (4, 2)), zorder=4)
    bx.axhline(0.5, color=MUDO, lw=0.6, ls=(0, (1, 2)), zorder=0)
    bx.text(0.01, 0.5, '50 % de la fuerza máxima', transform=bx.get_yaxis_transform(), ha='left', va='bottom',
            fontsize=7, color=SEC)
    bx.set_ylim(0, 1.05)
    bx.set_xlabel('Profundidad menos la máxima (nm)'); bx.set_ylabel('Fuerza / fuerza máxima'); eje_es(bx)
    bx.set_title(f'(b) Descarga del ensayo {i_uc},\ncon curvatura anómala de la descarga', loc='left',
                 color=TINTA, fontsize=8, pad=5)
    fig.legend(handles=[Line2D([], [], color=VALIDO, lw=1.6, label='Ensayos válidos de la misma película'),
                        Line2D([], [], color=DESC, lw=1.6, label='Ensayo con el defecto'),
                        Line2D([], [], color=SEC, lw=1.0, ls=(0, (4, 2)),
                               label='Ajuste a la mitad superior de la descarga, prolongado')],
               loc='upper center', ncol=2, bbox_to_anchor=(0.5, 0.0))
    guarda(fig, 11)
except Exception:
    fallo(11)

try:
    npel = M.groupby('batch_num').sample_num.nunique().reindex(lotes)
    w = npel.values / npel.sum()
    xs = np.r_[0, np.cumsum(w)]
    fig = plt.figure(figsize=(ANCHO, 4.35))
    gs = fig.add_gridspec(3, 1, height_ratios=[2.4, 0.42, 0.42], hspace=0.78)
    ax = fig.add_subplot(gs[0])
    gap = 0.004
    for r_, lt in enumerate(lotes):
        for c_, lc in enumerate(lotes):
            ax.add_patch(Rectangle((xs[c_] + gap, len(lotes) - 1 - r_ + 0.12), w[c_] - 2 * gap, 0.76,
                                   color=DESC if lc == lt else '#e4e2dc', lw=0))
    for c_, lc in enumerate(lotes):
        ax.text((xs[c_] + xs[c_ + 1]) / 2, len(lotes) + 0.1, f'{int(lc)}\n{int(npel[lc])} pel.',
                ha='center', va='bottom', fontsize=7, color=SEC)
    for r_ in range(len(lotes)):
        ax.text(-0.012, len(lotes) - 1 - r_ + 0.5, f'Ronda {r_ + 1}', ha='right', va='center',
                fontsize=7.5, color=SEC)
    XE, XP = 1.105, 1.315
    ax.text(XE, len(lotes) + 0.1, 'Ensayos de\nentrenamiento', ha='center', va='bottom', fontsize=7, color=SEC)
    ax.text(XP, len(lotes) + 0.1, 'Descartados por\nel especialista', ha='center', va='bottom', fontsize=7,
            color=SEC)
    for r_, lt in enumerate(lotes):
        tr_ = M[M.batch_num != lt]
        ax.text(XE, len(lotes) - 1 - r_ + 0.5, nf(len(tr_), 0), ha='center', va='center', fontsize=7.5,
                color=TINTA)
        ax.text(XP, len(lotes) - 1 - r_ + 0.5, f'{nf(100 * tr_["_y"].mean(), 1)} %', ha='center',
                va='center', fontsize=7.5, color=TINTA)
    ax.text(0.5, len(lotes) + 1.18, 'Campañas de fabricación (ancho proporcional al número de películas)',
            ha='center', va='bottom', fontsize=7.5, color=TINTA)
    ax.text(-0.11, len(lotes) + 1.18, '(a)', ha='left', va='bottom', fontsize=9, color=TINTA)
    ax.set_xlim(-0.11, 1.42); ax.set_ylim(-0.1, len(lotes) + 1.75); ax.axis('off')
    ax.legend(handles=[Patch(color='#e4e2dc', label='Campañas con las que se entrena'),
                       Patch(color=DESC, label='Campaña en la que se evalúa')],
              loc='upper center', bbox_to_anchor=(0.43, 0.02), ncol=2, fontsize=7.5)
    # (b) Ejemplo con valores inventados: 20 ensayos en cada fila, con la probabilidad de descarte
    # que les daría el modelo y si los descartó el especialista (1) o no (0).
    UMB_EJ = 0.645
    EJ_ENT = [(0.97, 1), (0.93, 1), (0.88, 1), (0.81, 0), (0.76, 1), (0.70, 1), (0.60, 0), (0.52, 0), (0.45, 1),
              (0.40, 0), (0.35, 0), (0.31, 0), (0.28, 0), (0.25, 0), (0.22, 0), (0.20, 0), (0.18, 0), (0.15, 0),
              (0.10, 0), (0.05, 0)]
    EJ_EVA = [(0.95, 1), (0.90, 1), (0.84, 1), (0.78, 0), (0.72, 1), (0.69, 0), (0.67, 0), (0.58, 0), (0.50, 1),
              (0.45, 0), (0.38, 0), (0.33, 0), (0.30, 0), (0.27, 0), (0.24, 0), (0.19, 0), (0.16, 0), (0.12, 0),
              (0.09, 0), (0.06, 0)]
    filas_b = ((fig.add_subplot(gs[1]), EJ_ENT, 'Campañas de entrenamiento'),
               (fig.add_subplot(gs[2]), EJ_EVA, 'Campaña evaluada'))
    for k_, (bx, pts, et) in enumerate(filas_b):
        pts = sorted(pts)
        for j, (pr, dsc) in enumerate(pts):
            bx.scatter(pr, 0.32 if j % 2 == 0 else 0.72, s=24, lw=0, zorder=3, color=DESC if dsc else VALIDO_P)
        nd = sum(d_ for _, d_ in pts); nu = sum(1 for pr, _ in pts if pr > UMB_EJ)
        bx.axvline(UMB_EJ, color=SEC, lw=0.9, ls=(0, (3, 2)), zorder=2)
        bx.set_xlim(0, 1); bx.set_ylim(0, 1.05)
        bx.set_yticks([]); bx.spines['left'].set_visible(False)
        if k_ == 0:
            bx.text(0.0, 1.12, '(b)', transform=bx.transAxes, ha='left', va='bottom', fontsize=9, color=TINTA)
        bx.text(0.04 if k_ == 0 else 0.0, 1.12, f'{et}: el especialista descarta {nd} de {len(pts)}'
                + (f' ({nf(100 * nd / len(pts), 0)} %)' if k_ == 0 else ''),
                transform=bx.transAxes, ha='left', va='bottom', fontsize=7.5, color=TINTA)
        bx.text(1.0, 1.12, f'{nu} por encima del umbral', transform=bx.transAxes, ha='right', va='bottom',
                fontsize=7.5, color=SEC)
        if k_ == 0:
            bx.set_xticklabels([])
            bx.text(UMB_EJ + 0.008, 1.0, 'Umbral', transform=bx.get_xaxis_transform(), ha='left', va='top',
                    fontsize=7, color=SEC)
        else:
            bx.set_xlabel('Probabilidad de descarte que asigna el modelo (valores inventados)')
            eje_es(bx, 'x', 1)
        bx.xaxis.set_major_locator(FixedLocator([0, 0.5, 1]))
    filas_b[1][0].legend(handles=[Line2D([], [], marker='o', ls='', ms=4.5, color=DESC, label='Descartado por el especialista'),
                                  Line2D([], [], marker='o', ls='', ms=4.5, color=VALIDO_P, label='Conservado por el especialista')],
                         loc='upper center', bbox_to_anchor=(0.5, -1.25), ncol=2, fontsize=7.5)
    guarda(fig, 12)
except Exception:
    fallo(12)

try:
    lo_m, hi_m = IV9['mediana'][1], IV9['mediana'][2]
    FILAS = [('B', 'Modelo B', NARANJA, PT_IV['B'], *np.percentile(GL_IV['B'], [2.5, 97.5]), T[bB]),
             ('A', 'Modelo A', AZUL, PT_IV['A'], *np.percentile(GL_IV['A'], [2.5, 97.5]), T[bA]),
             ('mediana', 'Mediana de los\n64 ensayos', REF, IV9['mediana'][0], lo_m, hi_m, T9['mediana'])]
    nfi = B_[FIAB].groupby('batch').size()
    EST = None
    if os.path.exists(F_ESTAB):
        EST = pd.read_csv(F_ESTAB)
        EST = EST[EST['rep'] > 0]
    fig = plt.figure(figsize=(ANCHO, 5.0))
    gs = fig.add_gridspec(2, 1, height_ratios=[3.0, 4.4], hspace=0.62)
    ax = fig.add_subplot(gs[0]); bx = fig.add_subplot(gs[1])
    for k, (key, et, col, pt, lo_, hi_, pl) in enumerate(FILAS[::-1]):
        ax.plot([lo_, hi_], [k, k], color=col, lw=2.2, zorder=2)
        ax.plot([pt], [k], 'o', ms=8, color=col, zorder=3, mec='white', mew=1.5)
        pl = pd.Series(pl).astype(float).values
        ax.scatter(pl, np.full(len(pl), k - 0.30), s=10, color=col, alpha=0.45, lw=0, zorder=2)
        ax.text(pt, k + 0.2, nf(pt, 2), va='bottom', ha='center', fontsize=8.5, color=TINTA)
    ax.set_yticks(range(len(FILAS))); ax.set_yticklabels([f[1] for f in FILAS[::-1]])
    ax.set_ylim(-0.6, len(FILAS) - 0.35)
    ax.axvline(0, color=EJE, lw=0.8, zorder=0); ax.axvline(1, color=EJE, lw=0.8, zorder=0)
    ax.axvline(0.25, color=SEC, lw=0.8, ls=(0, (3, 2)), zorder=0)
    ax.text(0.26, 0.99, 'Umbral de F1', transform=ax.get_xaxis_transform(), ha='left', va='top',
            fontsize=7, color=SEC)
    ax.text(1, 1.0, 'Valor del\nespecialista', transform=ax.get_xaxis_transform(), ha='center',
            va='bottom', fontsize=7, color=SEC)
    ax.text(0, 1.0, 'Media sin\nlimpiar', transform=ax.get_xaxis_transform(), ha='center', va='bottom',
            fontsize=7, color=SEC)
    ax.annotate('(a)', xy=(0.0, 1.02), xycoords=('figure fraction', 'axes fraction'), ha='left',
                va='bottom', fontsize=9)
    ax.set_xlabel('Fracción de la corrección del especialista que se recupera')
    allv = np.r_[[f[4] for f in FILAS], [f[5] for f in FILAS],
                 np.concatenate([pd.Series(f[6]).astype(float).values for f in FILAS])]
    ax.set_xlim(min(-0.08, np.nanmin(allv) - 0.06), max(1.12, np.nanmax(allv) + 0.06))
    eje_es(ax, 'x')
    # (b) Barras agrupadas por campaña: mediana de los 64 ensayos, modelo A y modelo B
    RANGO_B = {}
    WB = 0.26
    SERIES_B = [('Mediana de los 64 ensayos', REF, lambda lt: float(T9.loc[lt, 'mediana'])),
                ('Modelo A', AZUL, lambda lt: float(T.loc[lt, bA])),
                ('Modelo B', NARANJA, lambda lt: float(T.loc[lt, bB]))]
    for j, lt in enumerate(lotes):
        for q, (et, col, fv) in enumerate(SERIES_B):
            bx.bar(j + (q - 1) * WB, fv(lt), width=WB - 0.03, color=col, lw=0, zorder=2)
        if EST is not None and f'B_{int(lt)}' in EST.columns:
            r0_, r1_ = float(EST[f'B_{int(lt)}'].min()), float(EST[f'B_{int(lt)}'].max())
            RANGO_B[lt] = (r0_, r1_)
            xb_ = j + WB
            bx.plot([xb_, xb_], [r0_, r1_], color=DESC, lw=0.9, zorder=3)
            for r__ in (r0_, r1_):
                bx.plot([xb_ - 0.05, xb_ + 0.05], [r__, r__], color=DESC, lw=0.9, zorder=3)
    bx.axhline(0, color=EJE, lw=0.8, zorder=1)
    bx.axhline(1, color=EJE, lw=0.8, zorder=1)
    bx.axhline(0.25, color=SEC, lw=0.8, ls=(0, (3, 2)), zorder=1)
    bx.text(1.0, 1.0, ' Valor del especialista', transform=bx.get_yaxis_transform(), ha='left', va='center',
            fontsize=7, color=SEC)
    bx.text(1.0, 0.25, ' Umbral de F1', transform=bx.get_yaxis_transform(), ha='left', va='center',
            fontsize=7, color=SEC)
    bx.set_xticks(range(len(lotes)))
    bx.set_xticklabels([f'{int(lt)}\n{int(nfi.get(lt, 0))} pel.' for lt in lotes])
    bx.tick_params(axis='x', length=0)
    bx.set_xlim(-0.6, len(lotes) - 0.4)
    ymax_ = max([1.0] + [v_[1] for v_ in RANGO_B.values()])
    bx.set_ylim(min(0, min(float(T9.loc[lt, 'mediana']) for lt in lotes)) - 0.02, ymax_ + 0.08)
    bx.set_ylabel('Recuperación de la corrección'); bx.set_xlabel('Campaña evaluada y número de películas evaluables')
    eje_es(bx, 'y', 1)
    ley11 = [Patch(color=c_, label=e_) for e_, c_, _ in SERIES_B]
    if RANGO_B:
        ley11.append(Line2D([], [], color=DESC, lw=0.9, label='Modelo B con ruido de redondeo'))
    bx.legend(handles=ley11, loc='lower left', fontsize=7.5, ncol=2, bbox_to_anchor=(0.0, 1.02),
              borderaxespad=0.2, columnspacing=1.4, handletextpad=0.4)
    bx.annotate('(b)', xy=(0.0, 1.02), xycoords=('figure fraction', 'axes fraction'), ha='left',
                va='bottom', fontsize=9)
    guarda(fig, 13)
except Exception:
    fallo(13)

try:
    E2 = EA.astype(float).copy()
    E9 = EA9.reindex(E2.index)
    for c_ in ('mediana', 'mad3'):
        E2[c_] = E9[c_].astype(float)
    ruido = E2['ruido'] if 'ruido' in E2.columns else E2['sem'] * FAC_H
    MET = [('cruda', 'Sin limpiar', REF), ('base', 'Media recortada', REF),
           ('mediana', 'Mediana de los 64 ensayos', REF),
           ('mad3', 'Regla de la desviación\nabsoluta mediana', REF),
           ('A', 'Modelo A', AZUL), ('B', 'Modelo B', NARANJA)]
    fiE = E2['fiable'].astype(bool)
    fig = plt.figure(figsize=(ANCHO, 6.3))
    gs = fig.add_gridspec(3, 1, height_ratios=[6, 5, 2.4], hspace=0.62)
    ax = fig.add_subplot(gs[0]); bx = fig.add_subplot(gs[1]); cx = fig.add_subplot(gs[2])
    rng = np.random.default_rng(7)
    d = E2[fiE]
    todo = np.concatenate([(100 * d[c] / d['E']).values for c, _, _ in MET])
    LIMX = float(np.nanpercentile(todo, 97))
    paso = 1 if LIMX <= 6 else (2 if LIMX <= 12 else (5 if LIMX <= 30 else 10))
    LIMX = paso * np.ceil(LIMX * 1.02 / paso)
    for k, (c, et, col) in enumerate(MET[::-1]):
        v = (100 * d[c] / d['E']).values
        ax.scatter(np.clip(v, 0, LIMX), k + rng.uniform(-0.17, 0.17, len(v)), s=8, color=col, alpha=0.55,
                   lw=0, zorder=2)
        md = float(np.nanmedian(v))
        ax.plot([md, md], [k - 0.3, k + 0.3], color=TINTA, lw=2, zorder=3)
        ax.text(md, k + 0.33, nf(md, 1), ha='center', va='bottom', fontsize=7, color=TINTA)
    ax.set_yticks(range(len(MET))); ax.set_yticklabels([m[1] for m in MET[::-1]])
    ax.set_xlim(-0.02 * LIMX, LIMX); ax.set_ylim(-0.6, len(MET) - 0.2)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=7, steps=[1, 2, 5, 10]))
    ax.set_xlabel('Distancia al valor del especialista (% de su dureza)')
    ax.set_title(f'(a) Películas evaluables ({int(fiE.sum())})', loc='left', color=TINTA, fontsize=8, pad=4)
    eje_es(ax, 'x')
    d = E2[~fiE]; rn = ruido[~fiE]
    NAL = {c: int((d[c] > rn).sum()) for c, _, _ in MET}
    pct = [100 * NAL[c] / len(d) for c, _, _ in MET[::-1]]
    bx.barh(range(len(MET)), pct, color=[m[2] for m in MET[::-1]], height=0.55, lw=0)
    for k, (v, (c, _, _)) in enumerate(zip(pct, MET[::-1])):
        bx.text(v + 0.012 * max(max(pct), 10), k, f'{nf(v, 0)} % ({NAL[c]})', va='center', fontsize=7.5,
                color=TINTA)
    bx.set_yticks(range(len(MET))); bx.set_yticklabels([m[1] for m in MET[::-1]])
    bx.set_xlim(0, max(max(pct) * 1.25, 10))
    bx.set_xlabel('% de esas películas en las que el método aleja el valor más que la incertidumbre')
    bx.set_title(f'(b) Películas no evaluables ({int((~fiE).sum())})', loc='left', color=TINTA, fontsize=8,
                 pad=4)
    eje_es(bx, 'x')

    apA = np.array([det[bA][l]['pr_auc'] for l in lotes], float)
    apB = np.array([det[bB][l]['pr_auc'] for l in lotes], float)
    azar = np.array([M.loc[M.batch_num == l, '_y'].mean() for l in lotes], float)
    rng_c = np.random.default_rng(11)
    cx.axvspan(azar.min(), azar.max(), color=BANDA, lw=0, zorder=0)
    cx.text(azar.mean(), 0.5, 'Orden\nal azar', ha='center', va='center', fontsize=7, color=SEC, linespacing=1.1)
    for k, (v, col) in enumerate(((apA, AZUL), (apB, NARANJA))):
        cx.scatter(v, k + rng_c.uniform(-0.12, 0.12, len(v)), s=14, color=col, lw=0, zorder=2)
        md = float(np.median(v))
        cx.plot([md, md], [k - 0.3, k + 0.3], color=TINTA, lw=2, zorder=3)
        cx.text(md, k + 0.33, nf(md, 3), ha='center', va='bottom', fontsize=7, color=TINTA)
    cx.set_yticks([0, 1]); cx.set_yticklabels(['Modelo A', 'Modelo B'])
    cx.set_xlim(0.2, 1.0); cx.set_ylim(-0.6, 1.75)
    cx.set_xlabel('Precisión media en cada campaña')
    cx.set_title('(c) Detección ensayo a ensayo en las siete campañas, con todos sus ensayos',
                 loc='left', color=TINTA, fontsize=8, pad=4)
    eje_es(cx, 'x')
    print(f'    ilustración 14 (c): A {nf(apA.min(), 3)} a {nf(apA.max(), 3)}, mediana {nf(np.median(apA), 3)}; '
          f'B {nf(apB.min(), 3)} a {nf(apB.max(), 3)}, mediana {nf(np.median(apB), 3)}; '
          f'azar {nf(100 * azar.min(), 1)} a {nf(100 * azar.max(), 1)} %')
    guarda(fig, 14)
except Exception:
    fallo(14)

try:
    det_t = []
    for c in POR6:
        m_ = (NIV6[c] >= 2).values
        if m_.sum() < 20:
            continue
        fr = []
        for nom in (bA, bB):
            pv = pred_all[nom].values; uu = M.batch_num.map(UMB[nom]).values.astype(float)
            fr.append(100 * float(np.mean(pv[m_] >= uu[m_])))
        det_t.append((c, int(m_.sum()), fr[0], fr[1]))
    DT = pd.DataFrame(det_t, columns=['tipo', 'n', 'A', 'B']).sort_values('n').reset_index(drop=True)
    IMPd = pd.read_csv(F_IMP, index_col=0) if os.path.exists(F_IMP) else imp
    g = IMPd.groupby(IMPd['familia'].map(lambda f: ZONA.get(f, 'Otras')))['media'].sum().clip(lower=0)
    g = 100 * g / g.sum()
    ZP = None
    if os.path.exists(F_ZONA):
        ZP = pd.read_csv(F_ZONA)
        ZP['zona_m'] = ZP['zona'].map(lambda f: ZONA.get(f, 'Otras'))
        ZP = ZP.groupby('zona_m').agg(perdida=('perdida', 'max'), n_var=('n_var', 'sum'))
    filas_z = [z for z in ORDEN_ZONA if z in g.index and (z != 'Otras' or g[z] >= 0.5)]
    filas_z = filas_z[::-1]
    fig = plt.figure(figsize=(ANCHO, 5.4))
    gs = fig.add_gridspec(2, 2, height_ratios=[len(DT), len(filas_z)], hspace=0.62, wspace=0.08,
                          width_ratios=[1, 0.8])
    ax = fig.add_subplot(gs[0, :]); bx = fig.add_subplot(gs[1, 0]); cx = fig.add_subplot(gs[1, 1], sharey=bx)
    for k, r in DT.iterrows():
        ax.plot([r['A'], r['B']], [k, k], color=EJE, lw=1.6, zorder=1)
        ax.plot([r['A']], [k], 'o', ms=7, color=AZUL, zorder=3, mec='white', mew=1.2)
        ax.plot([r['B']], [k], 'o', ms=7, color=NARANJA, zorder=4, mec='white', mew=1.2)
        izq, der = sorted([r['A'], r['B']])
        if round(izq) == round(der):
            ax.text(der + 2.5, k, f'{nf(der, 0)} %', ha='left', va='center', fontsize=7, color=SEC)
        else:
            ax.text(izq - 2.5, k, f'{nf(izq, 0)} %', ha='right', va='center', fontsize=7, color=SEC)
            ax.text(der + 2.5, k, f'{nf(der, 0)} %', ha='left', va='center', fontsize=7, color=SEC)
    ax.set_yticks(range(len(DT)))
    ax.set_yticklabels([f'{TIPO[c]} ({nf(n_, 0)})' for c, n_ in zip(DT['tipo'], DT['n'])])
    ax.set_xlim(min(0, DT[['A', 'B']].values.min() - 12), 112); ax.set_ylim(-0.6, len(DT) - 0.4)
    ax.set_xlabel('% de los ensayos con ese defecto que el modelo descarta')
    ax.legend(handles=[Line2D([], [], marker='o', ls='', ms=7, color=AZUL, label='Modelo A'),
                       Line2D([], [], marker='o', ls='', ms=7, color=NARANJA, label='Modelo B')],
              loc='lower left', fontsize=7.5, ncol=2, bbox_to_anchor=(0.0, 1.0), borderaxespad=0.2)
    eje_es(ax, 'x'); ax.text(-0.02, 1.02, '(a)', transform=ax.transAxes, ha='right', va='bottom', fontsize=9)
    gv = g.reindex(filas_z).values
    cols_ = [AZUL if z == 'Valores del laboratorio' else (REF if z == 'Otras' else NARANJA) for z in filas_z]
    bx.barh(range(len(filas_z)), gv, color=cols_, height=0.55, lw=0)
    for k, v in enumerate(gv):
        bx.text(v + 0.015 * np.nanmax(gv), k, f'{nf(v, 1) if v < 10 else nf(v, 0)} %', va='center',
                fontsize=7.5, color=TINTA)
    bx.set_yticks(range(len(filas_z))); bx.set_yticklabels(filas_z)
    bx.set_xlim(0, np.nanmax(gv) * 1.2)
    bx.set_xlabel('Peso en las decisiones del\nmodelo B (% del total)')
    eje_es(bx, 'x'); bx.text(-0.02, 1.02, '(b)', transform=bx.transAxes, ha='right', va='bottom', fontsize=9)
    PERD = {}
    if ZP is not None:
        for k, z in enumerate(filas_z):
            if z in ZP.index and z != 'Valores del laboratorio':
                v = float(ZP.loc[z, 'perdida']); PERD[z] = v
                cx.barh(k, max(v, 0), color=NARANJA if v >= 0.02 else VALIDO_P, height=0.55, lw=0)
                cx.text(max(v, 0) + 0.0008, k, nf(v, 4), va='center', fontsize=7, color=TINTA)
            else:
                cx.text(0.0008, k, 'No se reentrena', va='center', fontsize=7, color=SEC)
        xm_ = max(0.03, max(PERD.values(), default=0) * 1.3)
        cx.axvline(0.02, color=SEC, lw=0.8, ls=(0, (3, 2)), zorder=0)
        cx.text(0.02, 1.0, '0,02', transform=cx.get_xaxis_transform(), ha='center', va='bottom',
                fontsize=7, color=SEC)
        cx.set_xlim(0, xm_)
    cx.tick_params(labelleft=False, axis='y', length=0)
    cx.set_xlabel('Pérdida de precisión media al\nreentrenar sin la zona')
    eje_es(cx, 'x', 2)
    guarda(fig, 15)
except Exception:
    fallo(15)

try:
    Zs = np.load(F_SHAP, allow_pickle=True)
    sv, dat, cols_s, yv = Zs['valores'], Zs['datos'], list(Zs['columnas']), Zs['y']
    imp_abs = np.abs(sv).mean(axis=0)
    top = list(np.argsort(imp_abs)[::-1][:2])
    LZ = 8
    fig, axs = plt.subplots(1, 2, figsize=(ANCHO, 2.9), sharey=True, gridspec_kw=dict(wspace=0.20))
    for ax, j, le in zip(axs, top, ('(a)', '(b)')):
        x = np.clip(dat[:, j].astype(float), -LZ + 0.05, LZ - 0.05); s_ = sv[:, j].astype(float)
        okk = np.isfinite(x)
        for sel, col, z in ((yv == 0, VALIDO_P, 2), (yv == 1, DESC, 3)):
            ax.scatter(x[sel & okk], s_[sel & okk], s=7, color=col, alpha=0.75, lw=0, zorder=z)
        ax.axhline(0, color=EJE, lw=0.8, zorder=1)
        qb = np.unique(np.quantile(x[okk], np.linspace(0, 1, 21))); mid, mm = [], []
        for a_, b_ in zip(qb[:-1], qb[1:]):
            q = okk & (x >= a_) & (x <= b_)
            if q.sum() >= 5:
                mid.append(np.median(x[q])); mm.append(np.median(s_[q]))
        mid, mm = np.array(mid), np.array(mm)
        cz = np.flatnonzero(np.sign(mm[:-1]) != np.sign(mm[1:]))
        sube = float(np.corrcoef(x[okk], s_[okk])[0, 1]) > 0
        xc = np.nan
        if len(cz):
            c0 = cz[0] if sube else cz[-1]
            xc = mid[c0] + (0 - mm[c0]) * (mid[c0 + 1] - mid[c0]) / (mm[c0 + 1] - mm[c0])
            ax.axvline(xc, color=SEC, lw=0.8, ls=(0, (3, 2)), zorder=1)
            ax.text(xc, 1.0, (' Empuja a descartar →' if sube else '← Empuja a descartar '),
                    transform=ax.get_xaxis_transform(), ha='left' if sube else 'right', va='bottom',
                    fontsize=7, color=SEC)
        ax.set_xlim(-LZ, LZ); ax.set_xticks([-8, -4, 0, 4, 8])
        ax.set_xticklabels(['≤ −8', '−4', '0', '4', '≥ 8'])
        ax.set_xlabel(envuelve(mayus(nombre_var(cols_s[j])), 44) + '\n(en unidades de dispersión de su película)')
        ax.text(-0.02, 1.0, le, transform=ax.transAxes, ha='right', va='bottom', fontsize=9)
        eje_es(ax, 'y')
    axs[0].set_ylabel('Aportación a la salida del modelo\n(log-odds; positiva: hacia descartar)')
    fig.legend(handles=[Line2D([], [], marker='o', ls='', ms=4, color=DESC, label='Descartado por el especialista'),
                        Line2D([], [], marker='o', ls='', ms=4, color=VALIDO_P, label='Válido')],
               loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.94))
    guarda(fig, 16)
except Exception:
    fallo(16)

try:
    fig, ax = plt.subplots(figsize=(ANCHO, 2.75))
    n_ = 8
    for i in range(n_):
        for j in range(n_):
            if i + 1 < n_:
                ax.plot([j, j], [i, i + 1], color=REJ, lw=1.0, zorder=1)
            if j + 1 < n_:
                ax.plot([j, j + 1], [i, i], color=REJ, lw=1.0, zorder=1)
    gx, gy = np.meshgrid(range(n_), range(n_))
    ax.scatter(gx.ravel(), gy.ravel(), s=13, color=VALIDO_P, lw=0, zorder=2)
    EJ = [((0, 7), 's', 'Esquina: 2 vecinos (4 ensayos)'),
          ((4, 7), '^', 'Borde: 3 vecinos (24 ensayos)'),
          ((3, 3), 'o', 'Interior: 4 vecinos (36 ensayos)')]
    for (x, y), mk, _ in EJ:
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            u, v = x + dx, y + dy
            if 0 <= u < n_ and 0 <= v < n_:
                ax.plot([x, u], [y, v], color=DESC, lw=1.6, zorder=3)
                ax.scatter([u], [v], s=15, color=DESC, lw=0, zorder=4)
        ax.scatter([x], [y], s=48, marker=mk, color=DESC, lw=0, zorder=5)
    for k, ((x, y), mk, et) in enumerate(EJ):
        yy = 6.6 - 1.05 * k
        ax.scatter([9.0], [yy], s=40, marker=mk, color=DESC, lw=0)
        ax.text(9.45, yy, et, va='center', ha='left', fontsize=7.5, color=TINTA)
    for k, t in enumerate(['4 × 2 + 24 × 3 + 36 × 4 = 224 vecindades', 'Cada pareja cuenta dos veces: 112 parejas',
                           '224 / 64 = 3,5 vecinos por ensayo']):
        ax.text(8.8, 2.9 - 0.95 * k, t, va='center', ha='left', fontsize=7.5, color=SEC)
    ax.text(3.5, -0.85, 'Rejilla de 8 × 8 ensayos de una película', ha='center', va='center', fontsize=7.5,
            color=SEC)
    ax.set_xlim(-0.5, 19.0); ax.set_ylim(-1.2, 7.5); ax.set_aspect('equal', adjustable='datalim'); ax.axis('off')
    guarda(fig, 17)
except Exception:
    fallo(17)

print(f'\n  ilustraciones dibujadas: {len(HECHAS)} de {len(FICH)}')


  figuras en /content/figuras



  Ilustracion  1 · ilustracion_01_anatomia_ensayo.png


  Ilustracion  2 · ilustracion_02_esquema_contacto.png


  Ilustracion  3 · ilustracion_03_fases_del_trabajo.png


  Ilustracion  4 · ilustracion_04_tipos_de_defecto.png


  Ilustracion  5 · ilustracion_05_como_se_mide_el_acierto.png


... (11 outputs left out)

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


  Ilustracion 16 · ilustracion_16_como_decide.png


Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


  Ilustracion 17 · ilustracion_17_vecinos_de_la_rejilla.png

  ilustraciones dibujadas: 17 de 17


## Descarga de las figuras

### Celda 19. Descarga de las figuras

Solo en Colab: comprime la carpeta `/content/figuras`, que escribe la celda 18, y descarga el fichero `figuras_comprimidas.zip`.

In [ ]:
from google.colab import files

output_filename = '/content/figuras_comprimidas.zip'
shutil.make_archive(output_filename.replace('.zip', ''), 'zip', '/content/figuras')
files.download(output_filename)